# iGEM Pooled NLP - Notebook 2: Clustering, Sub-themes, LLM Labels & Analytics

**Loads Notebook 1 artifacts only - never re-embeds.** Seeded UMAP, multi-criterion K, Ward, sub-themes, validation, one grounded LLM labelling pass, award join, per-response sentiment, and the sub-theme->success forest plot. Saves all analytic artifacts and writes `cluster_manifest.json`. The preflight fails loudly if Notebook 1 artifacts are missing.

## Stage 1 — Environment

Recommended Colab runtime: an L4 GPU with High-RAM (Runtime, Change runtime type). Installs the libraries and sets the determinism environment variables that must exist before CUDA starts. Parametric UMAP and TensorFlow are not used: this pipeline relies on standard, seeded UMAP, which is reproducible. Run once per session.

In [ ]:
# @title
# ==============================================================================
# STAGE 1  -  ENVIRONMENT
# Installs the libraries the pipeline needs, sets the determinism environment
# variables that must exist before CUDA initialises, and prints a version
# fingerprint for the methods section. Run once per session.
# Default embedding is Qwen3-Embedding-4B (needs recent transformers /
# sentence-transformers). Parametric UMAP and TensorFlow are intentionally NOT
# installed: this rebuild uses standard, seeded UMAP (bit-for-bit reproducible).
# ==============================================================================
import os

# Determinism vars set before torch / CUDA touch the GPU.
os.environ["PYTHONHASHSEED"]        = "42"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"   # deterministic cuBLAS matmuls
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import subprocess, sys
def _pip(pkgs):
    subprocess.run(f"pip install -q {pkgs}", shell=True, check=False)

# One install pass. Pinned where a known-good version matters (kaleido 0.1.x
# exports static Plotly images headlessly on Colab).
_pip("'sentence-transformers>=3.3.0' 'transformers>=4.51.0' umap-learn scikit-learn scipy")
_pip("torch bitsandbytes accelerate")
_pip("lingua-language-detector contractions 'spacy>=3.8'")
_pip("plotly 'kaleido==0.2.1' matplotlib seaborn colorcet gdown")
subprocess.run("python -m spacy download en_core_web_sm -q", shell=True, check=False)
_pip("nltk statsmodels")

import torch
from importlib.metadata import version, PackageNotFoundError
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"
print("=" * 64)
print("  STAGE 1  ENVIRONMENT")
print("=" * 64)
print(f"  Device : {gpu}")
print("  Versions (record these for the methods section / requirements.txt):")
for p in ["torch", "transformers", "sentence-transformers", "umap-learn",
          "scikit-learn", "scipy", "numpy", "plotly", "kaleido"]:
    try:    print(f"    {p:22s} {version(p)}")
    except PackageNotFoundError: print(f"    {p:22s} (not found)")
print("=" * 64)


## Stage 2 — Drive and model cache

This pipeline does not need Google Drive. Model weights and results both live on Colab local disk, so by default Drive is not mounted at all (no connect step, no auth popup, no Drive errors), and nothing fills your Drive. The survey data is fetched by its public share link, which also needs no mount. Set RESULTS_ON_DRIVE=True only if you want results to persist across sessions, in which case Drive is mounted with a safe fallback to local disk.

In [ ]:
# @title
# ==============================================================================
# STAGE 2  -  STORAGE, MODEL CACHE & PERSISTENT ARTIFACTS  (rewritten)
# Two concerns are kept strictly apart:
#   * MODEL CACHE  (HF_HOME)       -> ALWAYS local disk. The embedder / LLM weights
#                                     are many GB and must never be written to Drive.
#   * ARTIFACT STORE (CHECKPOINT)  -> local disk by default, with an OPTIONAL
#                                     persistent MIRROR (Google Drive on Colab, or
#                                     any folder you pick) for files <= the size cap.
# This implements the plan exactly: every artifact is written LOCALLY and also
# MIRRORED to a persistent place, EXCEPT a single file larger than
# ARTIFACT_SIZE_CAP_MB, which stays LOCAL-ONLY (so a big embedding/UMAP matrix
# never tries to fill Drive). Notebook 2 reads the SAME store: local first, then
# the mirror, so you never recompute embeddings just because the runtime reset.
# ==============================================================================
import os, shutil

# ------------------------------ EDIT HERE -------------------------------------
USE_PERSISTENT_MIRROR = True     # True = also copy small artifacts to a persistent folder
PERSIST_ROOT          = None     # None = auto (mount Google Drive on Colab).
                                 # Or set a path, e.g. "/content/drive/MyDrive"
                                 # or a local persistent disk / mounted folder.
ARTIFACT_SIZE_CAP_MB  = 100      # a single file larger than this is saved LOCAL ONLY
# ----------------------------- END EDIT ---------------------------------------

ON_COLAB   = os.path.isdir("/content")
LOCAL_BASE = "/content" if ON_COLAB else os.path.abspath(".")

# Model weights -> ALWAYS local disk.
os.environ["HF_HOME"] = os.path.join(LOCAL_BASE, "hf_cache")
os.makedirs(os.environ["HF_HOME"], exist_ok=True)

# Local artifact root (fast; may be ephemeral on Colab).
CHECKPOINT_DIR = os.path.join(LOCAL_BASE, "iGEM_Pipeline_Checkpoints_Pooled")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Optional persistent mirror (Drive on Colab, or a folder you choose).
MIRROR_ROOT = None
if USE_PERSISTENT_MIRROR:
    _root = PERSIST_ROOT
    if _root is None and ON_COLAB:
        try:
            from google.colab import drive
            try:
                drive.mount("/content/drive", force_remount=False)
            except ValueError:
                pass  # already mounted
            _root = "/content/drive/MyDrive"
        except Exception as e:
            print(f"  Drive mount unavailable ({type(e).__name__}); mirror OFF, local only.")
            _root = None
    if _root:
        MIRROR_ROOT = os.path.join(_root, "iGEM_Pipeline_Checkpoints_Pooled")
        try:
            os.makedirs(MIRROR_ROOT, exist_ok=True)
        except Exception as e:
            print(f"  Could not create mirror dir ({type(e).__name__}); local only.")
            MIRROR_ROOT = None

# Notebook identity (stamped into every manifest).
NOTEBOOK_NAME    = "Notebook 2 - Cluster + Subthemes + LLM + Analytics"
NOTEBOOK_VERSION = "2.0"

print("=" * 64)
print("  STAGE 2  STORAGE & MODEL CACHE")
print("=" * 64)
print(f"  notebook        : {NOTEBOOK_NAME}  (v{NOTEBOOK_VERSION})")
print(f"  model cache     : {os.environ['HF_HOME']}  (LOCAL, always)")
print(f"  artifact local  : {CHECKPOINT_DIR}")
print(f"  artifact mirror : {MIRROR_ROOT or 'OFF (local only)'}")
print(f"  size rule       : files > {ARTIFACT_SIZE_CAP_MB} MB are LOCAL-ONLY (never mirrored)")
print("=" * 64)


## Stage 3 — Control Panel

The single source of truth. Every tunable lives here and is frozen into one immutable `CFG`. `set_all_seeds` fixes Python, NumPy and PyTorch randomness; `CFG.fingerprint` builds the per-stage cache key. Change a value, re-run, and only the affected stages recompute.

In [ ]:
# @title
# ==============================================================================
# STAGE 3  -  CONTROL PANEL  (the single source of truth)
# Every tunable lives here and nowhere else (Golden Rule 3). Edit a value in the
# EDIT block, re-run this cell and the cells below; the content-hash cache
# (Stage 4) recomputes only the stages a change actually affects. The values are
# then frozen into one immutable CFG object that also carries:
#   set_all_seeds()  - fixes Python, NumPy and PyTorch randomness (Golden Rule 1)
#   CFG.fingerprint() - the per-stage cache key (Golden Rule 2)
# ==============================================================================
from dataclasses import dataclass, field, asdict
import os, json, hashlib, random
import numpy as np

# ----------------------------- EDIT HERE --------------------------------------
SEED            = 42
CACHE_VERSION   = "v1"              # bump to "v2","v3"... to force EVERY cached stage to recompute

# Data
DATA_URL        = "https://drive.google.com/uc?id=YOUR_SURVEY_DATA_FILE_ID"
TEXT_COLS       = ("Connection and belonging",
                   "Reflect on how your team worked together",
                   "Beyond iGEM",
                   "Supporting and limiting conditions",
                   "One word or phrase that captures what iGEM meant to you")
SHORT_LABELS    = ("Connection", "Team Work", "Beyond iGEM", "Conditions", "One Word")

# Embedding   (Qwen3-Embedding-4B by default, 2560-d; switch to "bge-m3" to compare)
EMBED_BACKEND   = "qwen3-8b"        # "qwen3-8b"=Qwen3-Embedding-8B (4096-d, for L4 High-RAM) | "qwen3"=4B (2560-d) | "bge-m3"=1024-d
EMBED_BATCH     = 16

# UMAP  (seeded, standard; no Parametric UMAP)
UMAP_NEIGHBORS  = 50
UMAP_MIN_DIST   = 0.0               # tight geometry for clustering
CLUSTER_DIM     = 5                 # dimensionality of the clustering space

# Ward clustering
K_MODE          = "auto"            # "auto" = data-driven over [K_MIN, K_MAX]; "fixed" = K_FIXED
K_FIXED         = 16
K_MIN           = 4
K_MAX           = 30

# Sub-theme drill-down + validation
SUBTHEME_MIN    = 30                # split any macro theme with at least this many responses
SUBTHEME_TARGET = 50                # aim ~this many responses per sub-theme (LOWER = MORE sub-themes)
SUBTHEME_MAX_K  = 12                # cap sub-themes per macro theme
N_SEEDS         = 10                # multi-seed stability runs
N_BOOTSTRAP     = 100               # bootstrap resamples for Jaccard stability
TRUST_K         = 15                # neighbours for trustworthiness / continuity

# Language-model theme labelling
LLM_ENABLE      = True
LLM_MODEL       = "Qwen/Qwen2.5-14B-Instruct"   # A100: "Qwen/Qwen2.5-32B-Instruct"
LLM_N_QUOTES    = 7
LLM_FORCE_REGEN = False             # True forces re-labelling even if cached

# Team awards (structured metadata, never embedded)
AWARD_YEAR      = 2025
# ------------------------------ END EDIT --------------------------------------

# Embedding registry: name + output dimension for each backend.
_EMBED_REGISTRY = {
    "qwen3":      ("Qwen/Qwen3-Embedding-4B",   2560),  # default: strong + multilingual, fits an L4
    "qwen3-8b":   ("Qwen/Qwen3-Embedding-8B",   4096),  # A100: highest accuracy
    "qwen3-0.6b": ("Qwen/Qwen3-Embedding-0.6B", 1024),  # light alternative
    "bge-m3":     ("BAAI/bge-m3",               1024),  # previous model, kept for comparison
}
# Qwen3-Embedding is instruction-aware: a short task description sharpens the
# geometry for our use (thematic clustering). BGE-M3 ignores this (encoded plain).
EMBED_INSTRUCTION = ("Represent this iGEM participant's survey response BY ITS TOPIC for "
                     "thematic clustering, independent of sentiment or how positive or "
                     "negative it is:")
_EMBED_MODEL, _EMBED_DIM = _EMBED_REGISTRY[EMBED_BACKEND]

# Which settings affect which stage. A stage's cache key is built only from the
# settings above it in the dependency chain, so changing K re-runs Ward but not
# the embedding, and changing UMAP re-runs UMAP, Ward and everything downstream.
_RELEVANT = {
    "data":    ["data_url"],
    "embed":   ["seed", "embed_model", "embed_dim", "embed_batch"],
    "umap":    ["seed", "embed_model", "umap_neighbors", "umap_min_dist", "cluster_dim"],
    "ward":    ["seed", "embed_model", "umap_neighbors", "umap_min_dist", "cluster_dim",
                "k_mode", "k_fixed", "k_min", "k_max"],
    "subtheme":["seed", "embed_model", "umap_neighbors", "umap_min_dist", "cluster_dim",
                "k_mode", "k_fixed", "k_min", "k_max", "subtheme_min", "subtheme_target", "subtheme_max_k"],
}


@dataclass(frozen=True)
class Config:
    seed: int
    cache_version: str
    data_url: str
    text_cols: tuple
    short_labels: tuple
    embed_backend: str
    embed_model: str
    embed_dim: int
    embed_batch: int
    umap_neighbors: int
    umap_min_dist: float
    cluster_dim: int
    k_mode: str
    k_fixed: int
    k_min: int
    k_max: int
    subtheme_min: int
    subtheme_target: int
    subtheme_max_k: int
    n_seeds: int
    n_bootstrap: int
    trust_k: int
    llm_enable: bool
    llm_model: str
    llm_n_quotes: int
    llm_force_regen: bool
    award_year: int
    checkpoint_dir: str

    def fingerprint(self, tag, n_rows, extra=None):
        """Short, stable cache key for `tag`. Built from the settings relevant to
        that stage, the row count, and an optional content signature `extra`
        (e.g. a hash of the actual input), so the cache is genuinely content
        addressed and never returns a stale object."""
        keys = _RELEVANT.get(tag, sorted(asdict(self).keys()))
        payload = {k: getattr(self, k) for k in keys}
        payload["n_rows"] = int(n_rows)
        payload["cache_version"] = self.cache_version
        if extra is not None:
            payload["extra"] = str(extra)
        blob = json.dumps(payload, sort_keys=True, default=str)
        return hashlib.md5(blob.encode()).hexdigest()[:10]


CFG = Config(
    seed=SEED, cache_version=CACHE_VERSION, data_url=DATA_URL, text_cols=TEXT_COLS, short_labels=SHORT_LABELS,
    embed_backend=EMBED_BACKEND, embed_model=_EMBED_MODEL, embed_dim=_EMBED_DIM,
    embed_batch=EMBED_BATCH, umap_neighbors=UMAP_NEIGHBORS, umap_min_dist=UMAP_MIN_DIST,
    cluster_dim=CLUSTER_DIM, k_mode=K_MODE, k_fixed=K_FIXED, k_min=K_MIN, k_max=K_MAX,
    subtheme_min=SUBTHEME_MIN, subtheme_target=SUBTHEME_TARGET, subtheme_max_k=SUBTHEME_MAX_K, n_seeds=N_SEEDS, n_bootstrap=N_BOOTSTRAP, trust_k=TRUST_K,
    llm_enable=LLM_ENABLE, llm_model=LLM_MODEL, llm_n_quotes=LLM_N_QUOTES,
    llm_force_regen=LLM_FORCE_REGEN, award_year=AWARD_YEAR,
    checkpoint_dir=CHECKPOINT_DIR,
)


def set_all_seeds(seed=None):
    """Fix every random source the pipeline uses. Call at the top of any stage
    that draws random numbers (UMAP, bootstrap, sampling)."""
    s = CFG.seed if seed is None else int(seed)
    os.environ["PYTHONHASHSEED"] = str(s)
    random.seed(s)
    np.random.seed(s)
    try:
        import torch
        torch.manual_seed(s)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(s)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except Exception:
        pass


set_all_seeds()
print("=" * 64)
print("  STAGE 3  CONTROL PANEL")
print("=" * 64)
print(f"  seed            : {CFG.seed}")
print(f"  embedding       : {CFG.embed_model}  ({CFG.embed_dim}-d, backend={CFG.embed_backend})")
print(f"  UMAP            : n_neighbors={CFG.umap_neighbors}, min_dist={CFG.umap_min_dist}, "
      f"cluster_dim={CFG.cluster_dim}  (seeded, standard)")
print(f"  K (legacy knob) : mode={CFG.k_mode}, range=[{CFG.k_min}, {CFG.k_max}]  -- NOT the analytical K")
print(  "  analytical K    : 3-level hierarchy Main/Mid/Sub set in NB2 Stage 21-23c (bootstrap-Jaccard within interpretability bands); this legacy range is unused")
print(f"  LLM labelling   : {CFG.llm_model if CFG.llm_enable else 'disabled'}")
print(f"  cache fingerprint demo (embed, n=900) : {CFG.fingerprint('embed', 900)}")
print("=" * 64)


## Stage 4 — Library

Define-once utilities used everywhere: the content-hash cache `cached(tag, key, compute)` that replaces a manual fresh/load switch, the named result store `psave`/`pload`, the house style, the colour-blind-safe `palette`, and `save_fig`, which writes a 300 dpi PNG and a vector SVG for every figure.

In [ ]:
# @title
# ==============================================================================
# STAGE 4  -  LIBRARY  (define once, used everywhere)  [rewritten, storage-aware]
# Same house style + helpers as the monolith, with upgrades:
#   1. psave / pload / cached write LOCAL and MIRROR (<= size cap), read local
#      first then mirror, so artifacts survive a runtime reset and Notebook 2/3
#      find them automatically.
#   2. Every save is logged so write_manifest() can stamp a reproducible manifest.
#   3. artifact_diagnostic() reports which artifacts are present / missing.
#   4. validate_columns() is completed (monolith truncated it) and checks
#      Participant / Team.
#   5. invalidate() deletes an artifact so its stage recomputes next run.
# ==============================================================================
import os, pickle, hashlib, json, time, shutil
import numpy as np, pandas as pd
import matplotlib as mpl, matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import colorsys

POOLED_DIR = os.path.join(CFG.checkpoint_dir, "pooled_combined")
os.makedirs(POOLED_DIR, exist_ok=True)

MIRROR_POOLED_DIR = None
if "MIRROR_ROOT" in globals() and MIRROR_ROOT:
    MIRROR_POOLED_DIR = os.path.join(MIRROR_ROOT, "pooled_combined")
    try:
        os.makedirs(MIRROR_POOLED_DIR, exist_ok=True)
    except Exception:
        MIRROR_POOLED_DIR = None

_ARTIFACT_LOG = {}


def _cap_mb():
    return float(globals().get("ARTIFACT_SIZE_CAP_MB", 100))


def _mirror_file(path_local, mirror_dir):
    size_mb = os.path.getsize(path_local) / 1e6
    if not mirror_dir:
        return False, size_mb
    if size_mb > _cap_mb():
        print(f"    (LOCAL-ONLY: {os.path.basename(path_local)} = {size_mb:.0f} MB > {_cap_mb():.0f} MB cap)")
        return False, size_mb
    try:
        shutil.copy2(path_local, os.path.join(mirror_dir, os.path.basename(path_local)))
        return True, size_mb
    except Exception as e:
        print(f"    (mirror failed for {os.path.basename(path_local)}: {type(e).__name__})")
        return False, size_mb


def _log_artifact(name, path, mirrored):
    _ARTIFACT_LOG[name] = dict(path=path, bytes=int(os.path.getsize(path)),
                               size_mb=round(os.path.getsize(path) / 1e6, 3),
                               mirrored=bool(mirrored))


def cached(tag, key, compute):
    fname = f"{tag}_{key}.pkl"
    local = os.path.join(CFG.checkpoint_dir, fname)
    if os.path.exists(local):
        with open(local, "rb") as f:
            obj = pickle.load(f)
        print(f"  [{tag}] loaded from local cache  ({fname})")
        return obj
    if MIRROR_ROOT:
        mp = os.path.join(MIRROR_ROOT, fname)
        if os.path.exists(mp):
            try:
                shutil.copy2(mp, local)
                with open(local, "rb") as f:
                    obj = pickle.load(f)
                print(f"  [{tag}] restored from mirror  ({fname})")
                return obj
            except Exception:
                pass
    print(f"  [{tag}] computing (no cache for key {key}) ...")
    obj = compute()
    with open(local, "wb") as f:
        pickle.dump(obj, f)
    m, _ = _mirror_file(local, MIRROR_ROOT)
    _log_artifact(tag, local, m)
    print(f"  [{tag}] saved  ({fname})")
    return obj


def psave(name, obj):
    local = os.path.join(POOLED_DIR, f"{name}.pkl")
    with open(local, "wb") as f:
        pickle.dump(obj, f)
    m, _ = _mirror_file(local, MIRROR_POOLED_DIR)
    _log_artifact(name, local, m)


def pload(name):
    local = os.path.join(POOLED_DIR, f"{name}.pkl")
    if os.path.exists(local):
        with open(local, "rb") as f:
            return pickle.load(f)
    if MIRROR_POOLED_DIR:
        mp = os.path.join(MIRROR_POOLED_DIR, f"{name}.pkl")
        if os.path.exists(mp):
            try:
                shutil.copy2(mp, local)
                with open(local, "rb") as f:
                    return pickle.load(f)
            except Exception:
                pass
    return None


save_ckpt = psave
def load_ckpt(name):
    return pload(name)


def get_df():
    """Safe loader for the cleaned frame: df_final if present else df_cleaned.
    Avoids `pload('df_final') or pload('df_cleaned')`, which raises ValueError
    because a DataFrame's truth value is ambiguous."""
    d = pload("df_final")
    return d if d is not None else pload("df_cleaned")


def invalidate(name):
    """Delete a named artifact (local + mirror) and any content-hash cache tagged
    with it, so the producing stage recomputes next run. The monolith referenced
    this helper but never defined it; provided here."""
    import glob as _glob
    removed = []
    for d in (POOLED_DIR, MIRROR_POOLED_DIR):
        if not d:
            continue
        p = os.path.join(d, f"{name}.pkl")
        if os.path.exists(p):
            try:
                os.remove(p); removed.append(p)
            except Exception:
                pass
    for d in (CFG.checkpoint_dir, (MIRROR_ROOT if "MIRROR_ROOT" in globals() else None)):
        if not d:
            continue
        for p in _glob.glob(os.path.join(d, f"{name}_*.pkl")):
            try:
                os.remove(p); removed.append(p)
            except Exception:
                pass
    _ARTIFACT_LOG.pop(name, None)
    print(f"  invalidate('{name}'): removed {len(removed)} file(s)")
    return removed


def data_sig(x):
    h = hashlib.md5()
    if isinstance(x, np.ndarray):
        h.update(np.ascontiguousarray(x).view(np.uint8)[::max(1, x.size // 100000)].tobytes())
        h.update(str(x.shape).encode())
    else:
        joined = "\x1f".join(map(str, x))
        h.update(joined.encode("utf-8", "ignore"))
    return h.hexdigest()[:10]


# ---------- house style ----------
# neutrals aligned with the NB3 design system (softer ink, cooler grid)
NAT_BG, NAT_TEXT, NAT_GRID = "#FFFFFF", "#14202B", "#E7EAEE"
mpl.rcParams.update({
    "figure.facecolor": "white", "savefig.facecolor": "white", "savefig.edgecolor": "white",
    "axes.facecolor": "#FCFCFE", "axes.edgecolor": "#D9D9E0", "axes.linewidth": 0.9,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 11.5, "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.titlepad": 8,
    "axes.labelcolor": "#333333", "axes.labelsize": 9.5,
    "xtick.color": "#666666", "ytick.color": "#666666",
    "axes.axisbelow": True, "grid.color": "#EEEEF2", "grid.linewidth": 0.8,
    "legend.frameon": False, "font.size": 10, "figure.titleweight": "bold",
})
# ---------------------------------------------------------------------------
# UNIFIED SOFT-MODERN COLOUR SYSTEM (round 2). One system across NB1/NB2/NB3.
# Categorical set derived from Paul Tol's "muted" qualitative scheme
# (colourblind-validated; Tol 2021) - lower saturation than the old VIVID reds,
# no yellow on white, and the SAME hue keeps the same meaning in every figure.
# Semantic tokens (C_GOOD/C_WARN/C_BAD/C_NEUT) replace ad-hoc hexes in cells.
# ---------------------------------------------------------------------------
HIGHLIGHT = "#882255"          # soft wine accent (replaces the harsh maroon)
VIVID = ["#44AA99", "#CC6677", "#6699CC", "#AA4499", "#D9A44E", "#117733",
         "#882255", "#332288", "#5E81AC", "#B07AA1", "#996B4F", "#7A9E7E",
         "#C98D66", "#4C6B8A", "#8A7CA8", "#6FAF9F"]
C_GOOD, C_WARN, C_BAD, C_NEUT = "#44AA99", "#D9A44E", "#CC6677", "#9AA5B1"
try:
    import colorcet as _cc
    _GLASBEY = list(_cc.glasbey_category10)
except Exception:
    _GLASBEY = []


def _not_yellow(c):
    r, g, b = mcolors.to_rgb(c)
    h, sat, v = colorsys.rgb_to_hsv(r, g, b)
    return not (0.11 <= h <= 0.22 and sat > 0.30)


def apply_house_style():
    mpl.rcParams.update({
        "figure.facecolor": NAT_BG, "axes.facecolor": NAT_BG,
        "savefig.facecolor": NAT_BG, "axes.edgecolor": NAT_GRID,
        "axes.grid": True, "grid.color": NAT_GRID, "grid.linewidth": 0.6,
        "axes.axisbelow": True, "axes.titleweight": "bold",
        "text.color": NAT_TEXT, "axes.labelcolor": NAT_TEXT,
        "xtick.color": NAT_TEXT, "ytick.color": NAT_TEXT,
        "font.size": 11, "figure.dpi": 110, "savefig.dpi": 300,
    })


def palette(n):
    pool = list(VIVID)
    if n > len(pool):
        for c in _GLASBEY:
            hx = mcolors.to_hex(c)
            if _not_yellow(hx) and hx not in pool:
                pool.append(hx)
    if len(pool) < n:
        pool = pool * (n // len(pool) + 1)
    return [mcolors.to_hex(c) for c in pool[:n]]


def lighten(hex_color, amount):
    r, g, b = mcolors.to_rgb(hex_color)
    return mcolors.to_hex((r + (1 - r) * amount, g + (1 - g) * amount, b + (1 - b) * amount))


Q_COLORS = dict(zip(CFG.short_labels, palette(len(CFG.short_labels))))
# magnitude ramp: soft teal sequential (replaces the aggressive white->ruby ramp;
# same ramp NB3 uses, so heatmap intensity reads identically across the report)
# ---- TYPEFACE  (one place, inherited by every figure in this notebook) --------
# A figure's type is part of its argument, and the matplotlib default dates a
# paper on sight. Inter is the modern open-licence choice for data display: drawn
# for screens, tabular digits so columns of numbers line up, legible at the 7-9pt
# a journal figure actually uses. Several sources are tried because a single raw
# URL is not reliable from every runtime, and the result is VERIFIED - the first
# version of this block announced success without checking that the family had
# actually registered, which is how it silently fell back.
import matplotlib as _mpl, matplotlib.font_manager as _fm
import os as _os, urllib.request as _ur, subprocess as _sp

def _resolves(name):
    try:
        _fm.findfont(_fm.FontProperties(family=name), fallback_to_default=False)
        return True
    except Exception:
        return False

WANT = ["Inter", "Inter Tight", "IBM Plex Sans", "Source Sans 3", "Roboto",
        "Open Sans", "Lato"]
FIG_FONT = next((f for f in WANT if _resolves(f)), None)

if FIG_FONT is None:
    _dir = "/content/_fonts"; _os.makedirs(_dir, exist_ok=True)
    _SOURCES = [
        ("Inter", [
            ("Inter-Variable.ttf",
             "https://github.com/google/fonts/raw/main/ofl/inter/Inter%5Bopsz,wght%5D.ttf"),
            ("Inter-Regular.otf",
             "https://raw.githubusercontent.com/rsms/inter/master/docs/font-files/Inter-Regular.otf"),
        ]),
        ("Roboto", [
            ("Roboto-Regular.ttf",
             "https://github.com/google/fonts/raw/main/apache/roboto/static/Roboto-Regular.ttf"),
        ]),
    ]
    for _fam, _files in _SOURCES:
        for _fn, _u in _files:
            _p = _os.path.join(_dir, _fn)
            try:
                if not _os.path.exists(_p):
                    _ur.urlretrieve(_u, _p)
                _fm.fontManager.addfont(_p)
            except Exception:
                continue
        if _resolves(_fam):
            FIG_FONT = _fam
            print(f"  typeface: {_fam} downloaded and registered")
            break
    if FIG_FONT is None:                      # last resort: the distribution's own
        try:
            _sp.run(["apt-get", "-qq", "install", "-y", "fonts-roboto"],
                    check=True, capture_output=True, timeout=180)
            _fm.fontManager.__init__()        # rebuild the font list after apt
            if _resolves("Roboto"):
                FIG_FONT = "Roboto"
                print("  typeface: Roboto installed from the distribution")
        except Exception:
            pass
else:
    print(f"  typeface: using '{FIG_FONT}' (already available)")

if FIG_FONT is None:
    FIG_FONT = "DejaVu Sans"
    print("  typeface: no modern family could be installed; staying on DejaVu Sans.")
    print("            Only the lettering differs - every number and figure is the same.")

_mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": [FIG_FONT, "DejaVu Sans"],
    "mathtext.fontset": "dejavusans",
    "axes.unicode_minus": False,     # a real minus sign, not a hyphen
    "pdf.fonttype": 42, "ps.fonttype": 42,   # embed as TrueType, editable downstream
    "svg.fonttype": "none",                  # SVG keeps live text, not outlines
})

GRAD = mcolors.LinearSegmentedColormap.from_list(
    "teal_seq", ["#F3FAF7", "#CDEDE1", "#8FD9BE", "#3FB98D", "#1D9E75", "#0F6E56"])
# publication-standard static export: SVG keeps TEXT AS TEXT (editable in
# Illustrator/Inkscape/Affinity), instead of outlining every glyph to paths.
mpl.rcParams["svg.fonttype"] = "none"


def save_fig(fig, name, interactive=False, static=True, slide_16x9=True):
    """Save a figure (hi-res PNG + SVG), mirror it, AND emit a 1920x1080 16:9
    slide-ready PNG ({name}_16x9.png) for Canva / PowerPoint. Letterboxing keeps the
    plot undistorted; only white margins are added to reach an exact 16:9 canvas."""
    base = os.path.join(POOLED_DIR, name)
    wrote = []
    if hasattr(fig, "write_html"):
        if interactive:
            fig.write_html(base + ".html", include_plotlyjs="cdn", full_html=True); wrote.append("HTML")
        if static:
            try:
                fig.write_image(base + ".png", scale=2); fig.write_image(base + ".svg"); wrote += ["PNG", "SVG"]
            except Exception as e:
                print(f"  (Plotly static export needs kaleido; skipped: {type(e).__name__})")
    else:
        fig.savefig(base + ".png", dpi=220, bbox_inches="tight", facecolor="white")
        fig.savefig(base + ".svg", bbox_inches="tight", facecolor="white"); wrote += ["PNG", "SVG"]
    if slide_16x9 and os.path.exists(base + ".png"):
        try:
            from PIL import Image
            im = Image.open(base + ".png").convert("RGB"); CW, CH = 1920, 1080
            r = min(CW / im.width, CH / im.height); nw, nh = max(1, int(im.width * r)), max(1, int(im.height * r))
            canvas = Image.new("RGB", (CW, CH), "white")
            canvas.paste(im.resize((nw, nh), Image.LANCZOS), ((CW - nw) // 2, (CH - nh) // 2))
            canvas.save(base + "_16x9.png"); wrote.append("16x9")
        except Exception as e:
            print(f"  (16:9 slide export skipped: {type(e).__name__})")
    for ext in (".png", ".svg", ".html", "_16x9.png"):
        p = base + ext
        if os.path.exists(p) and MIRROR_POOLED_DIR:
            _mirror_file(p, MIRROR_POOLED_DIR)
    try:
        from IPython.display import display as _display; _display(fig)
    except Exception:
        pass
    if wrote:
        print(f"  saved figure: {os.path.basename(base)}  ({' + '.join(wrote)})")
    return base


def validate_columns(df):
    missing = [c for c in CFG.text_cols if c not in df.columns]
    if missing:
        raise KeyError(f"Survey is missing expected text columns: {missing}. "
                       "Check DATA_URL / the export; the pipeline will not guess column names.")
    for c in ("Participant", "Team"):
        if c not in df.columns:
            raise KeyError(f"Survey is missing required column '{c}' "
                           "(needed for embedding metadata and the team-award join).")


def _now():
    return time.strftime("%Y-%m-%d %H:%M:%S")


def write_manifest(filename, fingerprints, n_rows):
    man = dict(
        notebook=NOTEBOOK_NAME, notebook_version=NOTEBOOK_VERSION, timestamp=_now(),
        seed=CFG.seed, cache_version=CFG.cache_version, embed_model=CFG.embed_model,
        n_rows=int(n_rows), config_fingerprint=CFG.fingerprint("all", n_rows),
        fingerprints=dict(fingerprints),
        artifact_store=dict(local=POOLED_DIR, mirror=MIRROR_POOLED_DIR, size_cap_mb=_cap_mb()),
        artifacts=[dict(name=k, **v) for k, v in sorted(_ARTIFACT_LOG.items())],
    )
    p_local = os.path.join(CFG.checkpoint_dir, filename)
    with open(p_local, "w") as f:
        json.dump(man, f, indent=2, default=str)
    if MIRROR_ROOT:
        try:
            shutil.copy2(p_local, os.path.join(MIRROR_ROOT, filename))
        except Exception:
            pass
    print(f"  manifest written: {filename}  ({len(man['artifacts'])} artifacts logged, "
          f"config={man['config_fingerprint']})")
    return man


def artifact_status(name):
    lp = os.path.join(POOLED_DIR, f"{name}.pkl")
    if os.path.exists(lp):
        return "local", os.path.getsize(lp) / 1e6
    if MIRROR_POOLED_DIR:
        mp = os.path.join(MIRROR_POOLED_DIR, f"{name}.pkl")
        if os.path.exists(mp):
            return "mirror", os.path.getsize(mp) / 1e6
    return "MISSING", 0.0


def artifact_diagnostic(required, optional=()):
    print("=" * 64)
    print("  ARTIFACT DIAGNOSTIC")
    print("=" * 64)
    print(f"  {'artifact':34s} {'where':8s} {'MB':>9}")
    print("  " + "-" * 54)
    missing = []
    for n in required:
        where, mb = artifact_status(n)
        if where == "MISSING":
            missing.append(n)
        flag = "" if where != "MISSING" else "   <-- MISSING"
        print(f"  {n:34s} {where:8s} {mb:9.2f}{flag}")
    for n in optional:
        where, mb = artifact_status(n)
        print(f"  {n:34s} {where:8s} {mb:9.2f}   (optional)")
    print("  " + "-" * 54)
    if missing:
        print(f"  {len(missing)} REQUIRED artifact(s) MISSING: {missing}")
    else:
        print("  all required artifacts present.")
    print("=" * 64)
    return missing


# ---------- canonical artifact registry (single source of truth; Fix 1/6/7) ----
ART_NB1 = ["df_cleaned", "df_final", "embeddings_dict", "metadata_dict",
           "pooled_embeddings", "pooled_origin", "pooled_meta"]
ART_NB2 = ["pooled_umap_nd", "pooled_umap5d", "pooled_umap3d", "pooled_umap2d",
           "pooled_hdbscan_screen", "pooled_hdbscan_choice", "pooled_hdbscan_hierarchy",
           "pooled_K", "pooled_levels", "pooled_canonical_level",
           "pooled_labels", "pooled_labels_main7", "pooled_labels_sub",
           "pooled_sub_labels", "pooled_sub_of_macro",
           "pooled_hier_maps", "pooled_hier_crosswalk",
           "pooled_cluster_meta", "pooled_analytical_unit",
           "pooled_stability", "validation_main7", "validation_sub",
           "pooled_main_llm", "pooled_sub_llm", "pooled_theme_llm",
           "pooled_main_name", "pooled_sub_name",
           "pooled_main_terms", "pooled_sub_terms",
           "pooled_main_size", "pooled_sub_size",
           "pooled_naming_audit", "pooled_label_verification",
           "pooled_award", "pooled_team_org", "pooled_master",
           "pooled_sentiment_response", "pooled_sentiment_team_long",
           "pooled_sentiment_team_wide", "pooled_projection",
           "pooled_validity_xq", "cluster_reality", "pooled_convergent",
           "pooled_embed_validation",
           "pooled_correlation_forest", "pooled_correlation_forest_main",
           "pooled_correlation_forest_sub"]
PRODUCER = {
    "df_cleaned": "NB1 Stage 12", "df_final": "NB1 Stage 16",
    "embeddings_dict": "NB1 Stage 13b", "metadata_dict": "NB1 Stage 13b",
    "pooled_embeddings": "NB1 Stage 18", "pooled_origin": "NB1 Stage 18",
    "pooled_meta": "NB1 Stage 18",
    "pooled_umap_nd": "NB2 Stage 21H", "pooled_umap5d": "NB2 Stage 21H",
    "pooled_umap3d": "NB2 Stage 21H", "pooled_umap2d": "NB2 Stage 21H",
    "pooled_hdbscan_screen": "NB2 Stage 21H", "pooled_hdbscan_choice": "NB2 Stage 21H",
    "pooled_hdbscan_hierarchy": "NB2 Stage 23H",
    "pooled_K": "NB2 Stage 23H", "pooled_levels": "NB2 Stage 23H",
    "pooled_canonical_level": "NB2 Stage 23H", "pooled_labels": "NB2 Stage 23H",
    "pooled_labels_main7": "NB2 Stage 23H", "pooled_labels_sub": "NB2 Stage 23H",
    "pooled_sub_of_macro": "NB2 Stage 23H", "pooled_hier_maps": "NB2 Stage 23H",
    "pooled_sub_labels": "NB2 Stage 46 (alias of the canonical level)",
    "pooled_hier_crosswalk": "NB2 Stage 46 (from Stage 23H's crosswalk)",
    "pooled_cluster_meta": "NB2 Stage 46", "pooled_analytical_unit": "NB2 Stage 46",
    "pooled_stability": "NB2 Stage 25", "validation_main7": "NB2 Stage 25",
    "validation_sub": "NB2 Stage 25",
    "pooled_projection": "NB2 Stage 23d",
    "pooled_embed_validation": "NB2 Stage 24", "pooled_validity_xq": "NB2 Stage 26",
    "cluster_reality": "NB2 Stage 27", "pooled_convergent": "NB2 Stage 29",
    "pooled_sentiment_response": "NB2 Stage 26B",
    "pooled_sentiment_team_long": "NB2 Stage 26B",
    "pooled_sentiment_team_wide": "NB2 Stage 26B",
    "pooled_main_llm": "NB2 Stage 30 (verified in 30v)",
    "pooled_sub_llm": "NB2 Stage 30 (verified in 30v)",
    "pooled_theme_llm": "NB2 Stage 30 (verified in 30v)",
    "pooled_main_name": "NB2 Stage 30 (verified in 30v)",
    "pooled_sub_name": "NB2 Stage 30 (verified in 30v)",
    "pooled_main_terms": "NB2 Stage 30", "pooled_sub_terms": "NB2 Stage 30",
    "pooled_main_size": "NB2 Stage 30", "pooled_sub_size": "NB2 Stage 30",
    "pooled_naming_audit": "NB2 Stage 30",
    "pooled_label_verification": "NB2 Stage 30v",
    "pooled_award": "NB2 Stage 32", "pooled_team_org": "NB2 Stage 33",
    "pooled_master": "NB2 Stage 33b",
    "pooled_correlation_forest": "NB2 Stage 40",
    "pooled_correlation_forest_main": "NB2 Stage 40",
    "pooled_correlation_forest_sub": "NB2 Stage 40",
}

def require(*names, stage=""):
    """Stage-local dependency guard (Fix 4): raise a clear error naming each
    missing artifact and the stage that produces it. Returns the loaded object(s)."""
    missing = [n for n in names if pload(n) is None]
    if missing:
        prod = "; ".join(f"{n} <- {PRODUCER.get(n, '?')}" for n in missing)
        raise RuntimeError(f"[{stage or 'stage'}] missing required artifact(s): {prod}. "
                           "Run the producing stage(s) first.")
    objs = [pload(n) for n in names]
    return objs[0] if len(objs) == 1 else objs


def artifact_inventory(names=None):
    """One place to see every saved artifact, where it lives, and which stage made it."""
    names = names or (ART_NB1 + ART_NB2)
    print("=" * 70)
    print("  ARTIFACT INVENTORY   (name | where | MB | producing stage)")
    print("=" * 70)
    for n in names:
        where, mb = artifact_status(n)
        print(f"  {n:28s} {where:7s} {mb:8.2f}   {PRODUCER.get(n, '?')}")
    print("=" * 70)


print("  Stage 4 library ready: storage-aware cache + psave/pload + manifest + diagnostics.")


## Stage 5 — Compatibility bridge

A thin bridge that exposes the names the analysis stages were written against, sourced from `CFG`, so the pipeline keeps one source of truth and needs no run-mode anywhere.

In [ ]:
# @title
# ==============================================================================
# STAGE 5  -  b  -  COMPATIBILITY BRIDGE
# The analysis stages were written against an earlier set of bare constant names.
# Rather than touch every one, we expose those names here, sourced from the one
# Control Panel (CFG) so there is still a single source of truth. Cleaning and
# outlier thresholds that were never clustering-critical live here as plain
# constants. There is no RUN_MODE: caching is automatic (Stage 4).
# ==============================================================================
# Core identifiers from CFG
SEED            = CFG.seed
DATA_URL        = CFG.data_url
TEXT_COLS       = list(CFG.text_cols)
SHORT_LABELS    = list(CFG.short_labels)
ONE_WORD_COL    = TEXT_COLS[4]
EMBED_MODEL     = CFG.embed_model
EMBED_DIM       = CFG.embed_dim
EMBED_BATCH     = CFG.embed_batch
TRUST_K         = CFG.trust_k
UMAP_QUALITY_METRIC = "euclidean"

# Clustering names used by the K-selection / Ward / validation cells
AGGLO_LINKAGE   = "ward"
AGGLO_METRIC    = "euclidean"
AGGLO_K_RANGE   = (CFG.k_min, CFG.k_max)
AGGLO_K_OVERRIDE = CFG.k_fixed if CFG.k_mode == "fixed" else None

# Language-model + awards
LLM_ENABLE      = CFG.llm_enable
LLM_MODEL_NAME  = CFG.llm_model
LLM_N_QUOTES    = CFG.llm_n_quotes
LLM_FORCE_REGEN = CFG.llm_force_regen
AWARD_TSV       = __import__("os").path.join(CFG.checkpoint_dir, "team_awards.tsv")
AWARD_YEAR      = CFG.award_year
AWARD_TEAMID_OVERRIDES = {"GreatBay_SCIE": "GreatBay-SCIE"}
MEDAL_RANK      = {"Gold": 3, "Silver": 2, "Bronze": 1, "None": 0}

# Colours: reuse the house palette so the old names resolve to the new style
# Capture the Stage 4 palette FUNCTION under a private name: several analysis
# cells reassign a local variable called `palette` to a list, so get_cluster_colors
# must not depend on the mutable global of that name.
_PALETTE_FN     = palette
NAT_COLORS      = _PALETTE_FN(len(SHORT_LABELS))
HIGHLIGHT_COLOR = HIGHLIGHT
def get_cluster_colors(n):
    return _PALETTE_FN(n)

# Text-cleaning / keyword sanitation constants (rarely tuned)
MIN_TOKEN_LENGTH = 2
HELSINKI_MODEL   = "Helsinki-NLP/opus-mt-mul-en"
KEYWORD_LATIN_ONLY = True
WC_DOMAIN_STOPS = {
    "igem", "team", "project", "experience", "nan", "thing", "lot", "way", "day",
    "time", "people", "part", "year", "place", "point", "fact", "kind", "group",
    "number", "case", "end", "question", "item", "work", "teammate", "new",
}
WC_SHORT_WHITELIST = {"lab", "phd", "dna", "rna", "fun", "pcr", "gel", "gmo",
                      "bio", "aid", "gap", "hub", "key", "aim"}
EXTRA_MULTILANG_STOPS = {
    "avant", "apres", "pendant", "environ", "depuis", "aprovado", "tudo", "tambm",
    "jamais", "aller", "venir", "passer", "rester", "devenir", "thing", "things",
    "something", "someone", "everyone", "everything", "really", "actually",
    "definitely", "probably", "maybe",
}

# Response-level outlier thresholds (Stage 10)
OUTLIER_MIN_ALPHA_RATIO  = 0.5
OUTLIER_MIN_ALPHA_TOKENS = 1
OUTLIER_MIN_VOWEL_RATIO  = 0.15
OUTLIER_MAX_INVALID_Q    = 3
SAME_WORD_THRESHOLD      = 3

print("  Stage 4b bridge ready: old config names resolve to CFG (no RUN_MODE).")


## Stage 6 — Imports

Shared third-party imports used across the analysis stages.

In [ ]:
# @title
# ════════════════════════════════════════════════════════
# STAGE 6  -  C3_IMPORTS
# INPUT:  -
# OUTPUT: all libraries available
# ════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import re
import warnings
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import umap as umap_lib
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter
from scipy.stats import spearmanr
from scipy.stats import spearmanr, ttest_ind, ttest_1samp
from scipy.spatial.distance import pdist
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d
from sklearn.manifold import trustworthiness
from sklearn.feature_extraction.text import TfidfVectorizer

warnings.filterwarnings('ignore')
print('Imports complete.')

## Notebook 2 - Preflight (load upstream artifacts)
Load the pooled artifacts from Notebook 1 (local first, then mirror) and verify the fingerprint chain. Fails loudly if anything required is missing.

In [ ]:
# @title
# ==============================================================================
# NOTEBOOK 2  -  PREFLIGHT: LOAD NOTEBOOK 1 ARTIFACTS  (fail loudly if missing)
# This notebook NEVER recomputes embeddings. It loads the pooled artifacts that
# Notebook 1 produced (local first, then the persistent mirror). If a required
# artifact is missing it stops with a clear message instead of recomputing or
# crashing deep in a later stage (Golden Rule 10).
# ==============================================================================
import os, json
import numpy as np

REQUIRED_FROM_NB1 = ART_NB1   # canonical list from the shared library (Fix 1)
missing = artifact_diagnostic(REQUIRED_FROM_NB1)
if missing:
    raise RuntimeError(
        "Notebook 1 artifacts are missing: " + ", ".join(missing) + ".\n"
        "FIX: run Notebook 1 to the end FIRST, OR point this notebook at the same "
        "store. If Notebook 1 ran on a different machine/session, set PERSIST_ROOT "
        "in Stage 2 to the persistent folder where Notebook 1 saved (the mirror), "
        "then re-run from Stage 2.")

# Load into the names the analysis cells expect.
X_pool          = np.asarray(pload("pooled_embeddings"), dtype=np.float32)
origin_pool     = np.asarray(pload("pooled_origin"))
pooled_origin   = origin_pool   # canonical name some downstream cells reference directly
meta_pool       = pload("pooled_meta")
embeddings_dict = pload("embeddings_dict")
metadata_dict   = pload("metadata_dict")
df = pload("df_final")
if df is None:
    df = pload("df_cleaned")
if df is None:
    raise RuntimeError("Need df_final or df_cleaned from Notebook 1 "
                       "(its bert_* columns feed sub-themes, LLM naming and sentiment).")

# Verify the upstream fingerprint chain: did the Control Panel change since NB1?
man = None
for p in [os.path.join(CFG.checkpoint_dir, "embed_manifest.json"),
          (os.path.join(MIRROR_ROOT, "embed_manifest.json") if MIRROR_ROOT else None)]:
    if p and os.path.exists(p):
        try:
            man = json.load(open(p)); break
        except Exception:
            pass
if man:
    same_data = (man.get("fingerprints", {}).get("dataset_embeddings") == data_sig(X_pool))
    cfg_now = CFG.fingerprint("all", man.get("n_rows", len(meta_pool)))
    same_cfg = (cfg_now == man.get("config_fingerprint"))
    print(f"  upstream manifest : {man.get('notebook')} v{man.get('notebook_version')} @ {man.get('timestamp')}")
    print(f"  embed model       : {man.get('embed_model')}  | seed={man.get('seed')} | cache={man.get('cache_version')}")
    print(f"  dataset fingerprint {'MATCHES' if same_data else 'DIFFERS  <-- embeddings are not the ones NB1 saved!'}")
    print(f"  config  fingerprint {'MATCHES' if same_cfg else 'DIFFERS  <-- Control Panel changed since NB1'}")
    if not (same_data and same_cfg):
        print("  -> If you changed seed / model / data, re-run Notebook 1 so the artifacts agree.")
else:
    print("  (no embed_manifest.json found; proceeding on the artifacts present.)")

print("=" * 64)
print(f"  PREFLIGHT OK  -  pooled set loaded: X={X_pool.shape}, rows={len(meta_pool)}, "
      f"questions={len(CFG.short_labels)}")
print("  config ownership: NB2 OWNS the clustering knobs (UMAP / K / sub-theme / LLM /")
print("  award). Embedding + data knobs are INHERITED + frozen from NB1; editing them")
print("  here does nothing unless you re-run NB1. The shared CFG only proves the chain.")
print("=" * 64)


## Stage 19 — UMAP parameter sweep (choose best settings)

A diagnostic that chooses the UMAP settings by evidence. It sweeps n_neighbors x min_dist and the clustering dimension, scoring each by neighbourhood faithfulness (trustworthiness + continuity; Venna and Kaski 2001), not by how separated clusters look. It recommends the most faithful settings to put in the Control Panel, keeping min_dist low for clustering. Run once, read the recommendation, set the Control Panel, re-run UMAP.

In [ ]:
# @title
# ==============================================================================
# STAGE 19  -  UMAP PARAMETER SWEEP  (choose the best UMAP by evidence)
# The "best" UMAP is the one most FAITHFUL to the original embedding, measured by
# trustworthiness and continuity (Venna and Kaski 2001) - how well near neighbours
# are preserved - NOT by how separated the clusters look (separation can be
# inflated by aggressive settings). We sweep n_neighbors x min_dist at the
# clustering dimension, and the clustering dimension itself, then recommend the
# settings that maximise neighbourhood preservation. For CLUSTERING we keep
# min_dist low (UMAP authors' guidance) and confirm here it costs no faithfulness.
# Read the recommendation, set the winners in the Control Panel, and re-run UMAP.
# This is a diagnostic; it does not change anything by itself.
# ==============================================================================
import numpy as np
import umap
import matplotlib.pyplot as plt
from sklearn.manifold import trustworthiness

X = np.asarray(pload("pooled_embeddings"))
if X is None:
    raise RuntimeError("Run the pooling stage first (need pooled_embeddings).")
set_all_seeds()
rng = np.random.default_rng(CFG.seed)
m = rng.choice(len(X), min(700, len(X)), replace=False) if len(X) > 700 else np.arange(len(X))
Xs = X[m]
TK = CFG.trust_k

def _umap(nn, md, nc):
    return umap.UMAP(n_neighbors=nn, min_dist=md, n_components=nc, metric="cosine",
                     random_state=CFG.seed, n_jobs=1).fit_transform(Xs)

def _faith(Y):
    t = trustworthiness(Xs, Y, n_neighbors=TK)   # neighbours preserved going in
    c = trustworthiness(Y, Xs, n_neighbors=TK)   # continuity (dual)
    return float(t), float(c)

NN = [10, 15, 30, 50]
MD = [0.0, 0.1, 0.25]
grid = np.zeros((len(NN), len(MD)))
for i, nn in enumerate(NN):
    for j, md in enumerate(MD):
        t, c = _faith(_umap(nn, md, CFG.cluster_dim))
        grid[i, j] = (t + c) / 2

# for clustering we use min_dist = 0.0; pick the n_neighbors best at that column
j0 = MD.index(0.0)
best_nn = NN[int(np.argmax(grid[:, j0]))]
# dimensionality sweep at (best_nn, 0.0)
DIMS = [2, 5, 8, 12, 16, 20]
dim_f = [(_faith(_umap(best_nn, 0.0, nc))) for nc in DIMS]
dim_score = [(t + c) / 2 for t, c in dim_f]
dmax = max(dim_score)
best_dim = next(d for d, v in zip(DIMS, dim_score) if v >= dmax - 0.01)   # knee within 0.01

print("=" * 64)
print("  UMAP PARAMETER SWEEP  (faithfulness = trustworthiness + continuity)")
print("=" * 64)
print("  n_neighbors x min_dist faithfulness (clustering dim "
      f"{CFG.cluster_dim}):")
print("            " + "  ".join(f"md={md:<4}" for md in MD))
for i, nn in enumerate(NN):
    print(f"    nn={nn:<3} " + "  ".join(f"{grid[i, j]:.3f}  " for j in range(len(MD))))
print(f"\n  dimensionality at nn={best_nn}, min_dist=0.0:")
for d, s in zip(DIMS, dim_score):
    print(f"    {d:>2}-D : {s:.3f}" + ("   <- knee (recommended)" if d == best_dim else ""))
print("\n  RECOMMENDED for clustering (set in the Control Panel, then re-run UMAP):")
print(f"    UMAP_NEIGHBORS = {best_nn}")
print(f"    UMAP_MIN_DIST  = 0.0     (low for clustering; faithfulness cost is negligible)")
print(f"    CLUSTER_DIM    = {best_dim}")
cur = (CFG.umap_neighbors, CFG.umap_min_dist, CFG.cluster_dim)
print(f"  current settings: n_neighbors={cur[0]}, min_dist={cur[1]}, cluster_dim={cur[2]}"
      + ("   (already optimal)" if cur == (best_nn, 0.0, best_dim) else "   (consider updating)"))
psave("umap_sweep", dict(grid=grid, NN=NN, MD=MD, DIMS=DIMS, dim_score=dim_score,
                         best_nn=best_nn, best_min_dist=0.0, best_dim=best_dim))

from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.patches as _mpatch
NT = globals().get("NAT_TEXT", "#1E1E1E")
fig, ax = plt.subplots(1, 2, figsize=(16, 9), facecolor=NAT_BG,
                       gridspec_kw=dict(width_ratios=[1.05, 1.0], wspace=0.34))
im = ax[0].imshow(grid, cmap=GRAD, aspect="auto", vmin=grid.min(), vmax=grid.max())
ax[0].set_xticks(range(len(MD))); ax[0].set_xticklabels([f"min_dist={md}" for md in MD], fontsize=10.1)
ax[0].set_yticks(range(len(NN))); ax[0].set_yticklabels([f"nn={nn}" for nn in NN], fontsize=10.1)
for i in range(len(NN)):
    for j in range(len(MD)):
        ax[0].text(j, i, f"{grid[i, j]:.3f}", ha="center", va="center", fontsize=10.1,
                   color="white" if grid[i, j] > (grid.min() + grid.max()) / 2 else "#333")
_ri, _rj = NN.index(best_nn), MD.index(0.0)
ax[0].add_patch(_mpatch.Rectangle((_rj - 0.5, _ri - 0.5), 1, 1, fill=False, edgecolor="#111", lw=2.4))
ax[0].set_title("A  Faithfulness over n_neighbors x min_dist\n(black box = chosen for clustering)",
                fontsize=12.3, fontweight="bold", loc="left", color=NT)
_div = make_axes_locatable(ax[0]); _cax = _div.append_axes("right", size="4.5%", pad=0.12)
_cb = plt.colorbar(im, cax=_cax); _cb.set_label("faithfulness", fontsize=10.1); _cb.ax.tick_params(labelsize=9)
_dscore = dict(zip(DIMS, dim_score))
ax[1].plot(DIMS, dim_score, "-o", color=HIGHLIGHT, lw=2.2, ms=6, zorder=3)
ax[1].axvline(best_dim, color="#999", ls=":", lw=1.3)
ax[1].scatter([best_dim], [_dscore[best_dim]], s=160, facecolor="none", edgecolor="#111", lw=2, zorder=4)
ax[1].annotate(f"knee {best_dim}-D (recommended)", xy=(best_dim, _dscore[best_dim]),
               xytext=(best_dim + 2.4, _dscore[best_dim] - 0.010), fontsize=10.1, color=NT,
               arrowprops=dict(arrowstyle="->", color="#555"))
ax[1].set_xlabel("clustering dimension"); ax[1].set_ylabel("faithfulness")
ax[1].set_title(f"B  Faithfulness vs dimensionality (nn={best_nn})",
                fontsize=12.3, fontweight="bold", loc="left", color=NT)
ax[1].grid(True, color="#F0F0F0"); ax[1].set_axisbelow(True); ax[1].set_facecolor("white")
for sp in ("top", "right"):
    ax[1].spines[sp].set_visible(False)
fig.suptitle("UMAP parameter sweep: choosing the most faithful projection",
             fontsize=15.1, fontweight="bold", color=NT, y=1.03)
fig.text(0.5, -0.04, "Faithfulness = (trustworthiness + continuity)/2 (Venna & Kaski 2001); UMAP (McInnes et al. 2018). "
         "Real UMAP fits on a 700-response subsample.", ha="center", fontsize=9, style="italic", color="#888")
plt.tight_layout(); save_fig(fig, "chart_umap_sweep")
plt.close(fig)
print("=" * 64)


## Stage 20 — UMAP (seeded, standard)

Standard UMAP (McInnes, Healy, Melville 2018) with `random_state` and `n_jobs=1` is bit-for-bit reproducible, unlike the neural Parametric UMAP it replaces. UMAP minimises the cross-entropy between the high- and low-dimensional fuzzy neighbour graphs. We keep an N-dimensional space for clustering and 2-D and 3-D maps for viewing, cached, with quality read by trustworthiness (Venna and Kaski 2001).

In [ ]:
# @title
# ==============================================================================
# STAGE 20  -  UMAP  (seeded, standard; the reproducibility fix)
# Projects the pooled embeddings to a low-dimensional clustering space and a 2-D
# and 3-D map for viewing. We use STANDARD UMAP (McInnes, Healy, Melville 2018)
# with random_state=CFG.seed and n_jobs=1, which is bit-for-bit reproducible. The
# earlier Parametric UMAP trained a neural network and therefore moved between
# runs; that is why the themes were not stable. The projection is content-hash
# cached, so a re-run with the same settings loads the identical coordinates.
# Quality is read with trustworthiness and continuity (Venna and Kaski 2001):
#   how faithfully near neighbours in the embedding stay near in the projection.
# ==============================================================================
import numpy as np
import umap
from sklearn.manifold import trustworthiness

X_pool = np.asarray(X_pool, dtype=np.float32)

def _umap(n_components, X):
    set_all_seeds()
    return umap.UMAP(
        n_neighbors=CFG.umap_neighbors, min_dist=CFG.umap_min_dist,
        n_components=n_components, metric="cosine",
        random_state=CFG.seed, n_jobs=1,
    ).fit_transform(X).astype(np.float32)

def _compute_umap():
    return {"nd": _umap(CFG.cluster_dim, X_pool),
            "d3": _umap(3, X_pool),
            "d2": _umap(2, X_pool)}

_key = CFG.fingerprint("umap", len(X_pool), extra=data_sig(X_pool))
_um = cached("umap", _key, _compute_umap)
# A restored UMAP is keyed on its parameters, not on the data it was built from,
# so re-running over a different corpus silently returns the OLD projection. On the
# five-question pool this handed back a 1134-row map for 1505 responses. Check the
# length and recompute rather than carry it forward.
_Xn = len(np.asarray(pload("pooled_embeddings")))
if len(np.asarray(_um["nd"])) != _Xn:
    print(f"  the restored UMAP has {len(np.asarray(_um['nd']))} rows but "
          f"pooled_embeddings has {_Xn} - it belongs to a previous corpus.")
    print("  recomputing rather than reusing it.")
    for _k in list(globals().get("_UMAP_CACHE_KEYS", []) or []):
        try:
            invalidate(_k)
        except Exception:
            pass
    _um = _umap_all(X_pool) if "_umap_all" in globals() else _um
    if len(np.asarray(_um["nd"])) != _Xn:
        raise RuntimeError(
            f"UMAP still has {len(np.asarray(_um['nd']))} rows for {_Xn} responses. "
            f"Delete the umap_*.pkl checkpoints and re-run this cell.")

pooled_umap_nd = _um["nd"]
pooled_umap3d  = _um["d3"]
pooled_umap2d  = _um["d2"]
# Name kept for the downstream analysis cells (the N-D clustering coordinates).
pooled_umap5d = pooled_umap_nd
for _n, _o in [("pooled_umap5d", pooled_umap5d), ("pooled_umap3d", pooled_umap3d),
               ("pooled_umap2d", pooled_umap2d)]:
    psave(_n, _o)

# Projection quality (higher is better; ~1.0 is excellent).
set_all_seeds()
tw_2d = trustworthiness(X_pool, pooled_umap2d, n_neighbors=CFG.trust_k)
tw_nd = trustworthiness(X_pool, pooled_umap_nd, n_neighbors=CFG.trust_k)

print("=" * 64)
print("  STAGE 20  UMAP (seeded, standard)")
print("=" * 64)
print(f"  clustering space : {pooled_umap_nd.shape}  (used by Ward)")
print(f"  maps             : 3-D {pooled_umap3d.shape}, 2-D {pooled_umap2d.shape}")
print(f"  trustworthiness  : 2-D={tw_2d:.3f}, {CFG.cluster_dim}-D={tw_nd:.3f}  (k={CFG.trust_k})")

# Per the pipeline design the UMAP stage shows ONLY a 2-D STATIC validation
# map (below). The interactive 2-D and 3-D explorers are dedicated stages
# that run after the themes are labelled.

# ── 2-D STATIC validation map (questions intermix; trustworthiness reported) ──
import matplotlib.pyplot as plt
NB = globals().get("NAT_BG", "#FFFFFF"); NT = globals().get("NAT_TEXT", "#1E1E1E")
_origin = pooled_origin if "pooled_origin" in globals() else pload("pooled_origin")
_QC = globals().get("Q_COLORS", None)
figm, ax = plt.subplots(figsize=(16, 9), facecolor=NB)
if _origin is not None:
    _origin = np.asarray(_origin)
    for _i, q in enumerate(CFG.short_labels):
        c = (_QC[q] if _QC else palette(len(CFG.short_labels))[_i])
        m = _origin == q
        ax.scatter(pooled_umap2d[m, 0], pooled_umap2d[m, 1], s=11, alpha=0.62,
                   color=c, label=q, edgecolors="none")
    ax.legend(frameon=False, fontsize=10.6, markerscale=2, loc="upper left",
              bbox_to_anchor=(1.01, 1.0), title="survey question", title_fontsize=11.2)
else:
    ax.scatter(pooled_umap2d[:, 0], pooled_umap2d[:, 1], s=11, alpha=0.62)
# clip the VIEW to the central bulk so the cluster fills the frame (a few extreme
# outliers are pushed off-view; they remain in the data and the clustering).
_x = np.asarray(pooled_umap2d[:, 0]); _y = np.asarray(pooled_umap2d[:, 1])
_xlo, _xhi = np.percentile(_x, [2, 99.3]); _ylo, _yhi = np.percentile(_y, [1, 99])
_px = (_xhi - _xlo) * 0.05; _py = (_yhi - _ylo) * 0.05
ax.set_xlim(_xlo - _px, _xhi + _px); ax.set_ylim(_ylo - _py, _yhi + _py)
ax.set_xticks([]); ax.set_yticks([]); ax.set_facecolor("white")
for sp in ("top", "right", "bottom", "left"):
    ax.spines[sp].set_color("#E0E0E0")
ax.text(0.015, 0.015,
        f"trustworthiness  2-D = {tw_2d:.3f}\nclustering {CFG.cluster_dim}-D = {tw_nd:.3f}   (k={CFG.trust_k})",
        transform=ax.transAxes, fontsize=10.1, va="bottom", ha="left",
        bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="#D9A7B6", alpha=0.92))
figm.suptitle("Pooled UMAP 2-D map  (seeded, reproducible)", fontsize=16.2, fontweight="bold", color=NT, y=0.97)
figm.text(0.5, 0.005, "Seeded standard UMAP (McInnes et al. 2018); faithfulness by trustworthiness "
          "(Venna & Kaski 2001). Questions intermix (the pooling rationale). View focuses on the main mass; the separate 21-response theme C4 projects far left and is shown in the theme maps.",
          ha="center", fontsize=9, style="italic", color="#888")
plt.tight_layout(rect=[0, 0.03, 1, 0.94]); save_fig(figm, "chart_stage20_umap_map")
plt.close(figm)
print("=" * 64)


## Stage 21H — UMAP × HDBSCAN screening

Ward is gone. It had to place every response in a cluster, so leftover material was forced together into groups that could not be named. HDBSCAN can leave a response as **noise**, which is the honest outcome for an answer that belongs to no theme.

This screens UMAP and HDBSCAN settings **together**, because they interact, and picks by a rule written into the cell **before any result is seen**: noise ≤ 35%, 10–60 clusters, smallest cluster ≥ 8, then the highest DBCV with ties broken on bootstrap stability. Every configuration tried is written to `hdbscan_screen.csv`, including the failures and why they failed. If nothing is admissible the cell **refuses to choose** rather than relaxing the rule.

*Note carried into the methods:* UMAP compacts scattered points toward structure, so UMAP + HDBSCAN systematically **under**-reports noise.


In [ ]:
# @title
# ==============================================================================
# STAGE 21H   UMAP x HDBSCAN SCREENING          (replaces the Ward K-selection)
# ------------------------------------------------------------------------------
# WHY HDBSCAN AND NOT WARD
#   Ward assigns EVERY response to a cluster. It has no way to say "this answer
#   belongs to no theme", so leftover material is forced into a cluster, and that
#   cluster is then impossible to name because its members share nothing. HDBSCAN
#   can leave a point as noise, which is the honest outcome for such answers.
#
# WHAT THIS CELL DOES
#   Screens UMAP and HDBSCAN settings TOGETHER - they interact, so tuning one with
#   the other fixed would be arbitrary - and picks by a rule written down BEFORE
#   any result is seen. Every configuration tried is saved, including the ones that
#   fail, so the choice can be audited rather than taken on trust.
#
# ------------------------------------------------------------------------------
# WHAT CHANGED IN THIS VERSION, AND WHY IT IS NOT POST-HOC TUNING
#
#   THE DEFECT. The old rule read "maximise DBCV subject to noise <= 35%". It
#   treats every noise point as costing the same. It does not. A one-word answer
#   left as noise costs this study nothing - there is no theme for "Community" to
#   belong to. An 83-word answer describing an advisor, the judges and the team
#   costs a great deal, because that is exactly the material the study is about.
#   The old objective cannot tell those two apart, so it optimises a quantity
#   nobody cares about and is silent on the one that matters.
#
#   THE FIX, IN TWO PARTS.
#
#   (1) MEASURE THE RIGHT THING. Noise is now measured separately on SUBSTANTIVE
#       responses (>= SUBSTANTIVE_MIN_WORDS words) and on THIN ones. The threshold
#       is fixed here, before any configuration is scored, and it is a property of
#       the response, not of any clustering - so no choice of settings can move it.
#       Both numbers are recorded for every configuration in the grid.
#
#   (2) SELECT WITH A PRE-REGISTERED TOLERANCE, NOT BY SWAPPING THE OBJECTIVE.
#       Simply switching the objective to "least substantive noise" after seeing
#       that DBCV gave 27% noise would be exactly the post-hoc tuning this cell
#       exists to prevent. Instead this uses the ONE-STANDARD-ERROR RULE (Breiman,
#       Friedman, Olshen & Stone 1984, section 3.4.3; the standard convention in
#       cross-validated model selection):
#
#           threshold = best DBCV  -  1 bootstrap standard error of that DBCV,
#                       but never lower than (1 - BAND_MAX_DROP) x best DBCV
#           among every admissible, stable configuration scoring at or above the
#           threshold, take the one with the LEAST SUBSTANTIVE NOISE.
#
#       The second clause is a guard, not a second criterion. With 15 refits the
#       standard error is itself noisy, and an over-large estimate would let the
#       band reach down to the min_samples=1 configurations that give up ~40% of
#       the DBCV. The cap makes the rule strictly more conservative and can only
#       ever shrink the band.
#
#       The logic is the same as it is in cross-validation: DBCV is an estimate
#       with sampling error, so configurations within one standard error of the
#       best are not distinguishable from it on that evidence. When the internal
#       index cannot separate them, the tie is broken by the criterion the study
#       actually cares about. The rule cannot select a materially worse partition,
#       because it is bounded by the best DBCV's own uncertainty; and if no
#       configuration inside that band has better substantive coverage, it returns
#       the DBCV maximiser unchanged. Both outcomes are printed either way.
#
#       This is also what arXiv 2509.09839 (2025) recommends after showing that
#       DBCV-optimised HDBSCAN settings can miss real structure: prefer a
#       task-aligned criterion over a generic internal index alone. Here DBCV
#       still sets the admissible band; the task criterion only breaks ties
#       inside it.
#
#   WHAT THIS CANNOT DO. It cannot invent coverage that the density structure does
#   not support. The grid on this corpus contains no configuration with both low
#   noise and high DBCV - every low-noise option uses min_samples=1, which removes
#   the smoothing from the density estimate and costs about 40% of the DBCV. The
#   1-SE band is deliberately narrow, so if that trade-off is the only one on
#   offer, this rule will decline it and say so. Declining it is the correct
#   result and belongs in the paper.
#
# THE SELECTION RULE, FIXED IN ADVANCE
#   admissible  = noise <= MAX_NOISE
#                 AND substantive noise <= MAX_SUBSTANTIVE_NOISE
#                 AND N_CLUSTERS_RANGE[0] <= clusters <= N_CLUSTERS_RANGE[1]
#                 AND smallest cluster >= MIN_CLUSTER_FLOOR
#                 AND bootstrap stability >= STAB_MIN
#   selected    = among the admissible whose DBCV is within one bootstrap standard
#                 error of the best (and never more than BAND_MAX_DROP below it),
#                 the least substantive noise (ties -> DBCV).
#
#   WHY STABILITY IS A FLOOR AND NOT A TIE-BREAK. An earlier version of this rule
#   ranked on DBCV and used stability only to settle ties. On this corpus that
#   chose a partition with DBCV 0.393 / stability 0.600 / noise 34% over one with
#   DBCV 0.362 / stability 0.707 / noise 27% - it bought 0.03 of an internal
#   density score by accepting a partition that a 20% change of sample would
#   substantially rearrange. A result that does not survive resampling is not a
#   result. Hennig (2007) treats 0.6-0.75 as a "pattern" and below 0.6 as not
#   real, and this project already applies exactly that floor to the bootstrap
#   Jaccard elsewhere; the same standard applies here.
#   If NOTHING is admissible the cell says so and picks nothing. It does not
#   relax the rule to produce an answer.
#
# HONEST PRIOR: an earlier run of this project applied HDBSCAN with default-ish
# settings and got 2 clusters. That is the failure this screening exists to
# detect or to fix - in particular cluster_selection_method='leaf', which returns
# the fine leaves of the tree instead of a few large excess-of-mass clusters.
#
# Outputs: pooled_hdbscan_screen (every configuration), pooled_hdbscan_choice,
#          pooled_substantive (the a-priori mask, reused by Stage 23d),
#          hdbscan_screen.csv, chart_hdbscan_screen
# Refs: Moulavi et al. (2014) DBCV; Hennig (2007) cluster-wise stability;
#       Breiman et al. (1984) the one-standard-error rule; arXiv 2509.09839 (2025).
# ==============================================================================
import os, sys, time, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

try:
    import hdbscan as _hdb
except Exception:
    print("  installing hdbscan (needed for DBCV; the sklearn build does not expose it)")
    os.system(f"{sys.executable} -m pip install -q hdbscan")
    import hdbscan as _hdb
import umap

# ---------------------------- EDIT HERE ---------------------------------------
# The grid. Every combination is tried and recorded.
UMAP_NCOMP      = [5, 10, 15]
UMAP_NNEIGH     = [15, 30, 50]
MIN_CLUSTER     = [8, 12, 15, 20, 25, 30, 40]
MIN_SAMPLES     = [None, 1, 5, 10]        # None = same as min_cluster_size
SELECT_METHOD   = ["eom", "leaf"]
SELECT_EPSILON  = [0.0, 0.1, 0.2]

# The rule. Written before any result is seen; do not edit it after looking.
MAX_NOISE          = 0.35     # more than this and too much of the corpus is lost
MAX_SUBSTANTIVE_NOISE = 0.35  # the SAME ceiling, applied to the responses that
                              # carry content. Deliberately not stricter: a
                              # tighter number could only be justified by looking
                              # at the results first, which is the thing this cell
                              # refuses to do. It binds only if the substantive
                              # rows are LESS well covered than the corpus as a
                              # whole - which would itself be a finding.
SUBSTANTIVE_MIN_WORDS = 10    # a response with fewer words than this cannot carry
                              # a theme on its own. Fixed here, a property of the
                              # text alone, so no setting can move it. Chosen to
                              # match the band structure found by reading the
                              # unplaced responses by hand (<=9 words: too thin;
                              # 10-29: one specific subject; >=30: several).
N_CLUSTERS_RANGE   = (10, 60) # fewer is not a theme structure, more is fragmentation
MIN_CLUSTER_FLOOR  = 8        # a cluster below this cannot carry a claim
STAB_MIN           = 0.70     # Hennig (2007): below this a partition is not a
                              # finding. A FLOOR, not a tie-break - see above.
BAND_MAX_DROP      = 0.10     # HARD CAP on the 1-SE band. With only N_BOOT refits
                              # the standard error is itself a noisy estimate, and
                              # an over-large SE would let the band swallow the
                              # whole shortlist - including the min_samples=1
                              # configurations that cost ~40% of the DBCV. The
                              # band may never reach further than this fraction
                              # below the best DBCV, whatever the SE says. Fixed
                              # in advance; it makes the rule strictly more
                              # conservative, never less.
N_BOOT             = 15       # bootstrap refits per shortlisted configuration
SHORTLIST          = 20       # top-DBCV configs that get the stability test. Must
                              # be wide enough that the floor can actually bite;
                              # at 6 the floor could only ever pick within the
                              # DBCV leaders, which is the bug it exists to fix.
SEED               = int(globals().get("SEED", 42))
# --------------------------- END EDIT -----------------------------------------

X = np.asarray(pload("pooled_embeddings"), dtype=np.float32)
if X is None or not len(X):
    raise RuntimeError("pooled_embeddings missing - run Notebook 1 first.")
N = len(X)

# ---- the a-priori substantive mask -------------------------------------------
# Word counts come from pooled_meta if NB1 recorded them, otherwise from the raw
# text resolved exactly the way Stage 33b resolves it. If neither works this cell
# STOPS rather than quietly dropping the criterion: a selection rule that silently
# switches itself off is worse than no rule at all.
_meta = require("pooled_meta", stage="Stage 21H").reset_index(drop=True)
if len(_meta) != N:
    raise RuntimeError(f"[Stage 21H] pooled_embeddings has {N} rows but pooled_meta "
                       f"has {len(_meta)} - they are from different runs.")
if "n_words" in _meta.columns:
    NWORDS = np.asarray(_meta["n_words"].values, dtype=float)
    _wsrc = "pooled_meta['n_words']"
else:
    _df = df if ("df" in globals() and df is not None) else get_df()
    _RAW = dict(zip(CFG.short_labels, CFG.text_cols))
    _pid = {}
    for _, _r in _df.iterrows():
        _pid.setdefault(_r["Participant"], _r)
    _txt = []
    for _, m in _meta.iterrows():
        r = _pid.get(m["Participant"])
        v = "" if r is None else r.get(_RAW.get(m["question"]), "")
        _txt.append("" if (v is None or str(v).strip().lower() == "nan") else str(v))
    NWORDS = np.array([len(t.split()) for t in _txt], dtype=float)
    _wsrc = "raw text via CFG.text_cols"
if not np.isfinite(NWORDS).all() or NWORDS.sum() <= 0:
    raise RuntimeError("[Stage 21H] word counts could not be resolved, so the "
                       "substantive-noise criterion cannot be evaluated. Fix the "
                       "text join rather than running without the criterion.")
SUBST = NWORDS >= SUBSTANTIVE_MIN_WORDS
n_sub, n_thin = int(SUBST.sum()), int((~SUBST).sum())

print("=" * 78)
print("  STAGE 21H   UMAP x HDBSCAN SCREENING")
print("=" * 78)
print(f"  {N} responses x {X.shape[1]}-d")
print(f"  word counts from {_wsrc}")
print(f"  SUBSTANTIVE (>= {SUBSTANTIVE_MIN_WORDS} words) : {n_sub:>5}  ({n_sub/N:.1%})")
print(f"  THIN        (<  {SUBSTANTIVE_MIN_WORDS} words) : {n_thin:>5}  ({n_thin/N:.1%})")
print("  The split is a property of the text and is fixed before any configuration")
print("  is scored. Losing a THIN response to noise costs nothing; losing a")
print("  SUBSTANTIVE one is the failure this criterion exists to price.")
print(f"  grid: {len(UMAP_NCOMP)}x{len(UMAP_NNEIGH)} UMAP  x  "
      f"{len(MIN_CLUSTER)}x{len(MIN_SAMPLES)}x{len(SELECT_METHOD)}x{len(SELECT_EPSILON)}"
      f" HDBSCAN  =  "
      f"{len(UMAP_NCOMP)*len(UMAP_NNEIGH)*len(MIN_CLUSTER)*len(MIN_SAMPLES)*len(SELECT_METHOD)*len(SELECT_EPSILON)}"
      f" configurations")
print(f"  admissible: noise <= {MAX_NOISE:.0%}, substantive noise <= "
      f"{MAX_SUBSTANTIVE_NOISE:.0%}, clusters in {N_CLUSTERS_RANGE},")
print(f"              smallest >= {MIN_CLUSTER_FLOOR}, stability >= {STAB_MIN}")
print("  then: best DBCV, and within one bootstrap SE of it, least substantive noise.")

def _fit(E, mcs, ms, meth, eps, mst=True):
    return _hdb.HDBSCAN(min_cluster_size=int(mcs),
                        min_samples=None if ms is None else int(ms),
                        cluster_selection_method=meth,
                        cluster_selection_epsilon=float(eps),
                        metric="euclidean", gen_min_span_tree=mst,
                        core_dist_n_jobs=1).fit(E)

# ---- phase A: the UMAP embeddings, computed once each -------------------------
EMB, t0 = {}, time.time()
for nc in UMAP_NCOMP:
    for nn in UMAP_NNEIGH:
        EMB[(nc, nn)] = umap.UMAP(n_components=nc, n_neighbors=nn, min_dist=0.0,
                                  metric="cosine", random_state=SEED,
                                  verbose=False).fit_transform(X).astype(np.float32)
        print(f"    umap n_components={nc:<3} n_neighbors={nn:<3} done "
              f"({time.time()-t0:.0f}s)")

# ---- phase B: every HDBSCAN configuration ------------------------------------
rows = []
for (nc, nn), E in EMB.items():
    for mcs in MIN_CLUSTER:
        for ms in MIN_SAMPLES:
            for meth in SELECT_METHOD:
                for eps in SELECT_EPSILON:
                    try:
                        cl = _fit(E, mcs, ms, meth, eps)
                        lab = cl.labels_
                        k = int(lab.max()) + 1
                        noise = float((lab == -1).mean())
                        # the two halves of the noise, measured separately
                        ns = float((lab[SUBST] == -1).mean()) if n_sub else np.nan
                        nt = float((lab[~SUBST] == -1).mean()) if n_thin else np.nan
                        sizes = np.bincount(lab[lab >= 0], minlength=max(k, 1)) if k else np.array([])
                        try:
                            dbcv = float(cl.relative_validity_)
                        except Exception:
                            dbcv = np.nan
                        rows.append(dict(n_components=nc, n_neighbors=nn,
                                         min_cluster_size=mcs,
                                         min_samples=(-1 if ms is None else ms),
                                         method=meth, epsilon=eps,
                                         n_clusters=k, noise=noise,
                                         noise_subst=ns, noise_thin=nt,
                                         n_subst_lost=int((lab[SUBST] == -1).sum()) if n_sub else 0,
                                         min_size=int(sizes.min()) if k else 0,
                                         median_size=int(np.median(sizes)) if k else 0,
                                         max_size=int(sizes.max()) if k else 0,
                                         dbcv=dbcv, error=""))
                    except Exception as e:
                        rows.append(dict(n_components=nc, n_neighbors=nn,
                                         min_cluster_size=mcs,
                                         min_samples=(-1 if ms is None else ms),
                                         method=meth, epsilon=eps, n_clusters=0,
                                         noise=1.0, noise_subst=1.0, noise_thin=1.0,
                                         n_subst_lost=n_sub,
                                         min_size=0, median_size=0,
                                         max_size=0, dbcv=np.nan,
                                         error=f"{type(e).__name__}"))
    print(f"    hdbscan grid on umap({nc},{nn}) done ({time.time()-t0:.0f}s)")

G = pd.DataFrame(rows)
G["admissible"] = ((G.noise <= MAX_NOISE)
                   & (G.noise_subst <= MAX_SUBSTANTIVE_NOISE)
                   & (G.n_clusters >= N_CLUSTERS_RANGE[0])
                   & (G.n_clusters <= N_CLUSTERS_RANGE[1])
                   & (G.min_size >= MIN_CLUSTER_FLOOR)
                   & G.dbcv.notna())
G.to_csv(f"{POOLED_DIR}/hdbscan_screen.csv", index=False)

print("\n" + "-" * 78)
print(f"  {len(G)} configurations tried, {int(G.admissible.sum())} admissible")
print(f"  why the rest failed:")
for lbl, m in (("noise above the ceiling", G.noise > MAX_NOISE),
               ("SUBSTANTIVE noise above the ceiling", G.noise_subst > MAX_SUBSTANTIVE_NOISE),
               ("too few clusters", G.n_clusters < N_CLUSTERS_RANGE[0]),
               ("too many clusters", G.n_clusters > N_CLUSTERS_RANGE[1]),
               ("a cluster below the floor", G.min_size < MIN_CLUSTER_FLOOR),
               ("DBCV could not be computed", G.dbcv.isna()),
               ("the fit raised", G.error != "")):
    print(f"    {int(m.sum()):>5}  {lbl}")
print(f"  clusters found across the whole grid: min {int(G.n_clusters.min())}, "
      f"max {int(G.n_clusters.max())}, median {int(G.n_clusters.median())}")
print(f"  noise across the whole grid: min {G.noise.min():.0%}, "
      f"max {G.noise.max():.0%}, median {G.noise.median():.0%}")

# Is noise actually concentrated in the substantive rows, or in the thin ones?
# This is the diagnostic that says whether the new criterion has anything to bite
# on, and it is printed whether the answer is convenient or not.
_ok = G[G.dbcv.notna() & (G.n_clusters > 0)]
if len(_ok):
    _d = (_ok.noise_subst - _ok.noise_thin)
    print(f"\n  substantive minus thin noise, across the grid: median {_d.median():+.1%}"
          f"  (range {_d.min():+.1%} to {_d.max():+.1%})")
    print("    negative = substantive responses are BETTER covered than thin ones,")
    print("    which is the expected and desirable direction.")

CHOICE = None
if not G.admissible.any():
    print("\n  NOTHING IS ADMISSIBLE. HDBSCAN does not produce a usable theme")
    print("  structure on this embedding under the rule fixed above. The rule is not")
    print("  relaxed here to manufacture an answer - inspect hdbscan_screen.csv and")
    print("  decide explicitly whether to change the rule or the method.")
    if (G.n_clusters <= 3).mean() > .5:
        print("  NOTE: over half the grid collapsed to 3 clusters or fewer, which is the")
        print("  same failure this project saw before. That is a property of the data's")
        print("  density, not of the settings.")
else:
    A = G[G.admissible].sort_values("dbcv", ascending=False).head(SHORTLIST).copy()
    print(f"\n  bootstrap stability on the top {len(A)} by DBCV "
          f"({N_BOOT} refits on 80% subsamples, adjusted Rand on the shared rows)")
    from sklearn.metrics import adjusted_rand_score
    rng = np.random.default_rng(SEED)
    stab = []
    for r in A.itertuples():
        E = EMB[(r.n_components, r.n_neighbors)]
        base = _fit(E, r.min_cluster_size,
                    None if r.min_samples < 0 else r.min_samples,
                    r.method, r.epsilon, mst=False).labels_
        sc = []
        for _ in range(N_BOOT):
            idx = rng.choice(N, int(.8 * N), replace=False)
            lb = _fit(E[idx], r.min_cluster_size,
                      None if r.min_samples < 0 else r.min_samples,
                      r.method, r.epsilon, mst=False).labels_
            sc.append(adjusted_rand_score(base[idx], lb))
        stab.append(float(np.mean(sc)))
        print(f"    umap({r.n_components},{r.n_neighbors}) mcs={r.min_cluster_size} "
              f"ms={r.min_samples} {r.method} eps={r.epsilon}: "
              f"k={r.n_clusters} noise={r.noise:.0%} subst={r.noise_subst:.0%} "
              f"DBCV={r.dbcv:+.3f} stability={stab[-1]:.3f}")
    A["stability"] = stab
    A["stable_enough"] = A.stability >= STAB_MIN
    _pass = A[A.stable_enough]
    if not len(_pass):
        print(f"\n  NO configuration reaches the stability floor of {STAB_MIN}. The best")
        print(f"  seen is {A.stability.max():.3f}. A partition this sensitive to an 80%")
        print(f"  resample is not a finding, so nothing is chosen. Read hdbscan_screen.csv")
        print(f"  and decide explicitly whether to lower the floor and say so in the paper.")
        CHOICE = None
        psave("pooled_hdbscan_screen", G); psave("pooled_hdbscan_choice", None)
        raise SystemExit
    _rej = A[~A.stable_enough]
    if len(_rej):
        print(f"\n  {len(_rej)} shortlisted configuration(s) rejected on stability alone:")
        for r in _rej.sort_values("dbcv", ascending=False).head(4).itertuples():
            print(f"    k={r.n_clusters:<3} noise={r.noise:.0%} DBCV={r.dbcv:+.3f} "
                  f"stability={r.stability:.3f}  <- below {STAB_MIN}")

    # ---------------------------------------------------------------------
    # THE ONE-STANDARD-ERROR RULE
    # The bootstrap SE of DBCV is measured on the DBCV MAXIMISER only - that is
    # the reference model whose uncertainty defines the band (Breiman et al.
    # 1984). Refits are on 80% subsamples with the minimum spanning tree built,
    # because relative_validity_ needs it.
    # ---------------------------------------------------------------------
    A = _pass.sort_values(["dbcv", "stability"], ascending=False)
    REF = A.iloc[0]
    _dv = []
    for _ in range(N_BOOT):
        idx = rng.choice(N, int(.8 * N), replace=False)
        try:
            c = _fit(EMB[(REF.n_components, REF.n_neighbors)][idx], REF.min_cluster_size,
                     None if REF.min_samples < 0 else REF.min_samples,
                     REF.method, REF.epsilon, mst=True)
            _dv.append(float(c.relative_validity_))
        except Exception:
            pass
    DBCV_SE = float(np.std(_dv, ddof=1)) if len(_dv) >= 3 else 0.0
    _th_se  = float(REF.dbcv) - DBCV_SE
    _th_cap = float(REF.dbcv) * (1.0 - BAND_MAX_DROP)
    THRESH  = max(_th_se, _th_cap)          # the tighter of the two always wins
    print("\n" + "-" * 78)
    print("  ONE-STANDARD-ERROR RULE  (Breiman et al. 1984)")
    print(f"    best admissible DBCV                 {REF.dbcv:+.4f}")
    print(f"    its bootstrap SE ({len(_dv)} refits)          {DBCV_SE:.4f}")
    print(f"    1-SE floor                           {_th_se:+.4f}")
    print(f"    hard cap floor ({BAND_MAX_DROP:.0%} of best)          {_th_cap:+.4f}")
    print(f"    band floor actually used             {THRESH:+.4f}"
          f"   ({'hard cap binds' if _th_cap > _th_se else '1-SE binds'})")
    print("    configurations at or above that are not distinguishable from the")
    print("    best on this evidence; among those the least substantive noise wins.")
    if DBCV_SE == 0.0:
        print("    NOTE: the SE could not be estimated (too few successful refits), so")
        print("    the band collapses to the maximiser and the rule reduces to best DBCV.")
    BAND = A[A.dbcv >= THRESH].sort_values(["noise_subst", "dbcv"],
                                           ascending=[True, False])
    print(f"\n    {len(BAND)} configuration(s) inside the band:")
    for r in BAND.itertuples():
        mark = "  <- CHOSEN" if r.Index == BAND.index[0] else (
               "  (DBCV maximiser)" if r.Index == REF.name else "")
        print(f"      k={r.n_clusters:<3} DBCV={r.dbcv:+.4f} stab={r.stability:.3f} "
              f"noise={r.noise:.1%} subst={r.noise_subst:.1%} "
              f"({int(r.n_subst_lost)} substantive lost){mark}")
    B = BAND.iloc[0]
    _by = ("best DBCV (no better substantive coverage inside the 1-SE band)"
           if B.name == REF.name else
           "least substantive noise within one SE of the best DBCV")
    if B.name != REF.name:
        print(f"\n    The rule moved off the DBCV maximiser: substantive noise "
              f"{REF.noise_subst:.1%} -> {B.noise_subst:.1%} "
              f"({int(REF.n_subst_lost - B.n_subst_lost)} more substantive responses "
              f"placed) for {REF.dbcv - B.dbcv:.4f} of DBCV, which is inside its own "
              f"standard error.")
    else:
        print("\n    The rule did NOT move: no configuration within one standard error")
        print("    of the best DBCV covers the substantive responses any better. The")
        print("    coverage that remains is a property of the density structure, not")
        print("    of the settings, and is handled downstream in Stage 23d - not by")
        print("    loosening the clustering.")

    CHOICE = dict(n_components=int(B.n_components), n_neighbors=int(B.n_neighbors),
                  min_cluster_size=int(B.min_cluster_size),
                  min_samples=(None if B.min_samples < 0 else int(B.min_samples)),
                  method=str(B.method), epsilon=float(B.epsilon),
                  n_clusters=int(B.n_clusters), noise=float(B.noise),
                  noise_subst=float(B.noise_subst), noise_thin=float(B.noise_thin),
                  n_subst_lost=int(B.n_subst_lost),
                  dbcv=float(B.dbcv), dbcv_se=DBCV_SE, dbcv_best=float(REF.dbcv),
                  stability=float(B.stability),
                  min_size=int(B.min_size), median_size=int(B.median_size),
                  substantive_min_words=int(SUBSTANTIVE_MIN_WORDS),
                  selected_by=_by)
    print("\n" + "-" * 78)
    print("  CHOSEN, by the rule fixed above:")
    for k, v in CHOICE.items():
        print(f"    {k:<22} {v}")
    print(f"\n  {int(B.noise * N)} of {N} responses ({B.noise:.0%}) are left as NOISE:")
    print(f"    {int(B.n_subst_lost):>5} of {n_sub} SUBSTANTIVE ({B.noise_subst:.1%}) "
          f"- these are what Stage 23d exists to recover")
    print(f"    {int(round(B.noise_thin*n_thin)):>5} of {n_thin} THIN "
          f"({B.noise_thin:.1%}) - correctly left out; too short to carry a theme")
    print("  They are not forced into a theme. Later stages report them as their own")
    print("  group and no prevalence or claim counts them as a theme member.")
    # the chosen embedding becomes the one every later stage uses
    E = EMB[(CHOICE["n_components"], CHOICE["n_neighbors"])]
    psave("pooled_umap_nd", E)
    psave("pooled_umap5d", E)          # historical name; later stages read this
    for d, nm in ((2, "pooled_umap2d"), (3, "pooled_umap3d")):
        psave(nm, umap.UMAP(n_components=d, n_neighbors=CHOICE["n_neighbors"],
                            min_dist=0.0, metric="cosine",
                            random_state=SEED).fit_transform(X).astype(np.float32))
    print(f"  pooled_umap_nd / 5d set to the chosen {CHOICE['n_components']}-d "
          f"embedding; 2-d and 3-d maps recomputed at the same n_neighbors.")

psave("pooled_hdbscan_screen", G)
psave("pooled_hdbscan_choice", CHOICE)
# the a-priori mask, so Stage 23d and the methods text use the SAME definition
# rather than each re-deriving it
psave("pooled_substantive", pd.DataFrame(dict(row_id=np.arange(N),
                                              n_words=NWORDS.astype(int),
                                              is_substantive=SUBST)))

# ------------------------------- figure ---------------------------------------
try:
    import matplotlib.pyplot as plt
    ok = G[G.dbcv.notna() & (G.n_clusters > 0)]
    fig, ax = plt.subplots(1, 3, figsize=(13.6, 3.6))
    a = ax[0]
    a.scatter(ok.n_clusters, 100 * ok.noise, s=9, c="#9AA5B1", alpha=.55,
              label="tried")
    adm = ok[ok.admissible]
    a.scatter(adm.n_clusters, 100 * adm.noise, s=16, c="#44AA99", label="admissible")
    if CHOICE:
        a.scatter([CHOICE["n_clusters"]], [100 * CHOICE["noise"]], s=70,
                  facecolor="none", edgecolor="#CC6677", linewidth=1.8, label="chosen")
    a.axhline(100 * MAX_NOISE, color="#14202B", ls="--", lw=.9)
    a.axvline(N_CLUSTERS_RANGE[0], color="#14202B", ls=":", lw=.9)
    a.axvline(N_CLUSTERS_RANGE[1], color="#14202B", ls=":", lw=.9)
    a.set_xlabel("clusters found", fontsize=9)
    a.set_ylabel("responses left as noise (%)", fontsize=9)
    a.legend(fontsize=7, frameon=False)
    b = ax[1]
    b.scatter(ok.dbcv, 100 * ok.noise, s=9, c="#9AA5B1", alpha=.55)
    b.scatter(adm.dbcv, 100 * adm.noise, s=16, c="#44AA99")
    if CHOICE:
        b.scatter([CHOICE["dbcv"]], [100 * CHOICE["noise"]], s=70, facecolor="none",
                  edgecolor="#CC6677", linewidth=1.8)
    b.set_xlabel("DBCV   (density-based validation; higher is better)", fontsize=9)
    b.set_ylabel("responses left as noise (%)", fontsize=9)
    # the panel the new criterion is about: the axis the study actually cares about
    c = ax[2]
    c.scatter(ok.dbcv, 100 * ok.noise_subst, s=9, c="#9AA5B1", alpha=.55)
    c.scatter(adm.dbcv, 100 * adm.noise_subst, s=16, c="#44AA99")
    if CHOICE:
        c.axvspan(CHOICE["dbcv_best"] - CHOICE["dbcv_se"], ok.dbcv.max(),
                  color="#DDCC77", alpha=.22, lw=0, zorder=0)
        c.scatter([CHOICE["dbcv"]], [100 * CHOICE["noise_subst"]], s=70,
                  facecolor="none", edgecolor="#CC6677", linewidth=1.8)
        c.annotate("within 1 SE\nof best DBCV",
                   xy=(CHOICE["dbcv_best"] - CHOICE["dbcv_se"], 100 * ok.noise_subst.max()),
                   fontsize=7, ha="right", va="top", color="#7a6a1f")
    c.axhline(100 * MAX_SUBSTANTIVE_NOISE, color="#14202B", ls="--", lw=.9)
    c.set_xlabel("DBCV", fontsize=9)
    c.set_ylabel(f"SUBSTANTIVE (>={SUBSTANTIVE_MIN_WORDS}w) left as noise (%)", fontsize=9)
    for _a in ax:
        for sp in ("top", "right"):
            _a.spines[sp].set_visible(False)
        _a.tick_params(length=2, labelsize=8)
    if "fig_title" in globals():
        fig_title(fig, "Every UMAP and HDBSCAN setting tried, and the one chosen")
    else:
        fig.text(.5, .97, "Every UMAP and HDBSCAN setting tried, and the one chosen",
                 ha="center", va="top", fontsize=11.5, fontweight="bold")
    fig.subplots_adjust(top=.86, bottom=.18, left=.06, right=.99, wspace=.34)
    (save_fig(fig, "chart_hdbscan_screen") if "save_fig" in globals()
     else fig.savefig(f"{POOLED_DIR}/chart_hdbscan_screen.png", dpi=300,
                      bbox_inches="tight"))
    plt.close(fig)
except Exception as e:
    print(f"  (figure skipped: {type(e).__name__}: {str(e)[:90]})")

print(f"\n  -> hdbscan_screen.csv  ({len(G)} rows - every configuration, including")
print("     the ones that failed and why)  ·  pooled_hdbscan_choice")
print("     ·  pooled_substantive  (the a-priori mask, reused by Stage 23d)")
print("=" * 78)

## Stage 23 - Adaptive granularity (automatic: top-level OR macro->sub)
If the finest stable clustering already has >= TARGET_TOPLEVEL themes, those ARE the analytical unit. Otherwise the stable layer is kept as a macro scaffold and each macro is auto-split into sub-themes (silhouette-chosen, size/cap safeguarded); the global sub partition becomes the validated unit. Sub-layer silhouette + bootstrap Jaccard are saved.

## Stage 23H — Two levels from one HDBSCAN fit

**MAIN** is the communication layer; **SUB** is the analytical unit. The mid level is removed: it existed to bridge a gap that no longer exists, and every extra level multiplies the multiple-testing burden without adding evidence.

**Neither level's cluster count is chosen by hand.** An earlier version of this cell asked for a preferred K (`SUB_PREFER`, `MAIN_PREFER`) and searched a window around it — but the DBCV-vs-K curve on this corpus peaks at COARSE K and falls off sharply toward finer partitions, so a hand-picked target for SUB sat far from anything the density structure actually supports. This version instead finds every **local maximum** of DBCV over K in the admissible grid — a K that beats both its coarser and finer neighbours is a real, data-supported candidate; a K rejected by its neighbours on both sides is not. SUB takes the FINEST such peak that also clears the stability floor; MAIN takes the highest-DBCV peak reachable by merging SUB upward, restricted to a K a reader can hold in mind and kept meaningfully separate from SUB. The full ladder — peaks marked — is printed before anything is picked, so the choice is visible against every alternative, not asserted.

Both levels still come from the **same fit**: `MAIN` is `SUB` with `cluster_selection_epsilon` raised and nothing else changed. Because epsilon only ever *merges*, the coarse partition is a coarsening of the fine one by construction, so every sub-theme sits inside exactly one main theme — and that is still measured and printed, not assumed. Merging is a step function of epsilon, so the sweep is coarse first and then **bisected inside the bracket** that contains the window, and DBCV is now tracked at every probed epsilon (not just K and noise), so MAIN's peak is found on the same evidence SUB's was.

**Stability is measured at both levels**, against the same Hennig (2007) floor Stage 21H uses. Stage 21H only bootstraps its top-20 by DBCV and does not write those numbers back into the screen grid, so a configuration picked from that grid could otherwise carry a stability nobody had ever measured. A candidate that fails the floor is rejected in favour of the next peak.

**Silhouette is reported here too**, at the moment the levels are created — a coarse level with no separation should be visible immediately, not three stages later.

Noise keeps label `-1`, is counted at both levels, and is never a theme member.


In [ ]:
# @title
# ==============================================================================
# STAGE 23H   TWO-LEVEL HIERARCHY   (SUB = HDBSCAN by DBCV;  MAIN = Ward on SUB)
# ------------------------------------------------------------------------------
# TWO LEVELS, NOT THREE
#   SUB   - the analytical unit, found by density, fine enough to separate
#           distinct experiences
#   MAIN  - the communication layer, a short list a reader can hold in mind,
#           built by agglomerating SUB
#
# NEITHER LEVEL'S K IS CHOSEN BY HAND
#   SUB. A K is a candidate only if it is a LOCAL MAXIMUM of best-DBCV over K in
#   Stage 21H's admissible grid - it beats both its coarser and its finer
#   neighbour. A K accepted by a window but rejected by both neighbours is an
#   artifact of the window, not a finding. Peaks resting on fewer than
#   SUB_MIN_CONFIGS grid configurations are dropped (a peak seen once is not
#   distinguishable from grid noise), and a peak must reach SUB_MIN_DBCV_FRAC of
#   the best peak's DBCV to be eligible - that floor caps how much density
#   validity extra granularity is allowed to cost. Among the survivors the
#   FINEST is taken - granularity is what the analytical level is for, and every
#   survivor has already been shown to be a real peak of a validated index.
#   Stability is measured for all of them and gates none (see below).
#
#   MAIN. Raising cluster_selection_epsilon does NOT work on this corpus and the
#   sweep proves it: every merge below K=19 put more than half the themed rows
#   into one theme (K=18 -> 54%, K=14 -> 73%, K=7 -> 85%, K=2 -> 99%) and
#   swallowed most of SUB's unplaced responses (K=17 absorbed 52%, K=7 96%).
#   That is what epsilon does - it links dense regions THROUGH the noise sea, so
#   the first thing a coarse merge produces is one container plus crumbs. There
#   is no honest reader-sized layer on that path.
#
#   MAIN is therefore built by SIZE-WEIGHTED WARD AGGLOMERATION OF THE SUB
#   CENTROIDS. With weights equal to the sub-theme sizes this is EXACTLY Ward on
#   the responses themselves under the constraint that a sub-theme is never
#   split - the Lance-Williams update d(A,B) = nA*nB/(nA+nB) * ||cA - cB||^2 is
#   the same criterion Ward applies to individual points, evaluated on groups.
#   Two consequences follow by construction, not by measurement:
#     * nesting is exact - every sub-theme lies in exactly one main theme;
#     * noise is untouched - a response SUB left unplaced stays unplaced, because
#       centroids carry no noise. MAIN's noise count now EQUALS SUB's, where the
#       epsilon route quietly rescued 81 responses at the coarse level only.
#   K_main is cut where merging further would cost the most - the standard way
#   to cut a dendrogram - subject to a hard cap on how much of the corpus one
#   main theme may hold. Two details matter and both were learned from a real
#   run rather than assumed:
#     * The jump is measured in ABSOLUTE Ward cost, h(k-1) - h(k), not as the
#       ratio h(k-1)/h(k). At the fine end of the tree the merge costs are tiny,
#       so their ratio is dominated by noise: on this corpus the ratio criterion
#       picked K=30 on a 1.95x ratio between two negligible costs, in preference
#       to a 2.31x ratio between two large ones. Thirty main themes over
#       thirty-nine sub-themes is not a communication layer.
#     * Ward merge heights are monotone, so the largest absolute jumps sit at
#       the coarse end. In practice this rule therefore selects THE COARSEST
#       ADMISSIBLE CUT, and that is stated plainly rather than dressed up: what
#       stops it collapsing to two containers is the share cap, not the jump.
#   Silhouette is reported but not used: measured on this structure it climbs
#   almost monotonically with k, so it would return a MAIN barely coarser than
#   SUB, which is the opposite of a communication layer.
#   Refs: Ward (1963); Murtagh & Legendre (2014) on the Lance-Williams form;
#         Grootendorst (2022) BERTopic hierarchical_topics, which agglomerates
#         topic representations for the same reason.
#
# STABILITY IS REPORTED, NOT USED AS A GATE - AND THAT IS THE LITERATURE'S RULE
#   An earlier version of this cell refused any SUB whose GLOBAL bootstrap ARI
#   fell below 0.70, citing Hennig (2007). That was a misreading of the source.
#   Hennig's thresholds (>=0.75 valid, 0.60-0.75 a pattern whose exact
#   membership is doubtful, <0.60 not real) are CLUSTER-WISE Jaccard values, and
#   his central point is the opposite of a global gate: "in the same clustering,
#   some clusters may be very stable and others may be extremely unstable."
#   Applying his number to one whole-partition ARI threw away partitions in
#   which most themes were perfectly usable - on the first real run it rejected
#   K=39 at 0.687 and then fell back to a near-identical partition while
#   printing an inherited 0.707, which is not a defensible way to report a floor.
#
#   So: the global ARI is MEASURED and PRINTED here as context, with its spread,
#   and it decides nothing. What decides which themes may carry a claim is the
#   per-theme bootstrap Jaccard computed in Stage 25, which applies Hennig's
#   thresholds where they belong - to each cluster, one at a time.
#   N_BOOT is also raised: at 15 refits the sampling noise on this statistic was
#   wider than the differences it was being asked to adjudicate.
#
# NOISE IS A RESULT, NOT A FAILURE. HDBSCAN may leave a response in no cluster.
# Noise carries label -1, is counted at both levels, and never enters a theme.
#
# Outputs: pooled_labels_sub, pooled_labels_main7, pooled_hier_maps, pooled_levels,
#          pooled_K, pooled_canonical_level, pooled_hdbscan_hierarchy,
#          chart_hdbscan_hierarchy
# ==============================================================================
import numpy as np, pandas as pd
import hdbscan as _hdb
from sklearn.metrics import adjusted_rand_score, silhouette_score

# ---------------------------- EDIT HERE ---------------------------------------
SUB_MIN_CONFIGS     = 3     # a K needs this many admissible configs to count as
                            # a peak candidate, not one lucky configuration
SUB_MIN_DBCV_FRAC   = 0.50  # a peak must reach this fraction of the best peak's
                            # DBCV; raise toward 1.0 to favour cluster quality,
                            # lower to favour granularity
MAIN_COARSE_FLOOR   = 5     # below this it is not a communication layer
MAIN_COARSE_CEIL    = 30    # above this a reader cannot hold the list in mind
MAIN_MAX_SHARE      = 0.40  # HARD: reject any MAIN whose largest theme holds more
                            # than this share of the themed rows - a group that
                            # big is a container, not a theme
MAIN_MIN_SEPARATION = 4     # MAIN must be at least this many clusters coarser
                            # than SUB (a preference; relaxed before any floor)
MAX_NOISE   = float(globals().get("MAX_NOISE", 0.35))
N_BOOT      = 40         # bootstrap refits used to MEASURE stability. Raised
                         # from 15: at 15 draws the sd of this statistic was
                         # wider than the differences being read off it.
# There is deliberately no STAB_MIN here. Hennig's threshold is cluster-wise and
# is applied in Stage 25, per theme, where it belongs.
SEED        = int(globals().get("SEED", 42))
# --------------------------- END EDIT -----------------------------------------

E = np.asarray(pload("pooled_umap_nd"), dtype=np.float32)
G = pload("pooled_hdbscan_screen")
CH = pload("pooled_hdbscan_choice")
if E is None or G is None:
    raise RuntimeError("run Stage 21H first (needs pooled_umap_nd and the screen grid).")
if CH is None:
    raise RuntimeError("Stage 21H found no admissible configuration - there is nothing "
                       "to build a hierarchy from. Read hdbscan_screen.csv and decide.")
N = len(E)
_rng = np.random.default_rng(SEED)

print("=" * 78)
print("  STAGE 23H   SUB = HDBSCAN (DBCV local maxima) | MAIN = Ward on SUB centroids")
print("=" * 78)
print(f"  {N} responses in the chosen {E.shape[1]}-d embedding "
      f"(n_neighbors={CH['n_neighbors']})")
print(f"  SUB filter : noise <= {MAX_NOISE:.0%} (Stage 21H admissibility); "
      f"stability is MEASURED, not gated")
print(f"               per-theme Jaccard in Stage 25 decides what may be claimed "
      f"(Hennig 2007)")
print(f"  MAIN limits: K {MAIN_COARSE_FLOOR}-{MAIN_COARSE_CEIL}, largest theme "
      f"<= {MAIN_MAX_SHARE:.0%}, >= {MAIN_MIN_SEPARATION} coarser than SUB")


def _run(mcs, ms, meth, eps, X=None, mst=False):
    return _hdb.HDBSCAN(min_cluster_size=int(mcs),
                        min_samples=(None if ms is None else int(ms)),
                        cluster_selection_method=str(meth),
                        cluster_selection_epsilon=float(eps),
                        metric="euclidean", gen_min_span_tree=mst,
                        core_dist_n_jobs=1).fit(E if X is None else X)


def _summary(lab):
    k = int(lab.max()) + 1
    noise = float((lab == -1).mean())
    if not k:
        return 0, noise, 0, 0, 0
    sz = np.bincount(lab[lab >= 0], minlength=k)
    return k, noise, int(sz.min()), int(np.median(sz)), int(sz.max())


def _stability(mcs, ms, meth, eps, n_boot=N_BOOT):
    """Bootstrap ARI, computed the way Stage 21H computes it so the number is
    comparable to the floor set there. Returns (mean, sd) - the sd is printed
    because a floor applied to a point estimate with an unstated spread is how
    0.687 and 0.707 came to mean different things."""
    base = _run(mcs, ms, meth, eps).labels_
    sc = []
    for _ in range(n_boot):
        idx = _rng.choice(N, int(.8 * N), replace=False)
        lb = _run(mcs, ms, meth, eps, X=E[idx]).labels_
        sc.append(adjusted_rand_score(base[idx], lb))
    return float(np.mean(sc)), float(np.std(sc))


def _sil(lab, X=None):
    Z = E if X is None else X
    m = lab >= 0
    if m.sum() < 3 or len(set(lab[m].tolist())) < 2:
        return float("nan")
    return float(silhouette_score(Z[m], lab[m]))


def _local_peaks(vals):
    """Indices where vals[i] >= both neighbours in the SEQUENCE. Gaps in K are
    fine - a neighbour is whichever entry is adjacent in the list."""
    n = len(vals)
    return [i for i in range(n)
            if (i == 0 or vals[i] >= vals[i - 1])
            and (i == n - 1 or vals[i] >= vals[i + 1])]


# ============================= SUB =============================================
adm = G[G.admissible
        & (G.n_components == CH["n_components"])
        & (G.n_neighbors == CH["n_neighbors"])].copy()

print("\n" + "-" * 78)
print(f"  SUB: the DBCV-vs-K ladder on umap({CH['n_components']},{CH['n_neighbors']})")
SUB_PARAMS = SUB_STAB = SUB_SD = SUB_SOURCE = None
if not len(adm):
    print(f"    NOTHING admissible on this embedding; taking Stage 21H's own choice.")
else:
    _ladder = (adm.groupby("n_clusters")
                  .agg(configs=("dbcv", "size"), best_dbcv=("dbcv", "max"),
                       min_noise=("noise", "min"))
                  .reset_index().sort_values("n_clusters"))
    _ladder = _ladder[_ladder.configs >= SUB_MIN_CONFIGS].reset_index(drop=True)
    if not len(_ladder):
        raise RuntimeError(f"no K has >= {SUB_MIN_CONFIGS} admissible configurations - "
                           f"lower SUB_MIN_CONFIGS or widen Stage 21H's grid.")
    _pk = set(_local_peaks(_ladder.best_dbcv.tolist()))
    print(f"    {'K':>4} {'configs':>8} {'best DBCV':>10} {'min noise':>10}")
    for i, r in enumerate(_ladder.itertuples()):
        print(f"    {int(r.n_clusters):>4} {int(r.configs):>8} {r.best_dbcv:>10.3f} "
              f"{r.min_noise:>9.0%}" + ("  <- local peak" if i in _pk else ""))

    peaks = _ladder.iloc[sorted(_pk)].sort_values("n_clusters", ascending=False)
    _best_peak = float(peaks.best_dbcv.max())
    _floor = SUB_MIN_DBCV_FRAC * _best_peak
    _rej = peaks[peaks.best_dbcv < _floor]
    peaks = peaks[peaks.best_dbcv >= _floor]
    print(f"\n  quality floor: a peak must reach {SUB_MIN_DBCV_FRAC:.0%} of the best "
          f"peak's DBCV ({_best_peak:.3f}), i.e. >= {_floor:.3f}")
    for r in _rej.sort_values("n_clusters", ascending=False).itertuples():
        print(f"    refused K={int(r.n_clusters):<3} DBCV={r.best_dbcv:.3f} "
              f"({r.best_dbcv / max(_best_peak, 1e-9):.0%} of the best peak)")

    print(f"\n  SUB: measuring stability of every eligible peak "
          f"({N_BOOT} refits on 80% subsamples). This is CONTEXT - no peak is")
    print(f"       rejected on it. The finest eligible peak is taken, and Stage 25")
    print(f"       then reports which of its themes are individually stable enough")
    print(f"       to carry a claim.")
    _meas = []
    for prow in peaks.itertuples():
        r = adm[adm.n_clusters == prow.n_clusters].sort_values(
            "dbcv", ascending=False).iloc[0]
        ms = None if r.min_samples < 0 else int(r.min_samples)
        st, sd = _stability(r.min_cluster_size, ms, r.method, r.epsilon)
        _meas.append((int(r.n_clusters), r, ms, st, sd))
        print(f"    k={int(r.n_clusters):<3} mcs={int(r.min_cluster_size):<3} ms={ms} "
              f"{r.method:<4} eps={r.epsilon:<4} DBCV={r.dbcv:+.3f} "
              f"noise={r.noise:.0%} global ARI={st:.3f} +/-{sd:.3f}")
    if _meas:
        _k, r, ms, st, sd = _meas[0]          # peaks are sorted finest-first
        SUB_PARAMS = dict(min_cluster_size=int(r.min_cluster_size), min_samples=ms,
                          method=str(r.method), epsilon=float(r.epsilon))
        SUB_STAB, SUB_SD = st, sd
        SUB_SOURCE = "finest DBCV local peak above the quality floor"
        print(f"    -> taking the finest: K={_k}")

if SUB_PARAMS is None:
    # Only reachable when the grid offered no eligible peak at all. The fallback
    # is RE-MEASURED here rather than inheriting Stage 21H's number, which came
    # from a different bootstrap draw.
    SUB_PARAMS = dict(min_cluster_size=CH["min_cluster_size"],
                      min_samples=CH["min_samples"],
                      method=CH["method"], epsilon=CH["epsilon"])
    SUB_STAB, SUB_SD = _stability(SUB_PARAMS["min_cluster_size"],
                                  SUB_PARAMS["min_samples"],
                                  SUB_PARAMS["method"], SUB_PARAMS["epsilon"])
    _inherited = float(CH.get("stability", float("nan")))
    SUB_SOURCE = "stage21h-choice, re-measured here (no eligible peak)"
    print(f"\n    no eligible peak on this embedding. Falling back to Stage 21H's own "
          f"configuration and RE-MEASURING it:")
    print(f"      mcs={SUB_PARAMS['min_cluster_size']} ms={SUB_PARAMS['min_samples']} "
          f"{SUB_PARAMS['method']} eps={SUB_PARAMS['epsilon']}  ->  "
          f"global ARI={SUB_STAB:.3f} +/-{SUB_SD:.3f}")
    if _inherited == _inherited:
        print(f"      (Stage 21H reported {_inherited:.3f} on a different draw; the "
              f"number used here is the one measured above)")

_sub_fit = _run(SUB_PARAMS["min_cluster_size"], SUB_PARAMS["min_samples"],
                SUB_PARAMS["method"], SUB_PARAMS["epsilon"], mst=True)
sub = _sub_fit.labels_.copy()
ks, ns, mn, md, mx = _summary(sub)
sub_sil = _sil(sub)
if ks < 2:
    raise RuntimeError(f"the chosen SUB configuration produced {ks} cluster(s).")
print(f"\n  SUB  K={ks}  noise={ns:.1%}  sizes min {mn} / median {md} / max {mx} "
      f"({mx / max(N, 1):.0%} of the corpus)")
print(f"       global ARI={SUB_STAB:.3f} +/-{SUB_SD:.3f} (context only)   "
      f"silhouette(themed rows)={sub_sil:+.3f}")
print(f"       [{SUB_SOURCE}]")
print(f"       Which of these {ks} themes may carry a claim is decided in Stage 25 by")
print(f"       per-theme Jaccard: >=0.75 claimable, 0.60-0.75 a pattern, <0.60 excluded.")

# ============================= MAIN ============================================
# Size-weighted Ward on the sub-theme centroids. With w = cluster size this is
# Ward on the responses under the constraint that a sub-theme is never split:
#   d(A,B) = (nA*nB / (nA+nB)) * ||cA - cB||^2      (Lance-Williams, Ward 1963)
# merge the closest pair, then cA <- (nA*cA + nB*cB)/(nA+nB), nA <- nA+nB.
print("\n" + "-" * 78)
print(f"  MAIN: size-weighted Ward agglomeration of the {ks} sub-theme centroids")
print(f"        (epsilon-merging was abandoned here: on this corpus every merge below")
print(f"         K=19 produced one container holding most of the corpus and absorbed")
print(f"         the responses SUB had left unplaced. Ward on centroids cannot do")
print(f"         either - centroids carry no noise, and Ward balances variance.)")


def ward_merge_order(cents, sizes):
    """Size-weighted Ward agglomeration of groups.
    Returns (assign_at, height): assign_at[k] maps each original group to a
    0..k-1 label, height[k] is the Ward cost paid to go from k+1 groups to k.
    Written out because scipy's linkage assumes unit weights, and unit weights
    would let a 10-response theme outvote a 115-response one. Verified against
    scipy: with all weights 1 this reproduces scipy's ward exactly for every k."""
    k0 = len(cents)
    cur = {i: dict(c=np.array(cents[i], dtype=np.float64),
                   n=float(sizes[i]), members=[i]) for i in range(k0)}
    assign_at = {k0: np.arange(k0)}
    height = {}
    while len(cur) > 1:
        keys = sorted(cur)
        best, bi, bj = None, None, None
        for ii in range(len(keys)):
            for jj in range(ii + 1, len(keys)):
                a, b = cur[keys[ii]], cur[keys[jj]]
                d = (a["n"] * b["n"] / (a["n"] + b["n"])) * float(
                    np.sum((a["c"] - b["c"]) ** 2))
                if best is None or d < best:
                    best, bi, bj = d, keys[ii], keys[jj]
        a, b = cur.pop(bi), cur.pop(bj)
        nn = a["n"] + b["n"]
        cur[bi] = dict(c=(a["n"] * a["c"] + b["n"] * b["c"]) / nn, n=nn,
                       members=a["members"] + b["members"])
        lab = np.empty(k0, dtype=int)
        for gi, node in enumerate(sorted(cur)):
            for m in cur[node]["members"]:
                lab[m] = gi
        assign_at[len(cur)] = lab
        height[len(cur)] = float(best)
    return assign_at, height


_sizes = np.bincount(sub[sub >= 0], minlength=ks).astype(float)
_cents = np.stack([E[sub == c].mean(0) for c in range(ks)])
_assign, _height = ward_merge_order(_cents, _sizes)

_hi_k = min(MAIN_COARSE_CEIL, ks - MAIN_MIN_SEPARATION)
_cand_ks = [k for k in sorted(_assign) if MAIN_COARSE_FLOOR <= k <= _hi_k]
if not _cand_ks:
    _cand_ks = [k for k in sorted(_assign) if MAIN_COARSE_FLOOR <= k <= MAIN_COARSE_CEIL
                and k < ks]
    if _cand_ks:
        print(f"\n  note: no K is {MAIN_MIN_SEPARATION}+ coarser than SUB (K={ks}); the")
        print(f"        separation preference is relaxed - MAIN and SUB will sit close")
        print(f"        together, so report them as one structure at two resolutions.")
if not _cand_ks:
    raise RuntimeError(f"SUB has only {ks} themes, so no MAIN between "
                       f"{MAIN_COARSE_FLOOR} and {min(MAIN_COARSE_CEIL, ks - 1)} exists. "
                       f"Report the SUB level alone.")

# WHICH CRITERION CUTS THE TREE, AND WHY IT IS NOT SILHOUETTE.
# Measured on this structure, silhouette climbs almost monotonically with K
# (+0.448 at K=5 rising to +0.581 at K=16 on a proxy run of this corpus),
# because it rewards compact globular pieces and a finer cut always looks
# tighter. Choosing by silhouette would therefore return a MAIN barely coarser
# than SUB - the opposite of what a communication layer is for. The native
# criterion for cutting a Ward tree is the MERGE COST: if the next merge costs
# far more than this one, the groups it would join are genuinely separate and
# this is where to stop. Gap ratio is primary; silhouette is printed beside it
# as a diagnostic, not as the decision.
rows = []
for k in _cand_ks:
    m = np.full(N, -1, dtype=int)
    a = _assign[k]
    for c in range(ks):
        m[sub == c] = a[c]
    sz = np.bincount(m[m >= 0], minlength=k)
    _h, _hn = _height.get(k, np.nan), _height.get(k - 1, np.nan)
    rows.append(dict(k=k, silhouette=_sil(m),
                     max_share=float(sz.max() / max(int((m >= 0).sum()), 1)),
                     min_size=int(sz.min()), merge_cost=_h,
                     gap_abs=(_hn - _h) if (_h == _h and _hn == _hn) else np.nan,
                     gap_ratio=(_hn / _h) if (_h and _h == _h and _hn == _hn)
                     else np.nan))
W = pd.DataFrame(rows).sort_values("k")
print(f"\n        jump = ABSOLUTE Ward cost of merging one step further, h(k-1)-h(k).")
print(f"        Large jump = the next merge would join two genuinely separate")
print(f"        groups. ratio is shown too but is NOT the criterion: at the fine")
print(f"        end both costs are tiny and their ratio is mostly noise.")
print(f"        {'K':>4} {'jump':>11} {'ratio':>7} {'silhouette':>11} {'biggest':>9} "
      f"{'smallest':>9}")
for r in W.itertuples():
    print(f"        {int(r.k):>4} {r.gap_abs:>11.1f} {r.gap_ratio:>7.2f} "
          f"{r.silhouette:>+11.3f} {r.max_share:>8.0%} {int(r.min_size):>9}"
          + ("   CONTAINER" if r.max_share > MAIN_MAX_SHARE else ""))

_okW = W[(W.max_share <= MAIN_MAX_SHARE) & W.gap_abs.notna()]
if not len(_okW):
    _best = W.loc[W.max_share.idxmin()]
    print(f"\n  !! every candidate exceeds the {MAIN_MAX_SHARE:.0%} cap, or no merge gap "
          f"could be computed. The least dominated (K={int(_best.k)}, "
          f"{_best.max_share:.0%})")
    print(f"     is taken and MUST be described as a presentation grouping, not a result.")
else:
    _best = _okW.loc[_okW.gap_abs.idxmax()]
    _refused = W[~W.k.isin(_okW.k)]
    if len(_refused):
        print(f"\n  refused: " + ", ".join(
            f"K={int(r.k)}" + (f" (container {r.max_share:.0%})"
                               if r.max_share > MAIN_MAX_SHARE else " (no jump)")
            for r in _refused.itertuples()))
    print(f"  MAIN cut where merging further costs most: K={int(_best.k)}  "
          f"jump={_best.gap_abs:.1f}  ratio={_best.gap_ratio:.2f}x  "
          f"largest theme {_best.max_share:.0%}  silhouette={_best.silhouette:+.3f}")
    print(f"  Ward heights are monotone, so this is in effect the COARSEST cut that")
    print(f"  is not a container - which is what a communication layer should be.")
    print(f"  The {MAIN_MAX_SHARE:.0%} share cap is what keeps it from collapsing further.")
    _rat_pick = int(_okW.loc[_okW.gap_ratio.idxmax(), "k"])
    _sil_pick = int(_okW.loc[_okW.silhouette.idxmax(), "k"])
    if _rat_pick != int(_best.k):
        print(f"  (the ratio criterion would have said K={_rat_pick} - rejected: at that "
              f"k both merge costs are negligible, so their ratio is noise.)")
    if _sil_pick != int(_best.k):
        print(f"  (silhouette alone would have said K={_sil_pick} - not used, it rewards "
              f"granularity and would return a MAIN barely coarser than SUB.)")

K_MAIN = int(_best.k)
main = np.full(N, -1, dtype=int)
_amap = _assign[K_MAIN]
for c in range(ks):
    main[sub == c] = _amap[c]
km, nm_, mn2, md2, mx2 = _summary(main)
main_sil = _sil(main)

# MAIN's stability is the stability of the WHOLE two-step procedure: refit SUB on
# the subsample, re-run the same Ward agglomeration to the same K, compare. That
# is the honest question - a reader is being shown main themes produced by both
# steps, not by HDBSCAN alone.
def _main_labels_from(Xs, subs):
    kk = int(subs.max()) + 1
    if kk < 2:
        return np.full(len(subs), -1, dtype=int)
    szs = np.bincount(subs[subs >= 0], minlength=kk).astype(float)
    cts = np.stack([Xs[subs == c].mean(0) for c in range(kk)])
    asg, _ = ward_merge_order(cts, szs)   # height not needed here
    tgt = min(K_MAIN, kk - 1) if kk > 1 else 1
    tgt = max(tgt, 1)
    if tgt not in asg:
        tgt = min(asg, key=lambda z: abs(z - K_MAIN))
    out = np.full(len(subs), -1, dtype=int)
    a = asg[tgt]
    for c in range(kk):
        out[subs == c] = a[c]
    return out


_mb = []
for _ in range(N_BOOT):
    idx = _rng.choice(N, int(.8 * N), replace=False)
    sb = _run(SUB_PARAMS["min_cluster_size"], SUB_PARAMS["min_samples"],
              SUB_PARAMS["method"], SUB_PARAMS["epsilon"], X=E[idx]).labels_
    _mb.append(adjusted_rand_score(main[idx], _main_labels_from(E[idx], sb)))
MAIN_STAB, MAIN_SD = float(np.mean(_mb)), float(np.std(_mb))

print(f"\n  MAIN K={km}  noise={nm_:.1%}  sizes min {mn2} / median {md2} / max {mx2} "
      f"({mx2 / max(N, 1):.0%} of the corpus)")
print(f"       stability={MAIN_STAB:.3f} +/-{MAIN_SD:.3f} (SUB refit + Ward re-run, "
      f"the full procedure)")
print(f"       silhouette(themed rows)={main_sil:+.3f}")
if MAIN_STAB < 0.60:
    print(f"       !! below 0.60. On Hennig's scale a grouping this unstable is not a")
    print(f"          finding; use MAIN for narrative framing only, never for a claim.")
elif MAIN_STAB < 0.75:
    print(f"       0.60-0.75 on Hennig's scale: a pattern, but which sub-themes sit in")
    print(f"          which main theme is not firmly determined. Frame accordingly.")

# ---- nesting and noise: both exact by construction, still printed -------------
rows = []
for c in range(ks):
    m = sub == c
    par = main[m]
    par = par[par >= 0]
    top = int(np.bincount(par).argmax()) if len(par) else -1
    pur = float((par == top).mean()) if len(par) else np.nan
    rows.append(dict(sub=c, main=top, purity=pur, n=int(m.sum())))
X_cross = pd.DataFrame(rows)
sub_to_main = {int(r.sub): int(r.main) for r in X_cross.itertuples()}
_impure = int((X_cross.purity < 1.0).sum())
print(f"\n  nesting: {ks - _impure} of {ks} sub-themes sit entirely inside one main "
      f"theme (exact by construction - Ward merges whole sub-themes)")
assert _impure == 0, "a sub-theme was split across main themes - impossible here"

print(f"  noise: {int((sub == -1).sum())} responses have no sub-theme "
      f"({(sub == -1).mean():.1%}); MAIN inherits exactly the same "
      f"{int((main == -1).sum())} ({(main == -1).mean():.1%}).")
print("    The epsilon route used to rescue some of them at the coarse level only,")
print("    which made MAIN look cleaner than the data it rests on. It no longer can.")

# ---- publish -----------------------------------------------------------------
psave("pooled_labels_sub", sub)
psave("pooled_sub_params", SUB_PARAMS)      # Stage 23d refits with these to test
                                            # HDBSCAN's own soft-membership rule
psave("pooled_labels_main7", main)          # historical name; later stages read it
psave("pooled_labels", sub)
psave("pooled_levels", ["main", "sub"])
psave("pooled_K", int(ks))
psave("pooled_canonical_level", "sub")
psave("pooled_hier_maps", dict(sub_to_main=sub_to_main))
psave("pooled_sub_of_macro", sub_to_main)
psave("pooled_hdbscan_hierarchy",
      dict(sub_params=SUB_PARAMS,
           main_method="ward-on-sub-centroids",   # Stage 25 branches on this
           main_criterion="largest absolute merge jump, share-capped",
           main_k=int(K_MAIN),
           main_epsilon=None,                     # no epsilon route any more
           main_ward_table=W,
           crosswalk=X_cross, K=dict(main=km, sub=ks),
           noise=dict(main=float((main == -1).mean()),
                      sub=float((sub == -1).mean())),
           silhouette=dict(main=main_sil, sub=sub_sil),
           stability=dict(main=MAIN_STAB, sub=SUB_STAB),
           stability_sd=dict(main=MAIN_SD, sub=SUB_SD),
           stability_n_boot=int(N_BOOT),
           sub_source=SUB_SOURCE, same_fit_both_levels=False,
           max_noise=MAX_NOISE, stability_is_a_gate=False,
           sub_min_configs=SUB_MIN_CONFIGS, sub_min_dbcv_frac=SUB_MIN_DBCV_FRAC,
           main_coarse_floor=MAIN_COARSE_FLOOR, main_coarse_ceil=MAIN_COARSE_CEIL,
           main_max_share=MAIN_MAX_SHARE, main_min_separation=MAIN_MIN_SEPARATION,
           persistence=dict(sub=_sub_fit.cluster_persistence_.tolist())))
X_cross.to_csv(f"{POOLED_DIR}/hierarchy_crosswalk.csv", index=False)
W.to_csv(f"{POOLED_DIR}/main_ward_sweep.csv", index=False)
print(f"\n  published: main={km} / sub={ks}, canonical level = sub")
print(f"  SUB from HDBSCAN (DBCV local maxima); MAIN from size-weighted Ward on SUB's")
print(f"  centroids. Nesting is structural, noise is identical at both levels.")

# ------------------------------- figure ---------------------------------------
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(9.6, 3.5))
    a = ax[0]
    a.plot(W.k, W.gap_abs, "-o", ms=3.5, color="#44AA99", lw=1.5)
    a.axvline(K_MAIN, color="#CC6677", lw=1.4)
    a.text(K_MAIN, a.get_ylim()[1], f"  chosen K={K_MAIN}", fontsize=7.5, va="top",
           color="#CC6677")
    a2 = a.twinx()
    a2.plot(W.k, 100 * W.max_share, "-s", ms=3, color="#9AA5B1", lw=1.0, alpha=.8)
    a2.axhline(100 * MAIN_MAX_SHARE, color="#9AA5B1", ls=":", lw=1.0)
    a2.set_ylabel("largest theme (% of themed)", fontsize=8, color="#5B6670")
    a2.tick_params(labelsize=7, colors="#5B6670")
    a.set_xlabel("main themes after Ward agglomeration", fontsize=9)
    a.set_ylabel("cost of merging one step further", fontsize=9)
    b = ax[1]
    sz = np.bincount(sub[sub >= 0], minlength=ks)
    b.bar(np.arange(ks), np.sort(sz), color="#6699CC", width=.85)
    b.axhline(np.median(sz), color="#14202B", ls="--", lw=.9)
    b.text(0, np.median(sz), f" median {int(np.median(sz))}", fontsize=7.5,
           va="bottom", color="#14202B")
    b.set_xlabel("sub-theme, smallest to largest", fontsize=9)
    b.set_ylabel("responses in the theme", fontsize=9)
    for _a in ax:
        for sp in ("top", "right"):
            _a.spines[sp].set_visible(False)
        _a.tick_params(length=2, labelsize=8)
    _t = (f"{km} main themes (Ward on sub centroids) over {ks} sub-themes "
          f"(HDBSCAN), {(sub == -1).mean():.0%} unplaced at both levels")
    if "fig_title" in globals():
        fig_title(fig, _t)
    else:
        fig.text(.5, .97, _t, ha="center", va="top", fontsize=11.5, fontweight="bold")
    fig.subplots_adjust(top=.86, bottom=.18, left=.08, right=.90, wspace=.34)
    (save_fig(fig, "chart_hdbscan_hierarchy") if "save_fig" in globals()
     else fig.savefig(f"{POOLED_DIR}/chart_hdbscan_hierarchy.png", dpi=300,
                      bbox_inches="tight"))
    plt.close(fig)
except Exception as e:
    print(f"  (figure skipped: {type(e).__name__}: {str(e)[:90]})")
print("=" * 78)

## Stage 23d — Post-hoc assignment of the held-out responses

Stage 18 kept the One-Word item and very short narrative answers out of the thematic **fitting** so they could not distort the structure. They were never discarded. Here each of them is assigned to a theme by nearest centroid **in the original embedding space**, with an explicit confidence **margin** (best minus second-best). A row is placed at the finest level whose margin clears the threshold — **sub, then main** — and if neither does, it stays *unassigned* rather than being forced.

Two levels, not three: the mid level was removed in Stage 23H. And the centroids are built from **themed rows only** — the responses HDBSCAN left with no theme (`-1`) are not a theme, so they are not a centroid and never a destination. A held-out row can receive a theme or stay unassigned; it can never be assigned to "noise".

The rule is **validated on the core themed rows**, which have a known theme, so its accuracy is measured rather than assumed — and because core rows are long and informative, that accuracy is reported as an *optimistic upper bound* for these shorter rows. All of these rows are flagged `is_core = False`.


In [ ]:
# @title
# ==============================================================================
# STAGE 23d  -  FINAL MEMBERSHIP:  WHO BELONGS TO WHICH THEME, AND ON WHAT EVIDENCE
# ------------------------------------------------------------------------------
# WHAT THIS CELL REPLACES, AND WHY
#
#   This stage used to be called "post-hoc assignment of the HELD-OUT responses".
#   It was written for the rows Notebook 1 Stage 18 kept out of the fitting - the
#   One-Word item and answers below a length floor. Notebook 1 no longer holds
#   anything out, so its own printed output read:
#
#       "No held-out rows found (pooled_held_* absent or empty)."
#
#   It was dead code. But the machinery inside it was exactly right, and Notebook
#   3 later re-implemented the same design from scratch on a different population.
#   Rather than keep two copies of one idea in two notebooks, this cell keeps the
#   machinery and gives it the population that actually needs it: the responses
#   HDBSCAN left as noise. Notebook 3's recovery cell is deleted. After this stage
#   every question of the form "does this response belong to a theme" is answered
#   in Notebook 2 and Notebook 3 only draws the answer.
#
#   Held-out rows are still handled if they ever reappear - they go through the
#   same calibrated rule and are marked with source="held-out".
#
# THE PROBLEM THIS SOLVES
#
#   HDBSCAN leaves a point as noise when it sits in no dense region. For a
#   one-word answer that is the correct and honest outcome. For an 83-word answer
#   describing an advisor, the judges, and the team it is a failure of
#   representation, not a finding: a single vector cannot sit inside three dense
#   regions at once, so it lands between them and belongs to none. Stage 21H
#   already measured that this cannot be fixed by tuning - every configuration
#   with materially lower noise gives up around 40% of the DBCV. So the recovery
#   happens here, after the structure is fixed, and it is measured rather than
#   assumed.
#
# HOW THE RULE IS CHOSEN:  THREE CANDIDATES, ONE CALIBRATION, MEASURED WINNER
#
#   There is no a-priori reason to believe any one assignment rule is best on this
#   corpus, so this cell does not assert one. It scores three, on the same rows,
#   under the same protocol, and prints all three:
#
#   R1  CENTROID COSINE + MARGIN
#       Cosine to each theme centroid in the ORIGINAL embedding space - not
#       through UMAP, whose transform of an out-of-distribution point is
#       unreliable. Confidence is the MARGIN between the best and second-best
#       theme, which is what distinguishes "clearly this theme" from "somewhere
#       between two themes". This is the rule the old cell used.
#
#   R2  HDBSCAN SOFT MEMBERSHIP
#       all_points_membership_vectors() asks the condensed tree itself how much
#       each point belongs to each cluster. It is the method's own account of
#       partial membership rather than a geometric proxy layered on top, so it
#       deserves to be tested. It is only admitted if a refit with identical
#       parameters reproduces pooled_labels_sub EXACTLY - otherwise its membership
#       vectors describe a different partition than the one this study reports,
#       and it is disqualified and said so.
#
#   R3  CONSENSUS ACROSS THE TOP ADMISSIBLE CONFIGURATIONS
#       A response left out by ONE good configuration may be placed - consistently
#       - by several others. That disagreement is information the single chosen
#       partition throws away. Each of the top CONSENSUS_TOPN admissible
#       configurations from Stage 21H's grid is refit on the SAME chosen UMAP
#       embedding, each of its clusters is mapped to the reference sub-theme its
#       members mostly carry, and a response is assigned when enough
#       configurations agree. This adapts the robustness-by-agreement principle of
#       arXiv 2603.18358 (2026) - which uses eleven embedding models - to a scale
#       this project can actually afford, using configurations instead of models.
#
#   CALIBRATION. Every rule is scored on the CORE THEMED rows, whose theme is
#   known: sweep its thresholds, keep only those reaching TARGET_ACC, and among
#   those take the one maximising the EXPECTED NUMBER OF CORRECTLY PLACED
#   RESPONSES (rows placed x measured accuracy). Then the rules are compared on
#   the same quantity. Maximising raw coverage instead would always reward the
#   loosest threshold that scrapes past the floor; discounting each extra row by
#   the precision it costs does not. The losing rules are printed with their
#   measured numbers, so the choice is auditable and can be overturned.
#
#   THE CALIBRATION SET IS THE HARD CORE ROWS, NOT ALL OF THEM. This was a real
#   defect in the first draft of this cell and it is worth stating plainly. Core
#   rows sit in dense regions by construction, so they are easy: on a test run
#   the loosest possible threshold - accept everything, no margin required at all
#   - still reproduced 96.9% of core labels. The accuracy floor therefore never
#   bit, the sweep always chose "place everything", and every noise row including
#   the one-word answers was forced into a theme. That is exactly the failure
#   Grootendorst (2022) warns about in outlier reduction, arrived at through an
#   accuracy number that looked reassuring.
#
#   The fix is to calibrate on the core rows that RESEMBLE the population being
#   assigned: the bottom CALIB_HARD_FRAC of core rows by peripherality (maximum
#   cosine to any centroid - low means the row sits at the edge of its theme, the
#   way a noise row does). The same rows are used for all three rules, so the
#   comparison stays fair, and accuracy on ALL core rows is printed alongside so
#   the difference between the easy and the honest number is visible.
#
#   LEAKAGE. R3's cluster-to-theme map is built from the reference labels of the
#   cluster's members, so scoring row i against a map that counted row i would be
#   circular. Every count is therefore LEAVE-ONE-OUT: row i's own label is
#   subtracted before its own vote is read.
#
#   EVEN THE HARD NUMBER IS AN UPPER BOUND, AND IS REPORTED AS ONE. Every
#   calibration row has a theme - that is what makes it scoreable - and the rows
#   being recovered are exactly the rows that did not. No claim below pretends
#   otherwise.
#
# WHAT IS NOT RECOVERED, AND WHY THAT IS CORRECT
#   Responses that no rule can place stay unassigned, with a reason recorded. On
#   this corpus most of them are one to nine words long, or describe a single
#   event nothing else in the survey resembles. Forcing them into a theme would be
#   fabrication - it is precisely the failure this project abandoned Ward to
#   escape. Unassigned is a result, not a gap.
#
# OUTPUT
#   pooled_membership - ONE row per response, the canonical answer to "which theme
#   does this belong to and how do we know". Stage 33b folds it into pooled_master
#   and Notebook 3 reads it from there. Columns:
#     row_id, Participant, Team, question, n_words, is_substantive,
#     sub_id, main_id        the primary partition, UNCHANGED (-1 stays -1)
#     tier                   core | assigned | unassigned   (23M adds assigned-multi)
#     assigned_sub/main      the recovered theme, or -1
#     confidence, rule       what decided it and how strongly
#     unassigned_reason      thin | rare | below_threshold   (never blank)
#     sub_wide / main_wide   sub_id where core, assigned_sub where assigned, else -1
#                            - the single column every downstream figure should use
#   Also: pooled_membership_calibration (all three rules' curves),
#         pooled_projection (kept for backward compatibility), membership.csv
#
# Refs: Moulavi et al. (2014) DBCV; McInnes & Healy (2017) HDBSCAN soft clustering;
#       Grootendorst (2022) on the risks of outlier reduction; arXiv 2603.18358
#       (2026) robustness by agreement; Hennig (2007) cluster-wise stability.
# ==============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------- EDIT HERE ------------------------------------
TARGET_ACC       = 0.90   # a rule must reproduce the known theme of the HARD core
                          # rows at least this often before it is allowed to place
                          # anything. Fixed before the sweep is run.
CALIB_HARD_FRAC  = 0.40   # the calibration set: the most peripheral this fraction
                          # of core rows, i.e. the ones that most resemble the
                          # rows being recovered. Calibrating on all core rows
                          # measures performance on easy cases and lets the floor
                          # be cleared by a threshold that places everything -
                          # see the note in the header.
GRID_MARGIN      = [0.00, 0.01, 0.02, 0.03, 0.05, 0.08, 0.12]   # R1
GRID_COS         = [0.00, 0.30, 0.35, 0.40, 0.45, 0.50]         # R1 floor
GRID_PROB        = [0.05, 0.10, 0.20, 0.30, 0.40, 0.50]         # R2
GRID_PMARGIN     = [0.00, 0.02, 0.05, 0.10]                     # R2
GRID_VOTE        = [0.50, 0.60, 0.70, 0.80, 0.90, 1.00]         # R3
CONSENSUS_TOPN   = 8      # admissible configurations used for R3. They must share
                          # the chosen UMAP settings so the saved embedding can be
                          # reused - refitting UMAP nine times would cost minutes
                          # for no additional evidence.
CONSENSUS_MIN_CFG = 3     # a response must be placed by at least this many of them
                          # before its votes are counted at all
MIN_CLUSTER_MEMBERS = 5   # a consensus cluster smaller than this cannot define a
                          # theme mapping
SEED             = int(globals().get("SEED", 42))
# ----------------------------- END EDIT ---------------------------------------

# ---- stale-artifact guard (the same block opens every stage below) ------------
# The 1134-vs-1505 class of failure: a label array left on disk by an EARLIER
# clustering is loaded silently and is silently mis-aligned with the current
# rows, and every number downstream is then computed against the wrong people.
def _labels(n_rows, stage, levels=("main", "sub")):
    _KEY = {"main": "pooled_labels_main7", "sub": "pooled_labels_sub"}
    out = {}
    for lv in levels:
        a = pload(_KEY[lv])
        if a is None:
            raise RuntimeError(f"[{stage}] {_KEY[lv]} is missing - run Stage 23H first.")
        a = np.asarray(a)
        if len(a) != n_rows:
            raise RuntimeError(
                f"[{stage}] {_KEY[lv]} has {len(a)} rows but the pooled set has "
                f"{n_rows}. That artifact belongs to an EARLIER clustering. Re-run "
                f"Stage 23H (then Stage 30) before this stage.")
        out[lv] = a
    return out


def unit(v):
    v = np.asarray(v, dtype=np.float32)
    return v / (np.linalg.norm(v, axis=-1, keepdims=True) + 1e-9)
# ------------------------------------------------------------------------------

E     = np.asarray(require("pooled_embeddings", stage="Stage 23d"), dtype=np.float32)
meta  = require("pooled_meta", stage="Stage 23d").reset_index(drop=True)
N     = len(meta)
if len(E) != N:
    raise RuntimeError(f"[Stage 23d] pooled_embeddings has {len(E)} rows but "
                       f"pooled_meta has {N} - the pooled set is not self-consistent.")
LAB   = _labels(N, "Stage 23d")
SUB, MAIN = np.asarray(LAB["sub"]).copy(), np.asarray(LAB["main"]).copy()
K     = int(SUB.max()) + 1 if (SUB >= 0).any() else 0
if K < 2:
    raise RuntimeError(f"[Stage 23d] the sub level has {K} theme(s) - nothing to assign to.")

# sub -> main is deterministic (MAIN is Ward on the SUB centroids), so a recovered
# sub-theme carries its main-theme with it. There is no second rule for MAIN and
# no way for the two levels to disagree.
_hm = pload("pooled_hier_maps") or {}
S2M = dict(_hm.get("sub_to_main") or pload("pooled_sub_of_macro") or {})
if not S2M:
    S2M = {int(c): int(np.bincount(MAIN[SUB == c][MAIN[SUB == c] >= 0]).argmax())
           for c in range(K) if (SUB == c).any() and (MAIN[SUB == c] >= 0).any()}
    print("  (sub_to_main not on disk; reconstructed from the label arrays)")
S2M = {int(k): int(v) for k, v in S2M.items()}

# word counts and the a-priori substantive mask, from Stage 21H so there is ONE
# definition in the notebook rather than one per cell
_SB = pload("pooled_substantive")
if _SB is not None and len(_SB) == N:
    NW    = np.asarray(_SB["n_words"].values, dtype=float)
    SUBST = np.asarray(_SB["is_substantive"].values, dtype=bool)
    SUBW  = int(pload("pooled_hdbscan_choice").get("substantive_min_words", 10)) \
            if isinstance(pload("pooled_hdbscan_choice"), dict) else 10
else:
    raise RuntimeError("[Stage 23d] pooled_substantive is missing or misaligned. "
                       "Re-run Stage 21H - this stage must use the SAME definition "
                       "of 'substantive' that the selection rule used, not its own.")

CORE  = SUB >= 0
NOISE = ~CORE

print("=" * 78)
print("  STAGE 23d   FINAL MEMBERSHIP")
print("=" * 78)
print(f"  {N} responses · {K} sub-themes · {len(set(S2M.values()))} main themes")
print(f"  placed by HDBSCAN (core)      {int(CORE.sum()):>5}  ({CORE.mean():.1%})")
print(f"  left as noise                 {int(NOISE.sum()):>5}  ({NOISE.mean():.1%})")
print(f"    of which SUBSTANTIVE (>={SUBW}w) {int((NOISE & SUBST).sum()):>5}"
      f"   <- what this stage exists to recover")
print(f"    of which THIN                {int((NOISE & ~SUBST).sum()):>5}"
      f"   <- correctly left out; too short to carry a theme")

if not NOISE.any():
    print("\n  No noise rows. Every response already has a theme; nothing to recover.")

# ---------- centroids from THEMED rows only ------------------------------------
# A centroid over -1 would be the mean of "everything that belongs nowhere". That
# is not a direction in the embedding space, and it would attract exactly the
# vague answers this stage is trying to judge.
En   = unit(E)
CENT = unit(np.vstack([En[SUB == c].mean(0) for c in range(K)]))
print(f"\n  centroids built from {int(CORE.sum())} themed rows; "
      f"{int(NOISE.sum())} noise rows excluded from every centroid.")
print("  Noise is not a theme, so it is never a destination: a row is placed in a")
print("  real theme or left unassigned.")


def _top2(Smat):
    """best index, best value, and the margin to the runner-up, per row."""
    Smat = np.asarray(Smat, dtype=np.float32)
    if Smat.shape[1] < 2:
        b = Smat.argmax(1)
        return b, Smat.max(1), np.full(len(Smat), np.inf, dtype=np.float32)
    part = np.argpartition(-Smat, 1, axis=1)[:, :2]
    top  = np.take_along_axis(Smat, part, 1)
    o    = np.argsort(-top, axis=1)
    part = np.take_along_axis(part, o, 1)
    top  = np.take_along_axis(top, o, 1)
    return part[:, 0], top[:, 0], top[:, 0] - top[:, 1]


# ==============================================================================
# R1   centroid cosine + margin
# ==============================================================================
R1_best, R1_sim, R1_mrg = _top2(En @ CENT.T)
RULES = {"R1 centroid-cosine": dict(
    best=R1_best,
    grid=[dict(cos=c, margin=m) for c in GRID_COS for m in GRID_MARGIN],
    ok=lambda p: (R1_sim >= p["cos"]) & (R1_mrg >= p["margin"]),
    conf=R1_mrg,
    desc=lambda p: f"cos>={p['cos']:.2f}, margin>={p['margin']:.2f}")}

# ==============================================================================
# R2   HDBSCAN soft membership
# ==============================================================================
print("\n" + "-" * 78)
print("  R2  HDBSCAN soft membership - admitted only if a refit reproduces the")
print("      published partition EXACTLY")
_r2_ok, _r2_why = False, ""
try:
    import hdbscan as _hdb
    U  = pload("pooled_umap_nd")
    SP = pload("pooled_sub_params")
    if U is None or len(np.asarray(U)) != N:
        _r2_why = "pooled_umap_nd missing or misaligned"
    elif not isinstance(SP, dict):
        _r2_why = ("pooled_sub_params not on disk - add "
                   "psave('pooled_sub_params', SUB_PARAMS) to Stage 23H and re-run it")
    else:
        U = np.asarray(U, dtype=np.float32)
        _cl = _hdb.HDBSCAN(min_cluster_size=int(SP["min_cluster_size"]),
                           min_samples=(None if SP.get("min_samples") is None
                                        else int(SP["min_samples"])),
                           cluster_selection_method=str(SP["method"]),
                           cluster_selection_epsilon=float(SP["epsilon"]),
                           metric="euclidean", gen_min_span_tree=True,
                           prediction_data=True, core_dist_n_jobs=1).fit(U)
        if not np.array_equal(np.asarray(_cl.labels_), SUB):
            _nd = int((np.asarray(_cl.labels_) != SUB).sum())
            _r2_why = (f"the refit differs from pooled_labels_sub on {_nd} row(s), so "
                       f"its membership vectors describe a DIFFERENT partition than "
                       f"the one this study reports")
        else:
            MV = np.asarray(_hdb.all_points_membership_vectors(_cl), dtype=np.float32)
            if MV.ndim == 1:
                MV = MV.reshape(-1, 1)
            if MV.shape[1] != K:
                _r2_why = f"membership vectors have {MV.shape[1]} columns, expected {K}"
            else:
                R2_best, R2_p, R2_m = _top2(MV)
                RULES["R2 soft-membership"] = dict(
                    best=R2_best,
                    grid=[dict(prob=p, margin=m) for p in GRID_PROB for m in GRID_PMARGIN],
                    ok=lambda p: (R2_p >= p["prob"]) & (R2_m >= p["margin"]),
                    conf=R2_p,
                    desc=lambda p: f"p>={p['prob']:.2f}, margin>={p['margin']:.2f}")
                _r2_ok = True
                print("      refit reproduces pooled_labels_sub exactly - R2 admitted.")
except Exception as e:
    _r2_why = f"{type(e).__name__}: {str(e)[:110]}"
if not _r2_ok:
    print(f"      R2 DISQUALIFIED: {_r2_why}")
    print("      It is excluded from the comparison rather than used with a caveat.")

# ==============================================================================
# R3   consensus across the top admissible configurations
# ==============================================================================
print("\n" + "-" * 78)
print("  R3  consensus across the top admissible configurations from Stage 21H")
_r3_ok, _r3_why, _r3_used = False, "", []
try:
    import hdbscan as _hdb
    G  = pload("pooled_hdbscan_screen")
    CH = pload("pooled_hdbscan_choice")
    U  = np.asarray(pload("pooled_umap_nd"), dtype=np.float32)
    if G is None or CH is None or len(U) != N:
        _r3_why = "pooled_hdbscan_screen / choice / umap_nd missing or misaligned"
    else:
        cand = G[(G.admissible) & (G.n_components == CH["n_components"])
                 & (G.n_neighbors == CH["n_neighbors"])] \
                .sort_values("dbcv", ascending=False).head(CONSENSUS_TOPN)
        if len(cand) < CONSENSUS_MIN_CFG:
            _r3_why = (f"only {len(cand)} admissible configuration(s) share the chosen "
                       f"UMAP settings; need at least {CONSENSUS_MIN_CFG}")
        else:
            VOTE  = np.zeros((N, K), dtype=np.float32)
            PLACED = np.zeros(N, dtype=np.int32)
            for r in cand.itertuples():
                lb = _hdb.HDBSCAN(min_cluster_size=int(r.min_cluster_size),
                                  min_samples=(None if r.min_samples < 0
                                               else int(r.min_samples)),
                                  cluster_selection_method=str(r.method),
                                  cluster_selection_epsilon=float(r.epsilon),
                                  metric="euclidean", gen_min_span_tree=False,
                                  core_dist_n_jobs=1).fit(U).labels_
                kk = int(lb.max()) + 1
                if kk < 1:
                    continue
                # cluster g -> how many of its members carry each reference theme
                CNT = np.zeros((kk, K), dtype=np.float32)
                for g in range(kk):
                    mm = (lb == g) & CORE
                    if mm.any():
                        CNT[g] = np.bincount(SUB[mm], minlength=K)
                size = np.array([(lb == g).sum() for g in range(kk)])
                for g in range(kk):
                    if size[g] < MIN_CLUSTER_MEMBERS or CNT[g].sum() <= 0:
                        continue
                    idx = np.where(lb == g)[0]
                    for i in idx:
                        c = CNT[g].copy()
                        if SUB[i] >= 0:          # LEAVE-ONE-OUT: no row votes for itself
                            c[SUB[i]] -= 1
                        if c.sum() <= 0:
                            continue
                        VOTE[i, int(c.argmax())] += 1.0
                        PLACED[i] += 1
                _r3_used.append(f"mcs={int(r.min_cluster_size)} ms={int(r.min_samples)} "
                                f"{r.method} eps={r.epsilon} k={int(r.n_clusters)}")
            R3_best = VOTE.argmax(1)
            with np.errstate(invalid="ignore", divide="ignore"):
                R3_share = np.where(PLACED > 0, VOTE.max(1) / np.maximum(PLACED, 1), 0.0)
            R3_enough = PLACED >= CONSENSUS_MIN_CFG
            RULES["R3 grid-consensus"] = dict(
                best=R3_best,
                grid=[dict(vote=v) for v in GRID_VOTE],
                ok=lambda p: R3_enough & (R3_share >= p["vote"]),
                conf=R3_share,
                desc=lambda p: f"agreement>={p['vote']:.0%} of >={CONSENSUS_MIN_CFG} configs")
            _r3_ok = True
            print(f"      {len(_r3_used)} configurations refit on the chosen embedding:")
            for u in _r3_used:
                print(f"        {u}")
            print(f"      noise rows placed by at least {CONSENSUS_MIN_CFG} of them: "
                  f"{int((R3_enough & NOISE).sum())} of {int(NOISE.sum())}")
except Exception as e:
    _r3_why = f"{type(e).__name__}: {str(e)[:110]}"
if not _r3_ok:
    print(f"      R3 DISQUALIFIED: {_r3_why}")

# ==============================================================================
# CALIBRATION - every rule, same rows, same protocol
# ==============================================================================
# the calibration set: the most peripheral core rows, the ones that look most
# like the rows being recovered. Peripherality is max cosine to any centroid,
# which is rule-independent, so all three rules are scored on the SAME rows.
_periph = np.full(N, np.inf, dtype=float)
_periph[CORE] = R1_sim[CORE]
_cut = float(np.quantile(R1_sim[CORE], CALIB_HARD_FRAC)) if CORE.any() else 0.0
HARD = CORE & (R1_sim <= _cut)
if int(HARD.sum()) < 50:                     # never calibrate on a handful of rows
    HARD = CORE.copy()
    print("\n  (too few peripheral core rows; calibrating on all core rows and "
          "treating the result as optimistic)")

print("\n" + "=" * 78)
print("  CALIBRATION on the core themed rows, whose theme is known")
print(f"  A rule must reach accuracy >= {TARGET_ACC:.0%} on the HARD subset; among the")
print("  thresholds that do, the one covering the most rows is kept. Then the rules")
print("  are compared.")
print(f"  hard subset = the {CALIB_HARD_FRAC:.0%} most peripheral core rows "
      f"({int(HARD.sum())} of {int(CORE.sum())}),")
print(f"  max-centroid-cosine <= {_cut:.3f}. These are the core rows that most")
print("  resemble the noise rows; scoring on ALL core rows measures easy cases and")
print("  lets a threshold that places everything clear the bar. Both are printed.")
print("=" * 78)

CAL, PICK = [], {}
for nm, R in RULES.items():
    rows = []
    for p in R["grid"]:
        ok = np.asarray(R["ok"](p))
        mh = ok & HARD
        ma = ok & CORE
        nh, na = int(mh.sum()), int(ma.sum())
        acc_hard = float((R["best"][mh] == SUB[mh]).mean()) if nh else np.nan
        acc_all  = float((R["best"][ma] == SUB[ma]).mean()) if na else np.nan
        rows.append(dict(rule=nm, desc=R["desc"](p), **p,
                         n_hard_scored=nh, accuracy=acc_hard,
                         n_core_scored=na, accuracy_all_core=acc_all,
                         coverage_core=na / max(int(CORE.sum()), 1),
                         n_noise_placed=int((ok & NOISE).sum()),
                         n_subst_noise_placed=int((ok & NOISE & SUBST).sum()),
                         coverage_noise=(float((ok & NOISE).mean())
                                         if NOISE.any() else 0.0)))
    for r in rows:
        # the objective: the EXPECTED NUMBER OF CORRECTLY PLACED responses. Raw
        # coverage would always reward the loosest threshold that scrapes past the
        # accuracy floor; this does not, because every extra row is discounted by
        # the precision it costs. Subject to the floor, it is the quantity the
        # study actually wants to maximise.
        r["expected_correct"] = (r["n_noise_placed"] * r["accuracy"]
                                 if r["accuracy"] == r["accuracy"] else np.nan)
    D = pd.DataFrame(rows)
    CAL.append(D)
    good = D[(D.accuracy >= TARGET_ACC) & (D.n_hard_scored >= 50)]
    print(f"\n  {nm}")
    if not len(good):
        print(f"    NO threshold reaches {TARGET_ACC:.0%} on the hard subset "
              f"(best {D.accuracy.max():.1%}; on all core rows it would look like "
              f"{D.accuracy_all_core.max():.1%}). This rule places nothing.")
        continue
    b = good.sort_values(["expected_correct", "accuracy"], ascending=False).iloc[0]
    PICK[nm] = b
    for r in D.sort_values("expected_correct", ascending=False).head(6).itertuples():
        mark = "  <-" if r.desc == b.desc else ""
        flag = " " if (r.accuracy == r.accuracy and r.accuracy >= TARGET_ACC) else "x"
        print(f"    {flag} {r.desc:<34} hard={r.accuracy:6.1%} "
              f"(all core {r.accuracy_all_core:5.1%})  places {r.n_noise_placed:>4}"
              f"  ~{r.expected_correct:5.0f} correct{mark}")
    print(f"    kept: {b.desc}  ->  hard accuracy {b.accuracy:.1%}, "
          f"{int(b.n_noise_placed)} noise rows placed "
          f"({int(b.n_subst_noise_placed)} of them substantive), "
          f"~{b.expected_correct:.0f} of them correctly")

CALDF = pd.concat(CAL, ignore_index=True) if CAL else pd.DataFrame()
psave("pooled_membership_calibration", CALDF)

if not PICK:
    raise RuntimeError(
        f"[Stage 23d] no rule reaches {TARGET_ACC:.0%} accuracy on the core rows. "
        f"Nothing is recovered and nothing is guessed. Read "
        f"pooled_membership_calibration, and if the target is genuinely too strict "
        f"for this corpus, lower it EXPLICITLY and say so in the paper.")

WIN = max(PICK.items(), key=lambda kv: (float(kv[1].expected_correct),
                                        float(kv[1].accuracy)))
WNAME, WP = WIN
print("\n" + "-" * 78)
print(f"  WINNER: {WNAME}   ({WP.desc})")
for nm, b in sorted(PICK.items(), key=lambda kv: -float(kv[1].expected_correct)):
    print(f"    {nm:<22} hard acc={b.accuracy:6.1%} (all core {b.accuracy_all_core:5.1%})"
          f"  places {int(b.n_noise_placed):>4}  ~{b.expected_correct:.0f} correct"
          + ("   <- used" if nm == WNAME else "   (not used)"))
print("  The losing rules are printed above with their measured numbers, so this")
print("  choice can be checked and, if a reader disagrees, overturned.")
print(f"\n  HONEST BOUND: {WP.accuracy:.1%} is measured on the {int(HARD.sum())} most")
print("  peripheral core rows - the ones that most resemble the rows being recovered,")
print(f"  not the easy ones (on all core rows the same threshold scores "
      f"{WP.accuracy_all_core:.1%}).")
print("  It is still an UPPER BOUND: every calibration row has a theme, and the rows")
print("  being recovered are exactly the rows that did not. Report it as a bound.")

# ==============================================================================
# APPLY - and build the one canonical membership table
# ==============================================================================
WR   = RULES[WNAME]
_pd  = {k: WP[k] for k in WR["grid"][0].keys()}
OKW  = np.asarray(WR["ok"](_pd))
ASG  = OKW & NOISE                      # the rows actually recovered

asg_sub  = np.where(ASG, WR["best"], -1).astype(int)
asg_main = np.array([S2M.get(int(s), -1) if s >= 0 else -1 for s in asg_sub], dtype=int)
conf     = np.asarray(WR["conf"], dtype=float)

tier = np.where(CORE, "core", np.where(ASG, "assigned", "unassigned")).astype(object)

# why each unassigned row is unassigned - never blank, so no reader has to guess
reason = np.full(N, "", dtype=object)
_un = tier == "unassigned"
reason[_un & ~SUBST] = "thin"
reason[_un & SUBST]  = "below_threshold"
# a substantive row that no configuration in R3 placed at all is not "below a
# threshold" - nothing in the corpus resembles it. Distinguish the two honestly.
if _r3_ok:
    reason[_un & SUBST & (PLACED == 0)] = "rare"

sub_wide  = np.where(CORE, SUB,  asg_sub)
main_wide = np.where(CORE, MAIN, asg_main)

M = pd.DataFrame(dict(
    row_id=np.arange(N),
    Participant=meta["Participant"].astype(str).values,
    Team=meta["Team"].astype(str).str.strip().values,
    question=meta["question"].values,
    n_words=NW.astype(int), is_substantive=SUBST,
    sub_id=SUB, main_id=MAIN,
    tier=tier,
    assigned_sub=asg_sub, assigned_main=asg_main,
    confidence=np.where(CORE, np.nan, conf),
    rule=np.where(ASG, WNAME, ""),
    unassigned_reason=reason,
    sub_wide=sub_wide, main_wide=main_wide,
    source="pooled",
))

# ---- held-out rows, if Notebook 1 ever reintroduces them ----------------------
E_held, meta_h = pload("pooled_held_embeddings"), pload("pooled_held_meta")
if E_held is not None and meta_h is not None and len(np.asarray(E_held)):
    E_held = np.asarray(E_held, dtype=np.float32)
    meta_h = meta_h.reset_index(drop=True)
    hb, hs, hm = _top2(unit(E_held) @ CENT.T)
    # held-out rows are not in the UMAP fit, so only the centroid rule can reach
    # them; the threshold is R1's calibrated one whether or not R1 won overall
    _p1 = PICK.get("R1 centroid-cosine")
    if _p1 is None:
        print("\n  held-out rows found, but R1 never reached the accuracy target, so "
              "they are recorded as unassigned rather than placed by an uncalibrated rule.")
        hok = np.zeros(len(E_held), dtype=bool)
    else:
        hok = (hs >= float(_p1["cos"])) & (hm >= float(_p1["margin"]))
    H = pd.DataFrame(dict(
        row_id=np.arange(N, N + len(E_held)),
        Participant=meta_h["Participant"].astype(str).values,
        Team=meta_h["Team"].astype(str).str.strip().values,
        question=meta_h["question"].values,
        n_words=(meta_h["n_words"].values if "n_words" in meta_h.columns else np.nan),
        is_substantive=False,
        sub_id=-1, main_id=-1,
        tier=np.where(hok, "assigned", "unassigned"),
        assigned_sub=np.where(hok, hb, -1),
        assigned_main=[S2M.get(int(c), -1) if o else -1 for c, o in zip(hb, hok)],
        confidence=hm, rule=np.where(hok, "R1 centroid-cosine (held-out)", ""),
        unassigned_reason=np.where(hok, "", "below_threshold"),
        sub_wide=np.where(hok, hb, -1),
        main_wide=[S2M.get(int(c), -1) if o else -1 for c, o in zip(hb, hok)],
        source="held-out"))
    M = pd.concat([M, H], ignore_index=True)
    print(f"\n  {len(H)} held-out row(s) also processed "
          f"({int(hok.sum())} placed) and marked source='held-out'.")

psave("pooled_membership", M)
M.to_csv(f"{POOLED_DIR}/membership.csv", index=False)
# backward compatibility: the old artifact name, restricted to the rows this
# stage placed, so nothing that reads pooled_projection breaks
_P = M[M.tier == "assigned"][["Participant", "Team", "question", "assigned_sub",
                              "confidence", "n_words", "source"]].copy()
_P["level"], _P["is_core"] = "sub", False
psave("pooled_projection", _P)
psave("pooled_projection_validation",
      {WNAME: dict(accuracy=float(WP.accuracy), threshold=str(WP.desc),
                   n_core_scored=int(WP.n_core_scored),
                   note="measured on core rows; an optimistic upper bound")})

# ------------------------------- report ---------------------------------------
_pool = M[M.source == "pooled"]
print("\n" + "=" * 78)
print("  FINAL MEMBERSHIP")
print("=" * 78)
print(f"  {'tier':<14}{'n':>7}{'% corpus':>11}{'substantive':>14}{'thin':>8}")
print("  " + "-" * 54)
for t in ("core", "assigned", "unassigned"):
    d = _pool[_pool.tier == t]
    print(f"  {t:<14}{len(d):>7}{len(d)/N:>10.1%}"
          f"{int(d.is_substantive.sum()):>14}{int((~d.is_substantive).sum()):>8}")
print("  " + "-" * 54)
print(f"  {'TOTAL':<14}{len(_pool):>7}{1.0:>10.1%}"
      f"{int(_pool.is_substantive.sum()):>14}{int((~_pool.is_substantive).sum()):>8}")

_sn = _pool[(_pool.tier != "core") & _pool.is_substantive]
_sr = _pool[(_pool.tier == "assigned") & _pool.is_substantive]
print(f"\n  SUBSTANTIVE responses HDBSCAN left out : {int((NOISE & SUBST).sum())}")
print(f"    recovered here                       : {len(_sr)}"
      f"   ({len(_sr)/max(int((NOISE & SUBST).sum()),1):.0%} of them)")
print(f"    still unassigned                     : {len(_sn) - len(_sr)}")
print("\n  why the remaining unassigned rows are unassigned:")
for r, lbl in (("thin", "too short to carry a theme - correctly left out"),
               ("rare", "no admissible configuration placed them; nothing in the "
                        "corpus resembles them"),
               ("below_threshold", "placeable in principle, but not confidently "
                                   "enough to claim")):
    n = int((_pool.unassigned_reason == r).sum())
    if n:
        _nsw = int(((_pool.unassigned_reason == r) & _pool.is_substantive).sum())
        print(f"    {r:<18}{n:>5}  ({_nsw} substantive)   {lbl}")
_res = _pool[(_pool.tier == "unassigned") & (_pool.n_words >= 30)]
print(f"\n  {len(_res)} unassigned response(s) are 30+ words. Those are the "
      f"multi-topic ones;")
print("  Stage 23M splits them into clauses and gives each response every theme its")
print("  clauses match. Nothing else is done to the rest, on purpose.")
print("\n  sub_wide / main_wide are the columns downstream figures should use: the")
print("  HDBSCAN theme where there is one, the recovered theme where there is one,")
print("  and -1 where this study genuinely does not know.")

# ------------------------------- figure ---------------------------------------
try:
    fig, ax = plt.subplots(1, 3, figsize=(14.0, 4.2))

    # A - the calibration curves, all rules together
    a = ax[0]
    _cols = {"R1 centroid-cosine": "#44AA99", "R2 soft-membership": "#6699CC",
             "R3 grid-consensus": "#D9A44E"}
    for nm in RULES:
        d = CALDF[CALDF.rule == nm].sort_values("n_noise_placed")
        a.plot(d.n_noise_placed, 100 * d.accuracy, "o-", ms=3.5, lw=1.2,
               color=_cols.get(nm, "#999"), label=nm)
    a.axhline(100 * TARGET_ACC, color="#14202B", ls="--", lw=1.0)
    a.text(a.get_xlim()[0], 100 * TARGET_ACC + 0.6, f" required {TARGET_ACC:.0%}",
           fontsize=7.5, va="bottom", color="#14202B")
    a.set_xlabel("noise rows the threshold would place", fontsize=8.5)
    a.set_ylabel("accuracy on core rows (%)", fontsize=8.5)
    a.legend(fontsize=7.2, frameon=False, loc="lower left")
    a.set_title("A  Three rules, one calibration", fontsize=10, fontweight="bold", loc="left")

    # B - confidence: core vs recovered
    b = ax[1]
    _c = conf[CORE]; _c = _c[np.isfinite(_c)]
    _r = conf[ASG];  _r = _r[np.isfinite(_r)]
    if len(_c) and len(_r):
        bins = np.linspace(0, float(max(np.percentile(_c, 99), _r.max(), 1e-3)), 40)
        b.hist(_c, bins=bins, color="#44AA99", alpha=.75,
               label=f"core (median {np.median(_c):.3f})")
        b.hist(_r, bins=bins, color="#CC6677", alpha=.75,
               label=f"recovered (median {np.median(_r):.3f})")
        b.legend(fontsize=7.5, frameon=False)
    b.set_xlabel(f"confidence under {WNAME}", fontsize=8.5)
    b.set_ylabel("responses", fontsize=8.5)
    b.set_title("B  Recovered rows are weaker, and it shows",
                fontsize=10, fontweight="bold", loc="left")

    # C - where the corpus ended up, split by substantive vs thin
    c = ax[2]
    tiers = ["core", "assigned", "unassigned"]
    sv = [int(((_pool.tier == t) & _pool.is_substantive).sum()) for t in tiers]
    tv = [int(((_pool.tier == t) & ~_pool.is_substantive).sum()) for t in tiers]
    xs = np.arange(3)
    c.bar(xs, sv, width=.55, color="#44AA99", edgecolor="white", label=f">={SUBW} words")
    c.bar(xs, tv, width=.55, bottom=sv, color="#C9CDD2", edgecolor="white",
          label=f"<{SUBW} words")
    for i, (s, t) in enumerate(zip(sv, tv)):
        c.text(i, s + t + N * .012, f"{s+t}", ha="center", fontsize=9, fontweight="bold")
        if s:
            c.text(i, s / 2, f"{s}", ha="center", va="center", fontsize=8, color="white")
    c.set_xticks(xs); c.set_xticklabels(tiers, fontsize=9)
    c.set_ylabel("responses", fontsize=8.5)
    c.set_ylim(0, max(np.array(sv) + np.array(tv)) * 1.16)
    c.legend(fontsize=7.5, frameon=False)
    c.set_title("C  Where every response ended up", fontsize=10, fontweight="bold", loc="left")

    for _a in ax:
        _a.set_facecolor("white")
        for sp in ("top", "right"):
            _a.spines[sp].set_visible(False)
        _a.tick_params(length=2, labelsize=8)
    fig.suptitle("Final membership: who belongs to which theme, and on what evidence",
                 x=.5, ha="center", fontsize=12, fontweight="bold")
    fig.text(.5, -.035,
             f"Rule chosen by measurement, not assumption: {WNAME} at {WP.desc}, "
             f"{WP.accuracy:.0%} accurate on core rows (an optimistic upper bound). "
             f"Unassigned rows are reported, never forced into a theme.",
             ha="center", fontsize=7.6, style="italic", color="#888")
    plt.tight_layout(rect=[0, .02, 1, .92])
    (save_fig(fig, "chart_final_membership") if "save_fig" in globals()
     else fig.savefig(f"{POOLED_DIR}/chart_final_membership.png", dpi=300,
                      bbox_inches="tight"))
    plt.close(fig)
except Exception as e:
    print(f"  (figure skipped: {type(e).__name__}: {str(e)[:90]})")

print("\n  -> pooled_membership · membership.csv · pooled_membership_calibration")
print("     · pooled_projection (compatibility) · chart_final_membership")
print("=" * 78)

In [ ]:
# @title
# ==============================================================================
# STAGE 23M   CLAUSE-LEVEL MULTI-LABEL FOR THE LONG UNPLACED RESPONSES
#             (a NEW cell, placed directly after Stage 23d)
#
# WHY THIS IS A SEPARATE CELL FROM STAGE 23d, DELIBERATELY
#   Stage 23d decides membership using vectors that already exist. This stage has
#   to EMBED new text - the clauses - so it loads the sentence encoder. Folding it
#   into 23d would force a model load every time you re-ran the recovery, which is
#   the part you will iterate on. Two cells, run in order, cost nothing and keep
#   the expensive step separable.
# ------------------------------------------------------------------------------
# WHY THIS STAGE EXISTS, AND WHY IT IS DELIBERATELY SMALL
#
#   HDBSCAN leaves 403 responses in no sub-theme. Notebook 3's recovery cell
#   rescues 200 of them with a calibrated centroid rule, leaving 203. Reading
#   those 203 by hand splits them cleanly by length, and the split is the whole
#   justification for this stage:
#
#     >= 30 words   34 responses   ~85% raise SEVERAL distinct subjects
#     10-29 words   89 responses   ~80% raise exactly ONE subject
#     <= 9 words    80 responses   too thin to carry a topic at all
#
#   The long ones are unplaced because a single vector cannot sit inside three
#   dense regions at once. Example, 83 words, verbatim: an advisor who "always
#   cheers us up", judges at the jamboree who "gave many great suggestions", and
#   a team that "communicate with each other often" - three themes in one answer,
#   so its embedding lands between all three and belongs to none.
#
#   The 10-29 band is NOT that. Those are single, specific, RARE experiences -
#   "Late nights in the lab, venting about failed experiments", "teammates from
#   5 secondary schools in Hong Kong", "I am not a Biology major, I specialised
#   in Maths". One subject each; they are unplaced because nothing else in the
#   corpus is near them, which is ordinary density noise. Splitting those would
#   cut one thought in half and post the halves to two different themes.
#
#   So this stage touches ONLY responses at or above MIN_WORDS. It is expected to
#   affect about 2% of the corpus. That is the point: a targeted repair where the
#   diagnosis supports it, not a corpus-wide change of representation.
#
# WHAT IT DOES
#   1. splits each long unplaced response into clauses (sentences, then discourse
#      markers, then the '|' separator this corpus actually contains);
#   2. embeds each clause through THE SAME encoding path NB1 used - identified
#      by test, not assumed, exactly as Stage 30v does;
#   3. assigns a clause to a sub-theme by cosine to that theme's centroid, using
#      thresholds CALIBRATED on clauses whose theme is already known;
#   4. gives the response every theme its clauses confidently matched, so a
#      three-subject answer is recorded as belonging to three themes.
#
# WHAT IT DOES NOT DO
#   It does not change pooled_labels_sub, any centroid, any name, or any existing
#   figure. It writes ONE new artifact. A response it touches keeps its -1 in the
#   primary partition and gains a separate multi-label record, so every existing
#   number stays exactly as it was and the addition is opt-in downstream.
#
# HONESTY NOTE ON THE CALIBRATION
#   Accuracy is measured as "does this clause's nearest centroid equal the theme
#   its PARENT response was placed in". For a genuinely multi-topic parent that
#   is the wrong target by construction - a clause about funding inside a
#   response placed in a teamwork theme SHOULD go to funding. The measured
#   accuracy is therefore a LOWER BOUND, and it is reported as one.
#
# KNOWN LIMITATION OF THE SPLITTER, measured on the real responses
#   It splits on sentence ends, on discourse markers (however / but / also /
#   moreover / in addition / on the other hand ...) and on '|'. It does NOT split
#   a comma-chained list inside one sentence. Verbatim example that stays whole:
#       "it turned textbook biology into hands-on problem-solving, boosting my
#        confidence to lead projects, and clarified I want to pursue grad school"
#   Three subjects, one clause, so that response gets one theme instead of three.
#   This is UNDER-splitting, and it is the safe direction: an under-split clause
#   is assigned exactly as the whole response would have been today, so nothing
#   regresses, whereas an over-split clause would post half a thought to a theme
#   it does not belong to. Measured behaviour on the nine long responses read by
#   hand: 2.8 clauses each, and every boundary matched a subject boundary a human
#   reader had already marked.
#
# Outputs: pooled_clause_multilabel, clause_multilabel.csv
# Refs: Grootendorst (2022) on outlier reduction and its risks; the calibration
#       design is the one Notebook 3's recovery cell already uses.
# ==============================================================================
import re, gc
import numpy as np, pandas as pd

# ------------------------------- EDIT HERE ------------------------------------
MIN_WORDS        = 30    # only responses at least this long are candidates.
                         # Set from the hand-read above: below 30 the residue is
                         # single-topic, so splitting would do harm.
MIN_CLAUSE_WORDS = 5     # a fragment shorter than this is merged into the one
                         # before it; 3 words cannot carry a topic
MAX_CLAUSES      = 8     # guard against a runaway split on a wall of text
TARGET_ACC       = 0.80  # required accuracy of the clause rule, measured below
GRID_COS         = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]
GRID_MARGIN      = [0.00, 0.01, 0.02, 0.03, 0.05]
CALIB_MAX        = 300   # long PLACED responses used to calibrate
# ----------------------------- END EDIT ---------------------------------------

LV = "sub"

# Stage 23d has already recovered everything a whole-response rule can reach.
# This stage only ever looks at what it could NOT place, so the two stages can
# never fight over the same row.
MEMB = pload("pooled_membership")
if MEMB is None:
    raise RuntimeError("[Stage 23M] pooled_membership is missing - run Stage 23d first.")

meta = require("pooled_meta", stage="Stage 23M").reset_index(drop=True)
X    = np.asarray(require("pooled_embeddings", stage="Stage 23M"), dtype=np.float32)
lab  = np.asarray(pload("pooled_labels_sub"))
NAMES = pload("pooled_sub_name") or {}
# sub -> main, so a multi-labelled row carries its main-theme too (MAIN is Ward on
# the SUB centroids, so this is a lookup, not a second decision)
_hm = pload("pooled_hier_maps") or {}
S2M = {int(k): int(v) for k, v in
       dict(_hm.get("sub_to_main") or pload("pooled_sub_of_macro") or {}).items()}
N = len(meta)
if not (len(X) == len(lab) == N):
    raise RuntimeError(f"[Stage 23M] misaligned: meta {N}, embeddings {len(X)}, "
                       f"labels {len(lab)}.")
K = int(lab.max()) + 1

# ---- raw text, exactly as every other stage resolves it ----------------------
_df = df if "df" in globals() and df is not None else get_df()
_RAW_OF_Q = dict(zip(CFG.short_labels, CFG.text_cols))
_pid = {}
for _, r in _df.iterrows():
    _pid.setdefault(r["Participant"], r)


# ==============================================================================
# WHICH TEXT DID NOTEBOOK 1 ACTUALLY EMBED?
# ------------------------------------------------------------------------------
# This cost a failed run, so it is written down. An earlier version resolved the
# response text through CFG.text_cols - the RAW survey column - and could only
# reproduce pooled_embeddings at cosine 0.9743 with 89% same-theme retrieval, so
# the guard stopped the run. The guard was right and the text was wrong.
#
# Stage 30v (cell 41) does the same identification and its stored output reads:
#
#     winner 'prefix-join + collapsed-text' on 200 real responses:
#     mean cosine 1.0000, retrieves the same theme 100.0% of the time
#
# Exact reproduction. The difference is one line: Stage 30v reads
#
#     v = r.get(f"bert_{q}", "")
#
# the PREPROCESSED column NB1 embedded, not the raw survey column. Same model,
# same dtype, same batch size - a different string.
#
# Both sources are still offered as candidates below rather than hard-coding the
# winner, because the point of this block is to identify the path by test. The
# difference now is that the correct candidate is in the list.
# ==============================================================================
def _text_of(mr):
    """Return (bert-collapsed, bert-raw, col-collapsed, col-raw) for one row."""
    r = _pid.get(mr["Participant"])
    q = mr["question"]

    def _clean(v):
        if v is None or (isinstance(v, float) and np.isnan(v)):
            return ""
        v = str(v)
        return "" if v.strip().lower() == "nan" else v

    b = _clean(r.get(f"bert_{q}", "")) if r is not None else ""
    col = _RAW_OF_Q.get(q)
    c = _clean(r.get(col, "")) if (r is not None and col is not None) else ""
    return " ".join(b.split()), b, " ".join(c.split()), c


_pairs = [_text_of(m) for _, m in meta.iterrows()]
texts_bert     = np.array([p[0] for p in _pairs], dtype=object)   # NB1's own column
texts_bert_raw = np.array([p[1] for p in _pairs], dtype=object)
texts          = np.array([p[2] for p in _pairs], dtype=object)   # raw survey column
texts_raw      = np.array([p[3] for p in _pairs], dtype=object)
_nb = int(sum(1 for t in texts_bert if t))
print(f"  text sources: bert_<question> non-empty on {_nb} of {len(texts_bert)} rows; "
      f"raw survey column on {int(sum(1 for t in texts if t))}")
if _nb == 0:
    print("    !! no bert_<question> column found on the response frame. Stage 30v "
          "reads that column and reproduced pooled_embeddings exactly from it, so "
          "its absence here means the frame is different - expect the path test "
          "below to fail, and fix the source rather than the threshold.")
# word counts come from Stage 21H so the whole notebook uses ONE definition of
# how long a response is; only fall back to counting here if that is unavailable.
_SB = pload("pooled_substantive")
if _SB is not None and len(_SB) == N:
    nwords = np.asarray(_SB["n_words"].values, dtype=int)
    print("  word counts from pooled_substantive (Stage 21H) - one definition "
          "notebook-wide")
else:
    nwords = np.array([len(t.split()) for t in texts])
    print("  word counts computed here (pooled_substantive unavailable)")

print("=" * 78)
print("  STAGE 23M   CLAUSE-LEVEL MULTI-LABEL FOR LONG UNPLACED RESPONSES")
print("=" * 78)
_unplaced = lab < 0
print(f"  {N} responses · {int(_unplaced.sum())} unplaced ({_unplaced.mean():.1%})")
for lo, hi, nm in ((0, 9, "<= 9"), (10, 29, "10-29"), (30, 10**6, ">= 30")):
    m = _unplaced & (nwords >= lo) & (nwords <= hi)
    print(f"    unplaced, {nm:>6} words : {int(m.sum())}")
# the candidate set is Stage 23d's residue, NOT the raw HDBSCAN noise: a row the
# recovery already placed must not be re-decided here.
_MB = MEMB[MEMB["source"] == "pooled"].sort_values("row_id").reset_index(drop=True)
if len(_MB) != N or not (_MB["row_id"].values == np.arange(N)).all():
    raise RuntimeError(f"[Stage 23M] pooled_membership has {len(_MB)} pooled rows but "
                       f"the pooled set has {N} - re-run Stage 23d.")
TIER = _MB["tier"].astype(str).values
_still = TIER == "unassigned"
print(f"  Stage 23d already placed {int((TIER == 'assigned').sum())} of them; "
      f"{int(_still.sum())} remain unassigned")
CAND = np.where(_still & (nwords >= MIN_WORDS))[0]
print(f"  candidates (STILL unassigned after Stage 23d, and >= {MIN_WORDS} words): "
      f"{len(CAND)}")
if not len(CAND):
    print("  nothing to do - no unassigned response is long enough to be worth")
    print("  splitting. pooled_membership is left exactly as Stage 23d wrote it,")
    print("  with an empty multi_subs column so Stage 33b finds what it expects.")
    if "multi_subs" not in MEMB.columns:
        MEMB = MEMB.copy()
        MEMB["multi_subs"] = ""
        psave("pooled_membership", MEMB)
    psave("pooled_clause_multilabel", pd.DataFrame())
    raise SystemExit

# ------------------------------------------------------------------ splitter
_MARK = (r"(?i)(?<=[\s,;])(?=(?:however|but|although|though|whereas|meanwhile|"
         r"moreover|furthermore|besides|additionally|in addition|"
         r"on the other hand|also|and also)\b)")


def split_clauses(t, min_w=MIN_CLAUSE_WORDS, cap=MAX_CLAUSES):
    """Sentences first, then discourse markers, then the corpus's own '|'.

    The pipe matters: several responses in this survey are two answers joined
    with ' | ', sometimes a translation of each other and sometimes two separate
    points. Splitting on it is a property of THIS corpus, observed in the data.
    """
    t = " ".join(str(t).split())
    parts = [p for p in re.split(r"\s*\|\s*", t) if p.strip()]
    out = []
    for p in parts:
        for sent in re.split(r"(?<=[.!?])\s+", p):
            sent = sent.strip()
            if not sent:
                continue
            if len(sent.split()) >= 2 * min_w:
                out.extend([c.strip() for c in re.split(_MARK, sent) if c.strip()])
            else:
                out.append(sent)
    merged = []
    for c in out:
        if merged and len(c.split()) < min_w:
            merged[-1] = merged[-1] + " " + c
        else:
            merged.append(c)
    if merged and len(merged[0].split()) < min_w and len(merged) > 1:
        merged[1] = merged[0] + " " + merged[1]
        merged = merged[1:]
    return merged[:cap]


# ------------------------------------------------- the encoding path, verified
def _unit(A):
    A = np.asarray(A, dtype=np.float32)
    return A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-12)


INSTR = globals().get("EMBED_INSTRUCTION", "") or ""
EMB_MODEL = globals().get("EMBED_MODEL", getattr(CFG, "embed_model", None))
_FMTS = [("prefix-join",    lambda t: f"{INSTR}{t}"),
         ("prefix-space",   lambda t: f"{INSTR} {t}"),
         ("prefix-newline", lambda t: f"{INSTR}\n{t}"),
         ("qwen3-official", lambda t: f"Instruct: {INSTR}\nQuery: {t}"),
         ("no-instruction", lambda t: t)]
# NB1 may have embedded the raw string or the whitespace-collapsed one. Stage 30v
# tries both and so does this stage - a stage that tried only one would report a
# worse match and blame the model for a text-handling difference.
_TEXTSRC = {"bert-collapsed": texts_bert, "bert-raw": texts_bert_raw,
            "col-collapsed": texts,       "col-raw": texts_raw}
# bert-collapsed first: Stage 30v reproduced pooled_embeddings EXACTLY from it on
# this corpus. It is still only a candidate - the screen below decides.
CANDIDATES = [(f"{ft} + {tt}", tt, fn)
              for ft, fn in _FMTS
              for tt in ("bert-collapsed", "bert-raw", "col-collapsed", "col-raw")
              if len(_TEXTSRC[tt]) and any(_TEXTSRC[tt])]

_enc_hook = globals().get("VERIFY_ENCODE_FN")
if _enc_hook is not None:
    _encode, _drop = _enc_hook, (lambda: None)
else:
    from sentence_transformers import SentenceTransformer
    import torch
    _dev = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"\n  loading {EMB_MODEL} on {_dev} ...")
    _m = SentenceTransformer(EMB_MODEL, device=_dev,
                             model_kwargs={"torch_dtype": torch.float16} if _dev == "cuda" else {})
    _bs = int(globals().get("EMBED_BATCH", 16))
    _encode = lambda xs: np.asarray(
        _m.encode(list(xs), batch_size=_bs, convert_to_numpy=True,
                  normalize_embeddings=True, show_progress_bar=False), dtype=np.float32)

    def _drop():
        global _m
        try:
            del _m
        except Exception:
            pass
        gc.collect()
        try:
            import torch as _t
            if _t.cuda.is_available():
                _t.cuda.empty_cache()
        except Exception:
            pass

# ==============================================================================
# IDENTIFY NB1's ENCODING PATH, AND TEST THE PROPERTY WE ACTUALLY DEPEND ON
# ------------------------------------------------------------------------------
# An earlier version of this cell demanded mean cosine >= 0.99 against the stored
# vectors and stopped the run at 0.9756. That gate was wrong, and it is worth
# saying why rather than quietly loosening it.
#
# A HIGH COSINE IS NOT THE REQUIREMENT. fp16 inference, batch size, tokenizer
# padding and library versions all move the cosine a little without moving
# anything that matters. The property this stage actually depends on is narrower
# and directly testable: a re-encoded response must land on THE SAME THEME its
# stored vector lands on. If that holds, a clause vector is comparable to the
# centroids and the assignments below mean what they say; if it does not, no
# cosine threshold would have saved it.
#
# So the test is the one Stage 30v already established and passed on this very
# corpus, at the same two thresholds, applied here unchanged:
#     mean cosine        >= 0.95
#     same-theme retrieval >= 0.98      <- the one that matters
# Screening is coarse-then-fine, exactly as Stage 30v does it, so ten candidate
# paths cost roughly one path's worth of GPU time.
# ==============================================================================
MIN_PATH_COS  = 0.95
MIN_PATH_SAME = 0.98
Xn = _unit(X)
CEN = _unit(np.stack([Xn[lab == c].mean(0) for c in range(K)]))

_rs = np.random.RandomState(int(globals().get("SEED", 42)))
_pool = np.where((lab >= 0) & (nwords >= 5))[0]
if len(_pool) < 30:
    _drop()
    raise RuntimeError("[Stage 23M] too few placed responses to identify the "
                       "encoding path.")
_coarse = _rs.choice(_pool, size=int(min(64, len(_pool))), replace=False)
print("  identifying NB1's encoding path (not assuming it):")
_scored = []
for tag, tt, fn in CANDIDATES:
    V = _unit(_encode([fn(_TEXTSRC[tt][i]) for i in _coarse]))
    _scored.append((float(np.mean(np.sum(V * Xn[_coarse], axis=1))), tag, tt, fn))
_scored.sort(reverse=True, key=lambda z: z[0])
print("    closest paths on 64 responses (mean cosine with the stored vectors):")
for c, tag, _, _ in _scored[:4]:
    print(f"      {c:.4f}  {tag}")
_c0, ENC_NAME, _tt, ENC_FN = _scored[0]

_fine = _rs.choice(_pool, size=int(min(200, len(_pool))), replace=False)
_V = _unit(_encode([ENC_FN(_TEXTSRC[_tt][i]) for i in _fine]))
_cos  = float(np.mean(np.sum(_V * Xn[_fine], axis=1)))
_same = float(np.mean(np.argmax(_V @ CEN.T, axis=1) ==
                      np.argmax(Xn[_fine] @ CEN.T, axis=1)))
print(f"    winner '{ENC_NAME}' on {len(_fine)} real responses:")
print(f"      mean cosine with the stored vector        {_cos:.4f}   "
      f"(needs >= {MIN_PATH_COS})")
print(f"      retrieves the SAME theme as that vector   {_same*100:.1f}%   "
      f"(needs >= {MIN_PATH_SAME*100:.0f}%)  <- the property this stage depends on")
if _cos < MIN_PATH_COS or _same < MIN_PATH_SAME:
    _drop()
    raise RuntimeError(
        f"[Stage 23M] cannot reproduce pooled_embeddings well enough to trust "
        f"clause geometry (best '{ENC_NAME}': cosine {_cos:.4f}, same-theme "
        f"retrieval {_same*100:.1f}%). Clause vectors would live in a different "
        f"space than the centroids, so every assignment below would be "
        f"meaningless. This is the same gate Stage 30v applies, and Stage 30v "
        f"reached cosine 1.0000 / 100% on this corpus using the bert_<question> "
        f"column, so a failure here means the text source is still wrong - check "
        f"that the response frame really carries bert_<question>. Fix the source, "
        f"not the threshold.")
ENC_TEXTS = _TEXTSRC[_tt]
print(f"  path confirmed: '{ENC_NAME}' - clauses will be encoded exactly the way "
      f"the responses were.")
psave("pooled_clause_encoding_path",
      dict(path=ENC_NAME, text_source=_tt, mean_cosine=_cos, same_theme=_same,
           n_probe=int(len(_fine))))


def assign(vecs):
    """(best theme, best cosine, margin over the runner-up) for each row."""
    Sm = _unit(vecs) @ CEN.T
    order = np.argsort(-Sm, axis=1)
    best = order[:, 0]
    top = Sm[np.arange(len(Sm)), order[:, 0]]
    second = Sm[np.arange(len(Sm)), order[:, 1]] if K > 1 else np.zeros(len(Sm))
    return best, top, top - second


# ------------------------------------------------- calibrate on KNOWN clauses
# Long responses that HDBSCAN did place: their theme is known, so a clause drawn
# from one gives a labelled example. See the honesty note in the header - for a
# multi-topic parent this target is wrong by construction, so the accuracy below
# is a lower bound on the rule's real quality.
_placed_long = np.where((lab >= 0) & (nwords >= MIN_WORDS))[0]   # core rows only
rng = np.random.default_rng(int(globals().get("SEED", 42)))
if len(_placed_long) > CALIB_MAX:
    _placed_long = rng.choice(_placed_long, CALIB_MAX, replace=False)
print(f"\n  CALIBRATION on clauses from {len(_placed_long)} long PLACED responses")

_cal_txt, _cal_y = [], []
for i in _placed_long:
    for c in split_clauses(ENC_TEXTS[i]):
        _cal_txt.append(c)
        _cal_y.append(lab[i])
if not _cal_txt:
    _drop()
    raise RuntimeError("[Stage 23M] no calibration clauses could be built.")
_cal_y = np.asarray(_cal_y)
print(f"    {len(_cal_txt)} clauses, {len(_cal_txt)/max(len(_placed_long),1):.1f} per response")
_cal_v = _encode([ENC_FN(c) for c in _cal_txt])
_cb, _ct, _cm = assign(_cal_v)

rows = []
for cmin in GRID_COS:
    for mmin in GRID_MARGIN:
        sel = (_ct >= cmin) & (_cm >= mmin)
        if sel.sum() < 30:
            continue
        rows.append(dict(min_cos=cmin, min_margin=mmin, n=int(sel.sum()),
                         accuracy=float((_cb[sel] == _cal_y[sel]).mean()),
                         kept=float(sel.mean())))
CAL = pd.DataFrame(rows)
if not len(CAL):
    _drop()
    raise RuntimeError("[Stage 23M] no threshold retains enough clauses to calibrate.")
_ok = CAL[CAL.accuracy >= TARGET_ACC]
_pick = (_ok.sort_values(["kept", "accuracy"], ascending=[False, False]).iloc[0]
         if len(_ok) else CAL.sort_values("accuracy", ascending=False).iloc[0])
MIN_COS, MIN_MARGIN, ACC = float(_pick.min_cos), float(_pick.min_margin), float(_pick.accuracy)
print(f"    {'cos':>6}{'margin':>8}{'clauses':>9}{'agrees w/ parent':>18}{'kept':>7}")
for r in CAL.sort_values(["kept"], ascending=False).head(8).itertuples():
    mark = "  <- chosen" if (r.min_cos == MIN_COS and r.min_margin == MIN_MARGIN) else ""
    print(f"    {r.min_cos:>6.2f}{r.min_margin:>8.2f}{r.n:>9}{r.accuracy:>17.1%}"
          f"{r.kept:>7.0%}{mark}")
if not len(_ok):
    print(f"    !! no threshold reaches {TARGET_ACC:.0%}; the best is {ACC:.1%} and is "
          f"used. Treat the multi-labels below as provisional.")
print(f"    agreement with the parent theme is a LOWER BOUND: a clause about funding "
      f"inside a\n    response placed in a teamwork theme SHOULD disagree.")

# ------------------------------------------------------------------- apply
print(f"\n  APPLYING to the {len(CAND)} long unplaced responses")
recs, ev = [], []
_all_txt, _own = [], []
for i in CAND:
    cs = split_clauses(ENC_TEXTS[i])          # the text source the path test chose
    for c in cs:
        _all_txt.append(c)
        _own.append(i)
if _all_txt:
    _v = _encode([ENC_FN(c) for c in _all_txt])
    _b, _t, _mg = assign(_v)
else:
    _b = _t = _mg = np.array([])

_by = {}
for j, i in enumerate(_own):
    _by.setdefault(i, []).append(j)

for i in CAND:
    js = _by.get(i, [])
    hits = [(int(_b[j]), float(_t[j]), float(_mg[j]), _all_txt[j])
            for j in js if _t[j] >= MIN_COS and _mg[j] >= MIN_MARGIN]
    themes, strength = [], {}
    for th, cos, mg, ctext in hits:
        strength[th] = max(strength.get(th, 0.0), cos)
        if th not in themes:
            themes.append(th)
        ev.append(dict(row_id=int(i), theme=th,
                       theme_name=str(NAMES.get(th, f"S{th}")),
                       cosine=round(cos, 4), margin=round(mg, 4), clause=ctext))
    prim = max(strength, key=strength.get) if strength else -1
    recs.append(dict(row_id=int(i),
                     Participant=meta.iloc[i]["Participant"],
                     Team=meta.iloc[i].get("Team", ""),
                     question=meta.iloc[i]["question"],
                     n_words=int(nwords[i]), n_clauses=len(js),
                     n_assigned=len(hits), n_themes=len(themes),
                     themes=";".join(str(t) for t in sorted(themes)),
                     theme_names=" | ".join(str(NAMES.get(t, f"S{t}")) for t in sorted(themes)),
                     primary=prim,
                     primary_name=str(NAMES.get(prim, "")) if prim >= 0 else "",
                     best_cos=round(max(strength.values()), 4) if strength else np.nan,
                     text=texts[i][:400]))

ML = pd.DataFrame(recs)
EV = pd.DataFrame(ev)
_multi = int((ML.n_themes >= 2).sum())
_one = int((ML.n_themes == 1).sum())
_none = int((ML.n_themes == 0).sum())
print(f"    clauses embedded          : {len(_all_txt)}")
print(f"    responses given >= 2 themes: {_multi}")
print(f"    responses given exactly 1  : {_one}")
print(f"    responses still unmatched  : {_none}")
if len(ML):
    print(f"    themes per matched response: mean "
          f"{ML.loc[ML.n_themes > 0, 'n_themes'].mean():.2f}")

print(f"\n  WORKED EXAMPLES (read these before trusting the rest)")
for _, r in ML[ML.n_themes >= 2].nlargest(3, "n_words").iterrows():
    print(f"\n    [{r['question']}] {r['n_words']}w -> {r['n_themes']} themes")
    for _, e in EV[EV.row_id == r["row_id"]].sort_values("cosine", ascending=False).iterrows():
        print(f"      S{e['theme']:<3} cos={e['cosine']:.3f}  {str(e['theme_name'])[:34]:<34} "
              f"| {str(e['clause'])[:70]}")

psave("pooled_clause_multilabel",
      dict(assignments=ML, evidence=EV, calibration=CAL,
           min_words=MIN_WORDS, min_cos=MIN_COS, min_margin=MIN_MARGIN,
           accuracy_lower_bound=ACC, encoding_path=ENC_NAME,
           n_candidates=int(len(CAND)), n_multi=_multi, n_single=_one, n_none=_none))
ML.to_csv(f"{POOLED_DIR}/clause_multilabel.csv", index=False)
EV.to_csv(f"{POOLED_DIR}/clause_multilabel_evidence.csv", index=False)
_drop()

# ==============================================================================
# FOLD THE RESULT BACK INTO pooled_membership
# ------------------------------------------------------------------------------
# Stage 23d wrote the canonical membership table; this stage is the only other
# thing allowed to touch it, and only on rows it left unassigned. A response that
# matched at least one theme becomes tier "assigned-multi": it takes the STRONGEST
# clause's theme as its single-theme identity (so it appears in ordinary figures
# instead of vanishing) and carries the full list in multi_subs.
# ==============================================================================
OUT = MEMB.copy()
if "multi_subs" not in OUT.columns:
    OUT["multi_subs"] = ""
OUT["multi_subs"] = OUT["multi_subs"].astype(object).fillna("")

_pos = {int(r): i for i, r in enumerate(OUT["row_id"].values)
        if str(OUT["source"].values[i]) == "pooled"}
_n_changed = 0
for _, r in ML.iterrows():
    if int(r["n_themes"]) < 1 or int(r["primary"]) < 0:
        continue                       # no clause cleared the threshold - leave it
    i = _pos.get(int(r["row_id"]))
    if i is None:
        continue
    if str(OUT.at[i, "tier"]) != "unassigned":
        raise RuntimeError(
            f"[Stage 23M] row {int(r['row_id'])} is tier '{OUT.at[i, 'tier']}' but this "
            f"stage was about to overwrite it. Stage 23d and Stage 23M must never "
            f"decide the same row. Re-run Stage 23d.")
    prim = int(r["primary"])
    OUT.at[i, "tier"]              = "assigned-multi"
    OUT.at[i, "assigned_sub"]      = prim
    OUT.at[i, "assigned_main"]     = int(S2M.get(prim, -1))
    OUT.at[i, "sub_wide"]          = prim
    OUT.at[i, "main_wide"]         = int(S2M.get(prim, -1))
    OUT.at[i, "confidence"]        = float(r["best_cos"])
    OUT.at[i, "rule"]              = "R4 clause-multilabel"
    OUT.at[i, "unassigned_reason"] = ""
    OUT.at[i, "multi_subs"]        = str(r["themes"])
    _n_changed += 1

# the same invariants Stage 23d guarantees must still hold afterwards
_p = OUT[OUT["source"] == "pooled"]
assert set(_p["tier"]) <= {"core", "assigned", "assigned-multi", "unassigned"}
assert (_p.loc[_p.tier == "core", "sub_id"] >= 0).all(), "a core row lost its theme"
assert (_p.loc[_p.tier != "core", "sub_id"] < 0).all(), "the primary partition moved"
assert (_p.loc[_p.tier == "assigned-multi", "sub_wide"] >= 0).all()
assert (_p.loc[_p.tier == "unassigned", "sub_wide"] < 0).all()
assert (_p.loc[_p.tier == "unassigned", "unassigned_reason"]
        .astype(str).str.strip() != "").all(), "an unassigned row lost its reason"
_bad = _p[(_p.sub_wide >= 0) & (_p.main_wide < 0)]
assert not len(_bad), f"{len(_bad)} row(s) have a sub-theme but no main-theme"

psave("pooled_membership", OUT)
OUT.to_csv(f"{POOLED_DIR}/membership.csv", index=False)

print(f"\n  -> clause_multilabel.csv ({len(ML)} responses) · "
      f"clause_multilabel_evidence.csv ({len(EV)} clauses)")
print(f"  -> pooled_membership updated: {_n_changed} row(s) moved from 'unassigned'")
print(f"     to 'assigned-multi'. {int((_p.tier == 'unassigned').sum())} row(s) remain")
print(f"     unassigned, each with a recorded reason.")
print(f"  pooled_labels_sub is UNCHANGED - the primary partition never moves. A")
print(f"  multi-labelled response keeps sub_id = -1 and carries its themes in")
print(f"  sub_wide (the strongest clause) and multi_subs (all of them).")
print("=" * 78)

## Stage 24 — Embedding-to-map validation

Projection faithfulness: trustworthiness and continuity (Venna and Kaski 2001), k-nearest-neighbour overlap, and the Shepard correlation between embedding and map distances, confirming the 2-D and N-D views preserve real neighbourhoods.

In [ ]:
# @title
# ════════════════════════════════════════════════════════
# STAGE 24:  PROJECTION VALIDATION   (embedding -> clustering space; STATIC)
# INPUT:  pooled_embeddings (X_pool), pooled_umap5d
# OUTPUT: pooled_embed_validation, chart_pooled_embed_validation.png (+ .svg)
# ════════════════════════════════════════════════════════
# Is the embedding -> clustering-space projection faithful? Reported with the
# standard set (not just trustworthiness), as one clean static figure:
#   A  Trustworthiness / Continuity / kNN-overlap across neighbourhood sizes k
#   B  Shepard diagram: high-D vs low-D pairwise distances (global shape)
#   C  Trustworthiness & Continuity as a function of k (are local maps faithful)
# Refs: Venna & Kaski (2001); Lee & Verleysen (2009); Kruskal (1964).

import numpy as np
from sklearn.manifold import trustworthiness
from sklearn.neighbors import NearestNeighbors
from scipy.spatial.distance import pdist
from scipy.stats import spearmanr
import matplotlib.pyplot as plt

try: _ = X_pool
except NameError: X_pool = pload('pooled_embeddings')
try: _ = pooled_umap5d
except NameError: pooled_umap5d = pload('pooled_umap5d')
if X_pool is None or pooled_umap5d is None:
    raise RuntimeError('Run the embedding stage and UMAP first.')

Xh = np.asarray(X_pool, dtype=float); Xl = np.asarray(pooled_umap5d, dtype=float)
n = len(Xh)
def _knn_overlap(Xa, Xb, k):
    na = NearestNeighbors(n_neighbors=k + 1).fit(Xa).kneighbors(return_distance=False)[:, 1:]
    nb = NearestNeighbors(n_neighbors=k + 1).fit(Xb).kneighbors(return_distance=False)[:, 1:]
    return float(np.mean([len(set(a) & set(b)) / k for a, b in zip(na, nb)]))

K_LIST = [k for k in (5, 10, 15, 30, 50) if k < n]
rng = np.random.default_rng(globals().get('SEED', 42))
sub = rng.choice(n, min(400, n), replace=False)
lo_metric = globals().get('UMAP_QUALITY_METRIC', 'euclidean')
dh = pdist(Xh[sub], metric='cosine'); dl = pdist(Xl[sub], metric=lo_metric)
sp_rho, _ = spearmanr(dh, dl)

rows = []
print('=' * 62); print('  PROJECTION VALIDATION (embedding -> clustering space)'); print('=' * 62)
print(f'  {"k":>4}  {"Trustworth.":>12}  {"Continuity":>11}  {"kNN overlap":>12}')
print('  ' + '-' * 46)
for k in K_LIST:
    tw = float(trustworthiness(Xh, Xl, n_neighbors=k))
    co = float(trustworthiness(Xl, Xh, n_neighbors=k))
    ov = _knn_overlap(Xh, Xl, k)
    rows.append(dict(k=k, trust=tw, cont=co, knn=ov))
    print(f'  {k:>4}  {tw:>12.3f}  {co:>11.3f}  {ov:>12.3f}')
print('  ' + '-' * 46)
print(f'  Shepard / Spearman rho (pairwise distances, n={len(sub)}): {sp_rho:.3f}')
psave('pooled_embed_validation', dict(rows=rows, spearman=float(sp_rho), n=n))

ks = [r['k'] for r in rows]
NB = globals().get('NAT_BG', '#FFFFFF'); NT = globals().get('NAT_TEXT', '#14202B')
# (round 3) 2 panels, not 3: the old grouped-bar panel showed the same three
# metrics as the line panel - twice the ink, zero extra information.
fig, ax = plt.subplots(1, 2, figsize=(14.5, 6.2), facecolor=NB,
                       gridspec_kw=dict(width_ratios=[1.0, 1.0], wspace=0.26))
a = ax[0]
a.axhspan(0.9, 1.0, color='#E9F7EF', zorder=0)
a.plot(ks, [r['trust'] for r in rows], '-o', color='#6699CC', lw=2.2, ms=6, label='Trustworthiness')
a.plot(ks, [r['cont'] for r in rows], '-s', color='#44AA99', lw=2.2, ms=6, label='Continuity')
a.plot(ks, [r['knn'] for r in rows], '-^', color='#D9A44E', lw=2.2, ms=6, label='kNN overlap')
for k_, r in zip(ks, rows):
    a.text(k_, r['knn'] - 0.035, f"{r['knn']:.2f}", ha='center', fontsize=8, color='#9A6B00')
a.set_ylim(min(0.55, min(r['knn'] for r in rows) - 0.08), 1.02)
a.set_xlabel('neighbourhood size k'); a.set_ylabel('score (higher better)')
a.set_title('A  Local faithfulness vs k  (green band = excellent >0.9)',
            fontsize=12, fontweight='bold', loc='left', color=NT)
a.legend(fontsize=9, frameon=False, loc='lower right')
b = ax[1]
b.scatter(dh, dl, s=4, alpha=0.16, color='#332288', edgecolors='none')
_edges = np.linspace(dh.min(), dh.max(), 19); _bx, _by = [], []
for _e in range(len(_edges) - 1):
    _sel = (dh >= _edges[_e]) & (dh < _edges[_e + 1])
    if _sel.sum() > 5:
        _bx.append((_edges[_e] + _edges[_e + 1]) / 2); _by.append(np.median(dl[_sel]))
b.plot(_bx, _by, color='#D9A44E', lw=2.6, zorder=5, label='binned median')
b.legend(fontsize=9, frameon=False, loc='upper left')
b.set_xlabel('high-D distance (cosine)'); b.set_ylabel('clustering-space distance')
b.set_title(f'B  Shepard diagram  (rho = {sp_rho:.3f})', fontsize=12, fontweight='bold', loc='left', color=NT)
for _a in ax:
    _a.set_facecolor('white')
    for sp in ('top', 'right'):
        _a.spines[sp].set_visible(False)
    _a.grid(True, axis='y', color='#F0F0F0'); _a.set_axisbelow(True)
fig.suptitle('Projection validation: embedding -> clustering space', fontsize=14.6, fontweight='bold', color=NT, y=1.0)
fig.text(0.5, -0.03, f'Local neighbours preserved well (A); global distances moderately (B, rho={sp_rho:.2f}) - expected for UMAP; clustering uses LOCAL structure. '
         'Refs: Venna & Kaski (2001); Lee & Verleysen (2009); Kruskal (1964).', ha='center', fontsize=9, style='italic', color='#888')
plt.tight_layout(rect=[0, 0.03, 1, 0.97])
save_fig(fig, 'chart_pooled_embed_validation')
plt.close(fig)
print('  Saved: chart_pooled_embed_validation.png/svg  (advanced static, no overlap)')
print('=' * 62)


## Stage 25 — Cluster validation: stability, per-theme Jaccard, silhouette

Rewritten for HDBSCAN. It used to bootstrap **Ward** across a grid of $K$ and score those refits against the pipeline's labels — measuring the reproducibility of a method this notebook no longer uses, against a partition it did not produce. It also passed the `-1` noise label to `silhouette_score` as if it were a cluster, which drags every reported silhouette down by construction.

Now, at **both** levels: HDBSCAN is refitted on 80% subsamples with the exact parameters Stage 23H used, and each refit is compared to the stored labels on the shared rows (adjusted Rand, scored on the rows the reference calls *themed* — a refit that turns one of those into noise counts against the partition). Per-theme bootstrap **Jaccard** matches themes optimally between reference and refit (Hennig 2007: >0.75 stable, 0.60–0.75 pattern). **Silhouette** is computed on themed rows only (Rousseeuw 1987).

The cell first checks that refitting on the full data reproduces the labels on disk, and stops if it does not — otherwise every number would describe a different partition. The $K$-selection evidence is *not* repeated here; Stage 21H already screened every configuration under a rule fixed in advance.


In [ ]:
# @title
# ==============================================================================
# STAGE 25:  CLUSTER VALIDATION   (stability + per-theme Jaccard + silhouette)
# INPUT:  pooled_umap_nd, pooled_hdbscan_hierarchy, pooled_labels_main7/_sub
# OUTPUT: pooled_stability, chart_pooled_stability
# ------------------------------------------------------------------------------
# WHY THIS CELL WAS REWRITTEN
#   It used to bootstrap WARD over a grid of K and score the Ward refits against
#   the pipeline's labels. Ward is gone (Stage 21H), so that measured the
#   reproducibility of a method this notebook no longer uses and compared it to a
#   partition it did not produce. It also fed the -1 noise label to
#   silhouette_score as if it were a cluster: a "cluster" made of everything that
#   belongs nowhere has no cohesion by construction, and it dragged every reported
#   silhouette down. Both are fixed here.
#
#   The K-selection evidence is NOT repeated here. Stage 21H already screened
#   every configuration and chose one by a rule fixed in advance; re-plotting a
#   stability-vs-K curve would be the same evidence drawn twice.
#
# WHAT IS MEASURED, at BOTH levels (main and sub)
#   A. Bootstrap stability. HDBSCAN is refitted on 80% subsamples with the exact
#      parameters Stage 23H used, and the refit is compared to the stored labels
#      on the shared rows (adjusted Rand). Reported on the rows the reference
#      calls THEMED: if a refit turns one of those rows into noise, that counts
#      against the partition, which is the honest accounting. The rate at which
#      the same rows are called noise again is reported separately.
#   B. Per-theme bootstrap Jaccard, themes matched optimally (Hungarian) between
#      the reference and each refit. Noise is not a candidate on either side.
#   C. Silhouette in the clustering space, THEMED ROWS ONLY, overall and per theme.
# Guides: Hennig (2007) Jaccard >0.75 stable, 0.60-0.75 pattern; Rousseeuw (1987)
#         silhouette >0.5 strong, 0.25-0.5 some structure.
# Refs: Hennig (2007); Hubert & Arabie (1985); Rousseeuw (1987); Campello (2013).
# ==============================================================================
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import adjusted_rand_score, silhouette_samples, silhouette_score
from scipy.optimize import linear_sum_assignment
import hdbscan as _hdb

# ------------------------------- EDIT HERE ------------------------------------
N_BOOT, FRAC = 25, 0.80          # bootstrap refits, and the subsample fraction
# ----------------------------- END EDIT ---------------------------------------

# ---- stale-artifact guard (the same block opens every stage below) ------------
def _labels(n_rows, stage, levels=("main", "sub")):
    _KEY = {"main": "pooled_labels_main7", "sub": "pooled_labels_sub"}
    out = {}
    for lv in levels:
        a = pload(_KEY[lv])
        if a is None:
            raise RuntimeError(f"[{stage}] {_KEY[lv]} is missing - run Stage 23H first.")
        a = np.asarray(a)
        if len(a) != n_rows:
            raise RuntimeError(
                f"[{stage}] {_KEY[lv]} has {len(a)} rows but the pooled set has "
                f"{n_rows}. That artifact belongs to an EARLIER clustering. Re-run "
                f"Stage 23H (then Stage 30) before this stage.")
        out[lv] = a
    return out

def _K(lab):
    lab = np.asarray(lab)
    return int(lab.max()) + 1 if (lab >= 0).any() else 0
# ------------------------------------------------------------------------------

X = pload("pooled_umap_nd")
if X is None:
    X = pload("pooled_umap5d")
if X is None:
    raise RuntimeError("[Stage 25] pooled_umap_nd missing - run Stage 21H first.")
X = np.asarray(X, dtype=np.float32)

# ---- the ORIGINAL embedding space, for the coherence block added below -------
# Stage 27 already reports the honest global figure - "silhouette of the SUB
# labels: original-embedding=0.130 vs UMAP-space=0.600 ... the original-space
# value is the honest one" - but only as ONE number for all 39 themes together.
# Nothing said WHICH themes carry that weakness, and a theme cannot be defended
# or doubted on a corpus-wide average. The per-theme decomposition is computed
# here, beside the per-theme Jaccard, so both live in one table.
_EO = pload("pooled_embeddings")
if _EO is None:
    raise RuntimeError("[Stage 25] pooled_embeddings missing - run Notebook 1.")
_EO = np.asarray(_EO, dtype=np.float32)
if len(_EO) != len(X):
    raise RuntimeError(f"[Stage 25] pooled_embeddings has {len(_EO)} rows but the "
                       f"clustering space has {len(X)} - they are from different "
                       f"runs. Re-run Notebook 1 Stage 18, then Stage 21H.")
_EOn = _EO / (np.linalg.norm(_EO, axis=1, keepdims=True) + 1e-9)
n = len(X)
LAB = _labels(n, "Stage 25")

H = pload("pooled_hdbscan_hierarchy")
if H is None:
    raise RuntimeError("[Stage 25] pooled_hdbscan_hierarchy missing - run Stage 23H first.")
SP = H["sub_params"]
CH = pload("pooled_hdbscan_choice") or {}

# HOW EACH LEVEL IS REBUILT HERE
# SUB is one HDBSCAN fit, so refitting it is just that fit with its recorded
# parameters. MAIN depends on how Stage 23H built it:
#
#   main_method == "ward-on-sub-centroids"  (current)
#       MAIN is size-weighted Ward agglomeration of SUB's centroids to K_main.
#       There is no epsilon that reproduces it, so a bootstrap of MAIN must
#       re-run BOTH steps on the subsample - refit SUB, then re-agglomerate. That
#       is also the honest question to ask: the reader is shown main themes
#       produced by two steps, so both belong inside the resample.
#
#   legacy epsilon route (main_epsilon is a number)
#       MAIN was SUB's fit with cluster_selection_epsilon raised. Kept so an old
#       artifact still validates rather than crashing this cell.
_MAIN_METHOD = str(H.get("main_method") or
                   ("epsilon" if H.get("main_epsilon") is not None else "unknown"))
_K_MAIN = int(H.get("main_k") or _K(LAB["main"]))

_PARAMS = {
    "sub":  dict(min_cluster_size=int(SP["min_cluster_size"]),
                 min_samples=SP["min_samples"], method=str(SP["method"]),
                 epsilon=float(SP["epsilon"])),
}
if _MAIN_METHOD == "epsilon":
    _PARAMS["main"] = dict(
        min_cluster_size=int(CH.get("min_cluster_size", SP["min_cluster_size"])),
        min_samples=CH.get("min_samples", SP["min_samples"]),
        method=str(CH.get("method", SP["method"])),
        epsilon=float(H["main_epsilon"]))
else:
    _PARAMS["main"] = dict(_PARAMS["sub"])      # MAIN starts from the SUB fit
_SAME_FIT = all(_PARAMS["sub"][k] == _PARAMS["main"][k]
                for k in ("min_cluster_size", "min_samples", "method"))


def _hdb_fit(E, p):
    return _hdb.HDBSCAN(min_cluster_size=p["min_cluster_size"],
                        min_samples=p["min_samples"],
                        cluster_selection_method=p["method"],
                        cluster_selection_epsilon=p["epsilon"],
                        metric="euclidean", gen_min_span_tree=False,
                        core_dist_n_jobs=1).fit(E).labels_


def _ward_assign(cents, sizes):
    """Size-weighted Ward on group centroids: d(A,B) = nA*nB/(nA+nB)*||cA-cB||^2.
    Identical to the routine in Stage 23H - duplicated so this cell can be run on
    its own. Returns {k: array mapping each original group to a 0..k-1 label}."""
    k0 = len(cents)
    cur = {i: dict(c=np.array(cents[i], dtype=np.float64), n=float(sizes[i]),
                   members=[i]) for i in range(k0)}
    out = {k0: np.arange(k0)}
    while len(cur) > 1:
        keys = sorted(cur)
        best = bi = bj = None
        for ii in range(len(keys)):
            for jj in range(ii + 1, len(keys)):
                a, b = cur[keys[ii]], cur[keys[jj]]
                d = (a["n"] * b["n"] / (a["n"] + b["n"])) * float(
                    np.sum((a["c"] - b["c"]) ** 2))
                if best is None or d < best:
                    best, bi, bj = d, keys[ii], keys[jj]
        a, b = cur.pop(bi), cur.pop(bj)
        nn = a["n"] + b["n"]
        cur[bi] = dict(c=(a["n"] * a["c"] + b["n"] * b["c"]) / nn, n=nn,
                       members=a["members"] + b["members"])
        lab = np.empty(k0, dtype=int)
        for gi, node in enumerate(sorted(cur)):
            for m in cur[node]["members"]:
                lab[m] = gi
        out[len(cur)] = lab
    return out


def _fit(E, lv):
    """Rebuild a level on the rows of E, the way Stage 23H built it."""
    subl = _hdb_fit(E, _PARAMS["sub"])
    if lv == "sub":
        return subl
    if _MAIN_METHOD == "epsilon":
        return _hdb_fit(E, _PARAMS["main"])
    kk = int(subl.max()) + 1
    if kk < 2:
        return np.full(len(subl), -1, dtype=int)
    szs = np.bincount(subl[subl >= 0], minlength=kk).astype(float)
    cts = np.stack([E[subl == c].mean(0) for c in range(kk)])
    asg = _ward_assign(cts, szs)
    tgt = _K_MAIN if _K_MAIN in asg else min(asg, key=lambda z: abs(z - _K_MAIN))
    out = np.full(len(subl), -1, dtype=int)
    a = asg[tgt]
    for c in range(kk):
        out[subl == c] = a[c]
    return out

NB = globals().get("NAT_BG", "#FFFFFF"); NT = globals().get("NAT_TEXT", "#1E1E1E")
HL = globals().get("HIGHLIGHT_COLOR", "#882255")
rng = np.random.default_rng(globals().get("SEED", 42))

print("=" * 74)
print(f"  STAGE 25  CLUSTER VALIDATION   (HDBSCAN, bootstrap {N_BOOT}x@{int(FRAC*100)}%)")
print("=" * 74)
print(f"  {n} responses")
p = _PARAMS["sub"]
print(f"    sub   mcs={p['min_cluster_size']:<3} ms={p['min_samples']} "
      f"{p['method']:<5} epsilon={p['epsilon']:.3f}   (HDBSCAN)")
if _MAIN_METHOD == "epsilon":
    q = _PARAMS["main"]
    print(f"    main  mcs={q['min_cluster_size']:<3} ms={q['min_samples']} "
          f"{q['method']:<5} epsilon={q['epsilon']:.3f}   (HDBSCAN, legacy epsilon route)")
    if not _SAME_FIT:
        print("  NOTE: the two levels come from DIFFERENT HDBSCAN configurations, so MAIN")
        print("        is not a coarsening of SUB by construction.")
else:
    print(f"    main  size-weighted Ward on the SUB centroids -> K={_K_MAIN}")
    print("  Each bootstrap below re-runs BOTH steps (refit SUB, re-agglomerate), so the")
    print("  MAIN figures describe the stability of the whole procedure, not of one fit.")

# ---- does a refit on the FULL data reproduce the stored labels? ---------------
# If it does not, the labels on disk did not come from these parameters and every
# number below would describe a different partition. Checked, not assumed.
for lv in ("main", "sub"):
    _agree = adjusted_rand_score(LAB[lv], _fit(X, lv))
    if _agree < 0.999:
        raise RuntimeError(
            f"[Stage 25] rebuilding the {lv} level from the recorded procedure does NOT "
            f"reproduce pooled_labels_{'main7' if lv == 'main' else 'sub'} (ARI {_agree:.4f}). "
            f"The stored labels and pooled_hdbscan_hierarchy disagree - one of them is "
            f"stale. Re-run Stage 23H.")
print("  refit check: both levels reproduce exactly from the stored procedure (ARI 1.000)")

STAB = {}
for lv in ("main", "sub"):
    ref = LAB[lv]
    K = _K(ref)
    themed = ref >= 0
    if K < 2:
        print(f"\n  {lv}: fewer than 2 themes - nothing to validate.")
        continue

    # ---- A + B: bootstrap ----------------------------------------------------
    aris, noise_agree = [], []
    jac = {c: [] for c in range(K)}
    for _ in range(N_BOOT):
        idx = rng.choice(n, int(n * FRAC), replace=False)
        new = _fit(X[idx], lv)
        r = ref[idx]
        m = r >= 0                                  # scored on rows the reference themed
        if m.sum() > 1:
            aris.append(adjusted_rand_score(r[m], new[m]))
        noise_agree.append(float(((r == -1) == (new == -1)).mean()))
        # per-theme Jaccard, noise excluded as a candidate on both sides
        nid = sorted({int(v) for v in new if v >= 0})
        if not nid:
            for c in range(K):
                jac[c].append(0.0)
            continue
        J = np.zeros((K, len(nid)))
        for c in range(K):
            rm = r == c
            if not rm.any():
                continue
            for j, nc in enumerate(nid):
                nm = new == nc
                u = np.logical_or(rm, nm).sum()
                J[c, j] = np.logical_and(rm, nm).sum() / u if u else 0.0
        r_i, c_i = linear_sum_assignment(-J)
        matched = {int(a): float(J[a, b]) for a, b in zip(r_i, c_i)}
        for c in range(K):
            jac[c].append(matched.get(c, 0.0))
    jac_mean = {c: float(np.mean(v)) for c, v in jac.items()}

    # ---- C: silhouette, themed rows only -------------------------------------
    sil_by, sil_overall, sil = {}, float("nan"), None
    if themed.sum() > K:
        sil = silhouette_samples(X[themed], ref[themed])
        sil_overall = float(silhouette_score(X[themed], ref[themed]))
        rt = ref[themed]
        sil_by = {c: float(sil[rt == c].mean()) for c in range(K) if (rt == c).any()}

    # ---- C2: THE SAME QUESTION, ASKED IN THE ORIGINAL EMBEDDING SPACE ---------
    # WHY BOTH, AND WHY THE SECOND ONE IS THE HONEST ONE.
    #   The silhouette above is computed on pooled_umap_nd - the space HDBSCAN
    #   itself optimised. A high value there says the algorithm found dense
    #   regions in the representation it was given; it cannot say the responses
    #   in a theme MEAN similar things, because that representation was built to
    #   make them separable. Measuring separation where the separation was
    #   manufactured is circular, and on this corpus the gap is large: Stage 27
    #   reports 0.600 in UMAP space against 0.130 in the original space.
    #
    #   So the same four quantities are computed again on pooled_embeddings,
    #   with COSINE distance - the metric the embedding model was trained under,
    #   and the one every other semantic comparison in this project uses.
    #
    #   NO THRESHOLD IS INVENTED. These are reported as a RANKING to be read
    #   beside each theme's Jaccard. Where a theme is stable AND coherent the
    #   verdict is strong; where the two disagree - reproducible under bootstrap
    #   but semantically loose, or the reverse - that disagreement is the finding
    #   and is printed rather than resolved by picking a cut-off.
    #
    #   WHICH OF THESE ACTUALLY DISCRIMINATES - measured, not assumed. Before
    #   shipping this block I built clusters with a KNOWN answer: two real
    #   themes, plus one deliberate mixture of two sub-topics, in four different
    #   geometries (4096-d and 64-d, near-orthogonal and correlated sub-topics).
    #     cohesion            flagged the mixture in 4 of 4
    #     original silhouette flagged the mixture in 4 of 4
    #     spread              flagged it in 3 of 4 - NOT reliable
    #   So COHESION and the original-space SILHOUETTE are the statistics to read;
    #   spread is reported because it is free and occasionally informative, but
    #   nothing should rest on it. (My own first draft of this comment claimed
    #   spread was the key signal for a two-lump theme. The test above disproved
    #   that: in high dimensions a mixture's members sit at a uniformly LOW
    #   cosine to the mid-point centroid rather than at a widely varying one.)
    #
    #   AND THE DYNAMIC RANGE IS SMALL. At 4096 dimensions cosine concentrates,
    #   so these numbers live in a narrow band - which is exactly why they are a
    #   RANKING to be read against each other and against the Jaccard, never
    #   against an absolute cut-off.
    sil_orig_by, coh_by, spread_by, sep_by = {}, {}, {}, {}
    sil_orig_overall = float("nan")
    if themed.sum() > K:
        _Et = _EOn[themed]
        _rt = ref[themed]
        try:
            _so = silhouette_samples(_Et, _rt, metric="cosine")
            sil_orig_overall = float(np.mean(_so))
            sil_orig_by = {c: float(_so[_rt == c].mean())
                           for c in range(K) if (_rt == c).any()}
        except Exception as _e:
            print(f"    (original-space silhouette skipped: {type(_e).__name__})")
        _cen = {}
        for c in range(K):
            m = _rt == c
            if m.any():
                v = _Et[m].mean(0)
                _cen[c] = v / (np.linalg.norm(v) + 1e-9)
        for c, cv in _cen.items():
            m = _rt == c
            _cos = _Et[m] @ cv
            coh_by[c] = float(_cos.mean())
            spread_by[c] = float(_cos.std())
            _oth = [float(cv @ ov) for o, ov in _cen.items() if o != c]
            sep_by[c] = float(max(_oth)) if _oth else float("nan")

    # ---- HENNIG'S THRESHOLD, APPLIED WHERE IT BELONGS: PER THEME --------------
    # Hennig (2007) is explicit that stability is a property of a cluster, not of
    # a partition: "in the same clustering, some clusters may be very stable and
    # others may be extremely unstable." His bands are therefore read off each
    # theme's own bootstrap Jaccard, and they decide what that theme may be used
    # for - not whether the whole clustering survives. An earlier version of
    # Stage 23H applied the 0.70 number to one global ARI and discarded whole
    # partitions in which most themes were perfectly usable; that gate is gone.
    #   >= 0.75  claimable  - a stable cluster
    #   0.60-0.75 pattern   - real structure, but exact membership is doubtful
    #   <  0.60  provisional - not a finding; describe, never claim
    def _tier(j):
        return "claimable" if j >= 0.75 else ("pattern" if j >= 0.60 else "provisional")

    tier_of = {c: _tier(v) for c, v in jac_mean.items()}
    n_claim = sum(1 for t in tier_of.values() if t == "claimable")
    n_patt = sum(1 for t in tier_of.values() if t == "pattern")
    n_prov = sum(1 for t in tier_of.values() if t == "provisional")

    STAB[lv] = dict(K=K, n_themed=int(themed.sum()), n_noise=int((~themed).sum()),
                    noise_rate=float((~themed).mean()),
                    ari_mean=float(np.mean(aris)) if aris else float("nan"),
                    ari_sd=float(np.std(aris)) if aris else float("nan"),
                    noise_agreement=float(np.mean(noise_agree)),
                    per_theme_jaccard=jac_mean,
                    mean_jaccard=float(np.mean(list(jac_mean.values()))),
                    silhouette_overall=sil_overall, per_theme_silhouette=sil_by,
                    silhouette_orig_overall=sil_orig_overall,
                    per_theme_silhouette_orig=sil_orig_by,
                    per_theme_cohesion=coh_by, per_theme_spread=spread_by,
                    per_theme_nearest_other=sep_by,
                    per_theme_tier=tier_of,
                    n_claimable=n_claim, n_pattern=n_patt, n_provisional=n_prov,
                    n_boot=N_BOOT, frac=FRAC)

    s = STAB[lv]
    print(f"\n  {lv.upper():<5} K={K}   themed {s['n_themed']} / noise {s['n_noise']} "
          f"({s['noise_rate']:.1%})")
    print(f"    A  bootstrap ARI on themed rows : {s['ari_mean']:.3f} +/- {s['ari_sd']:.3f}")
    print(f"       the same rows are called noise again {s['noise_agreement']:.1%} of the time")
    print(f"    B* CLAIM GATE (Hennig 2007, per theme, not per partition):")
    print(f"         claimable  (Jaccard >= 0.75) : {n_claim:>3} of {K}")
    print(f"         pattern    (0.60 - 0.75)     : {n_patt:>3} of {K}   "
          f"real, but membership is doubtful")
    print(f"         provisional(< 0.60)          : {n_prov:>3} of {K}   "
          f"describe only, never claim")
    if n_prov:
        _pv = sorted([c for c, t in tier_of.items() if t == "provisional"],
                     key=lambda c: jac_mean[c])
        print(f"         provisional ids: " +
              ", ".join(f"{lv[0]}{c}({jac_mean[c]:.2f})" for c in _pv[:12])
              + (" ..." if len(_pv) > 12 else ""))
    print(f"    B  mean per-theme Jaccard       : {s['mean_jaccard']:.3f}   "
          f"(>=0.75 stable, 0.60-0.75 pattern, <0.60 fragile)")
    _fr = sorted((c for c in jac_mean if jac_mean[c] < 0.60), key=lambda c: jac_mean[c])
    print(f"       {len(_fr)} of {K} theme(s) below 0.60"
          + (f": {', '.join(f'{lv[0]}{c}({jac_mean[c]:.2f})' for c in _fr[:8])}"
             + (" ..." if len(_fr) > 8 else "") if _fr else ""))
    print(f"    C  silhouette (themed rows)     : {s['silhouette_overall']:.3f}")

# The audit cells (Stage 46/47) look for one validation object per level under the
# historical names. They are written HERE, from the numbers just measured, so those
# names always carry THIS run's evidence instead of an artifact of the dead Ward
# stage - which is what they held before, silently, on any machine that had run it.
for lv, key in (("main", "validation_main7"), ("sub", "validation_sub")):
    if lv in STAB:
        s = STAB[lv]
        # method is recorded per level, not hardcoded: SUB is one HDBSCAN fit,
        # MAIN is now size-weighted Ward on SUB's centroids, and an audit trail
        # that says "hdbscan" for both would misdescribe how MAIN was built.
        _mth = ("hdbscan" if lv == "sub" else
                ("hdbscan+epsilon" if _MAIN_METHOD == "epsilon"
                 else "ward-on-sub-centroids"))
        psave(key, dict(level=lv, K=s["K"], method=_mth,
                        per_theme_tier=s.get("per_theme_tier", {}),
                        n_claimable=s.get("n_claimable"),
                        n_pattern=s.get("n_pattern"),
                        n_provisional=s.get("n_provisional"),
                        global_silhouette=s["silhouette_overall"],
                        mean_jaccard=s["mean_jaccard"],
                        per_theme_jaccard=s["per_theme_jaccard"],
                        per_theme_silhouette=s["per_theme_silhouette"],
                        bootstrap_ari=s["ari_mean"], noise_rate=s["noise_rate"],
                        n_themed=s["n_themed"], n_noise=s["n_noise"],
                        n_boot=N_BOOT, frac=FRAC))

STAB["params"] = _PARAMS
STAB["same_fit_both_levels"] = bool(_SAME_FIT)
STAB["method"] = "hdbscan"
# Back-compat: anything written against the old flat keys reads the ANALYTICAL
# level (sub), which is what those keys always meant.
if "sub" in STAB:
    STAB.update({k: STAB["sub"][k] for k in
                 ("per_theme_jaccard", "mean_jaccard", "silhouette_overall",
                  "per_theme_silhouette")})
# The claim gate is published on its own so downstream figures can filter on it
# without unpacking the whole stability dict. One row per theme per level.
# Iterate the LEVEL entries only. By this point STAB also carries "params",
# "method", "same_fit_both_levels" and the flat back-compat keys, so walking
# .items() blindly reaches a bool and dies on .get - which is exactly what it
# did on the first run.
_tier_rows = []
for _lv in ("main", "sub"):
    _s = STAB.get(_lv)
    if not isinstance(_s, dict):
        continue
    for _c, _t in _s.get("per_theme_tier", {}).items():
        _g = lambda k: float(_s.get(k, {}).get(_c, float("nan")))
        _tier_rows.append(dict(level=_lv, theme_id=int(_c),
                               jaccard=float(_s["per_theme_jaccard"][_c]),
                               tier=_t,
                               silhouette=_g("per_theme_silhouette"),
                               silhouette_orig=_g("per_theme_silhouette_orig"),
                               cohesion=_g("per_theme_cohesion"),
                               spread=_g("per_theme_spread"),
                               nearest_other=_g("per_theme_nearest_other")))
TIERS = pd.DataFrame(_tier_rows).sort_values(["level", "theme_id"])

# ---- report: where does each theme sit on BOTH axes? -------------------------
for _lv in ("main", "sub"):
    _s = STAB.get(_lv)
    if not isinstance(_s, dict) or not _s.get("per_theme_silhouette_orig"):
        continue
    _T = TIERS[TIERS.level == _lv]
    print(f"\n  {_lv.upper()}  semantic coherence in the ORIGINAL embedding space")
    print(f"    overall silhouette: UMAP {_s['silhouette_overall']:+.3f}  vs  "
          f"original {_s['silhouette_orig_overall']:+.3f}")
    print( "    UMAP is the space HDBSCAN optimised, so the original-space value is")
    print( "    the honest one. Per theme, weakest first:")
    print(f"    {'id':<5}{'jaccard':>9}{'tier':>13}{'sil_umap':>10}{'sil_orig':>10}"
          f"{'cohesion':>10}{'spread':>8}")
    for r in _T.sort_values("silhouette_orig").head(8).itertuples():
        print(f"    {_lv[0]}{r.theme_id:<4}{r.jaccard:>9.2f}{r.tier:>13}"
              f"{r.silhouette:>10.2f}{r.silhouette_orig:>10.2f}"
              f"{r.cohesion:>10.2f}{r.spread:>8.2f}")
    # the pair worth naming: geometry without semantics
    _gap = _T[(_T.silhouette > _T.silhouette.median())
              & (_T.silhouette_orig < _T.silhouette_orig.median())]
    if len(_gap):
        print(f"    {len(_gap)} theme(s) look well separated in UMAP but not in the")
        print( "    original space - the separation is a property of the projection")
        print( "    more than of the language:")
        for r in _gap.sort_values("silhouette_orig").head(6).itertuples():
            print(f"      {_lv[0]}{r.theme_id:<4} sil_umap {r.silhouette:+.2f} vs "
                  f"sil_orig {r.silhouette_orig:+.2f}  (jaccard {r.jaccard:.2f}, "
                  f"{r.tier})")
    _hi = _T.sort_values("spread", ascending=False).head(3)
    print( "    widest spread (a two-lump theme shows here before anywhere else):")
    for r in _hi.itertuples():
        print(f"      {_lv[0]}{r.theme_id:<4} spread {r.spread:.2f} at cohesion "
              f"{r.cohesion:.2f}  (jaccard {r.jaccard:.2f}, {r.tier})")
TIERS.to_csv(f"{POOLED_DIR}/theme_stability_tiers.csv", index=False)
psave("pooled_theme_stability", TIERS)
print(f"\n  -> theme_stability_tiers.csv - one row per theme with its Jaccard and")
print(f"     its tier (claimable / pattern / provisional). Filter claims on this.")

psave("pooled_stability", STAB)

# ---- the figure reports the ANALYTICAL unit (sub) at per-theme resolution -----
lv = "sub"
if lv in STAB:
    s = STAB[lv]
    ref = LAB[lv]; K = s["K"]
    jac_mean = s["per_theme_jaccard"]; sil_by = s["per_theme_silhouette"]
    sil_overall = s["silhouette_overall"]
    # names only if the label dict matches THIS partition; a stale one is ignored
    _llm = pload("pooled_sub_llm") or {}
    if len(_llm) != K:
        _llm = {}
    def _short(c):
        nm = ((_llm.get(c) or _llm.get(str(c)) or {}).get("name") or "").strip()
        return f"S{c}" if not nm or nm.startswith("[unlabelled") else f"S{c}: {nm[:16]}"

    jorder = sorted(range(K), key=lambda c: jac_mean.get(c, 0.0))   # weakest at bottom
    fig_h = float(np.clip(0.24 * K + 2.4, 5.0, 16.0))
    f1, ax = plt.subplots(1, 2, figsize=(15.5, fig_h), facecolor=NB, sharey=True,
                          gridspec_kw=dict(wspace=0.16))
    a = ax[0]
    a.axvspan(0.75, 1.02, color="#E8F3EA", zorder=0)
    a.axvspan(0.60, 0.75, color="#FBF3E2", zorder=0)
    a.axvspan(0.00, 0.60, color="#FBEAEA", zorder=0)
    for r, c in enumerate(jorder):
        v = jac_mean.get(c, 0.0)
        col = C_GOOD if v >= 0.75 else (C_WARN if v >= 0.60 else C_BAD)
        a.plot([0, v], [r, r], color=col, lw=1.6, zorder=2, alpha=0.8)
        a.scatter(v, r, s=40, color=col, edgecolors="white", lw=0.8, zorder=3)
        a.text(v + 0.015, r, f"{v:.2f}", va="center", fontsize=6.8, color="#666")
    a.set_yticks(np.arange(K)); a.set_yticklabels([_short(c) for c in jorder], fontsize=7.4)
    a.set_xlim(0, 1.10); a.set_ylim(-0.8, K - 0.2)
    a.set_xlabel("bootstrap Jaccard (reproducibility)")
    a.set_title("A  Stability - bootstrap Jaccard", fontsize=11.5, fontweight="bold",
                loc="left", color=NT)
    b = ax[1]
    b.axvspan(0.50, 1.02, color="#E3F0FA", zorder=0)
    b.axvspan(0.25, 0.50, color="#E8F3EA", zorder=0)
    b.axvspan(0.00, 0.25, color="#FBF3E2", zorder=0)
    b.axvspan(-0.30, 0.00, color="#FBEAEA", zorder=0)
    for r, c in enumerate(jorder):                                # SAME row order as A
        mu = sil_by.get(c, float("nan")); nc = int((ref == c).sum())
        if mu != mu:
            continue
        _zc = "#6699CC" if mu >= 0.5 else (C_GOOD if mu >= 0.25 else (C_WARN if mu >= 0 else C_BAD))
        b.plot([0, mu], [r, r], color=_zc, lw=1.6, zorder=2, alpha=0.8)
        b.scatter(mu, r, s=40, color=_zc, edgecolors="white", lw=0.8, zorder=3)
        b.text(max(mu, 0) + 0.015, r, f"{mu:.2f} · n={nc}", va="center", fontsize=6.8, color="#666")
    b.axvline(0, color="#BBB", lw=0.9, zorder=1)
    b.axvline(sil_overall, color=HL, ls="--", lw=1.4, zorder=4,
              label=f"overall {sil_overall:.2f}")
    b.set_xlim(-0.15, max(0.75, (max(sil_by.values()) if sil_by else 0.5) + 0.18))
    b.set_xlabel("mean silhouette (cohesion, themed rows only)")
    b.set_title("B  Cohesion - mean silhouette", fontsize=11.5, fontweight="bold",
                loc="left", color=NT)
    b.legend(fontsize=8.5, frameon=False, loc="lower right")
    for _a in ax:
        _a.set_facecolor("white"); _a.tick_params(length=0)
        for sp in ("top", "right", "left"):
            _a.spines[sp].set_visible(False)
    f1.suptitle(f"Per-sub-theme validation (K={K}, HDBSCAN bootstrap {N_BOOT}x@{int(FRAC*100)}%) - "
                "zones: >=0.75 stable · 0.60-0.75 pattern · <0.60 fragile (A) / Rousseeuw (B)",
                fontsize=13, fontweight="bold", color=NT, y=0.995)
    f1.text(0.5, -0.012,
            f"Rows sorted by Jaccard, weakest at the bottom: a theme low in BOTH panels should be "
            f"read with caution and is a merge candidate. The {s['n_noise']} responses HDBSCAN left "
            f"with no sub-theme ({s['noise_rate']:.0%}) are excluded from the silhouette - noise is "
            f"not a cluster. Refs: Hennig (2007); Rousseeuw (1987).",
            ha="center", fontsize=8.6, style="italic", color="#888")
    plt.tight_layout()
    save_fig(f1, "chart_pooled_stability"); plt.close(f1)

print("\n  -> pooled_stability (both levels)  ·  chart_pooled_stability")
print("=" * 74)

## Stage 26 — Validity without ground truth

With no human codes to score against, we test the themes by label-free checks. The main one: the Adjusted Rand Index between theme labels and the source question should be LOW, meaning the themes cut across questions rather than re-tracing the survey, which is the point of pooling.

Computed at **both** levels, on the responses that have a theme. The `-1` rows are dropped first: they are not a theme, and leaving them in would have handed the metric a large "cluster" whose only shared property is belonging nowhere — it draws from every question roughly in proportion, pushing the ARI toward zero and making the themes look *more* cross-cutting than they are. The dropped count is printed. The figure reports SUB (the analytical unit); MAIN is a one-line summary.


In [ ]:
# @title
# ==============================================================================
# STAGE 26  -  VALIDITY WITHOUT GROUND TRUTH
# There is no human coding to score against, so we judge the themes by checks
# that need no external labels. The central one: a theme should CUT ACROSS the
# survey questions, not merely re-trace them. We measure the Adjusted Rand Index
# between the theme labels and the source question; a LOW value means the themes
# are genuine cross-cutting constructs (which is the whole point of pooling), not
# artefacts of the survey layout. We also report each theme's question-mixing
# entropy. Convergent evidence (other paradigms) is Stage 29; stability is
# Stage 25. Refs: Hubert and Arabie (1985, ARI); Shannon (1948, entropy).
# ------------------------------------------------------------------------------
# NOISE. HDBSCAN leaves some responses with no theme (label -1). They are NOT a
# theme, so they are dropped before the ARI, the NMI, the composition matrix and
# the entropy. Leaving them in would have handed the metric a 27%-of-the-corpus
# "cluster" whose only property is that its members belong nowhere - it draws
# from every question roughly in proportion, which pushes the ARI toward zero and
# would have made the themes look MORE cross-cutting than they are. The count is
# printed, not hidden.
# ------------------------------------------------------------------------------
# TWO LEVELS. The figure reports SUB (the analytical unit); MAIN is reported as a
# one-line summary, because a 15-row heatmap of the same measurement adds no
# evidence. The mid level is gone (Stage 23H).
# ==============================================================================
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

# ---- stale-artifact guard (the same block opens every stage below) ------------
def _labels(n_rows, stage, levels=("main", "sub")):
    _KEY = {"main": "pooled_labels_main7", "sub": "pooled_labels_sub"}
    out = {}
    for lv in levels:
        a = pload(_KEY[lv])
        if a is None:
            raise RuntimeError(f"[{stage}] {_KEY[lv]} is missing - run Stage 23H first.")
        a = np.asarray(a)
        if len(a) != n_rows:
            raise RuntimeError(
                f"[{stage}] {_KEY[lv]} has {len(a)} rows but the pooled set has "
                f"{n_rows}. That artifact belongs to an EARLIER clustering. Re-run "
                f"Stage 23H (then Stage 30) before this stage.")
        out[lv] = a
    return out

def _K(lab):
    lab = np.asarray(lab)
    return int(lab.max()) + 1 if (lab >= 0).any() else 0
# ------------------------------------------------------------------------------

origin = np.asarray(require("pooled_origin", stage="Stage 26"))
LAB = _labels(len(origin), "Stage 26")

# FIX: after Stage 18 the pool holds only the NARRATIVE questions. Normalising
# the cross-question entropy by log(5) while only 4 questions can occur would
# understate it, and the composition matrix would carry an all-zero column.
# Use the questions actually present in the pooled data.
QL = [q for q in CFG.short_labels if (origin == q).any()]
_absent = [q for q in CFG.short_labels if q not in QL]
if _absent:
    print(f"  questions not in the thematic pool (excluded at Stage 18): {_absent}")

print("=" * 66)
print("  STAGE 26  VALIDITY WITHOUT GROUND TRUTH")
print("=" * 66)

XQ = {}
for lv in ("main", "sub"):
    labels = LAB[lv]
    m = labels >= 0                      # themed rows only: noise is not a theme
    K = _K(labels)
    if K < 2 or m.sum() < 2:
        print(f"  {lv}: fewer than 2 themes - nothing to test.")
        continue
    XQ[lv] = dict(
        ari=float(adjusted_rand_score(origin[m], labels[m])),
        nmi=float(normalized_mutual_info_score(origin[m], labels[m])),
        K=K, n_themed=int(m.sum()), n_noise=int((~m).sum()),
        noise_rate=float((~m).mean()))

    comp = np.zeros((K, len(QL)))
    for c in range(K):
        for j, q in enumerate(QL):
            comp[c, j] = np.sum((labels == c) & (origin == q))
    frac = comp / comp.sum(1, keepdims=True).clip(min=1)
    ent = -(frac * np.log(frac + 1e-12)).sum(1) / np.log(len(QL))  # 0 = one question, 1 = mixed
    XQ[lv].update(entropy=ent.tolist(), composition=frac.tolist(),
                  mean_entropy=float(ent.mean()))
    XQ[lv]["_frac"] = frac
    XQ[lv]["_ent"] = ent

def _read(ari):
    return ("strongly cross-cutting" if ari < 0.20 else
            "mostly cross-cutting" if ari < 0.35 else "partly question-aligned")

for lv in ("main", "sub"):
    if lv not in XQ:
        continue
    d = XQ[lv]
    print(f"  {lv.upper():<5} K={d['K']:<3} on {d['n_themed']} themed rows "
          f"({d['n_noise']} noise rows dropped, {d['noise_rate']:.1%})")
    print(f"        ARI(theme, source question) = {d['ari']:.3f}   "
          f"(LOW = themes span questions, good)  ->  {_read(d['ari'])}")
    print(f"        NMI = {d['nmi']:.3f}   mean cross-question entropy = "
          f"{d['mean_entropy']:.2f}   (1.0 = a theme draws evenly from all "
          f"{len(QL)} pooled questions)")

psave("pooled_validity_xq", {lv: {k: v for k, v in d.items() if not k.startswith("_")}
                             for lv, d in XQ.items()})

# ------------------------------- figure ---------------------------------------
# SUB is the analytical unit, so it is the level drawn at per-theme resolution.
if "sub" in XQ:
    d = XQ["sub"]
    labels = LAB["sub"]; K = d["K"]; frac = d["_frac"]; ent = d["_ent"]; ari = d["ari"]
    xcut_label = _read(ari)
    NT = globals().get("NAT_TEXT", "#1E1E1E"); NB = globals().get("NAT_BG", "#FFFFFF")
    from mpl_toolkits.axes_grid1 import make_axes_locatable
    fig, ax = plt.subplots(1, 2, figsize=(16, min(0.55 * K + 2.5, 12)), facecolor=NB,
                           gridspec_kw=dict(width_ratios=[1.6, 1.0], wspace=0.30))
    im = ax[0].imshow(frac, cmap=GRAD, aspect="auto", vmin=0, vmax=1)
    ax[0].set_xticks(range(len(QL)))
    ax[0].set_xticklabels(QL, rotation=20, ha="right", fontsize=10.1)
    # question-identity colours: one colour = one question, everywhere in the report
    for _t, _q in zip(ax[0].get_xticklabels(), QL):
        _t.set_color(Q_COLORS.get(_q, NAT_TEXT)); _t.set_fontweight("bold")
    ax[0].set_yticks(range(K))
    ax[0].set_yticklabels([f"S{c}" for c in range(K)], fontsize=10.1)
    for c in range(K):
        for j in range(len(QL)):
            if frac[c, j] >= 0.18:
                ax[0].text(j, c, f"{frac[c, j]:.2f}", ha="center", va="center", fontsize=8.4,
                           color="white" if frac[c, j] > 0.55 else "#333")
    ax[0].set_title("A  Sub-theme composition by question (row-normalised)",
                    fontsize=12.3, fontweight="bold", loc="left", color=NT)
    _div = make_axes_locatable(ax[0]); _cax = _div.append_axes("right", size="3.5%", pad=0.10)
    plt.colorbar(im, cax=_cax).set_label("fraction of theme", fontsize=9)
    _order = np.argsort(ent)
    _cols = get_cluster_colors(K)
    ax[1].barh(range(K), ent[_order], color=[_cols[c] for c in _order], edgecolor="white")
    ax[1].set_yticks(range(K)); ax[1].set_yticklabels([f"S{c}" for c in _order], fontsize=10.1)
    for i, c in enumerate(_order):
        ax[1].text(ent[c] + 0.012, i, f"{ent[c]:.2f}", va="center", fontsize=9)
    ax[1].axvline(ent.mean(), color="#882255", ls="--", lw=1.2, label=f"mean {ent.mean():.2f}")
    ax[1].set_xlim(0, 1.10)
    ax[1].set_xlabel(f"cross-question entropy (1 = spans all {len(QL)} pooled questions)")
    ax[1].set_title("B  Which sub-themes are cross-cutting",
                    fontsize=12.3, fontweight="bold", loc="left", color=NT)
    ax[1].legend(fontsize=9, frameon=False, loc="lower right")
    ax[0].set_facecolor("white"); ax[1].set_facecolor("white")
    for sp in ("top", "right"):
        ax[1].spines[sp].set_visible(False)
    fig.suptitle(f"Validity without ground truth: sub-themes vs source question  "
                 f"(ARI {ari:.2f}, {xcut_label})",
                 fontsize=14.6, fontweight="bold", color=NT, y=1.02)
    fig.text(0.5, -0.03,
             f"Low theme-vs-question ARI = themes are not survey-section artefacts; entropy shows "
             f"the largest themes span questions while small themes track one. Computed on the "
             f"{d['n_themed']} responses that HAVE a sub-theme; the {d['n_noise']} left as noise "
             f"({d['noise_rate']:.0%}) are not a theme and are excluded "
             f"(Hubert & Arabie 1985; Shannon 1948).",
             ha="center", fontsize=9, style="italic", color="#888")
    plt.tight_layout(); save_fig(fig, "chart_stage26_validity_crosscut")
    plt.close(fig)
print("=" * 66)


## Stage 26B - Per-response sentiment, read by the instruction model

Not a lexicon. VADER was replaced here because it cannot see figurative language, which is the register respondents actually write in, and because it is English-only, so it marked every non-English response NaN. The model returns one of six ordered categories plus a confidence and must quote the span carrying the affect; the numeric score is derived in code on a fixed mapping. A verdict whose quoted span is not in the response is refused. Saved as `pooled_sentiment_response`, with team-level aggregates alongside.

In [ ]:
# @title
# ==============================================================================
# STAGE 26B  - SENTIMENT SCORING  (response + team level, LLM-read, not lexicon)
#              BATCHED + ADAPTIVE SELF-CONSISTENCY + CRASH-SAFE CACHE
# Produces:
#   - pooled_sentiment_response  : array aligned with pooled_meta rows
#   - pooled_sentiment_team_long : team x sub-theme mean sentiment (long)
#   - pooled_sentiment_team_wide : the same, pivoted
#   - pooled_sentiment_detail    : per-response category, confidence, rationale
#   - sentiment_response_audit.csv : every scored response, readable
# ------------------------------------------------------------------------------
# WHY THIS REPLACES VADER
#   VADER (Hutto & Gilbert 2014) is a lexicon plus hand-written intensity rules.
#   It matches words, boosts on "!!!" and ALL-CAPS, and sums. Two consequences
#   made it the wrong instrument for THIS corpus:
#     1. FIGURATIVE LANGUAGE IS INVISIBLE TO IT. "iGEM was a rollercoaster",
#        "we were thrown in at the deep end", "it broke me and rebuilt me" carry
#        clear affect that no lexicon entry encodes. Participants write in
#        exactly this register.
#     2. IT IS ENGLISH-ONLY. The previous version marked every non-English
#        response NaN - correct given the tool, but it discarded a real part of
#        the corpus, and the French subset has its own section in Notebook 3.
#   An instruction-following model reads the sentence instead of matching its
#   words, and is multilingual, so both limitations go at once.
#
# WHY A CATEGORY AND NOT A NUMBER
#   Asked for a float, an instruction model hedges toward the middle: scores pile
#   up near 0 and the scale loses its ends - the same compression this stage
#   already fought once (scoring cleaned text instead of raw). So the model is
#   asked for one of six ORDERED CATEGORIES plus a confidence, and the numeric
#   compound is DERIVED here, in code, on a fixed mapping. The model makes a
#   discrete judgement it can defend; the arithmetic is ours and is auditable.
#   Confidence shrinks a score toward that category's floor and can never flip
#   the sign the model committed to.
#
# GROUNDING, THE SAME STANDARD THE NAMING STAGE USES
#   Every verdict must quote the span of the response that carries the affect.
#   If the quoted span is not in the text, the verdict is refused and the
#   response is scored NaN rather than accepted on trust.
#
# ------------------------------------------------------------------------------
# WHAT CHANGED IN THIS VERSION, AND WHY (all three are speed, none is quality)
#   Measured on the first real run: 99 responses in 32 minutes = 19 s each, which
#   puts the full corpus at about 6.5 hours. Three causes, three fixes:
#
#   1. ONE RESPONSE AT A TIME. The previous version called generate() per
#      response, so the GPU sat idle between prompts. Generation is now BATCHED
#      (BATCH_SIZE prompts per call, left-padded as decoder-only generation
#      requires). This is the single biggest win and changes nothing about the
#      prompt or the answer.
#   2. THREE SAMPLES FOR EVERY RESPONSE. Self-consistency was spending 3x the
#      compute on every row, including the many where the model is plainly
#      certain and all three samples agree. Sampling is now ADAPTIVE: one greedy
#      pass over everything, then the extra samples ONLY for rows that failed to
#      parse or came back under CONF_SURE. Rows that never needed a second
#      opinion are recorded with agreement 1.0 from a single confident pass;
#      rows that did get the full vote they always got.
#   3. NOTHING WAS SAVED UNTIL THE END. psave ran after the loop, so interrupting
#      the cell threw away every verdict computed so far - which is exactly what
#      happened. The cache is now written every SAVE_EVERY responses and again in
#      a finally block, so an interrupt costs at most one chunk.
#
#   The prompt text and PROMPT_VERSION are UNCHANGED, so any verdict already in
#   the cache from a previous run is still valid and is reused.
#
# CACHING. Verdicts are keyed on sha1(text, PROMPT_VERSION, MODEL). The key does
# NOT include cluster labels, so re-running Stage 21H/23H and getting a different
# clustering does NOT invalidate sentiment - only the team x sub-theme
# aggregation at the bottom recomputes, which costs seconds and no LLM calls.
# Refs: Hutto & Gilbert (2014) on what VADER does and does not model;
#       Wang et al. (2023) self-consistency; Aggarwal et al. (2023) adaptive
#       consistency - stop sampling once the answer is stable.
# ==============================================================================
import os, re, json, gc, hashlib, time
import numpy as np
import pandas as pd

# ------------------------------- EDIT HERE ------------------------------------
SENT_MODEL   = globals().get("SENTIMENT_MODEL_NAME",
                             globals().get("LLM_MODEL_NAME", "Qwen/Qwen2.5-14B-Instruct"))
SENT_ENABLE  = bool(globals().get("SENTIMENT_LLM_ENABLE", True))
SENT_FORCE   = bool(globals().get("SENTIMENT_FORCE_REGEN", False))
BATCH_SIZE   = 16     # prompts per generate() call. Lower it if you hit OOM -
                      # the cell also halves it automatically and retries.
K_SELF       = 3      # MAXIMUM samples for a row the model is unsure about
CONF_SURE    = 0.85   # a greedy verdict at or above this confidence is accepted
                      # without spending the extra samples
TEMP         = 0.7    # temperature for the extra samples
MAX_CHARS    = 1200   # responses longer than this are truncated for the prompt
MAX_NEW      = 200    # generation cap per verdict
SAVE_EVERY   = 100    # write the cache to disk every N newly scored responses
# The ordered scale and its numeric anchors. The model never sees these numbers;
# it picks a label and the label maps here. Symmetric, so the mapping itself
# cannot push the corpus mean positive or negative.
SENT_SCALE = {
    "strongly_negative": -1.00,
    "negative":          -0.60,
    "mixed":              0.00,   # both directions present, not "no affect"
    "neutral":            0.00,   # no affect expressed
    "positive":           0.60,
    "strongly_positive":  1.00,
}
# Confidence shrinks toward the NEXT-MILDER anchor, never past it, so a low
# confidence never flips the sign.
SENT_SHRINK = {
    "strongly_negative": -0.60,
    "negative":          -0.20,
    "mixed":              0.00,
    "neutral":            0.00,
    "positive":           0.20,
    "strongly_positive":  0.60,
}
# ----------------------------- END EDIT ---------------------------------------
PROMPT_VERSION = "sent-v1-categorical"   # unchanged from the previous version on
                                         # purpose: the prompt is identical, so
                                         # cached verdicts stay valid

# ---- stale-artifact guard (the same block opens every stage below) ------------
def _labels(n_rows, stage, levels=("main", "sub")):
    _KEY = {"main": "pooled_labels_main7", "sub": "pooled_labels_sub"}
    out = {}
    for lv in levels:
        a = pload(_KEY[lv])
        if a is None:
            raise RuntimeError(f"[{stage}] {_KEY[lv]} is missing - run Stage 23H first.")
        a = np.asarray(a)
        if len(a) != n_rows:
            raise RuntimeError(
                f"[{stage}] {_KEY[lv]} has {len(a)} rows but the pooled set has "
                f"{n_rows}. That artifact belongs to an EARLIER clustering. Re-run "
                f"Stage 23H (then Stage 30) before this stage.")
        out[lv] = a
    return out
# ------------------------------------------------------------------------------

meta = require("pooled_meta", stage="Stage 26B").reset_index(drop=True)
N = len(meta)
sub = _labels(N, "Stage 26B", levels=("sub",))["sub"]
_df = df if "df" in globals() and df is not None else get_df()
if _df is None:
    raise RuntimeError("[Stage 26B] need df_final/df_cleaned for the raw survey text.")

# setdefault, not assignment: the same participant can appear in several rows of
# the cleaned frame, and every other stage in this notebook keeps the FIRST.
_pid = {}
for _, r in _df.iterrows():
    _pid.setdefault(r["Participant"], r)

# RAW text, not bert_*: minimal_clean() lowercases and strips ALL punctuation, and
# an LLM reads case and punctuation as signal exactly as a human does.
_RAW_OF_Q = dict(zip(CFG.short_labels, CFG.text_cols))

def _get_text(mr):
    r = _pid.get(mr["Participant"])
    if r is None:
        return ""
    col = _RAW_OF_Q.get(mr["question"])
    if col is None:
        return ""
    v = r.get(col, "")
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return ""
    v = str(v)
    return "" if v.strip().lower() == "nan" else " ".join(v.split())

texts = np.array([_get_text(mr) for _, mr in meta.iterrows()], dtype=object)
_n_blank = int((texts == "").sum())

print("=" * 74)
print(f"  STAGE 26B - SENTIMENT (LLM-read, categorical, batched)")
print(f"  model={SENT_MODEL}")
print("=" * 74)
print(f"  pooled responses : {N}   ({_n_blank} blank -> NaN, never 0.0)")
print(f"  batch size       : {BATCH_SIZE}   |  adaptive sampling: 1 greedy pass, "
      f"up to {K_SELF} only when confidence < {CONF_SURE}")
print(f"  multilingual     : non-English responses are SCORED, not NaN'd")

# ------------------------------- cache ----------------------------------------
_CACHE_KEY = "pooled_sentiment_llm_cache"
_cache = {} if SENT_FORCE else (pload(_CACHE_KEY) or {})
if not isinstance(_cache, dict):
    _cache = {}

def _key(t):
    return hashlib.sha1(f"{PROMPT_VERSION}|{SENT_MODEL}|{t}".encode("utf-8")).hexdigest()

_need = [i for i in range(N) if texts[i] and _key(texts[i]) not in _cache]
print(f"  cached           : {N - len(_need) - _n_blank} | to score: {len(_need)}")

# ------------------------------- prompts --------------------------------------
SYS = (
    "You rate the sentiment a survey respondent expresses about their own "
    "experience in the iGEM synthetic-biology student competition.\n\n"
    "Read what the sentence MEANS, not which words it contains. Respondents use "
    "metaphor, irony, understatement and figures of speech: \"a rollercoaster\", "
    "\"thrown in at the deep end\", \"it broke me and rebuilt me\". Judge the "
    "affect the writer is conveying, in context, the way an attentive human "
    "reader would.\n\n"
    "Respondents write in several languages. Rate the response in whatever "
    "language it is written; do not translate it first and do not penalise it "
    "for not being English.\n\n"
    "Choose exactly one label:\n"
    "  strongly_negative - clear, strong negative affect\n"
    "  negative          - negative affect, moderate\n"
    "  mixed             - genuinely BOTH positive and negative, e.g. hard but "
    "worth it\n"
    "  neutral           - factual or descriptive, no affect expressed\n"
    "  positive          - positive affect, moderate\n"
    "  strongly_positive - clear, strong positive affect\n\n"
    "Use 'mixed' only when both directions are really present. Use 'neutral' "
    "only when there is no affect at all - a flat description of what happened. "
    "Do not retreat to 'neutral' or 'mixed' because a response is short or hard "
    "to judge; commit to the reading the text best supports and lower your "
    "confidence instead.\n\n"
    "Quote the exact span of the response that carries the affect, copied "
    "character for character from the text. If nothing in the text carries "
    "affect, use 'neutral' and quote the span that is most representative."
)

def build_prompt(text, question):
    return (f"SURVEY QUESTION: {question}\n\n"
            f"RESPONSE:\n\"{text[:MAX_CHARS]}\"\n\n"
            "Return ONLY this JSON object:\n"
            '{"label": "<one of: strongly_negative|negative|mixed|neutral|'
            'positive|strongly_positive>", '
            '"confidence": <0.0-1.0>, '
            '"evidence": "<exact span copied from the response>", '
            '"reason": "<one short sentence>"}')

def parse_json(raw):
    if not raw:
        return None
    m = re.search(r"\{.*\}", raw, re.S)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except Exception:
        try:
            return json.loads(re.sub(r",\s*\}", "}", m.group(0).replace("'", '"')))
        except Exception:
            return None

def _norm(s):
    """Case- and whitespace-insensitive containment, so a model that re-wraps a
    quote is not punished but one that invents a quote still fails."""
    return re.sub(r"\s+", " ", str(s)).strip().lower()

# ------------------------------- model ----------------------------------------
gen_batch = None
mdl = tok = None
if _need and SENT_ENABLE:
    try:
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
        print(f"\n  loading {SENT_MODEL} in 4-bit ...")
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                 bnb_4bit_compute_dtype=torch.bfloat16,
                                 bnb_4bit_use_double_quant=True)
        tok = AutoTokenizer.from_pretrained(SENT_MODEL)
        # Decoder-only generation must be LEFT padded, otherwise the pad tokens
        # sit between the prompt and the first generated token and the model
        # continues from padding instead of from the prompt.
        tok.padding_side = "left"
        if tok.pad_token_id is None:
            tok.pad_token = tok.eos_token
        mdl = AutoModelForCausalLM.from_pretrained(SENT_MODEL, quantization_config=bnb,
                                                   device_map="auto",
                                                   torch_dtype=torch.bfloat16)
        mdl.eval()

        def gen_batch(usr_prompts, temperature=0.0, max_new=MAX_NEW, bs=None):
            """Generate for a LIST of prompts. Returns a list of strings, same
            order. Halves the batch and retries once on CUDA OOM."""
            bs = int(bs or BATCH_SIZE)
            out_all = []
            i = 0
            while i < len(usr_prompts):
                chunk = usr_prompts[i:i + bs]
                texts_in = [
                    tok.apply_chat_template(
                        [{"role": "system", "content": SYS},
                         {"role": "user", "content": u}],
                        add_generation_prompt=True, tokenize=False)
                    for u in chunk]
                enc = tok(texts_in, return_tensors="pt", padding=True,
                          truncation=True, max_length=2048)
                dev = next(mdl.parameters()).device
                enc = {k: v.to(dev) for k, v in enc.items()}
                kw = dict(max_new_tokens=max_new,
                          pad_token_id=(tok.pad_token_id or tok.eos_token_id))
                if temperature and temperature > 0:
                    kw.update(do_sample=True, temperature=float(temperature), top_p=0.9)
                else:
                    kw.update(do_sample=False)
                try:
                    with torch.no_grad():
                        out = mdl.generate(**enc, **kw)
                except torch.cuda.OutOfMemoryError:
                    torch.cuda.empty_cache()
                    if bs == 1:
                        raise
                    bs = max(1, bs // 2)
                    print(f"    (OOM - batch size reduced to {bs} and retrying)")
                    continue
                plen = enc["input_ids"].shape[1]
                out_all.extend(tok.batch_decode(out[:, plen:], skip_special_tokens=True))
                i += len(chunk)
            return out_all
        print("  model ready.")
    except Exception as e:
        print(f"  !! LLM unavailable ({type(e).__name__}: {str(e)[:110]})")
        print(f"     Sentiment is WITHHELD (NaN), not faked with a lexicon fallback:")
        print(f"     a silently different instrument is worse than a missing column.")
        gen_batch = None

# ------------------------------- score ----------------------------------------
def _verdict_from(samples, text):
    """Modal category across the samples that parsed, with the citation check."""
    rec = dict(label="", confidence=np.nan, evidence="", reason="",
               agreement=np.nan, value=np.nan, status="withheld", why="")
    good = [d for d in samples
            if isinstance(d, dict)
            and str(d.get("label", "")).strip().lower() in SENT_SCALE]
    if not good:
        rec["why"] = "no parsable verdict"
        return rec
    labs = [str(d["label"]).strip().lower() for d in good]
    modal = max(set(labs), key=labs.count)
    best = next(d for d, l in zip(good, labs) if l == modal)
    rec["label"] = modal
    rec["agreement"] = labs.count(modal) / len(labs)
    try:
        conf = float(best.get("confidence", 0.5))
    except Exception:
        conf = 0.5
    conf = min(max(conf, 0.0), 1.0)
    rec["confidence"] = conf
    rec["evidence"] = str(best.get("evidence", ""))[:300]
    rec["reason"] = str(best.get("reason", ""))[:200]
    if modal != "neutral" and rec["evidence"]:
        if _norm(rec["evidence"]) not in _norm(text):
            rec["why"] = "quoted span is not in the response"
            return rec
    anchor, floor = SENT_SCALE[modal], SENT_SHRINK[modal]
    rec["value"] = float(floor + (anchor - floor) * conf)
    rec["status"] = "scored"
    return rec


def _flush():
    psave(_CACHE_KEY, _cache)


if _need and gen_batch is not None:
    t0 = time.time()
    print(f"\n  scoring {len(_need)} responses in batches of {BATCH_SIZE} ...")
    _done = 0
    try:
        for start in range(0, len(_need), BATCH_SIZE):
            idxs = _need[start:start + BATCH_SIZE]
            prompts = [build_prompt(texts[i], str(meta.iloc[i]["question"])) for i in idxs]

            # --- pass 1: greedy, everything ---
            raw1 = gen_batch(prompts, temperature=0.0)
            samples = {i: [parse_json(r)] for i, r in zip(idxs, raw1)}

            # --- who actually needs a second opinion? ---
            def _unsure(i):
                d = samples[i][0]
                if not isinstance(d, dict):
                    return True
                if str(d.get("label", "")).strip().lower() not in SENT_SCALE:
                    return True
                try:
                    return float(d.get("confidence", 0.0)) < CONF_SURE
                except Exception:
                    return True

            hard = [i for i in idxs if _unsure(i)]
            for _ in range(max(0, K_SELF - 1)):
                if not hard:
                    break
                hp = [build_prompt(texts[i], str(meta.iloc[i]["question"])) for i in hard]
                rawk = gen_batch(hp, temperature=TEMP)
                for i, r in zip(hard, rawk):
                    samples[i].append(parse_json(r))

            for i in idxs:
                _cache[_key(texts[i])] = _verdict_from(samples[i], texts[i])

            _done += len(idxs)
            if _done % SAVE_EVERY < BATCH_SIZE:
                _flush()
            _rate = _done / max(time.time() - t0, 1e-9)
            _eta = (len(_need) - _done) / max(_rate, 1e-9)
            _ok = sum(1 for v in _cache.values() if v.get("status") == "scored")
            print(f"    {_done}/{len(_need)}   {_rate*60:5.1f}/min   "
                  f"eta {_eta/60:5.1f} min   ({len(hard)} of {len(idxs)} needed "
                  f"extra samples, {_ok} scored total)", flush=True)
    finally:
        # an interrupt must not throw away work already paid for
        _flush()
        print(f"  cache saved ({len(_cache)} verdicts on disk).")

# free the model before anything else loads one
try:
    del mdl, tok
except Exception:
    pass
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception:
    pass

# ------------------------------- assemble -------------------------------------
sent_scores = np.full(N, np.nan, dtype=float)
rows = []
_n_scored = _n_withheld = 0
for i in range(N):
    if not texts[i]:
        rows.append(dict(row_id=i, question=meta.iloc[i]["question"], n_words=0,
                         label="", confidence=np.nan, value=np.nan,
                         agreement=np.nan, status="blank", why="no text",
                         evidence="", reason="", text=""))
        continue
    v = _cache.get(_key(texts[i]))
    if not v:
        rows.append(dict(row_id=i, question=meta.iloc[i]["question"],
                         n_words=len(texts[i].split()), label="", confidence=np.nan,
                         value=np.nan, agreement=np.nan, status="withheld",
                         why="not scored (model unavailable or run interrupted)",
                         evidence="", reason="", text=texts[i][:300]))
        _n_withheld += 1
        continue
    if v.get("status") == "scored":
        sent_scores[i] = float(v["value"])
        _n_scored += 1
    else:
        _n_withheld += 1
    rows.append(dict(row_id=i, question=meta.iloc[i]["question"],
                     n_words=len(texts[i].split()), label=v.get("label", ""),
                     confidence=v.get("confidence"), value=v.get("value"),
                     agreement=v.get("agreement"), status=v.get("status"),
                     why=v.get("why", ""), evidence=v.get("evidence", ""),
                     reason=v.get("reason", ""), text=texts[i][:300]))

DET = pd.DataFrame(rows)
DET.to_csv(f"{POOLED_DIR}/sentiment_response_audit.csv", index=False)
psave("pooled_sentiment_detail", DET)
psave("pooled_sentiment_response", sent_scores)

corpus_mean = float(np.nanmean(sent_scores)) if _n_scored else 0.0
print(f"\n  scored     : {_n_scored}")
print(f"  withheld   : {_n_withheld}   -> NaN (not 0.0)")
print(f"  blank      : {_n_blank}   -> NaN")
print(f"  mean value : {corpus_mean:+.3f}")
if _n_scored:
    _d = DET[DET.status == "scored"]
    print(f"\n  DISTRIBUTION ACROSS THE SCALE  (this is the spread a lexicon compressed)")
    _order = sorted(SENT_SCALE, key=lambda k: (SENT_SCALE[k], k))
    for lb in _order:
        n = int((_d.label == lb).sum())
        if n:
            bar = "#" * max(1, int(40 * n / max(len(_d), 1)))
            print(f"    {lb:<18} {n:>5}  {bar}")
    print(f"    value range {_d.value.min():+.2f} .. {_d.value.max():+.2f}   "
          f"sd {_d.value.std():.3f}   mean self-consistency {_d.agreement.mean():.2f}")
    _q = (_d.groupby("question")["value"].agg(["size", "mean"]).round(3))
    print(f"\n  by question:")
    for q, r in _q.iterrows():
        print(f"    {str(q)[:22]:<24} n={int(r['size']):<5} mean={r['mean']:+.3f}")
_w = DET[DET.status == "withheld"]
if len(_w):
    print(f"\n  withheld reasons:")
    for why, n in _w["why"].value_counts().head(5).items():
        print(f"    {int(n):>5}  {why}")

# team-level aggregation per SUB-THEME (used by Stage 40 and NB3)
_themed = sub >= 0
print(f"\n  team x sub-theme aggregation over the {int(_themed.sum())} responses that "
      f"have a sub-theme")
print(f"  ({int((~_themed).sum())} noise responses excluded - '-1' is not a theme and "
      f"gets no row)")
if "Team" not in meta.columns:
    print("  WARNING: meta has no Team column; team-level sentiment not computed.")
else:
    tmp = meta.loc[_themed, ["Team"]].copy()
    tmp["_sub"] = sub[_themed]
    tmp["_sent_val"] = sent_scores[_themed]
    grp = (tmp.groupby(["Team", "_sub"])["_sent_val"].mean().dropna().reset_index())
    team_sent_df = pd.DataFrame({
        "Team": grp["Team"].astype(str),
        "sub_theme_id": grp["_sub"].astype(int),
        "mean_sentiment": grp["_sent_val"].astype(float),
    })
    if len(team_sent_df):
        team_sent_pivot = team_sent_df.pivot(index="Team", columns="sub_theme_id",
                                             values="mean_sentiment")
    else:
        team_sent_pivot = pd.DataFrame()
    assert (team_sent_df["sub_theme_id"] >= 0).all(), "noise leaked into the team table"
    psave("pooled_sentiment_team_long", team_sent_df)
    psave("pooled_sentiment_team_wide", team_sent_pivot)
    team_sent_df.to_csv(f"{POOLED_DIR}/pooled_sentiment_team_long.csv", index=False)
    team_sent_pivot.to_csv(f"{POOLED_DIR}/pooled_sentiment_team_wide.csv")
    print(f"  saved pooled_sentiment_team_long / wide  "
          f"({len(team_sent_df)} team x sub-theme cells, "
          f"{team_sent_df['Team'].nunique()} teams, "
          f"{team_sent_df['sub_theme_id'].nunique()} sub-themes)")

print(f"\n  -> sentiment_response_audit.csv (every verdict with its quoted evidence,")
print(f"     so any score can be checked against the sentence that produced it)")
print("=" * 74)
print("  STAGE 26B complete.")
print("=" * 74)

## Stage 27 — Cluster reality & embedding neutrality

Anti-bias validation that the themes are real, not manufactured, and that the embedding is not sorting by positivity. Three checks: a permutation null (real silhouette far above clusterings of shuffled embeddings = genuine structure, gap-style logic from Tibshirani 2001); original-embedding vs UMAP-space silhouette of the same labels (so we never claim more separation than the embeddings carry); and per-theme valence balance (each theme spans positive and negative responses, so clusters are topical, not positivity-sorted).

Every number is computed on **one consistent row set — the responses that have a sub-theme**. Passing `-1` to `silhouette_score` treats "everything that belongs nowhere" as a cluster: it is maximally incohesive by construction and made the honest separation look worse than it is. The excluded count is printed, and the main level's own silhouette is reported alongside.


In [ ]:
# @title
# ==============================================================================
# STAGE 27  -  CLUSTER REALITY & EMBEDDING NEUTRALITY  (anti-bias validation)
# Three honest checks that the themes are REAL structure, not manufactured, and
# that the embedding is NOT biased toward positive responses:
#   A. Permutation null. Silhouette of a clustering of the real embedding vs
#      clusterings of column-shuffled embeddings (joint structure destroyed).
#      Real far above the null = genuine structure, not an artefact (a
#      reference-null logic like the gap statistic; Tibshirani et al. 2001).
#   B. Original-embedding vs UMAP-space silhouette of the SAME labels: how much
#      the UMAP projection inflates apparent separation. We report BOTH, so we
#      never claim more separation than the embeddings actually carry.
#   C. Valence balance per theme. A sentiment-lexicon score shows each theme
#      spans positive AND negative responses, so the clusters are TOPICAL, not
#      sorted by positivity. If a theme were all-positive, that would signal bias.
# ------------------------------------------------------------------------------
# NOISE. Every number here is computed on the responses that HAVE a sub-theme.
# Feeding -1 to silhouette_score treats "everything that belongs nowhere" as a
# cluster: it is maximally incohesive by construction, so it dragged the reported
# separation down and made the honest number look worse than it is. The whole
# cell now works on one consistent row set - the themed rows - so panel A's null,
# panel B's two silhouettes and panel C's valence all describe the same people.
# The excluded count is printed.
# Refs: Tibshirani, Walther, Hastie (2001); Hennig (2007); Rousseeuw (1987).
# ==============================================================================
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

# ---- stale-artifact guard (the same block opens every stage below) ------------
def _labels(n_rows, stage, levels=("main", "sub")):
    _KEY = {"main": "pooled_labels_main7", "sub": "pooled_labels_sub"}
    out = {}
    for lv in levels:
        a = pload(_KEY[lv])
        if a is None:
            raise RuntimeError(f"[{stage}] {_KEY[lv]} is missing - run Stage 23H first.")
        a = np.asarray(a)
        if len(a) != n_rows:
            raise RuntimeError(
                f"[{stage}] {_KEY[lv]} has {len(a)} rows but the pooled set has "
                f"{n_rows}. That artifact belongs to an EARLIER clustering. Re-run "
                f"Stage 23H (then Stage 30) before this stage.")
        out[lv] = a
    return out

def _K(lab):
    lab = np.asarray(lab)
    return int(lab.max()) + 1 if (lab >= 0).any() else 0
# ------------------------------------------------------------------------------

E  = np.asarray(require("pooled_embeddings", stage="Stage 27"))
_x5 = pload("pooled_umap_nd")
if _x5 is None:
    _x5 = pload("pooled_umap5d")
if _x5 is None:
    raise RuntimeError("[Stage 27] pooled_umap_nd missing - run Stage 21H first.")
X5 = np.asarray(_x5)
meta = require("pooled_meta", stage="Stage 27").reset_index(drop=True)
N = len(meta)
if not (len(E) == len(X5) == N):
    raise RuntimeError(f"[Stage 27] row mismatch: embeddings={len(E)}, umap={len(X5)}, "
                       f"meta={N}. One of these is from an earlier run.")
LAB = _labels(N, "Stage 27")
labels = LAB["sub"]                      # the analytical unit
K = _K(labels)
THEMED = labels >= 0
set_all_seeds()
rng = np.random.default_rng(CFG.seed)

print("=" * 66)
print("  STAGE 27  CLUSTER REALITY & EMBEDDING NEUTRALITY")
print("=" * 66)
print(f"  sub K={K}; {int(THEMED.sum())} of {N} responses have a sub-theme, "
      f"{int((~THEMED).sum())} are noise ({(~THEMED).mean():.1%})")
print("  every silhouette below is computed on the themed rows only - '-1' is not")
print("  a cluster, and scoring it as one understates the real separation.")
if THEMED.sum() < 50 or K < 2:
    raise RuntimeError("[Stage 27] too few themed rows (or fewer than 2 themes) to "
                       "run the reality checks.")

# design-system sentiment endpoints (defined HERE: SENT_CMAP lives in NB3 only;
# this cell must be self-contained in NB2). Same ramp ends as every notebook.
_SENT_POS, _SENT_NEG = "#0F6E56", "#B84413"

# subsample the THEMED rows for the heavy cosine silhouette / Ward
_pool = np.where(THEMED)[0]
m = rng.choice(_pool, min(700, len(_pool)), replace=False) if len(_pool) > 700 else _pool
Es, lab_s, X5s = E[m], labels[m], X5[m]
Ks = int(len(set(lab_s.tolist())))

def _ward_sil(M, k, metric):
    lab = AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(M)
    return silhouette_score(M, lab, metric=metric) if len(set(lab)) > 1 else float("nan")

# A. permutation null in the ORIGINAL embedding space
sil_orig = silhouette_score(Es, lab_s, metric="cosine") if Ks > 1 else float("nan")
sil_struct = _ward_sil(Es, Ks, "cosine")   # FAIR null: the SAME operation as the shuffled null
null = []
for _ in range(15):
    Esh = Es.copy()
    for j in range(Esh.shape[1]):
        rng.shuffle(Esh[:, j])
    null.append(_ward_sil(Esh, Ks, "cosine"))
null = np.array([x for x in null if x == x])
z = (sil_struct - null.mean()) / (null.std() + 1e-9) if len(null) else float("nan")

# B. original vs UMAP silhouette of the SAME (final) labels
sil_umap = silhouette_score(X5s, lab_s) if Ks > 1 else float("nan")

# the same honest number for the communication layer, on its own themed rows
_mainT = LAB["main"] >= 0
_mm = rng.choice(np.where(_mainT)[0], min(700, int(_mainT.sum())), replace=False) \
      if _mainT.sum() > 700 else np.where(_mainT)[0]
sil_main = (silhouette_score(E[_mm], LAB["main"][_mm], metric="cosine")
            if len(set(LAB["main"][_mm].tolist())) > 1 else float("nan"))

# C. valence per response (lexicon) -> per theme
POS = {"good", "great", "enjoyed", "enjoy", "love", "loved", "helpful", "help", "support",
       "supportive", "rewarding", "fun", "amazing", "learned", "learn", "grateful",
       "inspiring", "confident", "proud", "friends", "friend", "community", "opportunity",
       "best", "happy", "wonderful", "valuable", "meaningful"}
NEG = {"difficult", "hard", "lack", "lacked", "expensive", "struggle", "struggled",
       "frustrating", "frustrated", "limited", "stress", "stressful", "problem", "problems",
       "challenge", "challenging", "tired", "exhausted", "conflict", "pressure", "never",
       "confusing", "disappointed", "overwhelming", "burnout", "unclear", "lost"}
_df = df if "df" in globals() and df is not None else get_df()
_pid = {}
for _, r in _df.iterrows():
    _pid.setdefault(r["Participant"], r)
val = np.zeros(N)
for i, (_, mr) in enumerate(meta.iterrows()):
    r = _pid.get(mr["Participant"]); q = mr["question"]
    t = "" if r is None else str(r.get(f"bert_{q}", "") or "")
    ws = t.lower().split()
    val[i] = sum(w in POS for w in ws) - sum(w in NEG for w in ws)
by_theme = [val[labels == c] for c in range(K)]
frac_pos = float(np.mean(val[THEMED] > 0)); frac_neg = float(np.mean(val[THEMED] < 0))

print(f"  A. structure(Ward@K={Ks}) real={sil_struct:.3f} vs shuffled-null "
      f"{null.mean():.3f}+/-{null.std():.3f}  ->  z = {z:.1f}")
print(f"     {'GENUINE structure (real >> shuffled)' if z > 3 else 'weak separation; read with caution'}")
print(f"  B. silhouette of the SUB labels: original-embedding={sil_orig:.3f}  vs  "
      f"UMAP-space={sil_umap:.3f}")
print(f"     the MAIN labels, original-embedding: {sil_main:.3f}")
print("     (UMAP can inflate visual separation; the original-space value is the honest one)")
print("     (A's z uses a DIRECT Ward-on-embedding reference: it only proves the embedding")
print("      space has some clusterable structure vs shuffled. B is the pipeline labels' own")
print("      honest separation - weak, as expected for open survey text.)")
print(f"  C. valence, themed rows: {100*frac_pos:.0f}% lean positive, {100*frac_neg:.0f}% negative")
allpos = [c for c in range(K) if len(by_theme[c])
          and np.median(by_theme[c]) > 0 and np.min(by_theme[c]) >= 0]
print(f"     themes that are ALL non-negative (possible positivity bias): "
      f"{len(allpos)}/{K}  -> {'none, themes span sentiment (neutral)' if not allpos else allpos}")
psave("cluster_reality", dict(sil_orig=float(sil_orig), sil_umap=float(sil_umap),
      sil_main_orig=float(sil_main),
      null_mean=float(null.mean()) if len(null) else None, z=float(z),
      frac_pos=frac_pos, frac_neg=frac_neg, K=K, n_themed=int(THEMED.sum()),
      n_noise=int((~THEMED).sum()), noise_rate=float((~THEMED).mean()),
      n_subsample=int(len(m))))

cols = get_cluster_colors(K)
_GREY = "#8A8F98"
fig, ax = plt.subplots(1, 3, figsize=(16, 9), facecolor=NAT_BG, gridspec_kw=dict(wspace=0.30))
def _titles(a, bold, sub):
    a.text(0.0, 1.11, bold, transform=a.transAxes, fontsize=14, fontweight="bold",
           color=NAT_TEXT, ha="left", va="bottom")
    a.text(0.0, 1.015, sub, transform=a.transAxes, fontsize=9.6, color=_GREY,
           ha="left", va="bottom")

# A. Is the structure real?  ->  null strip/box near 0 + real diamond far right + z headline
a = ax[0]
a.boxplot(null, vert=False, positions=[0.20], widths=0.11, patch_artist=True, showfliers=False,
          boxprops=dict(facecolor="#ECEDF1", edgecolor="#B6B9C0"), medianprops=dict(color="#888"),
          whiskerprops=dict(color="#B6B9C0"), capprops=dict(color="#B6B9C0"), zorder=1)
a.scatter(null, rng.normal(0.20, 0.022, len(null)), s=22, color="#C4C7CE", alpha=0.9,
          edgecolors="white", linewidths=0.5, zorder=2)
a.scatter([sil_struct], [0.20], s=200, marker="D", color=HIGHLIGHT, edgecolors="white",
          linewidths=1.6, zorder=5)
a.annotate("real", (sil_struct, 0.20), xytext=(sil_struct, 0.40), ha="center", fontsize=10.6,
           fontweight="bold", color=HIGHLIGHT,
           arrowprops=dict(arrowstyle="-", color=HIGHLIGHT, lw=1.1))
a.text(0.97, 0.84, f"{z:.0f} SD\nbeyond chance", transform=a.transAxes, ha="right", va="top",
       fontsize=14, fontweight="bold", color=HIGHLIGHT, linespacing=0.95)
a.text(0.03, 0.05, "shuffled-embedding nulls (15x)", transform=a.transAxes, fontsize=9, color=_GREY)
a.set_ylim(0.0, 0.55); a.set_yticks([]); a.set_xlim(-0.015, max(sil_struct * 1.3, 0.13))
a.set_xlabel("silhouette (cosine)")
_titles(a, "A.  Is the structure real?", "real clustering vs 15 shuffled-embedding nulls")

# B. How much does UMAP flatter it?  ->  original -> UMAP dumbbell
b = ax[1]
b.plot([sil_orig, sil_umap], [1, 1], color="#D2D3DA", lw=3.5, solid_capstyle="round", zorder=1)
b.scatter([sil_orig], [1], s=260, color="#332288", edgecolors="white", linewidths=1.7, zorder=3)
b.scatter([sil_umap], [1], s=260, color="#CC6677", edgecolors="white", linewidths=1.7, zorder=3)
b.text(sil_orig, 1.18, f"original\n{sil_orig:.2f}", ha="center", va="bottom", fontsize=11.2,
       fontweight="bold", color="#332288")
b.text(sil_umap, 1.18, f"UMAP\n{sil_umap:.2f}", ha="center", va="bottom", fontsize=11.2,
       fontweight="bold", color="#CC6677")
_infl = (sil_umap / sil_orig) if sil_orig > 0 else float("nan")
b.text(0.5, 0.30, f"UMAP makes it look ~{_infl:.0f}x cleaner.\nWe report the original as the honest value.",
       transform=b.transAxes, ha="center", fontsize=10.6, color=NAT_TEXT)
b.set_ylim(0.15, 1.6); b.set_yticks([]); b.set_xlim(0, max(sil_umap * 1.25, 0.6))
b.set_xlabel("silhouette of the same labels (themed rows)")
_titles(b, "B.  How much does UMAP flatter it?", "separation in the real space vs the UMAP picture")

# C. Topical, or just mood?  ->  per-theme MEAN valence, sorted (readable at any K)
cx = ax[2]
_mus = np.array([float(np.mean(v)) if len(v) else 0.0 for v in by_theme])
_o = np.argsort(_mus)
cx.axhline(0, color="#BBB", lw=1, zorder=1)
cx.scatter(range(K), _mus[_o], s=30,
           c=[_SENT_POS if mm > 0 else _SENT_NEG for mm in _mus[_o]],
           edgecolors="white", lw=0.6, zorder=3)
cx.set_xticks([]); cx.set_xlabel(f"{K} sub-themes, sorted by mean valence")
cx.set_ylabel("theme MEAN valence  (positive - negative words)")
_span = float(np.mean([(np.min(v) < 0 < np.max(v)) if len(v) else True for v in by_theme]))
_titles(cx, "C.  Topical, or just mood?",
        f"{_span:.0%} of themes contain BOTH signs; means hug 0 (dot = theme mean)")

for a_ in ax:
    a_.set_facecolor("white")
    for sp in ("top", "right"):
        a_.spines[sp].set_visible(False)
fig.text(0.5, -0.04,
         f"Permutation null (Tibshirani 2001) · original vs UMAP silhouette · valence balance per "
         f"theme.  Descriptive anti-bias check, computed on the {int(THEMED.sum())} responses that "
         f"have a sub-theme ({(~THEMED).mean():.0%} left as noise are excluded - noise is not a "
         f"cluster).", ha="center", fontsize=9.3, style="italic", color=_GREY)
plt.tight_layout()
save_fig(fig, "chart_cluster_reality")
plt.close(fig)

# (round 3) the interactive Plotly valence figure duplicated panel C exactly
# (same data, same message) - one figure per message; the static panel stays.
print("=" * 66)


## Stage 29 — Convergent validation: is the structure method-independent?

**This stage used to ask the question backwards.** It was written when Ward was the primary partition and HDBSCAN was the outside opinion; since Stage 21H, HDBSCAN *is* the partition, so the old cell compared HDBSCAN against HDBSCAN and called the agreement convergent evidence. It is inverted here.

Our themes come from a density method that chooses its own $K$ and may leave a response in no theme. Would a method built on completely different assumptions see the same groupings? Holding $K$ fixed at the HDBSCAN $K$, we run **Ward** (variance-minimising, assigns everything), **average** and **complete** linkage (same family, different merge rule), and **k-means** (centroid-based, spherical). Ward and k-means are the genuinely different paradigms; the linkages measure how much the answer depends on one merge rule.

Agreement is measured **only on the rows HDBSCAN gave a theme**. Every comparator assigns every response and HDBSCAN does not, so scoring all rows would charge HDBSCAN for the responses it honestly declined to place — a difference of contract, not of structure. The excluded fraction is reported next to each number. AMI is the headline (chance-adjusted, robust to the very uneven cluster sizes here); moderate agreement is convergent evidence, near-zero would mean the themes are an artifact of density clustering.


In [ ]:
# @title
# ==============================================================================
# STAGE 29  -  CONVERGENT VALIDATION  (is the structure method-independent?)
# ------------------------------------------------------------------------------
# THIS CELL USED TO ASK THE QUESTION BACKWARDS. It was written when Ward was the
# primary partition and HDBSCAN was the outside opinion. Since Stage 21H, HDBSCAN
# IS the partition - so the old cell compared HDBSCAN against HDBSCAN and called
# the agreement convergent evidence. It was circular, and it is inverted here.
#
# THE QUESTION: our themes come from a density method that chooses its own K and
# is allowed to leave a response in no theme. Would a method that works on
# completely different assumptions see the same groupings? We hold K fixed at the
# HDBSCAN K and run:
#   * Ward linkage        - variance-minimising, agglomerative, assigns everything
#   * average / complete  - same family, different merge rule (linkage robustness)
#   * k-means             - centroid/partitional, spherical assumption
# Ward and k-means are the two genuinely different paradigms; the other two
# measure how much the answer depends on one merge rule.
#
# NOISE, AND WHY IT DECIDES THE COMPARISON. Every comparator assigns EVERY
# response; HDBSCAN does not. Scoring them on all rows would charge HDBSCAN for
# the responses it honestly declined to place - it would look like disagreement
# when it is a difference of contract. Agreement is therefore measured only on
# the rows HDBSCAN gave a theme, and the excluded fraction is reported next to
# every number, so the reader sees what the comparison is and is not about.
#
# Deterministic apart from k-means, which is seeded. Refs: Ward (1963);
# Hubert & Arabie (1985) ARI; Vinh et al. (2010) AMI; Campello et al. (2013).
# ==============================================================================
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score
from sklearn.cluster import KMeans
from scipy.cluster.hierarchy import linkage, fcluster

# ---- stale-artifact guard (the same block opens every stage below) ------------
def _labels(n_rows, stage, levels=("main", "sub")):
    _KEY = {"main": "pooled_labels_main7", "sub": "pooled_labels_sub"}
    out = {}
    for lv in levels:
        a = pload(_KEY[lv])
        if a is None:
            raise RuntimeError(f"[{stage}] {_KEY[lv]} is missing - run Stage 23H first.")
        a = np.asarray(a)
        if len(a) != n_rows:
            raise RuntimeError(
                f"[{stage}] {_KEY[lv]} has {len(a)} rows but the pooled set has "
                f"{n_rows}. That artifact belongs to an EARLIER clustering. Re-run "
                f"Stage 23H (then Stage 30) before this stage.")
        out[lv] = a
    return out

def _K(lab):
    lab = np.asarray(lab)
    return int(lab.max()) + 1 if (lab >= 0).any() else 0
# ------------------------------------------------------------------------------

_x = pload("pooled_umap_nd")
if _x is None:
    _x = pload("pooled_umap5d")
if _x is None:
    raise RuntimeError("[Stage 29] pooled_umap_nd missing - run Stage 21H first.")
X = np.asarray(_x, dtype=float)
LAB = _labels(len(X), "Stage 29")
SEED = int(globals().get("SEED", CFG.seed))

# The comparator partitions depend only on X and K, so each is computed ONCE and
# reused for whichever level asks for that K.
_CACHE = {}
def _comparators(K):
    if K in _CACHE:
        return _CACHE[K]
    out = {}
    for meth in ("ward", "average", "complete"):
        out[f"{meth} linkage"] = fcluster(linkage(X, method=meth), K,
                                          criterion="maxclust") - 1
    out["k-means"] = KMeans(n_clusters=K, n_init=10, random_state=SEED).fit_predict(X)
    _CACHE[K] = out
    return out

print("=" * 78)
print("  STAGE 29  CONVERGENT VALIDATION   (HDBSCAN vs other paradigms)")
print("=" * 78)

res = {}
for lv in ("main", "sub"):
    lab = LAB[lv]
    K = _K(lab)
    if K < 2:
        print(f"  {lv}: fewer than 2 themes - nothing to compare.")
        continue
    themed = lab >= 0
    print(f"\n  {lv.upper():<5} HDBSCAN K={K}, agreement measured on the "
          f"{int(themed.sum())} themed rows "
          f"({int((~themed).sum())} noise rows excluded, {(~themed).mean():.0%})")
    print(f"  {'comparator':<18} {'ARI':>7} {'AMI':>7}   {'K':>4}")
    for name, la in _comparators(K).items():
        r = dict(level=lv,
                 ari=float(adjusted_rand_score(lab[themed], la[themed])),
                 ami=float(adjusted_mutual_info_score(lab[themed], la[themed])),
                 k=int(la.max()) + 1,
                 noise_excluded=float((~themed).mean()),
                 n_used=int(themed.sum()))
        res[f"{lv}: {name}"] = r
        print(f"  {name:<18} {r['ari']:>7.3f} {r['ami']:>7.3f}   {r['k']:>4}")

psave("pooled_convergent", res)
print("\n  reading: AMI is the fair headline (ARI is harsher when cluster SIZES differ,")
print("  and HDBSCAN's size profile is very uneven). Moderate agreement (>~0.3) from a")
print("  different paradigm = convergent structure; near-zero would mean the themes are")
print("  an artifact of density clustering. K is held fixed for the comparators, so this")
print("  tests the SHAPE of the partition, not the choice of K (that is Stage 21H).")

# ------------------------------- figure ---------------------------------------
names = list(res.keys())
if names:
    fig, ax = plt.subplots(figsize=(11.5, 0.62 * len(names) + 3.2), facecolor=NAT_BG)
    yv = np.arange(len(names))[::-1]
    h = 0.34
    for r_, name in zip(yv, names):
        r = res[name]
        ax.barh(r_ + h / 2 + 0.02, r["ami"], height=h, color=C_GOOD, edgecolor="white", zorder=3)
        ax.barh(r_ - h / 2 - 0.02, r["ari"], height=h, color="#6699CC", edgecolor="white", zorder=3)
        ax.text(r["ami"] + 0.012, r_ + h / 2 + 0.02, f'AMI {r["ami"]:.2f}', va="center",
                fontsize=9, color="#1D6152")
        ax.text(r["ari"] + 0.012, r_ - h / 2 - 0.02, f'ARI {r["ari"]:.2f}', va="center",
                fontsize=9, color="#3D5E86")
        ax.text(-0.02, r_, f'K={r["k"]} · n={r["n_used"]}', ha="right", va="center",
                fontsize=8, color="#888")
    ax.axvline(0.3, color="#999", ls=":", lw=1.2)
    ax.text(0.305, len(names) - 0.45, "convergent (>0.3)", fontsize=8, color="#888")
    ax.set_yticks(yv); ax.set_yticklabels(names, fontsize=10)
    ax.set_xlim(-0.30, 1.0); ax.set_xlabel("agreement with the HDBSCAN partition")
    ax.tick_params(length=0)
    for sp in ("top", "right", "left"):
        ax.spines[sp].set_visible(False)
    ax.xaxis.grid(True, color="#F0F0F0"); ax.set_axisbelow(True)
    ax.set_title("Convergent validation: HDBSCAN structure recovered by other paradigms",
                 fontsize=13, fontweight="bold", loc="left", color=NAT_TEXT, pad=12)
    _nz = ", ".join(f"{lv} {LAB[lv].tolist().count(-1) / len(LAB[lv]):.0%}"
                    for lv in ("main", "sub"))
    fig.text(0.5, -0.02,
             f"Comparators are cut at the HDBSCAN K and assign every response; agreement is "
             f"scored only on the rows HDBSCAN gave a theme (noise excluded: {_nz}). AMI is the "
             f"headline (chance-adjusted, robust to uneven cluster sizes). "
             f"Refs: Ward 1963; Vinh 2010; Campello 2013.",
             ha="center", fontsize=8.4, style="italic", color="#888")
    plt.tight_layout()
    save_fig(fig, "chart_stage29_convergent")
    plt.close(fig)
print("=" * 78)


## Stage 30 — Evidence-grounded theme naming (main / sub)

Two levels, not three — the mid level was removed in Stage 23H. And **noise is never named**: the responses HDBSCAN left with no theme (`-1`) are not members of any cluster, so the model is never shown them as a group. Naming that group would be the worst thing this stage could do, since its only shared property is belonging nowhere.

The labelling model reads the **actual responses** — MMR-diversified representatives plus the most atypical members, in raw text — rather than being handed keywords alone. Keywords are a supporting signal computed as a **log-odds ratio with an informative Dirichlet prior** (Monroe et al. 2008), which shrinks toward the corpus background and so does not inflate rare words in small clusters the way c-TF-IDF does.

Every label must then survive four independent checks: the model may **abstain** if the responses share no topic; it must **cite** the quotes supporting the name and those quotes are checked programmatically; the name is sampled several times and the **agreement** is recorded; a **held-out judge** sees members that were not used for naming; and finally each name is **embedded** and must sit closer to its own cluster than to any other. **Labels that fail are withheld, never invented** — the theme keeps a neutral placeholder and is flagged, so downstream figures can exclude it from interpretive claims. Full per-theme scores go to `theme_naming_audit.csv`.

This stage is also the source of the per-theme term lists (`pooled_main_terms` / `pooled_sub_terms`), the plain name lookups (`pooled_main_name` / `pooled_sub_name`) and the sizes. They used to come from the deleted Stage 23c, so on a clean machine Stage 30v silently repaired labels with an empty keyword list in front of the model.


In [ ]:
# @title
# ==============================================================================
# STAGE 30  -  EVIDENCE-GROUNDED THEME NAMING  (main / sub, one model load)
# ------------------------------------------------------------------------------
# TWO LEVELS, NOT THREE. The mid level was removed in Stage 23H, so this stage
# names MAIN (the communication layer) and SUB (the analytical unit) only. One
# fewer level is one fewer set of LLM calls, one fewer family of names to keep
# consistent, and one fewer multiple-testing burden downstream.
#
# NOISE IS NOT A THEME AND IS NEVER NAMED. HDBSCAN leaves some responses with no
# theme (label -1). Naming that group would be the single worst thing this stage
# could do: its only shared property is that its members belong nowhere, so the
# model would be handed an incoherent set and asked for a topic - exactly the
# failure the abstention rule exists to prevent. Cluster ids run 0..K-1 and the
# -1 rows are simply not members of any of them. Their count is printed.
#
# WHY THIS REPLACES THE PREVIOUS NAMING STAGE
# The old stage fed the model c-TF-IDF terms plus centroid-nearest quotes taken
# from the CLEANED text, and accepted whatever came back. Two consequences were
# measured in the audit: (i) on small clusters c-TF-IDF makes a word occurring
# 2-3 times look highly distinctive, and (ii) with no abstention and no
# verification, the model produced confident names and summaries that the member
# responses did not support.
#
# WHAT THIS STAGE DOES INSTEAD - the model reads the ACTUAL RESPONSES, and every
# label must survive four independent checks before it is accepted:
#   EVIDENCE   MMR-diversified representatives (central AND spread out) plus the
#              farthest members, in RAW text, so the model sees what is really in
#              the cluster - not nine near-identical medoids. Keywords are a
#              SUPPORTING signal only, computed as a log-odds ratio with an
#              informative Dirichlet prior (Monroe, Colaresi & Quinn 2008), which
#              shrinks toward the corpus background and therefore does not inflate
#              rare words in small clusters the way c-TF-IDF does.
#   ABSTENTION the prompt carries the cluster's own statistics (n, median length,
#              question mix) and an explicit escape hatch: if the responses share
#              no topic the model must return insufficient_evidence.
#   CITATION   the model must cite the numbered quotes that support the name; we
#              then check programmatically that those quotes exist and that the
#              name's content words actually occur in them.
#   AGREEMENT  the name is sampled K_SELF times; the modal name is kept and the
#              agreement rate is recorded as a reliability score.
#   JUDGE      a second pass shows members that were NOT used for naming and asks
#              whether the name describes them (held-out validation).
#   EMBEDDING  after the LLM is freed, each accepted name is embedded and must be
#              closer to its own cluster's responses than to any other cluster's
#              (a discriminativeness check that needs no human).
# A label that fails is WITHHELD, not invented: the theme keeps a neutral
# descriptive placeholder and is flagged so downstream figures can exclude it.
#
# THIS STAGE IS ALSO THE SOURCE OF THE PER-THEME TERM LISTS. Stage 30v needs each
# theme's own distinctive words to write a repair prompt, and used to read
# pooled_*_terms - an artifact of the deleted Stage 23c that nothing writes any
# more, so it silently fell back to an empty dict and repaired labels with no
# evidence in front of it. The log-odds keywords computed here are saved as
# pooled_main_terms / pooled_sub_terms, and the accepted names as
# pooled_main_name / pooled_sub_name, so no downstream stage depends on a dead
# artifact.
# Refs: Monroe et al. (2008); Grootendorst (2022); Wang et al. (2023) self-consistency.
# ==============================================================================
import os, re, json, gc, math
import numpy as np
import pandas as pd

# ------------------------------- EDIT HERE ------------------------------------
MODEL          = globals().get("LLM_MODEL_NAME", "Qwen/Qwen2.5-14B-Instruct")
ENABLE         = bool(globals().get("LLM_ENABLE", True))
FORCE          = bool(globals().get("LLM_FORCE_REGEN", False))
N_REP          = 14      # MMR-diversified representative responses shown
N_FAR          = 3       # farthest members shown, so heterogeneity is visible
N_JUDGE        = 8       # held-out members used by the judge
N_KEYWORDS     = 12      # supporting log-odds keywords
K_SELF         = 3       # self-consistency samples (1 = greedy only)
TEMP           = 0.7     # sampling temperature for the self-consistency samples
GROUND_MIN     = 0.25    # HARD floor: below this the label is withheld
GROUND_WEAK    = 0.40    # soft: below this it is accepted but flagged 'weak grounding'
EMB_MARGIN_MIN = -0.10   # HARD floor for the embedding check. A near-tie (e.g. -0.01)
                         # between two ADJACENT sub-themes is expected when dozens of
                         # themes are carved from one narrative space, and is not
                         # evidence the name is wrong; only a clear loss is.
EMB_MARGIN_WEAK = 0.0    # soft: between MIN and 0 the label is flagged 'ambiguous'
# The summary is held to the same evidence standard as the name. It is the text
# that reaches codebook.csv and every hover card downstream, so an ungrounded
# sentence there is read far more often than a bad label.
SUMMARY_MIN    = 0.30    # HARD: below this the summary is replaced, not published
SUMMARY_WEAK   = 0.50    # soft: accepted but flagged 'weak summary grounding'
# A key point is one short bullet, so it has very few content words and a SHARE
# floor is correspondingly weak on it: content_words() keeps only words longer
# than three letters, so "the team won a gold medal" reduces to three words and a
# single incidental match on "team" would clear 0.30 with nothing behind it.
# Key points therefore need at least half their content words in the evidence.
KEYPOINT_MIN   = 0.50
MMR_LAMBDA     = 0.6     # 1.0 = pure centrality, 0.0 = pure diversity
DIRICHLET_A0   = 500.0   # strength of the informative background prior
# ----------------------------- END EDIT ---------------------------------------
PROMPT_VERSION = "v6-two-level"   # bump whenever a THRESHOLD or a prompt changes -
                                  # the cache is keyed on this string, so a threshold
                                  # edit without a bump silently keeps the old verdicts.
                                  # v5 -> v6: the mid level is gone.

# ---- stale-artifact guard (the same block opens every stage below) ------------
def _labels(n_rows, stage, levels=("main", "sub")):
    _KEY = {"main": "pooled_labels_main7", "sub": "pooled_labels_sub"}
    out = {}
    for lv in levels:
        a = pload(_KEY[lv])
        if a is None:
            raise RuntimeError(f"[{stage}] {_KEY[lv]} is missing - run Stage 23H first.")
        a = np.asarray(a)
        if len(a) != n_rows:
            raise RuntimeError(
                f"[{stage}] {_KEY[lv]} has {len(a)} rows but the pooled set has "
                f"{n_rows}. That artifact belongs to an EARLIER clustering. Re-run "
                f"Stage 23H (then Stage 30) before this stage.")
        out[lv] = a
    return out

def _K(lab):
    lab = np.asarray(lab)
    return int(lab.max()) + 1 if (lab >= 0).any() else 0
# ------------------------------------------------------------------------------

meta_pool = require("pooled_meta", stage="Stage 30 (naming)").reset_index(drop=True)
E_pool    = np.asarray(require("pooled_embeddings", stage="Stage 30 (naming)"),
                       dtype=np.float32)
N = len(meta_pool)
if len(E_pool) != N:
    raise RuntimeError(f"[Stage 30] pooled_embeddings has {len(E_pool)} rows but "
                       f"pooled_meta has {N}.")
_LB      = _labels(N, "Stage 30 (naming)")
lab_main = _LB["main"]
lab_sub  = _LB["sub"]
_df = df if "df" in globals() and df is not None else get_df()
if _df is None:
    raise RuntimeError("Need df_final/df_cleaned for the raw response text.")

# ---------------- RAW response text, aligned to the pooled rows ---------------
RAWCOL = dict(zip(CFG.short_labels, CFG.text_cols))
_pid_row = {}
for _, r in _df.iterrows():
    _pid_row.setdefault(r["Participant"], r)
texts = []
for _, mr in meta_pool.iterrows():
    r = _pid_row.get(mr["Participant"])
    col = RAWCOL.get(mr["question"])
    v = "" if (r is None or col is None) else r.get(col, "")
    v = "" if (v is None or (isinstance(v, float) and np.isnan(v))) else str(v)
    texts.append("" if v.strip().lower() == "nan" else " ".join(v.split()))
texts = np.array(texts, dtype=object)
_n_empty = int((texts == "").sum())
if _n_empty:
    print(f"  note: {_n_empty} pooled row(s) had no raw text and will not be shown as evidence.")
nwords = np.array([len(t.split()) for t in texts])

# ---------------- log-odds ratio with informative Dirichlet prior -------------
from sklearn.feature_extraction.text import CountVectorizer
_cv = CountVectorizer(stop_words="english", ngram_range=(1, 2), min_df=3, max_features=8000)
try:
    _M = _cv.fit_transform([t.lower() for t in texts])
    _VOCAB = np.array(_cv.get_feature_names_out())
    _COUNTS = np.asarray(_M.sum(0)).ravel().astype(float)
    _TOTAL = float(_COUNTS.sum())
    _ALPHA = DIRICHLET_A0 * (_COUNTS / max(_TOTAL, 1.0))     # informative prior
    _A0 = float(_ALPHA.sum())
    _HAVE_LO = True
except ValueError:
    _HAVE_LO = False
    print("  (vocabulary too small for log-odds keywords; keywords disabled)")

def logodds_keywords(idx, top=N_KEYWORDS):
    """Monroe et al. (2008) z-scored log-odds vs the rest of the corpus."""
    if not _HAVE_LO or len(idx) == 0:
        return []
    yk = np.asarray(_M[idx].sum(0)).ravel().astype(float)
    nk = yk.sum()
    yr = _COUNTS - yk
    nr = _TOTAL - nk
    if nk <= 0 or nr <= 0:
        return []
    lk = np.log((yk + _ALPHA) / (nk + _A0 - yk - _ALPHA))
    lr = np.log((yr + _ALPHA) / (nr + _A0 - yr - _ALPHA))
    var = 1.0 / (yk + _ALPHA) + 1.0 / (yr + _ALPHA)
    z = (lk - lr) / np.sqrt(var)
    z[yk < 2] = -np.inf                     # a word must occur at least twice here
    order = np.argsort(-z)[:top]
    return [(_VOCAB[i], float(z[i])) for i in order if np.isfinite(z[i])]

# ---------------- MMR evidence selection --------------------------------------
def mmr_select(idx, k, lam=MMR_LAMBDA):
    """Representative AND diverse: central members that are not near-duplicates."""
    idx = np.asarray([i for i in idx if texts[i]])
    if len(idx) == 0:
        return []
    if len(idx) <= k:
        return list(idx)
    V = E_pool[idx]
    cen = V.mean(0)
    cen = cen / (np.linalg.norm(cen) + 1e-9)
    rel = V @ cen
    sel = [int(np.argmax(rel))]
    while len(sel) < k:
        S = V[sel]
        div = (V @ S.T).max(1)
        score = lam * rel - (1 - lam) * div
        score[sel] = -np.inf
        sel.append(int(np.argmax(score)))
    return [int(idx[i]) for i in sel]

def farthest_select(idx, k):
    idx = np.asarray([i for i in idx if texts[i]])
    if len(idx) == 0:
        return []
    V = E_pool[idx]
    cen = V.mean(0); cen = cen / (np.linalg.norm(cen) + 1e-9)
    order = np.argsort(V @ cen)[:k]
    return [int(idx[i]) for i in order]

STOP = {"and", "the", "of", "in", "to", "for", "with", "through", "a", "an", "by",
        "from", "their", "about", "into", "that", "this", "it", "on", "at", "as",
        "very", "more", "less", "new", "own", "its", "is", "are", "was", "were"}

def content_words(name):
    return [w for w in re.findall(r"[A-Za-z]+", str(name).lower())
            if w not in STOP and len(w) > 3]

def grounding(name, quotes, keywords):
    """Share of the name's content words that actually occur in the cited quotes
    or the supporting keywords. Prefix match to tolerate simple inflection."""
    kw = content_words(name)
    if not kw:
        return 0.0
    blob = " ".join(quotes).lower() + " " + " ".join(k for k, _ in keywords).lower()
    return sum(1 for w in kw if w[:5] in blob) / len(kw)

# ---------------- prompts ------------------------------------------------------
SYS = ("You are a meticulous qualitative-coding assistant for a study of the iGEM "
       "synthetic-biology student competition. You will be shown a set of survey "
       "responses that an algorithm grouped together, plus statistics about that "
       "group and keywords that distinguish it from the rest of the corpus.\n\n"
       "Your job is to decide whether these responses share a genuine topic, and "
       "if so to name it.\n\n"
       "CRITICAL: the grouping is algorithmic and can be wrong. If the responses "
       "do NOT share a recognisable topic - for example if they are simply short, "
       "or are a mix of unrelated remarks - you MUST return "
       "\"insufficient_evidence\": true and give no name. Returning a plausible "
       "name for an incoherent group is the single worst error you can make.\n\n"
       "If they DO share a topic, give a short Title-Case NAME of 3 to 6 words "
       "that states the TOPIC and makes the emotional TONE unmistakable: use a "
       "plainly positive word when the responses lean positive, name the "
       "difficulty plainly when they lean negative, show both sides when mixed, "
       "and use a plain topic name when neutral. Never use vague tone-hiding "
       "words such as dynamics, activities, experiences, outcomes, engagement, "
       "aspects or journey on their own. Ground every word in the responses "
       "shown; invent nothing. Cite the numbers of the responses that support "
       "your name.")

def build_prompt(stats, quotes, far, keywords):
    qb = "\n".join(f'[{i+1}] "{q}"' for i, q in enumerate(quotes))
    fb = "\n".join(f'[{len(quotes)+i+1}] "{q}"' for i, q in enumerate(far))
    kw = ", ".join(f"{w}" for w, _ in keywords) or "(none)"
    return (f"GROUP STATISTICS\n"
            f"  responses in group : {stats['n']}\n"
            f"  median length      : {stats['median_words']} words\n"
            f"  length range       : {stats['min_words']}-{stats['max_words']} words\n"
            f"  survey questions   : {stats['question_mix']}\n\n"
            f"DISTINGUISHING KEYWORDS (log-odds vs the rest of the corpus)\n  {kw}\n\n"
            f"REPRESENTATIVE RESPONSES\n{qb}\n\n"
            f"MOST ATYPICAL MEMBERS OF THE SAME GROUP\n{fb or '(none)'}\n\n"
            "Step 1. Read every response. Do they share a recognisable topic? "
            "Short, generic or unrelated remarks do NOT count as a topic.\n"
            "Step 2. If they do not, return exactly: "
            '{"insufficient_evidence": true, "reason": "<one short sentence>"}\n'
            "Step 3. If they do, decide the overall sentiment "
            "(positive | negative | mixed | neutral), then write the NAME, and "
            "cite the response numbers that support it.\n"
            "Return ONLY this JSON object:\n"
            '{"insufficient_evidence": false, "sentiment": "...", "name": "...", '
            '"summary": "1-2 even-handed sentences", "key_points": ["...","..."], '
            '"evidence": [1, 3, 7]}')

JUDGE_SYS = ("You validate qualitative codes. You are given a theme name and a set "
             "of survey responses that were NOT used to create that name. Decide "
             "honestly whether the name describes them. Return ONLY: "
             '{"verdict": "yes|partial|no", "reason": "<one short sentence>"}')

def build_judge(name, quotes):
    qb = "\n".join(f'- "{q}"' for q in quotes)
    return (f'THEME NAME: "{name}"\n\nRESPONSES (not used to create the name):\n{qb}\n\n'
            "Does the name describe these responses?")

def parse_json(raw):
    if not raw:
        return None
    m = re.search(r"\{.*\}", raw, re.S)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except Exception:
        try:
            return json.loads(re.sub(r",\s*\}", "}", m.group(0).replace("'", '"')))
        except Exception:
            return None

def clean_name(nm):
    nm = re.sub(r"\s+", " ", str(nm)).strip().strip('."')
    if not nm or len(nm) > 70 or nm.lower() in ("cluster", "theme", "n/a", "none", "unknown"):
        return None
    return nm

# ---------------- levels -------------------------------------------------------
LEVELS = [("main", lab_main, "pooled_main_llm", "M"),
          ("sub",  lab_sub,  "pooled_sub_llm",  "S")]
KOF = {lv: _K(lab) for lv, lab, _, _ in LEVELS}
for lv, lab, _, _ in LEVELS:
    if KOF[lv] < 1:
        raise RuntimeError(f"[Stage 30] the {lv} level has no themes at all - "
                           f"every response is noise. Re-run Stage 21H / 23H.")

caches = {lv: pload(art) for lv, _, art, _ in LEVELS}

def stale(cache, k):
    """A cache is usable only if it describes THIS partition: the same number of
    clusters AND exactly the ids 0..k-1. len() alone let a cache from a different
    clustering through whenever the two happened to have the same K."""
    if FORCE or not cache:
        return True
    try:
        ids = {int(x) for x in cache.keys()}
    except Exception:
        return True
    if ids != set(range(k)):
        return True
    return any((v or {}).get("prompt_version") != PROMPT_VERSION for v in cache.values())

need = {lv: stale(caches[lv], KOF[lv]) for lv, _, _, _ in LEVELS}

print("=" * 78)
print(f"  STAGE 30  EVIDENCE-GROUNDED NAMING   model={MODEL}  enabled={ENABLE}")
print(f"  levels: " + "  ".join(f"{lv}={KOF[lv]}" for lv, _, _, _ in LEVELS)
      + "   (the mid level was removed in Stage 23H)")
for lv, lab, _, _ in LEVELS:
    _nz = int((lab < 0).sum())
    print(f"    {lv:<5} {int((lab >= 0).sum())} responses in {KOF[lv]} themes; "
          f"{_nz} with no theme ({_nz / N:.1%}) - noise is never named")
print(f"  to (re)name: " + ", ".join(f"{lv}={need[lv]}" for lv in need))
print("=" * 78)

gen = None
if any(need.values()) and ENABLE:
    try:
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
        print(f"  loading {MODEL} in 4-bit ...")
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                 bnb_4bit_compute_dtype=torch.bfloat16,
                                 bnb_4bit_use_double_quant=True)
        tok = AutoTokenizer.from_pretrained(MODEL)
        mdl = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb,
                                                   device_map="auto", torch_dtype=torch.bfloat16)
        mdl.eval()
        def gen(sys_p, usr_p, temperature=0.0, max_new=320):
            msgs = [{"role": "system", "content": sys_p}, {"role": "user", "content": usr_p}]
            prompt = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
            enc = tok(prompt, return_tensors="pt")
            dev = next(mdl.parameters()).device
            enc = {k: v.to(dev) for k, v in enc.items()}
            kw = dict(max_new_tokens=max_new, pad_token_id=(tok.eos_token_id or tok.pad_token_id))
            if temperature and temperature > 0:
                kw.update(do_sample=True, temperature=float(temperature), top_p=0.9)
            else:
                kw.update(do_sample=False)
            with torch.no_grad():
                out = mdl.generate(**enc, **kw)
            return tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
        print("  model ready.")
    except Exception as e:
        print(f"  !! LLM unavailable ({type(e).__name__}: {str(e)[:110]}) - "
              f"labels will be WITHHELD, not invented.")
        gen = None

# ---------------- name one cluster --------------------------------------------
def name_cluster(idx, tag):
    stats = dict(n=len(idx),
                 median_words=int(np.median(nwords[idx])) if len(idx) else 0,
                 min_words=int(nwords[idx].min()) if len(idx) else 0,
                 max_words=int(nwords[idx].max()) if len(idx) else 0,
                 question_mix=", ".join(
                     f"{q} {int((meta_pool['question'].values[idx] == q).sum())}"
                     for q in CFG.short_labels
                     if (meta_pool["question"].values[idx] == q).any()))
    keywords = logodds_keywords(idx)
    rep_i = mmr_select(idx, N_REP)
    far_i = [i for i in farthest_select(idx, N_FAR) if i not in rep_i]
    used = set(rep_i) | set(far_i)
    hold_i = [i for i in idx if i not in used and texts[i]][:N_JUDGE]
    rep = [texts[i][:300] for i in rep_i]
    far = [texts[i][:300] for i in far_i]

    rec = dict(prompt_version=PROMPT_VERSION, n=len(idx), n_evidence=len(rep) + len(far),
               keywords=[w for w, _ in keywords], stats=stats,
               agreement=np.nan, grounding=np.nan, judge="", judge_reason="",
               emb_margin=np.nan, status="withheld", withheld_reason="", source="none",
               name="", llm_name="", summary="", key_points=[], sentiment="",
               evidence=[], flags=[])
    if gen is None:
        rec["withheld_reason"] = "no model available"
        return rec
    if not rep:
        rec["withheld_reason"] = "no member had usable raw text"
        return rec

    usr = build_prompt(stats, rep, far, keywords)
    samples = []
    for s in range(max(1, K_SELF)):
        d = parse_json(gen(SYS, usr, temperature=0.0 if s == 0 else TEMP))
        if d is not None:
            samples.append(d)
    if not samples:
        rec["withheld_reason"] = "model returned no parsable JSON"
        return rec

    # abstention wins if the model abstained at greedy or majority
    absts = sum(1 for d in samples if d.get("insufficient_evidence") is True)
    if absts * 2 >= len(samples):
        rec["status"] = "withheld"
        rec["withheld_reason"] = "model abstained: " + str(
            next((d.get("reason", "") for d in samples
                  if d.get("insufficient_evidence")), ""))[:110]
        rec["source"] = "llm"
        return rec

    named = [d for d in samples if not d.get("insufficient_evidence")
             and clean_name(d.get("name"))]
    if not named:
        rec["withheld_reason"] = "no valid name in any sample"
        rec["source"] = "llm"
        return rec
    keyf = lambda d: clean_name(d.get("name")).lower()
    modal = max({keyf(d) for d in named}, key=lambda k: sum(1 for d in named if keyf(d) == k))
    best = next(d for d in named if keyf(d) == modal)
    rec["agreement"] = sum(1 for d in named if keyf(d) == modal) / len(samples)
    rec["source"] = "llm"
    rec["name"] = clean_name(best.get("name"))
    rec["llm_name"] = rec["name"]      # preserved even if the label is withheld,
                                       # so thresholds can be re-applied without
                                       # paying for another LLM run
    rec["summary"] = re.sub(r"\s+", " ", str(best.get("summary", ""))).strip()
    kp = best.get("key_points") or []
    rec["key_points"] = [str(x).strip() for x in (kp if isinstance(kp, list) else [kp])][:3]
    rec["sentiment"] = str(best.get("sentiment", "")).strip().lower()

    # citation check
    ev = best.get("evidence") or []
    ev = [int(x) for x in ev if str(x).strip().lstrip("-").isdigit()]
    all_q = rep + far
    cited = [all_q[i - 1] for i in ev if 1 <= i <= len(all_q)]
    rec["evidence"] = ev
    if not cited:
        rec["withheld_reason"] = "no valid quote citation"
        return rec
    rec["grounding"] = grounding(rec["name"], cited, keywords)
    if rec["grounding"] < GROUND_MIN:
        rec["withheld_reason"] = f"name not grounded in cited quotes ({rec['grounding']:.2f})"
        return rec
    if rec["grounding"] < GROUND_WEAK:
        rec["flags"].append(f"weak grounding {rec['grounding']:.2f}")

    # held-out judge
    if hold_i:
        jd = parse_json(gen(JUDGE_SYS, build_judge(rec["name"], [texts[i][:260] for i in hold_i]),
                            temperature=0.0, max_new=120))
        rec["judge"] = str((jd or {}).get("verdict", "")).strip().lower()
        rec["judge_reason"] = str((jd or {}).get("reason", ""))[:120]
        if rec["judge"] == "no":
            rec["withheld_reason"] = "held-out judge rejected the name"
            return rec
    # ---- the SUMMARY and KEY POINTS are verified too --------------------------
    _shown = all_q                      # every response the model actually saw
    rec["llm_summary"] = rec["summary"]
    rec["summary_grounding"] = grounding(rec["summary"], cited + _shown, keywords)
    # A stated figure is invention by construction: the model is shown a sample of
    # the cluster and is never told the denominator, so it cannot know a count or
    # a percentage. Refuse those outright rather than scoring them.
    _nums = re.findall(r"\b\d+(?:\.\d+)?\s*%?", str(rec["summary"]))
    if _nums:
        rec["flags"].append("summary states figures the model could not know: "
                            + ", ".join(_nums[:3]))
    if rec["summary_grounding"] < SUMMARY_MIN or _nums:
        rec["summary"] = (
            "Summary withheld: the model's wording was not grounded in the "
            "responses it was shown"
            + (" and stated figures it had no way to know" if _nums else "")
            + f" (grounding {rec['summary_grounding']:.2f}). The theme's own top "
              f"terms are: " + ", ".join(w for w, _ in keywords[:6]) + ".")
        rec["flags"].append(f"summary withheld ({rec['summary_grounding']:.2f})")
    elif rec["summary_grounding"] < SUMMARY_WEAK:
        rec["flags"].append(f"weak summary grounding {rec['summary_grounding']:.2f}")
    # key points individually, so one invented bullet cannot ride along with two
    # good ones
    _kp_keep = [p for p in rec["key_points"]
                if grounding(p, cited + _shown, keywords) >= KEYPOINT_MIN
                and not re.search(r"\b\d", str(p))]
    if len(_kp_keep) != len(rec["key_points"]):
        rec["flags"].append(f"{len(rec['key_points']) - len(_kp_keep)} key point(s) "
                            f"dropped as ungrounded")
    rec["key_points"] = _kp_keep

    rec["status"] = "accepted"
    return rec

# ---------------- run both levels ---------------------------------------------
named = {}
for lv, lab, art, tag in LEVELS:
    K = KOF[lv]
    if not need[lv]:
        named[lv] = {int(k): v for k, v in caches[lv].items()}
        print(f"  {lv:<4} loaded from cache ({K} clusters)")
        continue
    out = {}
    for c in range(K):
        idx = np.where(lab == c)[0]          # -1 never matches, so noise is excluded
        if not len(idx):
            raise RuntimeError(f"[Stage 30] {lv} theme {c} has no members - the label "
                               f"array and its K disagree. Re-run Stage 23H.")
        out[c] = name_cluster(idx, f"{tag}{c}")
        st = out[c]["status"]
        mark = "ok " if st == "accepted" else "HOLD"
        _ag = out[c]["agreement"]; _gr = out[c]["grounding"]
        print(f"    [{mark}] {tag}{c:<3} n={out[c]['n']:<4} "
              f"agree={_ag if _ag == _ag else float('nan'):.2f} "
              f"grnd={_gr if _gr == _gr else float('nan'):.2f} "
              f"judge={out[c]['judge'] or '-':<7} "
              f"{out[c]['name'] or out[c]['withheld_reason'][:44]}")
    named[lv] = out
    psave(art, out)
    n_ok = sum(1 for v in out.values() if v["status"] == "accepted")
    print(f"  {lv:<4} named: {n_ok}/{K} accepted, {K - n_ok} withheld")

try:
    del mdl, tok
except Exception:
    pass
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception:
    pass

# ---------------- embedding discriminativeness check --------------------------
# Each accepted name is embedded and must be closer to its OWN cluster's
# responses than to the best competing cluster. Runs after the LLM is freed, so
# only one model occupies the GPU at a time. Centroids use members only, so the
# noise rows never enter a centroid.
try:
    if any(need.values()) and ENABLE:
        from sentence_transformers import SentenceTransformer
        print("\n  embedding check: loading the embedding model ...")
        _em = SentenceTransformer(CFG.embed_model, trust_remote_code=True,
                                  device="cuda" if __import__("torch").cuda.is_available() else None)
        for lv, lab, art, tag in LEVELS:
            if not need[lv]:
                continue
            out = named[lv]
            ids = [c for c in out if out[c]["status"] == "accepted"]
            if not ids:
                continue
            # The stored response vectors were produced with the embedding
            # instruction prefixed to the text. Stage 30v identifies that exact
            # path ('prefix-join': instruction + text, no separator) at cosine
            # 1.0000 against them, so encoding a bare name here compared two
            # different spaces. Names are encoded the way responses were.
            _INSTR = globals().get("EMBED_INSTRUCTION", "") or ""
            NE = _em.encode([f"{_INSTR}{out[c]['name']}" for c in ids],
                            normalize_embeddings=True,
                            batch_size=8, show_progress_bar=False)
            cents = {c: E_pool[lab == c].mean(0) for c in range(KOF[lv])}
            for c, ne in zip(ids, np.asarray(NE, dtype=np.float32)):
                sims = {k: float(ne @ (v / (np.linalg.norm(v) + 1e-9))) for k, v in cents.items()}
                own = sims[c]
                other = max(v for k, v in sims.items() if k != c) if len(sims) > 1 else -1.0
                mg = own - other
                out[c]["emb_margin"] = mg
                if mg < EMB_MARGIN_MIN:
                    out[c]["status"] = "withheld"
                    out[c]["withheld_reason"] = (f"name clearly fits another cluster better "
                                                 f"(margin {mg:+.3f})")
                elif mg < EMB_MARGIN_WEAK:
                    out[c].setdefault("flags", []).append(f"ambiguous vs a neighbour ({mg:+.3f})")
            psave(art, out)
            n_ok = sum(1 for v in out.values() if v["status"] == "accepted")
            print(f"    {lv:<4} after embedding check: {n_ok}/{len(out)} accepted")
        del _em
        gc.collect()
except Exception as e:
    print(f"  (embedding check skipped: {type(e).__name__}: {str(e)[:90]})")

# ---------------- placeholders for withheld labels + audit table --------------
rows = []
for lv, lab, art, tag in LEVELS:
    out = named[lv]
    for c in sorted(out):
        v = out[c]
        if v["status"] != "accepted":
            kws = ", ".join(v.get("keywords", [])[:4])
            v["name"] = f"[unlabelled {tag}{c}] top terms: {kws}" if kws else f"[unlabelled {tag}{c}]"
            v["summary"] = ("No label was accepted for this cluster: "
                            + (v.get("withheld_reason") or "verification failed")
                            + ". It is excluded from interpretive claims.")
        rows.append(dict(level=lv, id=c, n=v["n"], status=v["status"],
                         name=v["name"], llm_name=v.get("llm_name", ""),
                         flags="; ".join(v.get("flags", [])),
                         sentiment=v.get("sentiment", ""),
                         agreement=v.get("agreement"), grounding=v.get("grounding"),
                         judge=v.get("judge", ""), emb_margin=v.get("emb_margin"),
                         withheld_reason=v.get("withheld_reason", ""),
                         summary_grounding=v.get("summary_grounding"),
                         llm_summary=v.get("llm_summary", ""),
                         keywords="; ".join(v.get("keywords", [])[:8]),
                         summary=v.get("summary", "")))
    psave(art, out)

AUD = pd.DataFrame(rows)
AUD.to_csv(f"{POOLED_DIR}/theme_naming_audit.csv", index=False)
psave("pooled_naming_audit", AUD)

# ---- the plain per-theme lookups every later stage needs ---------------------
# Stage 30v wants each theme's own distinctive words; Stage 40 and NB3 want a
# name per id. Both used to read artifacts of the deleted Stage 23c. They are
# written HERE, from this run's evidence, so nothing downstream can pick up a
# dict from a different clustering.
for lv, lab, art, tag in LEVELS:
    out = named[lv]
    psave(f"pooled_{lv}_name",  {c: out[c]["name"] for c in sorted(out)})
    psave(f"pooled_{lv}_terms", {c: list(out[c].get("keywords", [])) for c in sorted(out)})
    psave(f"pooled_{lv}_size",  {c: int((lab == c).sum()) for c in sorted(out)})
print(f"\n  wrote pooled_main_name / pooled_sub_name, pooled_main_terms / pooled_sub_terms,")
print(f"  pooled_main_size / pooled_sub_size  (keyed 0..K-1; noise has no entry).")

pooled_main_llm, pooled_sub_llm = named["main"], named["sub"]
canon = pload("pooled_canonical_level") or "sub"
psave("pooled_theme_llm", named[canon])

# The summary floors print what they actually did on this corpus instead of
# asserting that they are right.
_sg = AUD["summary_grounding"].dropna() if "summary_grounding" in AUD.columns else []
if len(_sg):
    _held = int((AUD["flags"].fillna("").str.contains("summary withheld")).sum())
    print("\n" + "=" * 78)
    print("  SUMMARY GROUNDING  (calibrate SUMMARY_MIN from this)")
    print("=" * 78)
    print(f"  scored {len(_sg)} summaries: min {_sg.min():.2f}  q25 {_sg.quantile(.25):.2f}"
          f"  median {_sg.median():.2f}  q75 {_sg.quantile(.75):.2f}  max {_sg.max():.2f}")
    print(f"  withheld at SUMMARY_MIN={SUMMARY_MIN}: {_held} of {len(_sg)}"
          f"   |  flagged weak below {SUMMARY_WEAK}: "
          f"{int((_sg < SUMMARY_WEAK).sum()) - _held}")
    print("  the model's original wording is kept in llm_summary in the audit CSV,")
    print("  so a withheld summary can always be inspected rather than guessed at.")

print("\n" + "=" * 78)
print("  NAMING AUDIT")
print("=" * 78)
print(f"  {'level':<6}{'clusters':>9}{'accepted':>10}{'withheld':>10}{'meanAgree':>11}{'meanGrnd':>10}")
for lv, _, _, _ in LEVELS:
    d = AUD[AUD.level == lv]
    if not len(d):
        continue
    a = d[d.status == "accepted"]
    print(f"  {lv:<6}{len(d):>9}{len(a):>10}{len(d)-len(a):>10}"
          f"{a['agreement'].mean() if len(a) else float('nan'):>11.2f}"
          f"{a['grounding'].mean() if len(a) else float('nan'):>10.2f}")
_w = AUD[AUD.status != "accepted"]
if len(_w):
    print(f"\n  withheld ({len(_w)}) - these are NOT interpretive themes:")
    for _, r in _w.head(12).iterrows():
        print(f"    {r['level']:<5}{r['id']:<4} n={r['n']:<4} {r['withheld_reason'][:60]}")
print(f"\n  -> theme_naming_audit.csv  ·  pooled_main_llm / pooled_sub_llm")
print("  Every accepted label cited its evidence, was grounded in it, survived a")
print("  held-out judge and is closer to its own cluster than to any other.")
print("=" * 78)


## Stage 30v — Embedding-anchored label verification and repair

*(replaces the old Stage 30b and 30c)*

Stage 30 names one theme at a time, so it cannot see that it has already used similar wording for another cluster, and it sometimes writes a label whose meaning does not match the responses it is attached to. This cell **measures** that, in the same embedding space the clustering was built in, and repairs what fails.

* **Anchor** — each theme's centroid comes straight from `pooled_embeddings`, built from members only, so the `-1` responses never enter a centroid.
* **Same encoding path** — a label is only comparable to a centroid if it goes through the identical encoding NB1 used. The cell re-embeds real responses under each candidate path and keeps the one that reproduces the stored vectors; if none does, it **stops** rather than report numbers measured in a different space.
* **Optimal assignment** — whether a set of labels describes a set of themes is a bipartite matching question, so it is decided by the assignment maximising total similarity (Hungarian). A label passes only if the global optimum leaves it on its own theme *and* its own theme is among the few closest of all $K$.
* **Repair** — a failing label goes back to the LLM with the theme it was confused with, that theme's words, and this theme's own words and quotes. Anything still failing is then named from its **responses alone**, ignoring keywords. Every candidate a theme ever receives is scored and only the best is adopted, so the procedure is monotone — a repair that makes a label worse is simply not taken.
* **`NAME_OVERRIDE`** — your text beats both the model and the repair, but it is stamped with the $K$ it was written for and refused if the partition has changed, and it is measured like every other label rather than exempted.

Both levels — **main and sub**. The repaired names are written back into `pooled_main_llm` / `pooled_sub_llm`, `pooled_main_name` / `pooled_sub_name`, `pooled_theme_llm` and `pooled_master`, so Notebook 3 and every downstream figure pick them up automatically. Full per-label evidence: **`label_verification.csv`**.


In [ ]:
# @title
# ==============================================================================
# STAGE 30v  -  EMBEDDING-ANCHORED LABEL VERIFICATION AND REPAIR
#               (replaces Stage 30b and Stage 30c - delete both)
# ------------------------------------------------------------------------------
# The problem: the LLM names one cluster at a time, so it reuses wording across
# themes and sometimes writes a label whose meaning does not match the responses
# it is attached to. Nothing downstream can detect that.
#
# TWO LEVELS. main and sub only - the mid level was removed in Stage 23H. NOISE
# IS NOT A THEME: cluster ids run 0..K-1 at each level and the -1 rows are not
# members of any of them, so no centroid, no quote set and no label is ever built
# from responses that HDBSCAN declined to place.
#
# The method, using only the two models this pipeline already has:
#
#   1 ANCHOR. Each theme's centroid is computed directly from pooled_embeddings -
#     the same Qwen3 vectors the clustering was built on. No new embedding of the
#     corpus, no external model.
#
#   2 ENCODE THE LABEL THE SAME WAY. A label is only comparable to a centroid if
#     it goes through the identical encoding path as the responses did in NB1.
#     Because that path is not recorded in this notebook, the cell RE-EMBEDS a
#     sample of real responses under each candidate path and keeps the one that
#     reproduces the stored vectors (cosine ~ 1.0). If none reproduces them the
#     cell STOPS. It never reports numbers it cannot back up.
#
#   3 OPTIMAL ASSIGNMENT. Whether a set of labels describes a set of themes is a
#     bipartite matching question, so it is decided by the assignment of labels to
#     themes that maximises total similarity (Hungarian algorithm). A label passes
#     if that global optimum leaves it on its own theme. This matters twice over.
#     First, a short label and a long response differ in length and register,
#     which shifts similarity - but the shift applies to every centroid equally
#     for a given label and to every label equally for a given centroid, so a
#     matching-based decision cancels it. Second, requiring a two-way nearest
#     neighbour (mutual best match) would fail two good labels that happen to be
#     each other's nearest neighbour, even when each is the best label for its own
#     theme; the optimal assignment resolves that competition instead of
#     penalising both. The strict mutual flag is still reported alongside.
#
#   4 REPAIR. A failing label is sent back to the LLM together with the theme it
#     was confused with, that theme's own words, and this theme's words and real
#     quotes, with an instruction to write a label that cannot be confused with
#     it. The new label is re-tested. Iterate to a fixed budget.
#
#   5 REPORT. Rank-1 rate per direction and mutual-match rate, against the chance
#     baseline 1/K, before and after repair. Labels that never pass are reported,
#     never silently accepted.
#
# The per-theme term lists now come from Stage 30's own log-odds keywords
# (pooled_main_terms / pooled_sub_terms). They used to come from the deleted
# Stage 23c, so on a clean machine this cell silently repaired labels with an
# EMPTY keyword list in front of the model - the prompt demanded the new label
# use "at least one specific keyword" from a list that was not there.
#
# Nothing is invented. Every repair is grounded in the cluster's own terms and
# quotes, and any label that cannot be fixed is left for you in NAME_OVERRIDE.
# The encoder and the LLM are loaded one at a time and freed, so both fit.
#
# Outputs: label_verification.csv, chart_label_verification.png,
#          pooled_label_verification, and the repaired pooled_*_llm artifacts.
# ==============================================================================
import os, re, gc, json
import numpy as np, pandas as pd

# ---------------------------- EDIT HERE ---------------------------------------
# Your text always wins, but an override is keyed by CLUSTER ID, and a cluster id
# only means something inside the partition it was written for.
# To add your own: put the text here AND stamp the K it was written for, e.g.
#     NAME_OVERRIDE   = {"main": {}, "sub": {12: "Your Wording"}}
#     NAME_OVERRIDE_K = {"sub": 39}
# If the partition later changes, the stamp no longer matches and the override is
# refused with a message instead of being pinned to a different theme.
NAME_OVERRIDE   = {"main": {}, "sub": {}}
NAME_OVERRIDE_K = {}     # level -> number of clusters the overrides were written for
MAX_ROUNDS     = 2       # LLM repair rounds after the first verification
VERIFY_SAMPLE  = 200     # real responses used to identify NB1's encoding path
ENABLE_REPAIR  = True    # False = verify and report only, no LLM load
REUSE_PREVIOUS_REPAIRS = True   # reuse this cell's earlier LLM repairs (deterministic)
# --------------------------- END EDIT -----------------------------------------

EMB_MODEL = globals().get("EMBED_MODEL") or globals().get("_EMBED_MODEL") \
            or (CFG.embed_model if "CFG" in globals() else "Qwen/Qwen3-Embedding-8B")
INSTR     = globals().get("EMBED_INSTRUCTION", "")
LLM_NAME  = globals().get("LLM_MODEL_NAME") or globals().get("LLM_MODEL") \
            or "Qwen/Qwen2.5-14B-Instruct"

def _free():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception: pass

def _K(lab):
    lab = np.asarray(lab)
    return int(lab.max()) + 1 if (lab >= 0).any() else 0

print("=" * 78)
print("  STAGE 30v   EMBEDDING-ANCHORED LABEL VERIFICATION AND REPAIR")
print("=" * 78)

# ---- artifacts ---------------------------------------------------------------
X = np.asarray(pload("pooled_embeddings"), dtype=np.float32)
meta = pload("pooled_meta")
if X is None or meta is None:
    raise RuntimeError("Need pooled_embeddings and pooled_meta (run the preflight).")
meta = meta.reset_index(drop=True)
N = len(meta)
if len(X) != N:
    raise RuntimeError(f"[Stage 30v] pooled_embeddings has {len(X)} rows but pooled_meta "
                       f"has {N} - the pooled set is not self-consistent.")
_df = df if "df" in globals() and df is not None else get_df()
_pid = {}
for _, r in _df.iterrows():
    _pid.setdefault(r["Participant"], r)
texts, texts_raw = [], []
for _, mr in meta.iterrows():
    r = _pid.get(mr["Participant"]); q = mr["question"]
    v = "" if r is None else (r.get(f"bert_{q}", "") or "")
    v = "" if (v is None or str(v).lower() == "nan") else str(v)
    texts_raw.append(v); texts.append(" ".join(v.split()))
texts = np.array(texts, dtype=object); texts_raw = np.array(texts_raw, dtype=object)

LEVELS = [("main", "pooled_labels_main7", "pooled_main_llm", "pooled_main_terms"),
          ("sub",  "pooled_labels_sub",   "pooled_sub_llm",  "pooled_sub_terms")]
LAB, NAME0, TERMS, IDS = {}, {}, {}, {}
for lv, lk, nk, tk in LEVELS:
    L = pload(lk); D = pload(nk)
    if L is None:
        raise RuntimeError(f"[Stage 30v] {lk} is missing - run Stage 23H first.")
    if not D:
        raise RuntimeError(f"[Stage 30v] {nk} is missing - run Stage 30 first.")
    L = np.asarray(L)
    # ---- the stale guard, applied to BOTH the labels and the label dict -------
    if len(L) != N:
        raise RuntimeError(
            f"[Stage 30v] {lk} has {len(L)} rows but the pooled set has {N}. That "
            f"artifact belongs to an EARLIER clustering. Re-run Stage 23H, then 30.")
    LAB[lv] = L
    K = _K(L)
    ids = sorted(int(k) for k in D.keys())
    if set(ids) != set(range(K)):
        raise RuntimeError(
            f"[Stage 30v] {nk} covers ids {ids[:6]}...({len(ids)} of them) but the "
            f"{lv} partition has {K} themes (0..{K-1}). The labels were written for a "
            f"different clustering - re-run Stage 30.")
    _empty = [c for c in ids if not (L == c).any()]
    if _empty:
        raise RuntimeError(f"[Stage 30v] {lv} theme(s) {_empty} have no members; the "
                           f"label array and its K disagree. Re-run Stage 23H.")
    IDS[lv] = ids
    NAME0[lv] = {c: str(((D.get(c) if c in D else D.get(str(c))) or {}).get("name", ""))
                 for c in ids}
    T = pload(tk) or {}
    TERMS[lv] = {c: [str(x) for x in ((T.get(c) if c in T else T.get(str(c))) or [])]
                 for c in ids}
    _nz = int((L < 0).sum())
    _nt = sum(1 for c in ids if TERMS[lv][c])
    print(f"  {lv:<5} K={K:<3} labels loaded; {_nz} response(s) have no theme "
          f"({_nz / N:.1%}, never named); term lists present for {_nt}/{K} themes")
    if _nt == 0:
        print(f"    !! {tk} is empty - repair prompts will have no keywords to work "
              f"from. Re-run Stage 30, which now writes it.")

def _unit(M):
    M = np.asarray(M, dtype=np.float32)
    n = np.linalg.norm(M, axis=1, keepdims=True); n[n == 0] = 1.0
    return M / n

# Stage 30's log-odds keywords are computed on the raw text with English stopwords
# removed, so function words from the other languages in this corpus can survive
# ("avec", "ai", "tout", "les", "et", "ça"). They carry no meaning, they make a
# descriptive label unreadable, and they mislead the LLM when shown as keywords.
# Filtered here at the point of use; the stored artifacts are left untouched.
_STOP = set("""
a an the and or of in on for to with without at by from into during this that these those
it its their our your my his her we you they he she i is are was were be been being have
has had do does did not no very much more most so such then than there here what which who
whom when where why how all any both each few other some only own same too can will just
le la les un une des du de et ou au aux en dans pour par sur avec sans ce cette ces ai as
ont est etait sont etre avoir que qui quoi tout tous toute nous vous ils elles je tu il
elle on mais donc car ne pas plus tres bien ca cela ceci y ete fait comme
el los las unos unas del al con sin por para se lo su sus es son era muy pero como
der die das den dem ein eine und oder von zu mit ohne fur auf ist sind war sehr aber wie
lo gli di da su per che non piu molto
os um uma nao
""".split())
def _is_content(term):
    ws = [w for w in re.findall(r"[^\W\d_]+", str(term).lower(), re.UNICODE) if len(w) > 1]
    if not ws: return False
    norm = [w.translate(str.maketrans("àâäéèêëîïôöùûüçñáíóúãõ", "aaaeeeeiioouuucnaiouao"))
            for w in ws]
    return any(n not in _STOP for n in norm)

def _content_terms(lv, c, k=3):
    """The cluster's own most distinctive MEANINGFUL terms, function words removed."""
    out = [t.strip() for t in TERMS[lv].get(c, []) if _is_content(t)]
    if len(out) >= k:
        return out[:k]
    # nothing usable survived - fall back to the most frequent content words the
    # participants in this cluster actually wrote (still their own data)
    from collections import Counter
    cnt = Counter()
    for t in texts[LAB[lv] == c]:
        for w in re.findall(r"[^\W\d_]{3,}", str(t).lower(), re.UNICODE):
            if _is_content(w): cnt[w] += 1
    return (out + [w for w, _ in cnt.most_common(k) if w not in out])[:k]

Xn = _unit(X)
CEN, HAVE = {}, {}
for lv, _, _, _ in LEVELS:
    HAVE[lv] = list(IDS[lv])                      # verified non-empty above
    CEN[lv] = _unit(np.stack([Xn[LAB[lv] == c].mean(0) for c in HAVE[lv]]))
print(f"  centroids built from pooled_embeddings (members only, noise excluded): " +
      ", ".join(f"{lv} {len(HAVE[lv])}" for lv, _, _, _ in LEVELS) +
      f"   ({X.shape[1]}-d)")

# ---- the encoding path: identify it, do not assume it -----------------------
_FMTS = [("qwen3-official", lambda t: f"Instruct: {INSTR}\nQuery: {t}"),
         ("prefix-space",   lambda t: f"{INSTR} {t}"),
         ("prefix-newline", lambda t: f"{INSTR}\n{t}"),
         ("prefix-join",    lambda t: f"{INSTR}{t}"),
         ("no-instruction", lambda t: t)]
CANDIDATES = [(f"{ft} + {tt}-text", tt, fn)
              for ft, fn in _FMTS for tt in ("collapsed", "raw")]
TEXTSRC = {"collapsed": texts, "raw": texts_raw}
_hook_enc = globals().get("VERIFY_ENCODE_FN")

def _load_encoder():
    if _hook_enc is not None:
        return _hook_enc, (lambda: None)
    from sentence_transformers import SentenceTransformer
    import torch
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"  loading {EMB_MODEL} on {dev} ...")
    m = SentenceTransformer(EMB_MODEL, device=dev,
                            model_kwargs={"torch_dtype": torch.float16} if dev == "cuda" else {})
    _bs = int(globals().get("EMBED_BATCH", 16))       # match NB1's batching exactly
    fn = lambda xs: np.asarray(m.encode(list(xs), batch_size=_bs, convert_to_numpy=True,
                                        normalize_embeddings=True, show_progress_bar=False),
                               dtype=np.float32)
    def _drop():
        nonlocal m
        del m; _free()
    return fn, _drop

PATH = globals().get("_ANCHOR_PATH")   # kept in the session only (holds a function)
def _identify_path(enc):
    """Find NB1's encoding path, then check the property we actually depend on.

    A high cosine is not the requirement - what matters is that a re-embedded
    response still retrieves the SAME theme as its stored vector does. Small
    numerical differences (fp16 vs fp32, batching, tokenizer padding) move the
    cosine a little without changing any retrieval, so the decision is made on
    that functional agreement, measured and reported, not on an arbitrary
    similarity cut-off.
    """
    rs = np.random.RandomState(0)
    ok = np.array([len(t) >= 20 for t in texts])
    if ok.sum() < 30:
        raise RuntimeError("Too few usable responses to identify the encoding path.")
    n_fine = int(min(VERIFY_SAMPLE, ok.sum()))
    pool = np.where(ok)[0]
    coarse = rs.choice(pool, size=int(min(64, len(pool))), replace=False)
    scored = []
    for tag, tt, fmt in CANDIDATES:
        got = _unit(enc([fmt(TEXTSRC[tt][i]) for i in coarse]))
        scored.append((float(np.mean(np.sum(got * Xn[coarse], axis=1))), tag, tt, fmt))
    scored.sort(reverse=True, key=lambda z: z[0])
    print("    closest encoding paths (mean cosine with the stored vectors):")
    for c, tag, _, _ in scored[:3]:
        print(f"      {c:.4f}  {tag}")
    cos0, tag, tt, fmt = scored[0]

    idx = rs.choice(pool, size=n_fine, replace=False)
    got = _unit(enc([fmt(TEXTSRC[tt][i]) for i in idx]))
    cos = float(np.mean(np.sum(got * Xn[idx], axis=1)))
    Csub = CEN["sub"]
    same = float(np.mean(np.argmax(got @ Csub.T, axis=1) ==
                         np.argmax(Xn[idx] @ Csub.T, axis=1)))
    print(f"    winner '{tag}' on {n_fine} real responses: mean cosine {cos:.4f}, "
          f"retrieves the same theme as the stored vector {same*100:.1f}% of the time")
    if cos < 0.95 or same < 0.98:
        raise RuntimeError(
            f"Cannot reproduce pooled_embeddings well enough to trust label geometry "
            f"(best '{tag}': cosine {cos:.4f}, same-theme retrieval {same*100:.1f}%). "
            "Refusing to report numbers measured in a different space. Paste NB1's "
            "embedding function into this cell.")
    print(f"  encoding path confirmed: '{tag}' - labels are encoded exactly like the responses.")
    return (tag, cos, fmt, same)

# ---- verification: optimal assignment ----------------------------------------
from scipy.optimize import linear_sum_assignment

def _assert_partition(names, where=""):
    """Every level's label dict must cover EXACTLY the clusters that exist now.
    Subset was not enough: a dict with extra ids from an older, larger partition
    passed the old check and then verified labels for themes that do not exist."""
    for lv, _, _, _ in LEVELS:
        have, got = set(HAVE[lv]), set(names.get(lv, {}))
        if have != got:
            raise RuntimeError(
                f"[Stage 30v{(' ' + where) if where else ''}] labels for {lv} cover "
                f"{len(got)} clusters but the current partition has {len(have)} "
                f"(missing: {sorted(have - got)[:8]}; extra: {sorted(got - have)[:8]}). "
                f"The labels are from a different partition - re-run Stage 30, then "
                f"re-run this cell.")


def _verify(names, enc, fmt):
    """A label passes when the OPTIMAL one-to-one assignment of labels to themes
    puts it on its own theme.

    Mutual best match was too strict: two labels that are each other's nearest
    neighbour block one another even when both are the best label for their own
    theme, so one good label is failed for the other's sake. Whether a set of
    labels describes a set of themes is a bipartite matching question, and the
    right answer is the assignment that maximises total similarity (Hungarian
    algorithm). A label is wrong only if that global optimum moves it elsewhere.
    The directional rank and the strict mutual flag are still reported.
    """
    _assert_partition(names, "verify")
    flat = [(lv, c) for lv, _, _, _ in LEVELS for c in HAVE[lv]]
    E = _unit(enc([fmt(names[lv][c]) for lv, c in flat]))
    pos = {k: i for i, k in enumerate(flat)}
    out = {}
    for lv, _, _, _ in LEVELS:
        ids = HAVE[lv]; K = len(ids)
        N_ = E[[pos[(lv, c)] for c in ids]]
        S = N_ @ CEN[lv].T                                 # label x centroid
        rr, cc = linear_sum_assignment(-S)                 # optimal label -> theme
        assign = {int(a): int(b) for a, b in zip(rr, cc)}
        # The assignment is a perfect matching, so a useless label can still land on
        # its own theme purely because every other theme was taken. It therefore has
        # to clear a second, independent bar: its own theme must be among the few
        # closest of all K themes. Top 10%, never fewer than 2 and never more than 3.
        TOPK = max(2, min(3, int(np.ceil(K * 0.10))))
        for j, c in enumerate(ids):
            row = S[j]; col = S[:, j]
            # Pessimistic ranking: a tie is counted against the label, never for it.
            # Counting only strictly-greater rivals would hand rank 1 to a label that
            # is merely indistinguishable from others, which is the opposite of proof.
            r_lab = int((row > row[j]).sum()) + int((row == row[j]).sum())
            r_cen = int((col > col[j]).sum()) + int((col == col[j]).sum())
            keeps = (assign.get(j) == j) and (r_lab <= TOPK)
            rival = ids[assign.get(j, j)] if assign.get(j) != j else \
                    ids[int(np.argmax(np.where(np.arange(K) == j, -9, row)))]
            out[(lv, c)] = dict(level=lv, id=c, K=K, name=names[lv][c], topk=TOPK,
                                rank_label=r_lab, rank_centroid=r_cen,
                                assigned=bool(keeps),
                                mutual=bool(r_lab == 1 and r_cen == 1),
                                sim_own=float(row[j]),
                                margin=float(row[j] - np.max(np.where(np.arange(K) == j,
                                                                      -9, row))),
                                rival=int(rival))
    return out

# ---- repair: send the failure back to the LLM -------------------------------
_hook_llm = globals().get("VERIFY_LLM_FN")

def _load_llm():
    if _hook_llm is not None:
        return _hook_llm, (lambda: None)
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    print(f"  loading {LLM_NAME} in 4-bit for repair ...")
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.bfloat16,
                             bnb_4bit_use_double_quant=True)
    tk = AutoTokenizer.from_pretrained(LLM_NAME)
    md = AutoModelForCausalLM.from_pretrained(LLM_NAME, quantization_config=bnb,
                                              device_map="auto", torch_dtype=torch.bfloat16)
    md.eval()
    def fn(sys_p, usr_p):
        msgs = [{"role": "system", "content": sys_p}, {"role": "user", "content": usr_p}]
        p = tk.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
        e = tk(p, return_tensors="pt")
        e = {k: v.to(next(md.parameters()).device) for k, v in e.items()}
        with torch.no_grad():
            o = md.generate(**e, max_new_tokens=90, do_sample=False,
                            pad_token_id=(tk.eos_token_id or tk.pad_token_id))
        return tk.decode(o[0][e["input_ids"].shape[1]:], skip_special_tokens=True)
    def _drop():
        nonlocal md, tk
        del md, tk; _free()
    return fn, _drop

SYS_R = ("You are a qualitative-coding assistant. You rewrite one theme label so that it "
         "can no longer be confused with a named competing theme. Ground every word in the "
         "keywords and quotes you are given. Invent nothing. Reply with JSON only.")

def _quotes_for(lv, c, n=6):
    idx = np.where(LAB[lv] == c)[0]          # -1 never matches: noise is never quoted
    if not len(idx): return []
    cen = Xn[idx].mean(0)
    order = idx[np.argsort(-(Xn[idx] @ cen))]
    out = []
    for i in order:
        t = str(texts[i]).strip()
        if len(t) >= 20: out.append(t[:200])
        if len(out) >= n: break
    return out

def _repair_prompt(lv, c, rec, names):
    riv = rec["rival"]
    return (f'THIS THEME\'S OWN KEYWORDS: {", ".join(_content_terms(lv, c, 10)) or "(none)"}\n'
            f'THIS THEME\'S OWN RESPONSES:\n' +
            "\n".join(f'- "{q}"' for q in _quotes_for(lv, c)) +
            f'\n\nYOUR CURRENT LABEL: "{rec["name"]}"\n'
            f'PROBLEM: measured against the responses themselves, this label matches the '
            f'DIFFERENT theme "{names[lv].get(riv, "")}" '
            f'(whose own keywords are: {", ".join(_content_terms(lv, riv, 8))}) '
            f'more closely than it matches the responses above.\n\n'
            'Write a NEW Title-Case label of 3 to 6 words for the responses above that:\n'
            '  1. uses at least one specific keyword from THIS theme\'s keyword list;\n'
            '  2. could not be mistaken for the competing theme named above;\n'
            '  3. states the topic and the honest tone of these responses;\n'
            '  4. uses no word that is not supported by the keywords or quotes above.\n'
            'Return ONLY: {"name": "..."}')

def _parse_name(raw):
    if not raw: return None
    m = re.search(r'\{.*?\}', str(raw), re.S)
    if m:
        try:
            v = json.loads(m.group(0)).get("name", "")
            v = re.sub(r"\s+", " ", str(v)).strip().strip('."')
            if 2 < len(v) <= 70: return v
        except Exception: pass
    m = re.search(r'"?name"?\s*[:\-]\s*"?([^"\n}]+)', str(raw), re.I)
    if m:
        v = re.sub(r"\s+", " ", m.group(1)).strip().strip('",.')
        if 2 < len(v) <= 70: return v
    return None

# ---- the loop ----------------------------------------------------------------
# Always start from the ORIGINAL Stage 30 labels, snapshotted on the first run, so
# re-running this cell re-decides from scratch instead of compounding earlier edits.
PRIS = pload("pooled_names_pristine")

# A snapshot only describes the partition it was taken from. If the hierarchy has
# been recomputed since (a different K at any level, or the mid level removed),
# its cluster ids no longer correspond to anything. Detect the mismatch and
# re-snapshot from the CURRENT Stage 30 labels instead of crashing or, worse,
# carrying an old label across to a different cluster that shares its id.
_PART_OK = True
if PRIS is not None:
    _levels_now = {lv for lv, _, _, _ in LEVELS}
    _extra_levels = set(PRIS.keys()) - _levels_now
    PRIS = {lv: {int(k): v for k, v in (PRIS.get(lv) or {}).items()}
            for lv in _levels_now}
    _bad = {lv: (len(PRIS.get(lv, {})), len(NAME0[lv])) for lv, _, _, _ in LEVELS
            if set(PRIS.get(lv, {})) != set(NAME0[lv])}
    if _bad or _extra_levels:
        _PART_OK = False
        if _extra_levels:
            print(f"  the stored snapshot still carries the removed level(s) "
                  f"{sorted(_extra_levels)} - it predates Stage 23H's two-level hierarchy.")
        if _bad:
            print("  the stored snapshot belongs to a PREVIOUS partition ("
                  + "; ".join(f"{lv}: snapshot {a} vs current {b} clusters"
                              for lv, (a, b) in _bad.items())
                  + ").")
        print("  discarding it and every artifact derived from it, then "
              "re-snapshotting from the current Stage 30 labels.")
        for _stale in ("pooled_names_pristine", "pooled_names_repaired",
                       "pooled_quote_attempted", "pooled_names_verified"):
            try:
                invalidate(_stale)
            except Exception:
                pass
        _old_csv = f"{POOLED_DIR}/label_verification.csv"
        if os.path.exists(_old_csv):
            try:
                os.replace(_old_csv, _old_csv + ".stale")
                print("  previous label_verification.csv set aside as .stale "
                      "(its ids refer to the old partition).")
            except Exception:
                pass
        PRIS = None

if PRIS is None:
    PRIS = {lv: dict(NAME0[lv]) for lv, _, _, _ in LEVELS}
    # An earlier run of this cell may already have overwritten the artifacts. Its CSV
    # recorded the labels as they were before that run, so recover them from there
    # rather than treating edited labels as the starting point.
    _old = f"{POOLED_DIR}/label_verification.csv"
    # Only meaningful when the partition is unchanged: the CSV keys labels by
    # cluster id, and after a re-clustering the same id is a different theme.
    if _PART_OK and os.path.exists(_old):
        try:
            _p = pd.read_csv(_old)
            if "name_before" in _p.columns:
                n = 0
                for _, r in _p.iterrows():
                    lv, c, nb = str(r["level"]), int(r["id"]), str(r["name_before"]).strip()
                    if lv in PRIS and c in PRIS[lv] and nb and nb.lower() != "nan":
                        if PRIS[lv][c] != nb: n += 1
                        PRIS[lv][c] = nb
                print(f"  recovered {n} original Stage 30 label(s) from the previous "
                      f"label_verification.csv")
        except Exception as e:
            print(f"  (could not read the previous CSV: {type(e).__name__} - using "
                  "the labels currently in the artifacts)")
    psave("pooled_names_pristine", PRIS)
    print("  snapshotted the original Stage 30 labels (this cell is now re-runnable).")
else:
    print("  starting from the original Stage 30 labels, not from a previous run's edits.")
NAME0 = {lv: dict(PRIS[lv]) for lv, _, _, _ in LEVELS}
names = {lv: dict(PRIS[lv]) for lv, _, _, _ in LEVELS}

# The LLM repairs are deterministic (greedy decoding), so a re-run would spend the
# same GPU time to reach the same labels. They are cached and reused, which makes
# re-running this cell cost one encoder pass instead of several model loads.
_cached = pload("pooled_names_repaired") if REUSE_PREVIOUS_REPAIRS else None
_ROUNDS = MAX_ROUNDS
if _cached and set(_cached.keys()) == set(names.keys()) and \
        all({int(k) for k in _cached.get(lv, {})} == set(names[lv]) for lv in names):
    for lv in names:
        names[lv].update({int(k): v for k, v in _cached[lv].items()})
    _ROUNDS = 0
    print("  reusing the LLM repairs from the previous run (no LLM load needed). "
          "Set REUSE_PREVIOUS_REPAIRS = False to redo them.")

# Every candidate label a theme is ever given is scored and the BEST one is kept.
# Without this the loop accepts whatever the LLM last said, so a repair that makes a
# label worse is adopted anyway and the totals move up and down between runs. Keeping
# the best-so-far makes the procedure monotone: it can only improve.
#   score = (passes the assignment test, then how close its own theme ranks,
#            then prefer a readable label over a slash-separated term list)
BEST, BESTR = {}, {}      # best overall, and best READABLE (non term-list) candidate
def _score(v, nm):
    return (1 if v["assigned"] else 0, -v["rank_label"],
            1 if (" / " not in nm) else 0, round(v["margin"], 6))
def _record(r, tag):
    for k, v in r.items():
        lv, c = k
        nm = names[lv][c]
        s = _score(v, nm)
        rec = dict(score=s, name=nm, tag=tag, rank=v["rank_label"], assigned=v["assigned"])
        if k not in BEST or s > BEST[k]["score"]:
            BEST[k] = rec
        if " / " not in nm and (k not in BESTR or s > BESTR[k]["score"]):
            BESTR[k] = rec

history, res = [], None
for rnd in range(_ROUNDS + 1):
    enc, drop_enc = _load_encoder()
    if PATH is None:
        PATH = _identify_path(enc)
        globals()["_ANCHOR_PATH"] = PATH
    res = _verify(names, enc, PATH[2])
    drop_enc()
    _record(res, "stage30" if rnd == 0 else "llm-repaired")
    fails = [k for k, v in res.items() if not v["assigned"]]
    n_ok = len(res) - len(fails)
    history.append(dict(round=rnd, confirmed=n_ok, failing=len(fails)))
    print(f"  round {rnd}: optimal assignment keeps {n_ok}/{len(res)}"
          f"   (still failing {len(fails)})")
    if not fails or rnd == _ROUNDS or not ENABLE_REPAIR:
        break
    llm, drop_llm = _load_llm()
    n_new = 0
    for lv, c in fails:
        try:
            nm = _parse_name(llm(SYS_R, _repair_prompt(lv, c, res[(lv, c)], names)))
        except Exception as e:
            print(f"    {lv}{c}: repair failed ({type(e).__name__}) - keeping current label")
            nm = None
        if nm and nm != names[lv][c]:
            names[lv][c] = nm; n_new += 1
    drop_llm()
    print(f"    LLM proposed {n_new} new label(s)")
    if n_new == 0:
        break

# ---- last resort before giving up: name it from the RESPONSES, not the keywords --
# Some clusters fail only because their keyword list is useless - short,
# multilingual answers leave function words at the top ("avec", "ai", "tout"), so the
# repair prompt was feeding the model noise and demanding it use one of those words.
# The responses themselves are perfectly readable, so for anything still failing we
# ask once more with a dozen real responses and NO keyword requirement at all.
SYS_Q = ("You are a qualitative-coding assistant. You are given the actual responses that "
         "make up one theme. Name that theme from those responses alone. Ground every word "
         "in what you can read there. Invent nothing. Reply with JSON only.")

def _quote_prompt(lv, c, rec, names):
    qs = _quotes_for(lv, c, 12)
    return ("THE RESPONSES THAT MAKE UP THIS THEME:\n" +
            "\n".join(f'- "{q}"' for q in qs) +
            "\n\nThe automatic keyword list for this theme is unreliable, so ignore keywords "
            "entirely and read the responses above.\n"
            f'Your previous label "{rec["name"]}" did not describe them, and measured against '
            f'the responses it matched the different theme "{names[lv].get(rec["rival"], "")}".'
            "\n\nWrite a Title-Case label of 3 to 6 words naming what these responses have in "
            "common - the situation, the moment, or the topic they share - together with their "
            "honest tone. It must not be confusable with the theme named above. Use only words "
            "supported by the responses.\nReturn ONLY: {\"name\": \"...\"}")

_QNAMED = set()
# Themes already given a response-based name in an earlier run are not retried:
# decoding is greedy, so a second attempt returns the same text for nothing.
_TRIED = {(str(a), int(b)) for a, b in (pload("pooled_quote_attempted") or [])} \
         if REUSE_PREVIOUS_REPAIRS else set()
_TRIED = {(lv, c) for lv, c in _TRIED if lv in names and c in names[lv]}   # drop dead ids
_still = [k for k, v in res.items() if not v["assigned"] and k not in _TRIED]
if _still and ENABLE_REPAIR:      # runs even on a cached re-run, because the cache
                                  # cannot contain a label this step has not produced
                                  # yet; once it does, _still is empty and it is skipped
    print(f"  {len(_still)} theme(s) left; naming them from their responses instead of "
          f"their keywords ...")
    llm, drop_llm = _load_llm()
    n_q = 0
    for lv, c in _still:
        try:
            nm = _parse_name(llm(SYS_Q, _quote_prompt(lv, c, res[(lv, c)], names)))
        except Exception as e:
            print(f"    {lv}{c}: quote-based naming failed ({type(e).__name__})"); nm = None
        if nm and nm != names[lv][c]:
            names[lv][c] = nm; n_q += 1; _QNAMED.add((lv, c))
    drop_llm()
    psave("pooled_quote_attempted", sorted({(a, int(b)) for a, b in (_TRIED | set(_still))}))
    print(f"    LLM proposed {n_q} label(s) from the responses")
    if n_q:
        enc, drop_enc = _load_encoder()
        res = _verify(names, enc, PATH[2])
        _record(res, "llm-named-from-responses")
        drop_enc()
        print(f"  after response-based naming: optimal assignment keeps "
              f"{sum(v['assigned'] for v in res.values())}/{len(res)}")

psave("pooled_names_repaired", {lv: dict(names[lv]) for lv in names})

# ---- terminal step: adopt the best candidate each theme ever produced ---------
# Everything above only PROPOSES labels. Nothing is adopted until here, where every
# theme keeps the best candidate it was ever given. That is what makes the procedure
# monotone - a repair that made a label worse is simply not taken - and it is why the
# totals no longer move up and down between runs.
_enc_f, _drop_f = _load_encoder()

# One last candidate for anything still unconfirmed: a label built from the cluster's
# own distinctive terms. It is not imposed, it is measured like every other candidate
# and wins only if it really describes the theme better than the wording already there.
_open = [k for k, b in BEST.items() if not b["assigned"]]
if _open:
    _stash = {k: names[k[0]][k[1]] for k in _open}
    for lv, c in _open:
        t = _content_terms(lv, c, 3)
        if t: names[lv][c] = " / ".join(x.title() for x in t)[:70]
    _record(_verify(names, _enc_f, PATH[2]), "descriptive-fallback")
    for (lv, c), nm in _stash.items():
        names[lv][c] = nm
    print(f"  {len(_open)} unconfirmed theme(s): term-based label offered as a "
          f"candidate and kept only where it scored better.")

for (lv, c), b in BEST.items():
    names[lv][c] = b["name"]

# A label often failed only because a competitor was standing on its theme. Once the
# competitor has been resolved that label may be fine again, so every theme that ended
# on a term list is offered its best readable wording once more, and keeps it if it
# now passes. Without this a good name is lost to a competition that no longer exists.
_fb = [k for k, b in BEST.items() if b["tag"] == "descriptive-fallback" and k in BESTR]
if _fb:
    _keep = {k: names[k[0]][k[1]] for k in _fb}
    for lv, c in _fb:
        names[lv][c] = BESTR[(lv, c)]["name"]
    _r2 = _verify(names, _enc_f, PATH[2])
    n_back = 0
    for k in _fb:
        if _r2[k]["assigned"]:
            BEST[k] = dict(BESTR[k]); n_back += 1
        else:
            names[k[0]][k[1]] = _keep[k]
    if n_back:
        print(f"  {n_back} theme(s) recovered their readable label once the competing "
              f"label had moved away.")

# Cosmetic tidy happens BEFORE the final measurement, not after, so the numbers
# reported describe exactly the labels that get written out. Doing it afterwards
# meant a duplicate-disambiguating suffix could change a label's meaning without
# anything re-checking it.
_CONN = {"and","or","of","in","on","for","to","with","through","by","at","from",
         "the","a","an","into","during","vs"}
def _style(s):
    s = re.sub(r"\s+", " ", str(s)).strip().strip('."')
    s = re.sub(r"\b[Ii][Gg][Ee][Mm]\b", "iGEM", s)
    t = s.split(" ")
    s = " ".join([w if i == 0 or w.lower() not in _CONN or w != w.capitalize()
                  else w.lower() for i, w in enumerate(t)])
    return s.strip()
# Overrides are applied BEFORE the tidy, so an override goes through the same
# duplicate disambiguation as every other label instead of being able to collide
# with one silently. Each level's overrides are refused outright if they were
# written for a different number of clusters - the ids would point elsewhere.
_OVR = set()
for lv in names:
    _ov = NAME_OVERRIDE.get(lv) or {}
    if not _ov:
        continue
    _kw = (NAME_OVERRIDE_K or {}).get(lv)
    if _kw is not None and int(_kw) != len(names[lv]):
        print(f"  NAME_OVERRIDE[{lv}] was written for K={int(_kw)} but this run has "
              f"K={len(names[lv])}. Cluster ids do not carry across a re-clustering, "
              f"so those {len(_ov)} override(s) are REFUSED. Re-identify the clusters "
              f"in label_verification.csv and update the ids and NAME_OVERRIDE_K.")
        continue
    if _kw is None:
        print(f"  NAME_OVERRIDE[{lv}]: no NAME_OVERRIDE_K stamp, so it cannot be "
              f"checked against this partition (K={len(names[lv])}). Applying, but "
              f"confirm each one against its cluster in label_verification.csv.")
    for c, nm in _ov.items():
        if int(c) in names[lv] and str(nm).strip():
            names[lv][int(c)] = str(nm); _OVR.add((lv, int(c)))
if _OVR:
    print(f"  {len(_OVR)} label(s) replaced by your NAME_OVERRIDE; they are measured "
          f"like every other label, not exempted.")

# Remember the exact wording each theme had before the tidy. The tidy is meant to
# be cosmetic, so if it ever costs a label its theme we must be able to undo it.
_pre_tidy = {(lv, c): names[lv][c] for lv, _, _, _ in LEVELS for c in names[lv]}

n_style = 0
for lv, _, _, _ in LEVELS:
    for c in names[lv]:
        st = _style(names[lv][c])
        if st != names[lv][c]: names[lv][c] = st; n_style += 1
    seen = {}
    for c in sorted(names[lv]):
        k = names[lv][c].lower()
        if k in seen:
            for t in _content_terms(lv, c, 6):
                if t.lower() not in names[lv][c].lower():
                    names[lv][c] = _style(f"{names[lv][c]} ({t.title()})"); break
        seen[k] = c

res = _verify(names, _enc_f, PATH[2])

# ---- the tidy is cosmetic and must never cost a label its theme ---------------
# A label that passed before the tidy and fails after it has lost its theme to a
# change of capitalisation, not to anything about the data. Revert exactly those,
# and only where the two wordings are the SAME WORDS - that way a duplicate-
# disambiguating suffix, which really does change the label, is never undone.
# The revert is adopted only if the global assignment improves, so this cannot
# make the result worse than leaving the tidy alone.
def _same_words(a, b):
    return str(a).lower().split() == str(b).lower().split()

_n_now = sum(v["assigned"] for v in res.values())
_cosmetic_loss = [k for k, v in res.items()
                  if not v["assigned"] and k not in _OVR
                  and BEST.get(k, {}).get("assigned")
                  and _pre_tidy.get(k) is not None
                  and _pre_tidy[k] != names[k[0]][k[1]]
                  and _same_words(_pre_tidy[k], names[k[0]][k[1]])]
if _cosmetic_loss:
    _tidied = {k: names[k[0]][k[1]] for k in _cosmetic_loss}
    for lv, c in _cosmetic_loss:
        names[lv][c] = _pre_tidy[(lv, c)]
    _res_rev = _verify(names, _enc_f, PATH[2])
    _n_rev = sum(v["assigned"] for v in _res_rev.values())
    if _n_rev > _n_now:
        print(f"  {len(_cosmetic_loss)} label(s) lost their theme to the cosmetic tidy "
              f"alone and were reverted to their pre-tidy wording "
              f"({_n_now} -> {_n_rev} of {len(res)}):")
        for lv, c in sorted(_cosmetic_loss):
            print(f"    {lv}{c}: \"{_tidied[(lv, c)]}\" -> \"{names[lv][c]}\"")
        res, _n_now = _res_rev, _n_rev
        n_style -= len(_cosmetic_loss)
    else:
        for k, nm in _tidied.items():
            names[k[0]][k[1]] = nm

_drop_f()
SOURCE = {}
for k, v in res.items():
    tag = BEST[k]["tag"]
    if k in _OVR:
        SOURCE[k] = "manual override" + ("" if v["assigned"] else ", unverified")
        continue
    SOURCE[k] = ("llm-verified" if (v["assigned"] and tag == "stage30")
                 else tag if v["assigned"]
                 else "llm-verified (contested)" if v["rank_label"] <= v["topk"]
                 else "descriptive-fallback" if tag == "descriptive-fallback"
                 else "llm-label, unverified")
_n_final = sum(v["assigned"] for v in res.values())
print(f"  final: optimal assignment keeps {_n_final}/{len(res)}")

# The tidy and the overrides happen after the last verification round, so they can
# take a label that had already passed and break it. That must never be invisible:
# compare against round 0 and name anything that regressed.
_r0 = history[0]["confirmed"] if history else None
if _r0 is not None and _n_final < _r0:
    _lost = [k for k, v in res.items()
             if not v["assigned"] and BEST.get(k, {}).get("assigned")]
    print(f"  NOTE: {_r0}/{len(res)} passed at round 0 but {_n_final}/{len(res)} pass "
          f"now. The cosmetic tidy and/or NAME_OVERRIDE changed labels after the last "
          f"verification round.")
    for lv, c in sorted(_lost):
        _was = BEST[(lv, c)]["name"]
        print(f"    {lv}{c}: \"{_was}\" (passed) -> \"{names[lv][c]}\" (fails)"
              + ("   [NAME_OVERRIDE]" if (lv, c) in _OVR else "   [tidy]"))
    if not _lost:
        print("    (the assignment is global, so a changed label can also displace "
              "themes whose own wording did not change)")

# ---- write back --------------------------------------------------------------
canon = pload("pooled_canonical_level") or "sub"
for lv, _, nk, _ in LEVELS:
    D = pload(nk)
    for c, nm in names[lv].items():
        d = D.get(c) if c in D else D.get(str(c))
        if isinstance(d, dict): d["name"] = nm
    psave(nk, D)
    psave(f"pooled_{lv}_name", {c: names[lv][c] for c in sorted(names[lv])})
    if lv == canon: psave("pooled_theme_llm", D)
print("  pooled_main_llm / pooled_sub_llm and pooled_main_name / pooled_sub_name "
      "rewritten with the verified labels.")
M = pload("pooled_master")
if M is not None:
    if len(M) != N:
        print(f"  (pooled_master has {len(M)} rows but the pooled set has {N} - it is "
              f"from an earlier run and is NOT rewritten. Re-run Stage 33b.)")
    else:
        M = M.copy()
        for lv, _, _, _ in LEVELS:
            if f"{lv}_id" in M.columns:
                # -1 (no theme) is not in `names`, so map() leaves it NaN; it keeps the
                # explicit "(unassigned)" that Stage 33b wrote. A theme id must never
                # silently inherit the label of a row that has no theme.
                _mapped = M[f"{lv}_id"].map(names[lv])
                M[f"{lv}_name"] = _mapped.where(_mapped.notna(), M.get(f"{lv}_name"))
        psave("pooled_master", M); M.to_csv(f"{POOLED_DIR}/pooled_master.csv", index=False)
        print("  pooled_master rewritten with the verified labels.")
else:
    print("  (pooled_master not built yet - run Stage 33b after this cell.)")

rows = []
for (lv, c), v in sorted(res.items(), key=lambda kv: (kv[0][0], kv[0][1])):
    rows.append(dict(level=lv, id=c, name_final=names[lv][c], name_before=NAME0[lv][c],
                     label_source=SOURCE.get((lv, c), "llm-verified"),
                     n_responses=int((LAB[lv] == c).sum()),
                     verified=v["assigned"], mutual_best=v["mutual"],
                     rank_label_to_theme=v["rank_label"],
                     rank_theme_to_label=v["rank_centroid"], of_K=v["K"],
                     margin=round(v["margin"], 4), chance=round(1.0 / v["K"], 4),
                     confused_with=(f"{lv[0]}{v['rival']} {names[lv].get(v['rival'],'')}"
                                    if not v["assigned"] else ""),
                     top_terms=", ".join(_content_terms(lv, c, 6)),
                     top_terms_raw=", ".join(TERMS[lv].get(c, [])[:6])))
V = pd.DataFrame(rows)
V.to_csv(f"{POOLED_DIR}/label_verification.csv", index=False)
psave("pooled_label_verification", dict(table=V, history=history, path=PATH[0]))

# ---- report ------------------------------------------------------------------
print("-" * 78)
for lv, _, _, _ in LEVELS:
    s = V[V.level == lv]
    if not len(s): continue
    K = int(s.of_K.iloc[0])
    print(f"  {lv:<5} K={K:<3} verified {int(s.verified.sum()):>3}/{len(s)}"
          f"   label->theme rank-1 {int((s.rank_label_to_theme==1).sum()):>3}"
          f"   theme->label rank-1 {int((s.rank_theme_to_label==1).sum()):>3}"
          f"   (chance {100.0/K:.1f}%)")
print(f"\n  repair history: " +
      " -> ".join(f"r{h['round']} {h['confirmed']}/{len(V)}" for h in history) +
      f"   |  style fixes {n_style}")
print("  final label provenance: " + ", ".join(
    f"{k} {int(v)}" for k, v in V.label_source.value_counts().items()))
print("  ALL LABELS ARE FINAL. Nothing is waiting on you - continue to Notebook 3.")
bad = V[~V.verified]
if len(bad):
    print(f"\n  {len(bad)} label(s) still unverified. Evidence:")
    for _, r in bad.iterrows():
        print(f"   [{r['level']}{r['id']}] \"{r['name_final']}\"  "
              f"(label->theme {r['rank_label_to_theme']}/{r['of_K']}, "
              f"theme->label {r['rank_theme_to_label']}/{r['of_K']}, margin {r['margin']:+.3f})")
        print(f"        confused with: {r['confused_with'][:60]}")
        print(f"        its own words: {r['top_terms']}")
    _bad_ovr = bad[bad.label_source.str.startswith("manual override")]
    if len(_bad_ovr):
        print(f"\n  {len(_bad_ovr)} of these is YOUR override and does NOT fit its "
              f"cluster - it is not a descriptive fallback, it is text that was "
              f"pinned to the wrong theme. Remove it from NAME_OVERRIDE or re-point "
              f"it, then re-run this cell:")
        for _, r in _bad_ovr.iterrows():
            print(f"    {r['level']}{r['id']}: \"{r['name_final']}\"  vs its own "
                  f"words: {r['top_terms']}")
    _fbb = bad[bad.label_source == "descriptive-fallback"]
    _llm = bad[~bad.index.isin(_bad_ovr.index) & ~bad.index.isin(_fbb.index)]
    if len(_fbb):
        print(f"\n  {len(_fbb)} carry a descriptive label built from that cluster's own "
              f"terms, so nothing is required of you.")
    if len(_llm):
        print(f"\n  {len(_llm)} keep the model's own wording: it was still the best "
              f"candidate that theme ever received, so replacing it would be a "
              f"downgrade. They are flagged verified=False in label_verification.csv - "
              f"use them for description, not for an interpretive claim.")
    print("  Override any of them in NAME_OVERRIDE only if you want to "
          "(and set NAME_OVERRIDE_K to this run's K).")
else:
    print("\n  Every label is assigned to its own theme by the optimal matching.")

try:
    import matplotlib.pyplot as plt
    C_OK = globals().get("C_GOOD", "#44AA99"); C_BD = globals().get("C_BAD", "#CC6677")
    lvs = [l for l, _, _, _ in LEVELS if (V.level == l).any()]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.5, 4.2),
                                 gridspec_kw=dict(width_ratios=[1.15, 1.6]))
    ver = [100 * V[V.level == l].verified.mean() for l in lvs]
    a1.bar(range(len(lvs)), ver, color=C_OK, width=.6)
    for i, v in enumerate(ver):
        a1.text(i, v + 1.5, f"{v:.0f}%", ha="center", fontsize=10)
    a1.axhline(100 * np.mean([1 / int(V[V.level == l].of_K.iloc[0]) for l in lvs]),
               color="#9AA3AB", ls="--", lw=1)
    a1.text(len(lvs) - .5, 3, "chance", fontsize=8.5, color="#6B747C", ha="right")
    a1.set_xticks(range(len(lvs)))
    a1.set_xticklabels([f"{l}\n(K={int(V[V.level==l].of_K.iloc[0])})" for l in lvs], fontsize=9)
    a1.set_ylim(0, 108)
    a1.set_ylabel("Labels kept on their own theme by the\noptimal assignment (%)", fontsize=9)
    for s in ("top", "right"): a1.spines[s].set_visible(False)
    for yi, l in enumerate(lvs):
        s = V[V.level == l]
        for f, col, mk, sz in ((True, C_OK, "o", 26), (False, C_BD, "X", 62)):
            g = s[s.verified == f]
            if len(g):
                a2.scatter(g.margin, np.full(len(g), yi) + (np.arange(len(g)) % 5 - 2) * .045,
                           s=sz, color=col, marker=mk, alpha=.8,
                           edgecolor="white", linewidth=.5, zorder=3)
    a2.axvline(0, color="#9AA3AB", lw=1.1, ls="--")
    a2.set_yticks(range(len(lvs))); a2.set_yticklabels(lvs, fontsize=9.5)
    a2.set_ylim(-.6, len(lvs) - .4); a2.invert_yaxis()
    a2.set_xlabel("Similarity to its own theme minus similarity to the nearest other theme",
                  fontsize=9)
    xl = max(.02, float(np.nanmax(np.abs(V.margin))) * 1.18); a2.set_xlim(-xl, xl)
    for s in ("top", "right", "left"): a2.spines[s].set_visible(False)
    fig.suptitle("Embedding-anchored verification of the theme labels", fontsize=13, y=.98)
    fig.tight_layout(rect=[0, 0, 1, .95])
    try:    save_fig(fig, "chart_label_verification")
    except Exception:
        fig.savefig(f"{POOLED_DIR}/chart_label_verification.png", dpi=300, bbox_inches="tight")
    plt.close(fig)
    print("  figure saved: chart_label_verification")
except Exception as _e:
    print(f"  (figure skipped: {type(_e).__name__}: {str(_e)[:80]})")
print("  full table: label_verification.csv")
print("=" * 78)


## Stage 32 — Team award and medal join

Joins each team's iGEM award and medal as structured metadata that is never embedded, so success information stays separate from the text.

In [ ]:
# @title
# ════════════════════════════════════════════════════════
# STAGE 32:  TEAM AWARD / MEDAL JOIN   (robust + self-diagnosing)
# INPUT:  meta_pool (Team per pooled row); team_awards.tsv (auto-fetched from
#         Drive id AWARD_DRIVE_ID, or any *award*.tsv on the checkpoint/Drive path)
# OUTPUT: pooled_award (medal/special/grand per response), award_match_report,
#         award_unmatched.csv, chart_pooled_awards.html (+ .png)
# ════════════════════════════════════════════════════════
# Award outcomes are STRUCTURED labels joined to each response by its team, never
# embedded. This version is tolerant: it auto-detects the column names and table
# shape (tidy "AwardType/AwardTitle" rows OR a wide "one row per team with a Medal
# column"), strips separators so "TeamName"=="Team_Name", falls back to fuzzy
# matching, and if the Year filter removes everything it uses all rows. It then
# PRINTS what it found (columns, sample registry teams, sample survey teams, and
# the closest registry name for each unmatched survey team) so any mismatch is
# visible at a glance. Use TEAM_NAME_MAP = {'survey name':'registry name', ...} to
# pin any stragglers. Ref: iGEM 2025 results registry; descriptive use only.

import os, re, glob, csv, difflib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go

AWARD_DRIVE_ID = globals().get('AWARD_DRIVE_ID', 'YOUR_TEAM_AWARDS_FILE_ID')
AWARD_YEAR     = globals().get('AWARD_YEAR', 2025)
TEAM_NAME_MAP  = globals().get('TEAM_NAME_MAP', {})      # {'survey team':'registry team'}
MEDAL_RANK     = globals().get('MEDAL_RANK', {'Gold':3,'Silver':2,'Bronze':1,'None':0})
FUZZY_CUTOFF   = float(globals().get('AWARD_FUZZY_CUTOFF', 0.84))

try: _ = meta_pool
except NameError: meta_pool = pload('pooled_meta')
if meta_pool is None: meta_pool = pload('pooled_meta')

print('=' * 64); print('  TEAM AWARD / MEDAL JOIN  (robust)'); print('=' * 64)

# ── 1. get the file (Drive auto-download, then local search) ──
def _download_drive():
    """Fetch the award file from Drive automatically (no manual step needed).
    Cached locally for the session; a fresh Colab re-fetches from Drive each run."""
    dest = f'{CHECKPOINT_DIR}/team_awards.tsv'
    if globals().get('AWARD_TSV') and os.path.exists(str(globals().get('AWARD_TSV'))):
        return
    if os.path.exists(dest):
        print(f'  Using award file already present: {dest}')
        return
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    def _ok(p):
        if not os.path.exists(p) or os.path.getsize(p) < 100:
            return False
        head = open(p, 'rb').read(200).lstrip().lower()
        return not (head.startswith(b'<!doctype') or head.startswith(b'<html'))
    try:
        import gdown
    except Exception:
        import subprocess, sys, importlib
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'], check=False)
        try:
            import importlib; gdown = importlib.import_module('gdown')
        except Exception:
            gdown = None
    methods = []
    if gdown is not None:
        methods = [
            ('id',  lambda: gdown.download(id=AWARD_DRIVE_ID, output=dest, quiet=True, fuzzy=True)),
            ('url', lambda: gdown.download(f'https://drive.google.com/uc?id={AWARD_DRIVE_ID}',
                                           dest, quiet=True, fuzzy=True)),
        ]
    for tag, fn in methods:
        try:
            fn()
            if _ok(dest):
                print(f'  Auto-downloaded team_awards.tsv from Drive ({tag}) -> {dest}')
                return
            if os.path.exists(dest) and not _ok(dest):
                os.remove(dest)   # got an HTML/error page, try next method
        except Exception as e:
            print(f'  (Drive {tag} attempt failed: {type(e).__name__})')
    print('  !! Could not auto-download from Drive. Check the file is shared')
    print('     "Anyone with the link"; an upload button will appear as a fallback.')
_download_drive()

def _find():
    for p in [globals().get('AWARD_TSV'), f'{CHECKPOINT_DIR}/team_awards.tsv',
              '/content/team_awards.tsv', '/content/drive/MyDrive/team_awards.tsv']:
        if p and os.path.exists(p): return p
    for pat in [f'{CHECKPOINT_DIR}/*[Aa]ward*.*sv', '/content/*[Aa]ward*.*sv',
                '/content/drive/MyDrive/**/*[Aa]ward*.*sv']:
        h = glob.glob(pat, recursive=True)
        if h: return sorted(h)[0]
    return None
_path = _find()

# Colab fallback: if still not found, let the user pick the file (foolproof)
if _path is None:
    try:
        import google.colab.files as _cf
        print('  team_awards.tsv not found automatically. Use the button to upload it:')
        _up = _cf.upload()
        for _fn, _data in _up.items():
            if _fn.lower().endswith(('.tsv', '.csv')):
                os.makedirs(CHECKPOINT_DIR, exist_ok=True)
                open(f'{CHECKPOINT_DIR}/team_awards.tsv', 'wb').write(_data)
                print(f'  Uploaded {_fn} -> {CHECKPOINT_DIR}/team_awards.tsv')
        _path = _find()
    except Exception:
        pass

def _norm(s):                       # strip ALL separators so TeamName==Team_Name
    return re.sub(r'[^a-z0-9]', '', str(s).strip().lower())

teams_pool = meta_pool['Team'].astype(str).tolist()
n = len(teams_pool)
uniq_survey = sorted(set(teams_pool))

registry = {}; reg_cols = {}
if _path is None:
    print('  !! No award file found. Put team_awards.tsv at')
    print(f'       {CHECKPOINT_DIR}/team_awards.tsv   (or set AWARD_TSV=...)')
    print('     Awards set to "Unknown"; downstream award cells will note this.')
else:
    # auto-detect delimiter
    with open(_path, 'r', encoding='utf-8', errors='ignore') as fh:
        sample = fh.read(4096)
    sep = '\t' if sample.count('\t') >= sample.count(',') else ','
    raw = pd.read_csv(_path, sep=sep, dtype=str).fillna('')
    raw.columns = [c.strip() for c in raw.columns]
    print(f'  File: {_path}')
    print(f'  Columns detected: {list(raw.columns)}  ({len(raw)} rows, sep={"TAB" if sep==chr(9) else "COMMA"})')

    def _col(*pats):
        for p in pats:
            for c in raw.columns:
                if re.fullmatch(p, c, flags=re.I): return c
        for p in pats:
            for c in raw.columns:
                if re.search(p, c, flags=re.I): return c
        return None
    team_col  = _col(r'team.?name', r'team', r'name')
    year_col  = _col(r'year', r'season')
    type_col  = _col(r'award.?type', r'type', r'category', r'kind')
    title_col = _col(r'award.?title', r'medal', r'award', r'title', r'prize', r'result')
    decision_col = _col(r'award.?decision', r'decision')   # winner vs nominee (iGEM special prizes)
    reg_cols = dict(team=team_col, year=year_col, type=type_col, title=title_col, decision=decision_col)
    print(f'  Mapped -> team={team_col!r} year={year_col!r} type={type_col!r} title={title_col!r} decision={decision_col!r}')
    if team_col is None:
        print('  !! Could not find a team-name column. Rename one column to "Team".')
    else:
        df_aw = raw.copy()
        # year filter, with fallback if it empties the table
        if year_col:
            yr = df_aw[df_aw[year_col].astype(str).str.extract(r'(\d{4})')[0] == str(AWARD_YEAR)]
            if len(yr) == 0:
                yrs = sorted(set(df_aw[year_col].astype(str).str.extract(r'(\d{4})')[0].dropna()))
                print(f'  (No rows for year {AWARD_YEAR}; years present={yrs}. Using ALL rows.)')
            else:
                df_aw = yr
        MED = {'gold':'Gold','silver':'Silver','bronze':'Bronze','none':'None','no medal':'None'}
        def _medal_of(v):
            return MED.get(str(v).strip().lower(), None)
        # decide shape: tidy (type col with 'medal') vs wide (title col holds medals directly)
        tidy = bool(type_col) and df_aw[type_col].astype(str).str.contains('medal', case=False).any()
        for _, r in df_aw.iterrows():
            tname = r.get(team_col, '')
            if not str(tname).strip(): continue
            rec = registry.setdefault(_norm(tname),
                  {'medal':'None','n_special':0,'grand':False,'n_nominated':0,'raw':str(tname)})
            if tidy:
                at = str(r.get(type_col, '')).strip().lower()
                if at == 'medal':
                    m = _medal_of(r.get(title_col, ''))
                    if m and MEDAL_RANK.get(m,0) > MEDAL_RANK.get(rec['medal'],0): rec['medal'] = m
                elif at == 'special':
                    _dec = str(r.get(decision_col, '')).strip().lower() if decision_col else ''
                    if _dec == 'nominee':
                        rec['n_nominated'] += 1      # nominated for a special prize (did not win)
                    else:
                        rec['n_special'] += 1        # won a special prize
                elif at in ('grand-prize','grand prize','finalist','winner'): rec['grand'] = True
                elif at in ('nominated','nomination','nominee'): rec['n_nominated'] += 1
            else:   # wide: title/medal column holds the medal; any 'special' column counts
                m = _medal_of(r.get(title_col, '')) if title_col else None
                if m and MEDAL_RANK.get(m,0) > MEDAL_RANK.get(rec['medal'],0): rec['medal'] = m
                for c in raw.columns:
                    if re.search(r'special', c, re.I):
                        val = str(r.get(c,'')).strip()
                        if val and val.lower() not in ('0','no','false','none',''):
                            rec['n_special'] += (int(val) if val.isdigit() else 1)
                    if re.search(r'nominat', c, re.I):
                        val = str(r.get(c,'')).strip()
                        if val and val.lower() not in ('0','no','false','none',''):
                            rec['n_nominated'] += (int(val) if val.isdigit() else 1)
        print(f'  Registry teams ({AWARD_YEAR if year_col else "all years"}): {len(registry)}')
        print(f'  Sample registry teams : {[registry[k]["raw"] for k in list(registry)[:6]]}')
        print(f'  Sample survey teams   : {uniq_survey[:6]}')

reg_keys = list(registry.keys())
def _lookup(team):
    if team in TEAM_NAME_MAP: team = TEAM_NAME_MAP[team]
    key = _norm(team)
    if key in registry: return registry[key]
    if reg_keys:
        # substring containment (handles small prefixes like leading 'U')
        for rk in reg_keys:
            if key and (key in rk or rk in key) and abs(len(key)-len(rk)) <= 3:
                return registry[rk]
        cand = difflib.get_close_matches(key, reg_keys, n=1, cutoff=FUZZY_CUTOFF)
        if cand: return registry[cand[0]]
    return None

medal = np.array(['Unknown']*n, dtype=object); special=np.zeros(n,int)
nominated=np.zeros(n,int); grand=np.zeros(n,bool)
matched, unmatched = set(), set()
for i,t in enumerate(teams_pool):
    rec = _lookup(t)
    if rec is not None:
        medal[i]=rec['medal']; special[i]=rec['n_special']
        nominated[i]=rec['n_nominated']; grand[i]=rec['grand']; matched.add(t)
    elif registry: unmatched.add(t)

pooled_award = dict(medal=medal, special=(special>0), n_special=special,
                    nominated=(nominated>0), n_nominated=nominated,
                    grand=grand, gold=(medal=='Gold'))
report = dict(n_responses=n, file=_path, n_registry=len(registry),
              matched_teams=sorted(matched), unmatched_teams=sorted(unmatched),
              match_rate=float(np.mean(medal!='Unknown')))
psave('pooled_award', pooled_award); psave('award_match_report', report)

print(f'\n  Survey teams: {len(uniq_survey)} | matched: {len(matched)} | unmatched: {len(unmatched)}')
print(f'  Response-level award coverage: {100*report["match_rate"]:.1f}%')
if unmatched and registry:
    print('  Closest registry name for unmatched survey teams (build TEAM_NAME_MAP from this):')
    rows_csv = []
    for t in sorted(unmatched)[:15]:
        cm = difflib.get_close_matches(_norm(t), reg_keys, n=1, cutoff=0.0)
        near = registry[cm[0]]['raw'] if cm else '(none)'
        score = difflib.SequenceMatcher(None,_norm(t), cm[0]).ratio() if cm else 0.0
        print(f"      '{t}'  ~  '{near}'   (score {score:.2f})")
        rows_csv.append((t, near, round(score,3)))
    with open(f'{POOLED_DIR}/award_unmatched.csv','w',newline='') as fh:
        w=csv.writer(fh); w.writerow(['survey_team','closest_registry','score']); w.writerows(rows_csv)
    print(f'  (full list -> pooled_combined/award_unmatched.csv)')
order=['Gold','Silver','Bronze','None','Unknown']
dist={m:int(np.sum(medal==m)) for m in order if np.any(medal==m)}
print('  Medal distribution:', dist, '| special-prize responses:', int(np.sum(special>0)))

# ── visual: iGEM 2025 awards at TEAM level (each team counted once) ──
import matplotlib.pyplot as plt
REF = 'iGEM 2025 Jamboree results, aggregated to TEAM level (each team counted once; descriptive only).'
NT = NAT_TEXT
mc = {'Gold':'#D4AF37','Silver':'#95A5A6','Bronze':'#CD7F32','None':'#BDC3C7','Unknown':'#ECECEC'}
# aggregate responses -> teams (award is a team property, not a response property)
_uniq = sorted(set(teams_pool)); _firstmed = {}
_sp = {t: False for t in _uniq}; _nm = {t: False for t in _uniq}
for _i, _t in enumerate(teams_pool):
    _firstmed.setdefault(_t, medal[_i])
    if special[_i] > 0: _sp[_t] = True
    if nominated[_i] > 0: _nm[_t] = True
t_medal = np.array([_firstmed[t] for t in _uniq], dtype=object)
t_special = np.array([_sp[t] for t in _uniq]); t_nom = np.array([_nm[t] for t in _uniq])
T = len(_uniq)
order_m = ['Gold','Silver','Bronze','None']
cats = [m for m in order_m if np.any(t_medal == m)]; vals = [int(np.sum(t_medal == m)) for m in cats]
t_won = int(np.sum(t_special)); t_nomo = int(np.sum(~t_special & t_nom))
t_none = int(np.sum(~t_special & ~t_nom & (t_medal != 'Unknown')))
groups = ['Won special', 'Nominated', 'No special']; gcounts = [t_won, t_nomo, t_none]
gcol = {'Won special':'#882255', 'Nominated':'#D9A44E', 'No special':'#BDC3C7'}
t_status = np.where(t_special, 'Won special', np.where(t_nom, 'Nominated', 'No special'))
medal_cats = [m for m in order_m if np.any(t_medal == m)]
matM = np.array([[np.sum((t_status == g) & (t_medal == m)) for m in medal_cats] for g in groups], float)
matf = matM / matM.sum(1, keepdims=True).clip(min=1)
fig, ax = plt.subplots(1, 3, figsize=(16, 9), facecolor='white', gridspec_kw=dict(wspace=0.32))
ax[0].bar(cats, vals, color=[mc[m] for m in cats], edgecolor='#888', linewidth=0.5)
for _k, _v in enumerate(vals): ax[0].text(_k, _v, str(_v), ha='center', va='bottom', fontsize=10.1)
ax[0].set_title(f'A  Medal distribution ({T} teams)', fontsize=12.3, fontweight='bold', color=NT, loc='left')
ax[0].set_ylabel('teams')
ax[1].bar(groups, gcounts, color=[gcol[g] for g in groups], edgecolor='white')
for _k, _v in enumerate(gcounts): ax[1].text(_k, _v, str(_v), ha='center', va='bottom', fontsize=10.1)
ax[1].set_title('B  Special-prize outcome (team level)', fontsize=12.3, fontweight='bold', color=NT, loc='left')
ax[1].set_ylabel('teams'); ax[1].tick_params(axis='x', labelsize=10.1)
ypos = np.arange(len(groups))[::-1]; leftc = np.zeros(len(groups))
for mi, m in enumerate(medal_cats):
    ax[2].barh(ypos, matf[:, mi], left=leftc, color=mc[m], edgecolor='white', label=m, height=0.6)
    leftc += matf[:, mi]
for gi in range(len(groups)):
    ax[2].text(1.015, ypos[gi], f"{int(matM[gi].sum())} teams", va='center', fontsize=9, color='#666')
ax[2].set_yticks(ypos); ax[2].set_yticklabels(groups, fontsize=10.1); ax[2].set_xlim(0, 1)
ax[2].set_xlabel('medal composition (share of teams)')
ax[2].set_title('C  Do special-prize teams earn higher medals?', fontsize=12.3, fontweight='bold', color=NT, loc='left')
ax[2].legend(ncol=4, fontsize=9, frameon=False, loc='upper center', bbox_to_anchor=(0.5, -0.16))
for a in ax:
    a.set_facecolor('white')
    for sp in ('top', 'right'): a.spines[sp].set_visible(False)
ax[0].grid(True, axis='y', color='#F0F0F0'); ax[1].grid(True, axis='y', color='#F0F0F0')
ax[0].set_axisbelow(True); ax[1].set_axisbelow(True)
fig.suptitle(f'iGEM 2025 Jamboree awards by TEAM  ({T} teams: {t_won} won special, {t_nomo} nominated, {t_none} none)',
             fontsize=14.6, fontweight='bold', color=NT, y=1.06)
fig.text(0.5, -0.06, REF, ha='center', fontsize=9, style='italic', color='#888')
plt.tight_layout(); save_fig(fig, 'chart_pooled_awards'); plt.close(fig)
print(f'  TEAM-level (2025, {T} teams): medals {dict(zip(cats,vals))}; special won={t_won}, nominated={t_nomo}, none={t_none}')
print('  Saved: chart_pooled_awards.png/svg  (TEAM-level medal + special-prize)')
print('=' * 64)


## Stage 33 - Team organisation metadata join (moved from NB3)

Official registry metadata (Region, Country, Section, Kind, IsRemote, Track) joined per team - moved here so NB2 owns every metadata join and NB3 stays render-only.


In [ ]:
# @title
# ==============================================================================
# STAGE 33  -  TEAM ORGANIZATION METADATA JOIN  (moved here from NB3 per P1:
#             NB2 owns ALL metadata joins; NB3 is render-only)  (official iGEM registry)
# INPUT:  pooled_meta (Team per pooled row); team_meta_full.tsv (official registry:
#         Region/Country/Kind/Section/IsRemote/Track/InstitutionName). Auto-found
#         in the same checkpoint folder as team_awards.tsv.
# OUTPUT: pooled_team_org (one row per SURVEY team: Region, Country, Section,
#         Kind, IsRemote, Track, n_institutions, match method).
# ==============================================================================
# Brings in REAL team-organization structure (official registry, not derived
# from text) so it can be tested against theme content / prize. Same tolerant
# auto-locate + fuzzy team-name matching pattern as Stage 32 (award join), so it
# survives Drive path differences. A team can appear across many registry years;
# we prefer the row matching AWARD_YEAR, else the most recent year on file.
# Ref: iGEM official team registry (public); descriptive use only.
# ------------------------------------------------------------------------------
# THREE FIXES IN THIS ROUND
#   1. It reused whatever `meta_pool` happened to be in the session. On a kernel
#      that had run an earlier version of the pipeline that global could hold a
#      DIFFERENT pooled set, so the team list came from one run and every other
#      artifact from another. It now reloads from disk and checks the row count
#      against the current labels - the same stale-artifact guard every stage has.
#   2. The banner printed "STAGE 46".
#   3. An empty match set crashed on `.set_index("Team")` (no such column in an
#      empty frame) instead of reporting that nothing matched.
# This stage has no theme levels and no noise: it is a team -> registry join, one
# row per team, and every pooled response's team is looked up whether or not that
# response ended up in a theme.
# ==============================================================================
import os, glob, difflib
import numpy as np
import pandas as pd

AWARD_YEAR = globals().get("AWARD_YEAR", CFG.award_year)
ORG_COLS = ("Region", "Country", "Section", "Kind", "IsRemote", "Track")

# ---- stale-artifact guard ----------------------------------------------------
meta_pool = require("pooled_meta", stage="Stage 33 (team org)").reset_index(drop=True)
N = len(meta_pool)
_sub = pload("pooled_labels_sub")
if _sub is not None and len(np.asarray(_sub)) != N:
    raise RuntimeError(
        f"[Stage 33] pooled_labels_sub has {len(np.asarray(_sub))} rows but pooled_meta "
        f"has {N}. One of them is from an EARLIER run - re-run Stage 23H before the "
        f"metadata joins, or the team list and the themes describe different people.")
if "Team" not in meta_pool.columns:
    raise RuntimeError("[Stage 33] pooled_meta has no Team column.")

print("=" * 66)
print("  STAGE 33  TEAM ORGANIZATION METADATA JOIN")
print("=" * 66)

_MIRROR = globals().get("MIRROR_ROOT") or None

def _find_team_meta():
    ck = CFG.checkpoint_dir
    cands = [globals().get("TEAM_META_TSV"),
             os.path.join(ck, "team_meta_full.tsv"),
             (os.path.join(_MIRROR, "team_meta_full.tsv") if _MIRROR else None),
             "/content/team_meta_full.tsv",
             "/content/drive/MyDrive/team_meta_full.tsv"]
    for p in cands:
        if p and os.path.exists(p):
            return p
    for pat in [os.path.join(ck, "*team_meta*.*sv"),
                (os.path.join(_MIRROR, "*team_meta*.*sv") if _MIRROR else None),
                "/content/*team_meta*.*sv",
                "/content/drive/MyDrive/**/*team_meta*.*sv"]:
        if not pat:
            continue
        hits = glob.glob(pat, recursive=True)
        if hits:
            return sorted(hits)[0]
    return None

_path = _find_team_meta()
if _path is None:
    print(f"  !! team_meta_full.tsv not found (checked {CFG.checkpoint_dir}, the mirror,")
    print("     /content and Drive). Upload it next to team_awards.tsv, then re-run.")
    print("     pooled_team_org is set to None; Stage 33b will leave the org columns empty.")
    psave("pooled_team_org", None)
else:
    reg = pd.read_csv(_path, sep="\t")
    if "Team" not in reg.columns:
        raise RuntimeError(f"[Stage 33] {_path} has no 'Team' column "
                           f"(columns: {list(reg.columns)[:12]}).")
    print(f"  loaded {_path}  ({len(reg)} rows, {reg['Team'].nunique()} unique team names)")
    _miss = [c for c in ORG_COLS if c not in reg.columns]
    if _miss:
        print(f"  note: the registry file has no {_miss} column(s); those stay empty "
              f"rather than being guessed.")

    def _norm(s):
        return str(s).strip().lower().replace("_", "").replace("-", "").replace(" ", "")

    reg["_norm"] = reg["Team"].map(_norm)
    survey_teams = sorted(meta_pool["Team"].astype(str).str.strip().unique())
    reg_by_norm = {}
    for n_, grp in reg.groupby("_norm"):
        if "Year" in grp.columns:
            g = grp.sort_values("Year", ascending=False)   # prefer AWARD_YEAR, else most recent
            row = g[g["Year"] == AWARD_YEAR]
            reg_by_norm[n_] = (row.iloc[0] if len(row) else g.iloc[0])
        else:
            reg_by_norm[n_] = grp.iloc[0]

    TEAM_ORG_NAME_MAP = globals().get("TEAM_ORG_NAME_MAP", {})   # {'survey name':'registry name'}
    rows, matched, unmatched = [], [], []
    reg_norms = list(reg_by_norm.keys())
    for t in survey_teams:
        key = _norm(TEAM_ORG_NAME_MAP.get(t, t))
        row = reg_by_norm.get(key)
        method = "exact"
        if row is None:
            close = difflib.get_close_matches(key, reg_norms, n=1, cutoff=0.84)
            if close:
                row = reg_by_norm[close[0]]; method = "fuzzy"
        if row is None:
            unmatched.append(t); continue
        inst = str(row.get("InstitutionName") or "")
        n_inst = max(1, len([x for x in inst.replace("\r", "\n").split("\n") if x.strip()]))
        rec = dict(Team=t, RegistryYear=row.get("Year"), n_institutions=n_inst, match=method)
        for c in ORG_COLS:
            rec[c] = row.get(c) if c in reg.columns else np.nan
        rows.append(rec)
        matched.append(t)

    if not rows:
        print(f"  !! NONE of the {len(survey_teams)} survey teams matched the registry.")
        print("     That is a name-format mismatch, not a result. Check the Team column in")
        print("     both files and set TEAM_ORG_NAME_MAP = {'survey name':'registry name'}.")
        psave("pooled_team_org", None)
    else:
        org = pd.DataFrame(rows).set_index("Team")
        psave("pooled_team_org", org)
        print(f"  matched {len(matched)}/{len(survey_teams)} survey teams "
              f"({sum(r['match'] == 'exact' for r in rows)} exact, "
              f"{sum(r['match'] == 'fuzzy' for r in rows)} fuzzy)")
        if unmatched:
            print(f"  unmatched ({len(unmatched)}): {unmatched[:15]}"
                  f"{' ...' if len(unmatched) > 15 else ''}")
            print("  -> add these to TEAM_ORG_NAME_MAP = {'survey name':'registry name'} "
                  "and re-run if needed.")
        print("  coverage over the matched teams: " + " | ".join(
            f"{c} {org[c].notna().mean():.0%}" for c in ORG_COLS if c in org.columns))
        _resp_cov = float(meta_pool["Team"].astype(str).str.strip().isin(org.index).mean())
        print(f"  coverage over pooled RESPONSES (what the analyses actually use): "
              f"{_resp_cov:.0%} of {N}")
print("=" * 66)


## Stage 33b — `pooled_master`: the one canonical metadata table

One row per pooled response carrying text, **both** theme levels with their verified LLM names, sentiment, award and team organisation. **Every explorer hover and every descriptive analysis reads this single artifact** (also exported as CSV).

Noise is recorded, not hidden: a response HDBSCAN left with no theme has `main_id` / `sub_id` = `-1`, the name `(no theme)`, and `has_main` / `has_sub` = `False`. The row is kept — it is a real response with real text, sentiment and award — but it is not a member of any theme. **Every downstream count must filter on `has_sub` / `has_main`, not on the id.** Every input vector is length-checked against `pooled_meta` before it becomes a column, so a table one run out of date stops the build instead of being broadcast in.


In [ ]:
# @title
# ==============================================================================
# STAGE 33b  -  POOLED_MASTER  (P1 metadata hub: ONE canonical row-level table)
# Everything the explorers, hover cards and every descriptive analysis need, in
# one artifact: response text + BOTH theme levels with their verified LLM names +
# sentiment + award + team organisation (Region/Country/Section/Kind/IsRemote/
# Track) + team size. Saved as pooled_master.pkl + pooled_master.csv.
# NB3 reads ONLY this.
# ------------------------------------------------------------------------------
# TWO LEVELS. main and sub; the mid level was removed in Stage 23H, so mid_id /
# mid_name are gone from this table. Anything downstream that still reads them
# will now fail loudly instead of silently carrying a column of stale ids.
#
# NOISE IS RECORDED, NOT HIDDEN. HDBSCAN leaves some responses with no theme. In
# this table that is main_id / sub_id == -1, name "(no theme)", and the boolean
# has_main / has_sub == False. The row is kept - it is a real response with real
# text, sentiment and award - but it is NOT a member of any theme, so every
# prevalence, mean and count downstream must filter on has_sub (or has_main)
# rather than on the id. Writing the noise rows out with a blank name, as before,
# meant they were indistinguishable from a theme whose label had been withheld.
#
# EVERY INPUT IS LENGTH-CHECKED. This table is the single source for all
# downstream work, so a vector that is one run out of date must stop the build,
# not be broadcast into a column.
#
# ------------------------------------------------------------------------------
# THE WIDE VIEW NOW COMES FROM STAGE 23d, NOT FROM NOTEBOOK 3
#
#   Notebook 3 used to compute the recovery of the noise rows itself and write
#   four columns back onto this table. That put a membership DECISION inside the
#   figures notebook, duplicated machinery Stage 23d already had, and meant the
#   two notebooks could disagree about who belongs to what. Stage 23d now decides
#   it, once, and this stage carries the answer through under EXACTLY the column
#   names Notebook 3 already reads:
#
#     theme_tier      core | assigned | assigned-multi | unassigned
#     sub_assigned    the recovered theme id, or -1
#     sub_conf        the confidence behind that assignment
#     sub_multi       every theme the response clauses matched, ";"-separated
#     sub_id_wide     sub_id where core, sub_assigned where assigned, else -1
#     sub_name_wide / has_sub_wide      and the same three for main
#
#   Notebook 3 recovery cell can therefore be deleted with no other change.
#
#   ONE DELIBERATE IMPROVEMENT. Notebook 3 set the MAIN wide columns equal to the
#   core-only main columns, so a single figure showing both levels compared 1102
#   main rows against 1302 sub rows and printed the two Cramer V side by side as
#   if they were comparable. Here main_id_wide FOLLOWS sub_id_wide through the
#   sub -> main map, because MAIN is Ward on the SUB centroids and a recovered
#   sub-theme carries its main-theme with it. Both levels now describe the same
#   rows.
# ==============================================================================
import numpy as np, pandas as pd

NO_THEME = "(no theme)"

# ---- stale-artifact guard (the same block opens every stage) ------------------
def _labels(n_rows, stage, levels=("main", "sub")):
    _KEY = {"main": "pooled_labels_main7", "sub": "pooled_labels_sub"}
    out = {}
    for lv in levels:
        a = pload(_KEY[lv])
        if a is None:
            raise RuntimeError(f"[{stage}] {_KEY[lv]} is missing - run Stage 23H first.")
        a = np.asarray(a)
        if len(a) != n_rows:
            raise RuntimeError(
                f"[{stage}] {_KEY[lv]} has {len(a)} rows but the pooled set has "
                f"{n_rows}. That artifact belongs to an EARLIER clustering. Re-run "
                f"Stage 23H (then Stage 30) before this stage.")
        out[lv] = a
    return out

def _K(lab):
    lab = np.asarray(lab)
    return int(lab.max()) + 1 if (lab >= 0).any() else 0

def _vec(name, n_rows, default=None, dtype=float):
    """Load a per-response vector and refuse to use one of the wrong length."""
    v = pload(name)
    if v is None:
        if default is None:
            raise RuntimeError(f"[Stage 33b] {name} is missing.")
        return np.asarray(default)
    v = np.asarray(v, dtype=dtype) if dtype is not None else np.asarray(v)
    if len(v) != n_rows:
        raise RuntimeError(
            f"[Stage 33b] {name} has {len(v)} rows but the pooled set has {n_rows}. "
            f"It is from an EARLIER run - re-run the stage that produces it.")
    return v
# ------------------------------------------------------------------------------

meta = require("pooled_meta", stage="Stage 33b (master)").reset_index(drop=True)
N = len(meta)
LAB = _labels(N, "Stage 33b")
main, sub = LAB["main"], LAB["sub"]
KMAIN, KSUB = _K(main), _K(sub)

# ---- names: the dict must describe THIS partition ----------------------------
LLM = {}
for lv, key, K in (("main", "pooled_main_llm", KMAIN), ("sub", "pooled_sub_llm", KSUB)):
    d = pload(key) or {}
    d = {int(k): v for k, v in d.items()}
    if set(d) != set(range(K)):
        raise RuntimeError(
            f"[Stage 33b] {key} covers {len(d)} clusters but the {lv} partition has {K}. "
            f"The names were written for a different clustering - re-run Stage 30 (and "
            f"Stage 30v) before building the master table.")
    LLM[lv] = d

def _names_for(lv, ids):
    d = LLM[lv]
    return [NO_THEME if c < 0 else (d[int(c)].get("name", "") or "") for c in ids]

sent = _vec("pooled_sentiment_response", N, default=np.full(N, np.nan))
aw = pload("pooled_award") or {}
org = pload("pooled_team_org")

medal = np.asarray(aw.get("medal", np.array(["Unknown"] * N, dtype=object)), dtype=object)
special = np.asarray(aw.get("special", np.zeros(N, bool)), dtype=bool)
nominated = np.asarray(aw.get("nominated", np.zeros(N, bool)), dtype=bool)
for _nm_, _a_ in (("medal", medal), ("special", special), ("nominated", nominated)):
    if len(_a_) != N:
        raise RuntimeError(f"[Stage 33b] pooled_award['{_nm_}'] has {len(_a_)} rows but the "
                           f"pooled set has {N}. Re-run Stage 32.")

_df = get_df()
_pid = {}
for _, r in _df.iterrows():
    _pid.setdefault(r["Participant"], r)
RAW = dict(zip(CFG.short_labels, CFG.text_cols))
txt = []
for _, m in meta.iterrows():
    r = _pid.get(m["Participant"])
    v = "" if r is None else r.get(RAW[m["question"]], "")
    txt.append("" if (v is None or str(v).strip().lower() == "nan") else str(v))

M = pd.DataFrame(dict(
    row_id=np.arange(N),
    Participant=meta["Participant"].astype(str),
    Team=meta["Team"].astype(str).str.strip(),
    question=meta["question"],
    text=txt,
    main_id=main, main_name=_names_for("main", main),
    sub_id=sub,   sub_name=_names_for("sub", sub),
    has_main=(main >= 0), has_sub=(sub >= 0),
    sentiment=sent,
    medal=medal, special=special, nominated=nominated,
))
M["team_size"] = M.groupby("Team")["Participant"].transform("nunique")
# provenance: which rows defined the themes (core) vs were assigned post-hoc,
# and how long each response is - both are needed by the NB3 evidence guard.
M["n_words"] = (meta["n_words"].values if "n_words" in meta.columns
                else [len(str(t).split()) for t in txt])

# ---- the wide view, from Stage 23d --------------------------------------------
MB = pload("pooled_membership")
if MB is None:
    raise RuntimeError(
        "[Stage 33b] pooled_membership is missing. Run Stage 23d (and Stage 23M if "
        "you use it) before building the master table. This stage no longer invents "
        "a wide view of its own, and Notebook 3 no longer computes one either, so "
        "building without it would silently drop every recovered response.")
MB = MB[MB["source"] == "pooled"].sort_values("row_id").reset_index(drop=True)
if len(MB) != N:
    raise RuntimeError(
        f"[Stage 33b] pooled_membership has {len(MB)} pooled rows but the pooled set "
        f"has {N}. It is from an EARLIER clustering - re-run Stage 23d.")
if not (MB["row_id"].values == np.arange(N)).all():
    raise RuntimeError("[Stage 33b] pooled_membership row_ids are not 0..N-1 in order.")
if not (np.asarray(MB["sub_id"].values) == np.asarray(sub)).all():
    raise RuntimeError(
        "[Stage 33b] pooled_membership disagrees with pooled_labels_sub about the "
        "primary partition. One of them is stale - re-run Stage 23H then Stage 23d.")

M["theme_tier"]        = MB["tier"].astype(str).values
M["sub_assigned"]      = MB["assigned_sub"].astype(int).values
M["sub_conf"]          = MB["confidence"].astype(float).values
M["sub_multi"]         = (MB["multi_subs"].astype(str).values
                          if "multi_subs" in MB.columns else "")
M["assignment_rule"]   = MB["rule"].astype(str).values
M["unassigned_reason"] = MB["unassigned_reason"].astype(str).values
M["is_substantive"]    = MB["is_substantive"].astype(bool).values

# sub_wide, and main_wide FOLLOWING it through the hierarchy
_sw = MB["sub_wide"].astype(int).values
_mw = MB["main_wide"].astype(int).values
M["has_sub_wide"]   = _sw >= 0
M["sub_id_wide"]    = _sw
M["sub_name_wide"]  = _names_for("sub", _sw)
M["has_main_wide"]  = _mw >= 0
M["main_id_wide"]   = _mw
M["main_name_wide"] = _names_for("main", _mw)

# is_core is now MEASURED, not asserted. It used to be hard-coded True on every
# row, which made the provenance column useless: a recovered response was
# indistinguishable from one that defined its theme.
M["is_core"] = M["theme_tier"].values == "core"

if org is not None and len(org):
    _o = org.copy(); _o.index = _o.index.astype(str).str.strip()
    for col in ("Region", "Country", "Section", "Kind", "IsRemote", "Track"):
        M[col] = M["Team"].map(_o[col]) if col in _o.columns else np.nan
    _cov = float(M["Region"].notna().mean()) if "Region" in M.columns else 0.0
else:
    _cov = 0.0
    print("  (pooled_team_org missing - run Stage 33 first; org columns left empty)")

# ---- hard integrity checks: this table is the single source for ALL downstream work
assert len(M) == N == len(main) == len(sub), "pooled_master row misalignment"
assert (M["main_id"] >= -1).all() and (M["main_id"] < KMAIN).all(), "main_id out of range"
assert (M["sub_id"] >= -1).all() and (M["sub_id"] < KSUB).all(), "sub_id out of range"
assert (M.loc[M.has_main, "main_name"].astype(str).str.strip() != "").all(), \
    "a themed row has an empty main name - run Stage 30"
assert (M.loc[M.has_sub, "sub_name"].astype(str).str.strip() != "").all(), \
    "a themed row has an empty sub name - run Stage 30"
assert (M.loc[~M.has_main, "main_name"] == NO_THEME).all(), "noise row carries a theme name"
assert (M.loc[~M.has_sub, "sub_name"] == NO_THEME).all(), "noise row carries a theme name"
assert "mid_id" not in M.columns and "mid_name" not in M.columns, \
    "the mid level was removed in Stage 23H but is still being written"
# ---- the wide view must be self-consistent, or NB3 draws a lie ----------------
assert set(M["theme_tier"]) <= {"core", "assigned", "assigned-multi", "unassigned"}, \
    f"unexpected theme_tier value(s): {sorted(set(M['theme_tier']))}"
assert (M.loc[M.theme_tier == "core", "has_sub"]).all(), \
    "a row is tier core but has no sub-theme"
assert not (M.loc[M.theme_tier != "core", "has_sub"]).any(), \
    "a recovered or unassigned row still carries a core sub-theme"
assert not (M["has_sub_wide"] & (M["sub_id_wide"] < 0)).any(), \
    "a row is flagged wide-themed but carries sub_id_wide -1"
assert not (M["has_main_wide"] & (M["main_id_wide"] < 0)).any(), \
    "a row is flagged wide-themed but carries main_id_wide -1"
assert (M["has_sub_wide"] == M["has_main_wide"]).all(), \
    "the two levels disagree about which rows have a theme - main_id_wide must " \
    "follow sub_id_wide through the hierarchy, or every two-level figure is " \
    "comparing different row sets"
assert (M.loc[M.has_sub, "sub_id_wide"].values
        == M.loc[M.has_sub, "sub_id"].values).all(), \
    "the recovery changed a core row theme - it must only ever add"
assert (M.loc[M.theme_tier == "unassigned", "unassigned_reason"]
        .astype(str).str.strip() != "").all(), \
    "an unassigned row has no recorded reason"
assert (M["sub_id_wide"] < KSUB).all() and (M["main_id_wide"] < KMAIN).all(), \
    "a wide id is out of range"

psave("pooled_master", M)
M.to_csv(f"{POOLED_DIR}/pooled_master.csv", index=False)
print("=" * 70)
print(f"  POOLED_MASTER  {M.shape[0]} rows x {M.shape[1]} cols")
print(f"  teams={M['Team'].nunique()}  participants={M['Participant'].nunique()}")
print(f"  main K={KMAIN}: {int(M.has_main.sum())} rows in a theme, "
      f"{int((~M.has_main).sum())} with none ({(~M.has_main).mean():.1%})")
print(f"  sub  K={KSUB}: {int(M.has_sub.sum())} rows in a theme, "
      f"{int((~M.has_sub).sum())} with none ({(~M.has_sub).mean():.1%})")
print(f"  sentiment scored={int(np.isfinite(sent).sum())}  "
      f"award-known={(M['medal'] != 'Unknown').sum()}")
print(f"  org coverage (Region present): {_cov:.0%}")
print("-" * 70)
print("  THE WIDE VIEW (from Stage 23d; Notebook 3 no longer computes it)")
_tc = M["theme_tier"].value_counts()
for _t in ("core", "assigned", "assigned-multi", "unassigned"):
    if _t in _tc:
        _d = M[M.theme_tier == _t]
        print(f"    {_t:<16}{int(_tc[_t]):>6}  ({_tc[_t]/N:>5.1%})   "
              f"substantive {int(_d.is_substantive.sum()):>4}")
print(f"    core-only view : {int(M.has_sub.sum())} rows"
      f"   ({M.has_sub.mean():.1%})")
print(f"    wide view      : {int(M.has_sub_wide.sum())} rows"
      f"   ({M.has_sub_wide.mean():.1%})   <- has_sub_wide / sub_id_wide")
print("    Both levels describe the SAME rows: main_id_wide follows sub_id_wide")
print("    through the sub->main map, so a two-level figure is no longer comparing")
print("    1102 main rows against 1302 sub rows.")
_nm = int((M["sub_multi"].astype(str).str.contains(";")).sum())
if _nm:
    print(f"    {_nm} response(s) carry more than one theme (Stage 23M clause split)")
print(f"  NOTE for every downstream count: filter on has_sub / has_main, NOT on the id.")
print(f"       A row with id -1 is a real response that belongs to no theme; counting it")
print(f"       as a theme member would inflate that theme, and counting it as a theme of")
print(f"       its own would invent one.")
print(f"  -> pooled_master.pkl + pooled_master.csv   (NB3 + all analyses read THIS)")
print("=" * 70)

## Stage 40 — Award-success correlation COMPUTE

A prize is a **team** property, so the analysis is at the team level. Per cluster, Panel A regresses the team prize score (0/1/2) on the team's **mention rate** for that cluster, adjusted for team size; Panel B regresses it on the team's mean sentiment about that cluster, among teams that mentioned it. 95% CIs come from a team bootstrap and BH-FDR is reported alongside. Computed for **main and sub** (the mid level is gone, which also removes a third of the multiple-comparison burden). No plotting — NB3 renders; award optional, empty tables if there is no prize signal.

A response left as noise cannot be a "mention" of any theme, so it never enters the mention matrix — but it stays in the team-size **denominator**, because a mention rate is "what fraction of this team's people raised this topic" and a member whose answers were unclusterable is still a member. Teams whose award could not be matched in the registry are excluded entirely rather than scored as prize = 0, which would count missing data as a loss.


In [ ]:
# @title
# ==============================================================================
# STAGE 40  -  AWARD-SUCCESS CORRELATION *COMPUTE*  (TEAM-LEVEL OLS + team bootstrap)
# A prize is a TEAM property, so the analysis is at the TEAM level. The previous
# model `prize ~ mention + C(Team)` was mis-specified: prize is CONSTANT within a
# team, so the team dummies absorbed 100% of it and every topic beta collapsed to
# ~1e-14 (numerical zero). Here, per cluster, the predictor is the TEAM's mention
# RATE (fraction of the team's participants who mentioned the cluster) and the
# TEAM's mean sentiment about it; the outcome is the team prize score (0/1/2).
#   Panel A:  prize ~ mention_rate + team_size        (all teams; size-adjusted)
#   Panel B:  prize ~ team_theme_sentiment            (teams that mentioned it)
# 95% CI = team bootstrap (resample teams); significance = bootstrap CI excludes 0;
# BH-FDR also reported. Colour/`vader` = each cluster's OWN mean VADER (the theme's
# sentiment), NOT the mentioners' overall positivity. Computed for main and sub.
# ------------------------------------------------------------------------------
# TWO LEVELS. The mid level was removed in Stage 23H, so this stage models main
# and sub only - which also removes a third of the multiple-comparison burden the
# BH-FDR correction has to absorb.
#
# NOISE. A response HDBSCAN left with no theme (-1) cannot be a "mention" of any
# theme, so it never enters the mention matrix. It DOES stay in the team-size
# denominator: a mention rate is "what fraction of this team's people raised this
# topic", and a member whose answers were unclusterable is still a member of the
# team. Dropping them from the denominator would silently inflate every rate for
# teams with a lot of noise. Both counts are printed.
# ==============================================================================
import numpy as np, pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

MIN_TEAMS = 5      # a cluster must be mentioned by >= this many teams to be modelled
N_BOOT    = 2000   # team bootstrap resamples for the 95% CI
set_all_seeds()
_rng = np.random.default_rng(CFG.seed)

# ---- stale-artifact guard (the same block opens every stage) ------------------
def _labels(n_rows, stage, levels=("main", "sub")):
    _KEY = {"main": "pooled_labels_main7", "sub": "pooled_labels_sub"}
    out = {}
    for lv in levels:
        a = pload(_KEY[lv])
        if a is None:
            raise RuntimeError(f"[{stage}] {_KEY[lv]} is missing - run Stage 23H first.")
        a = np.asarray(a)
        if len(a) != n_rows:
            raise RuntimeError(
                f"[{stage}] {_KEY[lv]} has {len(a)} rows but the pooled set has "
                f"{n_rows}. That artifact belongs to an EARLIER clustering. Re-run "
                f"Stage 23H (then Stage 30) before this stage.")
        out[lv] = a
    return out

def _K(lab):
    lab = np.asarray(lab)
    return int(lab.max()) + 1 if (lab >= 0).any() else 0
# ------------------------------------------------------------------------------

meta  = require("pooled_meta", stage="Stage 40 (correlation compute)").reset_index(drop=True)
N = len(meta)
award = require("pooled_award", stage="Stage 40 (correlation compute)")
LAB = _labels(N, "Stage 40")
lab_main, lab_sub = LAB["main"], LAB["sub"]

sent_resp = pload("pooled_sentiment_response")
if sent_resp is None:
    raise RuntimeError("[Stage 40] missing pooled_sentiment_response <- NB2 Stage 26B. "
                       "Run it first.")
sent_resp = np.asarray(sent_resp, dtype=float)
if len(sent_resp) != N:
    raise RuntimeError(f"[Stage 40] pooled_sentiment_response has {len(sent_resp)} rows but "
                       f"the pooled set has {N} - re-run Stage 26B.")

# ---- display names: the VERIFIED label first, then Stage 30's raw name --------
NAMES = {}
for lv, K in (("main", _K(lab_main)), ("sub", _K(lab_sub))):
    llm = {int(k): v for k, v in (pload(f"pooled_{lv}_llm") or {}).items()}
    flat = {int(k): str(v) for k, v in (pload(f"pooled_{lv}_name") or {}).items()}
    if llm and set(llm) != set(range(K)):
        raise RuntimeError(f"[Stage 40] pooled_{lv}_llm covers {len(llm)} clusters but the "
                           f"{lv} partition has {K}. Re-run Stage 30 / 30v.")
    NAMES[lv] = {c: ((llm.get(c) or {}).get("name") or flat.get(c) or f"{lv[0].upper()}{c}")
                 for c in range(K)}

special   = np.asarray(award.get("special",   np.zeros(N, bool)), dtype=bool)
nominated = np.asarray(award.get("nominated", np.zeros(N, bool)), dtype=bool)
_medal_arr = np.asarray(award.get("medal", np.array(["Unknown"] * N, dtype=object)),
                        dtype=object)
for _nm_, _a_ in (("special", special), ("nominated", nominated), ("medal", _medal_arr)):
    if len(_a_) != N:
        raise RuntimeError(f"[Stage 40] pooled_award['{_nm_}'] has {len(_a_)} rows but the "
                           f"pooled set has {N}. Re-run Stage 32.")
prize_row = np.where(special, 2, np.where(nominated, 1, 0)).astype(float)
teams = meta["Team"].astype(str).str.strip().values
parts = meta["Participant"].astype(str).values

print("=" * 92)
for lv, lab in (("main", lab_main), ("sub", lab_sub)):
    print(f"  {lv:<5} K={_K(lab):<3}  {int((lab >= 0).sum())} responses in a theme, "
          f"{int((lab < 0).sum())} with none ({(lab < 0).mean():.1%}) - noise is never a mention")

# ROUND-2 FIX (fairness): teams whose award could NOT be matched in the registry
# carry medal == 'Unknown' with special/nominated == False, so they entered the
# models as prize = 0 - counting MISSING DATA as a LOSS and biasing every beta
# toward "themes don't matter". Exclude unknown-award rows from the award models.
_known = _medal_arr != "Unknown"
if (~_known).any():
    print(f"  award-unknown rows excluded from the models: {int((~_known).sum())} responses "
          f"/ {len(set(teams[~_known]))} team(s)  (missing registry match is not a loss)")
    prize_row = prize_row[_known]
    teams = teams[_known]
    parts = parts[_known]
    sent_resp = sent_resp[_known]
    lab_main = lab_main[_known]; lab_sub = lab_sub[_known]

def _boot_ci(X, y, col, B):
    """Fast team-bootstrap percentile CI for OLS coefficient `col` (X includes the intercept)."""
    n = len(y); p = X.shape[1]; out = np.empty(B); out[:] = np.nan
    for b in range(B):
        ix = _rng.integers(0, n, n)
        try:
            beta, _r, rank, _s = np.linalg.lstsq(X[ix], y[ix], rcond=None)
            if rank == p:
                out[b] = beta[col]
        except Exception:
            pass
    out = out[~np.isnan(out)]
    if len(out) < B * 0.5:
        return np.nan, np.nan
    return float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5))

def _forest_for(units, lv):
    U = _K(units)
    base = pd.DataFrame({"Team": teams, "Participant": parts, "prize": prize_row,
                         "sent": sent_resp, "unit": units})
    # the team frame is built from EVERY row: a participant whose answers were all
    # left as noise is still a member of the team and still in the denominator.
    team_prize = base.groupby("Team")["prize"].max()
    team_size  = base.groupby("Team")["Participant"].nunique()
    t_idx   = team_prize.index.values
    prize_t = team_prize.values.astype(float)
    sz      = team_size.reindex(t_idx).values.astype(float)
    size_z  = (sz - sz.mean()) / (sz.std() + 1e-9)
    T = len(t_idx)
    # mentions come from THEMED rows only - '-1' is not a topic anyone mentioned
    themed = base[base["unit"] >= 0]
    if not len(themed):
        return pd.DataFrame(), T
    ment_pp = (pd.crosstab(themed["Participant"], themed["unit"]) > 0).astype(int)
    p2t = base.drop_duplicates("Participant").set_index("Participant")["Team"]
    ment_pp["_t"] = p2t.reindex(ment_pp.index).values
    team_ment = ment_pp.groupby("_t").sum()                                  # teams x units
    team_rate = team_ment.div(team_size, axis=0).reindex(t_idx).fillna(0.0)  # teams x units

    rows = []
    for u in range(U):
        if u not in team_ment.columns:
            continue
        n_teams_u = int((team_ment[u].reindex(t_idx).fillna(0) > 0).sum())
        if n_teams_u < MIN_TEAMS:
            continue
        rate = team_rate[u].values.astype(float)
        theme_sent = float(np.nanmean(sent_resp[units == u]))   # NaN-safe (26B may emit NaN)
        # Panel A: prize ~ mention_rate + team_size
        ba = sea = pa = la = ha = np.nan
        if np.unique(rate).size > 1 and np.unique(prize_t).size > 1:
            d = pd.DataFrame({"prize": prize_t, "rate": rate, "size": size_z})
            try:
                fa = smf.ols("prize ~ rate + size", data=d).fit()
                ba, sea, pa = float(fa.params["rate"]), float(fa.bse["rate"]), float(fa.pvalues["rate"])
            except Exception:
                pass
            la, ha = _boot_ci(np.column_stack([np.ones(T), rate, size_z]), prize_t, 1, N_BOOT)
        # Panel B: prize ~ team theme-sentiment (only teams that mentioned the cluster)
        bb = seb = pb = lb = hb = np.nan
        tts = base[units == u].groupby("Team")["sent"].mean().reindex(t_idx)  # mean skips NaN
        mm = tts.notna().values
        if mm.sum() >= MIN_TEAMS and np.unique(prize_t[mm]).size > 1 and np.unique(tts.values[mm]).size > 1:
            d = pd.DataFrame({"prize": prize_t[mm], "sent": tts.values[mm]})
            try:
                fb = smf.ols("prize ~ sent", data=d).fit()
                bb, seb, pb = float(fb.params["sent"]), float(fb.bse["sent"]), float(fb.pvalues["sent"])
            except Exception:
                pass
            lb, hb = _boot_ci(np.column_stack([np.ones(int(mm.sum())), tts.values[mm]]),
                              prize_t[mm], 1, N_BOOT)
        rows.append(dict(unit=int(u), name=NAMES[lv].get(u, f"{lv[0].upper()}{u}"),
                         n_teams=n_teams_u, n_mentions=int((units == u).sum()),
                         beta_topic=ba, se_topic=sea, p_topic=pa, ci_topic_lo=la, ci_topic_hi=ha,
                         beta_sent=bb, se_sent=seb, p_sent=pb, ci_sent_lo=lb, ci_sent_hi=hb,
                         theme_sentiment=theme_sent, vader=theme_sent))
    if not rows:
        return pd.DataFrame(), T
    res = pd.DataFrame(rows)
    for pre in ("topic", "sent"):
        res[f"sig_{pre}"] = ((res[f"ci_{pre}_lo"] > 0) | (res[f"ci_{pre}_hi"] < 0)).fillna(False)
        m = res[f"p_{pre}"].notna(); res[f"q_{pre}"] = np.nan
        if m.any():
            res.loc[m, f"q_{pre}"] = multipletests(res.loc[m, f"p_{pre}"].values,
                                                   method="fdr_bh")[1]
    res = res.sort_values("beta_topic", ascending=False).reset_index(drop=True)
    res.attrs["n_teams"] = T
    return res, T

LEVELS = [("main", lab_main), ("sub", lab_sub)]

# the mid-level forest belongs to a hierarchy that no longer exists; leaving the
# artifact on disk lets NB3 render a table from a partition nothing else uses.
try:
    if pload("pooled_correlation_forest_mid") is not None:
        invalidate("pooled_correlation_forest_mid")
        print("  removed the stale pooled_correlation_forest_mid (the mid level is gone).")
except Exception:
    pass

if (prize_row > 0).sum() == 0:
    for lv, _ in LEVELS:
        psave(f"pooled_correlation_forest_{lv}", pd.DataFrame())
    psave("pooled_correlation_forest", pd.DataFrame())
    print("  STAGE 40  CORRELATION COMPUTE  -  SKIPPED (no Nominated/Special signal in pooled_award).")
    print("  Run Stage 32 with team_awards.tsv (award-type rows), then re-run. Saved empty tables.")
    print("=" * 92)
else:
    summary = {}
    for lv, units in LEVELS:
        res, T = _forest_for(units, lv)
        psave(f"pooled_correlation_forest_{lv}", res)
        if len(res):
            res.to_csv(f"{POOLED_DIR}/correlation_forest_{lv}.csv", index=False)
            summary[lv] = (len(res), int(res["sig_topic"].sum()), int(res["sig_sent"].sum()),
                           int((res["q_topic"] < 0.05).sum()) if "q_topic" in res else 0,
                           int((res["q_sent"] < 0.05).sum()) if "q_sent" in res else 0, T)
        else:
            summary[lv] = (0, 0, 0, 0, 0, T)
    psave("pooled_correlation_forest", pload("pooled_correlation_forest_sub"))  # back-compat alias
    print(f"  STAGE 40  AWARD-SUCCESS CORRELATION  (TEAM-LEVEL, {N_BOOT}x team bootstrap; "
          f"{summary['sub'][5]} teams)")
    print("=" * 92)
    print(f"  {'level':6s} {'clusters':>8} {'topicCI!=0':>11} {'topicFDR<.05':>13} "
          f"{'sentCI!=0':>10} {'sentFDR<.05':>12}")
    for lv, _ in LEVELS:
        nrow, sigT, sigS, qT, qS, T = summary[lv]
        print(f"  {lv:6s} {nrow:>8} {sigT:>11} {qT:>13} {sigS:>10} {qS:>12}")
    print("  FDR<.05 = honest count (BH multiple-comparison control); CI!=0 is uncorrected (~5%/theme false+).")
    print("  beta = team mention-rate effect on prize (size-adjusted); colour/vader = cluster mean VADER.")
    print("  -> saved pooled_correlation_forest_main / _sub  (Notebook 3 40B renders both).")
    print("=" * 92)


## Notebook 2 — Consistency audit & manifest

Row-alignment checks across every per-response artifact, then the two-level invariants: one lineage (`pooled_labels` *is* `pooled_labels_sub`), `K(main) < K(sub)`, every label value either `-1` or a real theme id, the sub → main crosswalk **re-derived from the labels** and compared against the one Stage 23H recorded, noise held to Stage 21H's ceiling, and names covering exactly the themes that exist.

It also **sweeps the dead artifacts**. The mid level and the Ward $K$-selection are gone, but their `.pkl` files survive on any machine that ran the old pipeline, and a later notebook that loads one gets a partition from a different clustering with no error. They are deleted here; the few historical names that still mean something are refreshed from the canonical artifacts. Then `cluster_manifest.json` and the final diagnostic.


In [ ]:
# @title
# ==============================================================================
# STAGE 46  -  AUTHORITATIVE INTEGRITY AUDIT + MANIFEST  (Cell C)
# ------------------------------------------------------------------------------
# REWRITTEN FOR THE TWO-LEVEL HDBSCAN CONTRACT. The previous version audited a
# world that no longer exists: it asserted MAIN < MID < SUB, read
# pooled_K_stability's hier_*_k targets, and required validation_mid - all of them
# produced by the deleted Stage 23c. On the current pipeline every one of those
# checks either crashed or, worse, passed against a file left behind by an older
# run. An audit that reads stale artifacts is not an audit.
#
# WHAT IS AUDITED NOW
#   1. row alignment    - every per-response artifact has exactly N rows
#   2. one lineage      - pooled_labels IS pooled_labels_sub, byte for byte
#   3. two levels       - pooled_levels == ['main','sub'], K(main) < K(sub)
#   4. label domain     - every value is -1 or a real theme id, every theme
#                         0..K-1 has members
#   5. nesting          - the sub -> main crosswalk is re-derived from the labels
#                         and compared to the one Stage 23H recorded
#   6. noise            - counted at both levels and held to Stage 21H's ceiling
#   7. names            - the LLM dicts cover exactly the themes that exist
#   8. master table     - built, aligned, two-level, with the has_* flags
#
# IT ALSO SWEEPS THE DEAD ARTIFACTS. The mid level is gone and so is the Ward
# K-selection, but their .pkl files survive on any machine that ran the old
# pipeline, and a later notebook that pload()s one gets a partition from a
# different clustering with no error. They are deleted here, and the handful of
# historical names that still have a meaning are REFRESHED from the canonical
# artifacts so nothing downstream reads a value from a previous run.
# ==============================================================================
import numpy as np

# ---- canonical inputs --------------------------------------------------------
meta_pool = require("pooled_meta", stage="Stage 46").reset_index(drop=True)
X_pool = np.asarray(require("pooled_embeddings", stage="Stage 46"))
N = len(meta_pool)

lab_main = np.asarray(require("pooled_labels_main7", stage="Stage 46"))
lab_sub  = np.asarray(require("pooled_labels_sub",  stage="Stage 46"))
labels   = np.asarray(require("pooled_labels",      stage="Stage 46"))
levels   = pload("pooled_levels") or []
maps     = pload("pooled_hier_maps") or {}
HIER     = pload("pooled_hdbscan_hierarchy") or {}
CHOICE   = pload("pooled_hdbscan_choice") or {}
K        = int(pload("pooled_K") or 0)
canon    = pload("pooled_canonical_level") or "sub"

def _K(lab):
    lab = np.asarray(lab)
    return int(lab.max()) + 1 if (lab >= 0).any() else 0

Kmain, Ksub = _K(lab_main), _K(lab_sub)

print("=" * 74)
print("  STAGE 46  INTEGRITY AUDIT   (two-level HDBSCAN hierarchy)")
print("=" * 74)

# ---- 0. dead-artifact sweep, BEFORE anything is audited ----------------------
DEAD = ["pooled_labels_mid", "pooled_mid_llm", "pooled_mid_name", "pooled_mid_terms",
        "pooled_mid_size", "pooled_mid_quotes", "pooled_mid_validation",
        "validation_mid", "pooled_correlation_forest_mid",
        "pooled_K_stability", "pooled_k_selection", "pooled_hier_participant",
        "pooled_super_of_theme", "pooled_dashboard_tables", "pooled_sub_validation"]
_removed = [n for n in DEAD if pload(n) is not None]
for n in _removed:
    try:
        invalidate(n)
    except Exception:
        pass
if _removed:
    print(f"  swept {len(_removed)} artifact(s) from the deleted three-level / Ward")
    print(f"  pipeline so nothing downstream can load them: {_removed}")
else:
    print("  dead-artifact sweep: nothing from the old pipeline is on disk.")

# ---- 0b. refresh the historical aliases that still mean something -------------
# Written from the canonical artifacts, never the other way round.
psave("pooled_sub_labels", lab_sub)                  # historical alias for the canonical level
psave("pooled_analytical_unit", canon)
psave("pooled_cluster_meta", dict(K=Ksub, level=canon, levels=list(levels),
                                  method="hdbscan", K_main=Kmain, K_sub=Ksub,
                                  noise_main=float((lab_main < 0).mean()),
                                  noise_sub=float((lab_sub < 0).mean())))
if "crosswalk" in HIER:
    psave("pooled_hier_crosswalk", HIER["crosswalk"])
print("  refreshed the aliases that survive the change: pooled_sub_labels, "
      "pooled_analytical_unit,")
print("  pooled_cluster_meta, pooled_hier_crosswalk  (all derived from this run's labels).")

checks = []
def _chk(name, ok):
    checks.append(bool(ok))
    print(f"  [{'OK ' if ok else 'XX '}] {name}")

# ---- 1. row alignment --------------------------------------------------------
print("\n  1. row alignment")
for nm, obj in (("pooled_labels_main7", lab_main), ("pooled_labels_sub", lab_sub),
                ("pooled_labels", labels)):
    _chk(f"{nm} aligned to rows ({len(obj)} == {N})", len(obj) == N)
for nm in ("pooled_sentiment_response", "pooled_master"):
    o = pload(nm)
    if o is None:
        print(f"  [-- ] {nm} not built yet (optional at this point)")
        continue
    _chk(f"{nm} aligned to rows ({len(o)} == {N})", len(o) == N)

# ---- 2. one lineage ----------------------------------------------------------
print("\n  2. one lineage")
_canon_arr = {"main": lab_main, "sub": lab_sub}.get(canon)
_chk(f"canonical level = '{canon}'", _canon_arr is not None)
if _canon_arr is not None:
    _chk("pooled_labels == the canonical level (byte-identical)",
         np.array_equal(labels, _canon_arr))
    _chk(f"pooled_K == K of the canonical level ({K} == {_K(_canon_arr)})",
         K == _K(_canon_arr))
    # the alias was just rewritten from _canon_arr, so this checks the round-trip
    # through psave/pload, not the pipeline's logic - it has caught dtype changes
    _chk("pooled_sub_labels survives the save/load round-trip unchanged",
         np.array_equal(np.asarray(pload("pooled_sub_labels")), _canon_arr))

# ---- 3. two levels -----------------------------------------------------------
print("\n  3. two levels, not three")
_chk(f"pooled_levels == ['main','sub']  (got {list(levels)})",
     list(levels) == ["main", "sub"])
_chk(f"K(main) < K(sub)   ({Kmain} < {Ksub})", Kmain < Ksub)
_chk("no mid-level artifact remains on disk",
     all(pload(n) is None for n in ("pooled_labels_mid", "pooled_mid_llm", "validation_mid")))

# ---- 4. label domain ---------------------------------------------------------
print("\n  4. label domain")
for lv, lab, Kx in (("main", lab_main, Kmain), ("sub", lab_sub, Ksub)):
    _bad = sorted({int(v) for v in np.unique(lab)} - ({-1} | set(range(Kx))))
    _chk(f"{lv}: every value is -1 or 0..{Kx-1}" + (f"  (offenders {_bad[:5]})" if _bad else ""),
         not _bad)
    _empty = [c for c in range(Kx) if not (lab == c).any()]
    _chk(f"{lv}: all {Kx} themes have members" + (f"  (empty {_empty[:5]})" if _empty else ""),
         not _empty)

# ---- 5. nesting, re-derived not trusted --------------------------------------
print("\n  5. nesting (sub -> main)")
_s2m_rec = {int(k): int(v) for k, v in (maps.get("sub_to_main") or {}).items()}
_s2m_now, _pure = {}, 0
for c in range(Ksub):
    par = lab_main[(lab_sub == c) & (lab_main >= 0)]
    if not len(par):
        _s2m_now[c] = -1
        continue
    vals, cnt = np.unique(par, return_counts=True)
    _s2m_now[c] = int(vals[cnt.argmax()])
    _pure += int(cnt.max() == len(par))
_chk(f"the recorded crosswalk covers all {Ksub} sub-themes",
     set(_s2m_rec) == set(range(Ksub)))
_disagree = [c for c in range(Ksub) if _s2m_rec.get(c) != _s2m_now.get(c)]
_chk("the recorded crosswalk matches one re-derived from the labels"
     + (f"  (differs at {_disagree[:5]})" if _disagree else ""), not _disagree)
_chk(f"every sub-theme maps into a real main theme (0..{Kmain-1})",
     all(0 <= v < Kmain for v in _s2m_now.values() if v >= 0))
print(f"       {_pure} of {Ksub} sub-themes sit ENTIRELY inside one main theme "
      f"({_pure/max(Ksub,1):.0%}); the rest are assigned to their majority parent "
      f"and the purity is in hierarchy_crosswalk.csv.")

# ---- 6. noise ----------------------------------------------------------------
print("\n  6. noise (a result, not a failure)")
_ceiling = float(HIER.get("max_noise", globals().get("MAX_NOISE", 0.35)))
for lv, lab in (("main", lab_main), ("sub", lab_sub)):
    nz = float((lab < 0).mean())
    _chk(f"{lv}: {int((lab < 0).sum())} of {N} responses have no theme ({nz:.1%}) "
         f"<= the Stage 21H ceiling {_ceiling:.0%}", nz <= _ceiling + 1e-9)
_both = int(((lab_main < 0) & (lab_sub < 0)).sum())
print(f"       {_both} response(s) belong to neither level; "
      f"{int(((lab_sub < 0) & (lab_main >= 0)).sum())} have a main theme but no sub-theme.")
print("       These carry -1 everywhere and are never counted as members of a theme.")

# ---- 7. names ----------------------------------------------------------------
print("\n  7. names")
for lv, Kx in (("main", Kmain), ("sub", Ksub)):
    d = pload(f"pooled_{lv}_llm")
    if d is None:
        print(f"  [-- ] pooled_{lv}_llm not built yet (run Stage 30)")
        continue
    ids = {int(k) for k in d.keys()}
    _chk(f"pooled_{lv}_llm covers exactly the {Kx} themes that exist",
         ids == set(range(Kx)))
    _acc = sum(1 for v in d.values() if (v or {}).get("status") == "accepted")
    print(f"       {lv}: {_acc}/{Kx} labels accepted, {Kx - _acc} withheld "
          f"(withheld themes carry a descriptive placeholder and are flagged)")

# ---- 8. master table ---------------------------------------------------------
print("\n  8. pooled_master")
M = pload("pooled_master")
if M is None:
    print("  [-- ] pooled_master not built yet (run Stage 33b)")
else:
    _chk(f"pooled_master has {N} rows", len(M) == N)
    _chk("pooled_master carries both levels and the has_* flags",
         {"main_id", "sub_id", "has_main", "has_sub"} <= set(M.columns))
    _chk("pooled_master carries NO mid columns",
         not ({"mid_id", "mid_name"} & set(M.columns)))
    _chk("has_sub agrees with the canonical labels",
         np.array_equal(np.asarray(M["has_sub"], dtype=bool), lab_sub >= 0))

n_fail = sum(1 for ok in checks if not ok)
print("\n" + "=" * 74)
print(f"  audit: {len(checks) - n_fail}/{len(checks)} passed   "
      f"(main={Kmain} / sub={Ksub}; canonical unit = {canon.upper()}; "
      f"noise sub {(lab_sub < 0).mean():.0%})")
if n_fail:
    print("  ^ FIX THE FAILING CHECKS BEFORE USING THESE ARTIFACTS. Each line above names")
    print("    the invariant and the numbers that broke it.")
print("=" * 74)

# ---- manifest + inventory ----------------------------------------------------
REQUIRED_NB2 = ["pooled_umap_nd", "pooled_umap2d", "pooled_umap3d",
                "pooled_hdbscan_screen", "pooled_hdbscan_choice",
                "pooled_hdbscan_hierarchy", "pooled_hier_maps", "pooled_hier_crosswalk",
                "pooled_K", "pooled_levels", "pooled_canonical_level",
                "pooled_labels", "pooled_labels_main7", "pooled_labels_sub",
                "pooled_sub_labels", "pooled_sub_of_macro",
                "pooled_stability", "validation_main7", "validation_sub",
                "pooled_cluster_meta", "pooled_analytical_unit"]
OPTIONAL_NB2 = ["pooled_main_llm", "pooled_sub_llm", "pooled_theme_llm",
                "pooled_main_name", "pooled_sub_name",
                "pooled_main_terms", "pooled_sub_terms",
                "pooled_main_size", "pooled_sub_size",
                "pooled_naming_audit", "pooled_label_verification",
                "pooled_award", "pooled_team_org", "pooled_master",
                "pooled_sentiment_response", "pooled_sentiment_team_long",
                "pooled_sentiment_team_wide", "pooled_projection",
                "pooled_validity_xq", "cluster_reality", "pooled_convergent",
                "pooled_embed_validation",
                "pooled_correlation_forest", "pooled_correlation_forest_main",
                "pooled_correlation_forest_sub"]

fingerprints = dict(config=CFG.fingerprint("all", N),
                    umap=CFG.fingerprint("umap", len(X_pool), extra=data_sig(X_pool)),
                    MAIN_K=int(Kmain), SUB_K=int(Ksub), analytical_unit=canon,
                    levels=list(levels), method="hdbscan",
                    hdbscan_choice={k: CHOICE.get(k) for k in
                                    ("n_components", "n_neighbors", "min_cluster_size",
                                     "min_samples", "method", "epsilon", "dbcv",
                                     "stability")},
                    main_epsilon=HIER.get("main_epsilon"),
                    noise_main=float((lab_main < 0).mean()),
                    noise_sub=float((lab_sub < 0).mean()))
write_manifest("cluster_manifest.json", fingerprints, n_rows=N)
artifact_diagnostic(REQUIRED_NB2, optional=OPTIONAL_NB2)
artifact_inventory(names=REQUIRED_NB2 + OPTIONAL_NB2)
print()
print(f"  STAGE 46 complete. ONE lineage, two levels, canonical analytical unit = "
      f"{canon.upper()} (K={Ksub}).")
print("  Run the post-fix assertion cell next - it turns these checks into hard asserts.")


## Stage 47 — Post-fix hard assertions (regression guard)

Crashes loudly if any invariant regresses: two levels and no mid artifact anywhere on disk, every per-response artifact exactly `N` rows (the 1134-vs-1505 guard, applied once to everything), one lineage with byte-identical canonical labels, `K(main) < K(sub)`, every label value in `{-1} ∪ 0..K-1` with no empty theme, noise present and under the ceiling, the nesting re-derived rather than trusted, names for exactly the themes that exist, and a `pooled_master` whose ids and `has_*` flags match the labels.

Keep this as the last NB2 cell. Trust it, not prints elsewhere.


In [ ]:
# @title
# ==============================================================================
# STAGE 47  -  POST-FIX HARD ASSERTIONS  (regression guard; crashes loudly)
# ------------------------------------------------------------------------------
# These asserts crash if any invariant of the CURRENT pipeline ever regresses.
# Keep this as the last NB2 cell; a green run certifies that the artifacts on
# disk are internally consistent: one lineage, two levels, a nested hierarchy,
# noise that is recorded and never counted as a theme, names for exactly the
# themes that exist, and no per-response vector left over from an earlier run.
# Trust THIS cell, not prints elsewhere.
#
# It was rewritten with the hierarchy. The previous version asserted
# MAIN < MID < SUB and read pooled_K_stability's hier_*_k targets - both from the
# deleted Stage 23c - so on the current pipeline it either crashed on a missing
# key or, on a machine that had run the old code, passed against files describing
# a completely different clustering. The whole point of a regression guard is
# that it cannot do that, so every assert below reads only artifacts this
# pipeline actually writes.
# ==============================================================================
import numpy as np, pandas as pd

meta = pload("pooled_meta")
assert meta is not None, "REGRESSION: pooled_meta missing."
N = len(meta)

main   = np.asarray(pload("pooled_labels_main7"))
sub    = np.asarray(pload("pooled_labels_sub"))
labels = np.asarray(pload("pooled_labels"))
levels = list(pload("pooled_levels") or [])
maps   = pload("pooled_hier_maps") or {}
HIER   = pload("pooled_hdbscan_hierarchy") or {}
canon  = pload("pooled_canonical_level") or "sub"
Kdisk  = int(pload("pooled_K") or 0)

def _K(lab):
    lab = np.asarray(lab)
    return int(lab.max()) + 1 if (lab >= 0).any() else 0

Kmain, Ksub = _K(main), _K(sub)
canon_arr = {"main": main, "sub": sub}[canon]
Kcanon = _K(canon_arr)

# 1. TWO LEVELS. The mid level is gone and must not come back, in the registry
#    or on disk. A stale pooled_labels_mid is a partition from another run.
assert levels == ["main", "sub"], \
    f"REGRESSION: pooled_levels is {levels}, expected ['main','sub']."
for _dead in ("pooled_labels_mid", "pooled_mid_llm", "pooled_mid_name",
              "pooled_mid_terms", "validation_mid", "pooled_correlation_forest_mid"):
    assert pload(_dead) is None, \
        f"REGRESSION: {_dead} is back on disk. The mid level was removed in Stage 23H; " \
        f"a file under that name is a partition from a different run. Run Stage 46, " \
        f"which sweeps it."

# 2. ROW ALIGNMENT. Every per-response artifact must have exactly N rows. This is
#    the 1134-vs-1505 guard, applied once to everything rather than per stage.
_PER_ROW = ["pooled_embeddings", "pooled_origin", "pooled_labels",
            "pooled_labels_main7", "pooled_labels_sub", "pooled_sub_labels",
            "pooled_sentiment_response", "pooled_master"]
for nm in _PER_ROW:
    o = pload(nm)
    assert o is not None, f"REGRESSION: {nm} missing."
    assert len(o) == N, \
        f"REGRESSION: {nm} has {len(o)} rows but pooled_meta has {N} - it is from an " \
        f"earlier run. Re-run the stage that produces it."

# 3. ONE LINEAGE: the canonical labels are byte-identical everywhere they appear.
assert np.array_equal(labels, canon_arr), \
    "REGRESSION: pooled_labels != the canonical level (fork)."
assert np.array_equal(np.asarray(pload("pooled_sub_labels")), canon_arr), \
    "REGRESSION: pooled_sub_labels != the canonical level (fork)."
assert Kdisk == Kcanon, \
    f"REGRESSION: pooled_K is {Kdisk} but the canonical level has {Kcanon} themes."
assert pload("pooled_analytical_unit") == canon, \
    "REGRESSION: pooled_analytical_unit != the canonical level."
_cmeta = pload("pooled_cluster_meta") or {}
assert int(_cmeta.get("K", -1)) == Kcanon, \
    "REGRESSION: pooled_cluster_meta.K != the canonical K."

# 4. STRICT ORDERING. Main is the coarse layer; it must really be coarser.
assert Kmain < Ksub, f"REGRESSION: not MAIN < SUB ({Kmain} < {Ksub})."

# 5. LABEL DOMAIN. Every value is -1 (no theme) or a real theme id, and every
#    theme has members. A theme id with no members means K and the labels
#    disagree, which is how an off-by-one silently shifts every name.
for lv, lab, K in (("main", main, Kmain), ("sub", sub, Ksub)):
    _vals = {int(v) for v in np.unique(lab)}
    assert _vals <= ({-1} | set(range(K))), \
        f"REGRESSION: {lv} has label values outside -1..{K-1}: " \
        f"{sorted(_vals - ({-1} | set(range(K))))[:5]}."
    _empty = [c for c in range(K) if not (lab == c).any()]
    assert not _empty, f"REGRESSION: {lv} theme(s) {_empty[:5]} have no members."

# 6. NOISE IS REAL AND IS NEVER A THEME. It must be present as -1 (HDBSCAN
#    leaving nothing out would mean the epsilon/parameters on disk are not the
#    ones that produced these labels), and it must stay under Stage 21H's ceiling.
_ceiling = float(HIER.get("max_noise", globals().get("MAX_NOISE", 0.35)))
for lv, lab in (("main", main), ("sub", sub)):
    _nz = float((lab < 0).mean())
    assert _nz <= _ceiling + 1e-9, \
        f"REGRESSION: {lv} noise is {_nz:.1%}, above the {_ceiling:.0%} ceiling the " \
        f"selection rule was fixed at."
assert (sub < 0).sum() > 0, \
    "REGRESSION: no response is noise. HDBSCAN leaving nothing out is possible in " \
    "principle but did not happen in this run - if it now does, the labels on disk " \
    "were not produced by the recorded parameters."

# 7. NESTING. Re-derived from the labels, not read from the record it is checking.
_rec = {int(k): int(v) for k, v in (maps.get("sub_to_main") or {}).items()}
assert set(_rec) == set(range(Ksub)), \
    f"REGRESSION: the sub->main crosswalk covers {len(_rec)} sub-themes, not {Ksub}."
for c in range(Ksub):
    par = main[(sub == c) & (main >= 0)]
    if not len(par):
        continue                     # a sub-theme entirely inside main-level noise
    vals, cnt = np.unique(par, return_counts=True)
    top = int(vals[cnt.argmax()])
    assert _rec[c] == top, \
        f"REGRESSION: sub{c} is recorded under main{_rec[c]} but its members are " \
        f"mostly in main{top} - the crosswalk and the labels disagree."
    assert 0 <= _rec[c] < Kmain, \
        f"REGRESSION: sub{c} maps to main{_rec[c]}, which is not a theme."

# 8. NAMES exist for exactly the themes that exist, at both levels.
for lv, K in (("main", Kmain), ("sub", Ksub)):
    d = pload(f"pooled_{lv}_llm")
    assert d is not None, f"REGRESSION: pooled_{lv}_llm missing - run Stage 30."
    _ids = {int(k) for k in d.keys()}
    assert _ids == set(range(K)), \
        f"REGRESSION: pooled_{lv}_llm covers {len(_ids)} clusters but the {lv} " \
        f"partition has {K}. The names belong to a different clustering."
    _nm = pload(f"pooled_{lv}_name") or {}
    assert {int(k) for k in _nm} == set(range(K)), \
        f"REGRESSION: pooled_{lv}_name does not cover exactly the {K} {lv} themes."
_theme_llm = pload("pooled_theme_llm")
assert _theme_llm is not None and len(_theme_llm) == Kcanon, \
    "REGRESSION: pooled_theme_llm is missing or is not the canonical level."

# 9. THE MASTER TABLE is two-level, aligned, and marks noise explicitly.
M = pload("pooled_master")
assert isinstance(M, pd.DataFrame) and len(M) == N, \
    "REGRESSION: pooled_master missing or misaligned - run Stage 33b."
assert not ({"mid_id", "mid_name"} & set(M.columns)), \
    "REGRESSION: pooled_master still carries mid columns."
assert {"main_id", "sub_id", "main_name", "sub_name", "has_main", "has_sub"} <= set(M.columns), \
    "REGRESSION: pooled_master is missing a theme column or a has_* flag."
assert np.array_equal(np.asarray(M["sub_id"]), sub) and \
       np.array_equal(np.asarray(M["main_id"]), main), \
    "REGRESSION: pooled_master's ids are not the canonical labels."
assert np.array_equal(np.asarray(M["has_sub"], dtype=bool), sub >= 0) and \
       np.array_equal(np.asarray(M["has_main"], dtype=bool), main >= 0), \
    "REGRESSION: pooled_master's has_* flags do not match the labels."
assert (M.loc[M["has_sub"], "sub_name"].astype(str).str.strip() != "").all(), \
    "REGRESSION: a themed row in pooled_master has no sub name."
assert (M.loc[~M["has_sub"], "sub_name"] == "(no theme)").all(), \
    "REGRESSION: a noise row in pooled_master carries a theme name."

# 9b. THE MEMBERSHIP TABLE. Stage 23d decides who belongs to which theme and
#     Stage 33b carries it into pooled_master under the names Notebook 3 reads.
#     If any of this is stale or inconsistent, Notebook 3 draws a picture of a
#     partition that does not exist - which is exactly the class of failure this
#     cell is here to make impossible.
MB = pload("pooled_membership")
assert MB is not None and len(MB), \
    "REGRESSION: pooled_membership missing - run Stage 23d."
_mb = MB[MB["source"] == "pooled"].sort_values("row_id").reset_index(drop=True)
assert len(_mb) == N, \
    f"REGRESSION: pooled_membership has {len(_mb)} pooled rows but the pooled set " \
    f"has {N} - it is from an EARLIER clustering."
assert np.array_equal(np.asarray(_mb["sub_id"]), sub), \
    "REGRESSION: pooled_membership disagrees with pooled_labels_sub about the " \
    "primary partition."
_TIERS = {"core", "assigned", "assigned-multi", "unassigned"}
assert set(_mb["tier"]) <= _TIERS, \
    f"REGRESSION: unexpected theme_tier value(s) {sorted(set(_mb['tier']) - _TIERS)}."
assert (_mb.loc[_mb.tier == "core", "sub_id"] >= 0).all(), \
    "REGRESSION: a row is tier 'core' but has no sub-theme."
assert (_mb.loc[_mb.tier != "core", "sub_id"] < 0).all(), \
    "REGRESSION: the recovery moved a row in the PRIMARY partition. It must only " \
    "ever add columns, never change sub_id."
assert (_mb.loc[_mb.tier == "unassigned", "unassigned_reason"]
        .astype(str).str.strip() != "").all(), \
    "REGRESSION: an unassigned row has no recorded reason - every one must say " \
    "whether it is thin, rare, or below the threshold."

_WIDE = {"theme_tier", "sub_assigned", "sub_conf", "sub_multi",
         "sub_id_wide", "sub_name_wide", "has_sub_wide",
         "main_id_wide", "main_name_wide", "has_main_wide",
         "is_substantive", "unassigned_reason", "is_core"}
assert _WIDE <= set(M.columns), \
    f"REGRESSION: pooled_master is missing the wide-view column(s) " \
    f"{sorted(_WIDE - set(M.columns))}. Notebook 3 reads these by name and will " \
    f"either crash or silently fall back to the core-only view."
assert np.array_equal(np.asarray(M["theme_tier"]), np.asarray(_mb["tier"])), \
    "REGRESSION: pooled_master's theme_tier does not match pooled_membership."
assert not (np.asarray(M["has_sub_wide"], dtype=bool)
            & (np.asarray(M["sub_id_wide"]) < 0)).any(), \
    "REGRESSION: a row is flagged wide-themed but carries sub_id_wide -1."
assert np.array_equal(np.asarray(M["has_sub_wide"], dtype=bool),
                      np.asarray(M["has_main_wide"], dtype=bool)), \
    "REGRESSION: the two levels disagree about which rows have a theme. " \
    "main_id_wide must follow sub_id_wide through the sub->main map, or every " \
    "two-level figure compares different row sets."
_hs = np.asarray(M["has_sub"], dtype=bool)
assert np.array_equal(np.asarray(M["sub_id_wide"])[_hs], sub[_hs]), \
    "REGRESSION: the wide view changed a CORE row's theme. It may only add."
assert np.array_equal(np.asarray(M["is_core"], dtype=bool),
                      np.asarray(M["theme_tier"]) == "core"), \
    "REGRESSION: is_core is not the measured tier. It used to be hard-coded True " \
    "on every row, which made the provenance column useless."
assert (np.asarray(M["sub_id_wide"]) < Ksub).all() and \
       (np.asarray(M["main_id_wide"]) < Kmain).all(), \
    "REGRESSION: a wide id is out of range for the current partition."
_wide_n = int(np.asarray(M["has_sub_wide"], dtype=bool).sum())
assert _wide_n >= int((sub >= 0).sum()), \
    "REGRESSION: the wide view covers FEWER rows than the core view - the " \
    "recovery is subtracting instead of adding."

# 10. THE VALIDATION OBJECTS exist for both levels and are finite.
for lv in ("main7", "sub"):
    v = pload(f"validation_{lv}")
    assert v is not None and np.isfinite(v.get("global_silhouette", np.nan)), \
        f"REGRESSION: validation_{lv} missing or non-finite - run Stage 25."

print("=" * 74)
print(f"  ALL POST-FIX ASSERTIONS PASSED.  ONE lineage, TWO levels, canonical unit = "
      f"{canon.upper()}")
print(f"  main K={Kmain} (noise {(main < 0).mean():.1%})   "
      f"sub K={Ksub} (noise {(sub < 0).mean():.1%})   rows={N}")
print(f"  {int((sub >= 0).sum())} responses carry a sub-theme; {int((sub < 0).sum())} carry "
      f"none and are marked has_sub=False, never counted as members.")
_tc = pd.Series(np.asarray(M["theme_tier"])).value_counts()
print(f"  membership tiers: "
      + "  ".join(f"{k}={int(v)}" for k, v in _tc.items()))
print(f"  core view {int((sub >= 0).sum())} rows -> wide view {_wide_n} rows "
      f"({_wide_n / N:.1%}); both levels describe the same rows.")
print(f"  {int((np.asarray(M['theme_tier']) == 'unassigned').sum())} response(s) belong "
      f"to no theme and each records why. That is a result, not a gap.")
print("  Names exist for both levels; the forest, the figures, the LLM labels and the")
print("  manifest all read one partition. Notebook 3 can consume pooled_master.")
print("=" * 74)